# DATA-DRIVEN 1D MECHANICAL EARTH MODEL - POSEIDON 2
## Increment 7.0.4 - Final End-to-End Corrective Patch

**Package version:** `p2mem 0.7.4`
**Assurance tier:** Tier C - Screening-Level / Uncalibrated Educational
**Approved wells:** Poseidon 2, Boreas 1, Poseidon North 1, Proteus 1ST2
**Baseline:** Increment 6.1.7 (`p2mem 0.6.8`), LOCKED and unmodified.

This notebook adds bulk-density QC, an explicit density-coverage qualification, and an
auditable framework for vertical overburden stress. It does **not** compute pore pressure,
fit a normal compaction trend, calculate effective stress, model elastic properties or rock
strength, estimate horizontal stresses, build a mud window, or assign a named lithology.
**Increment 8 has not been started.**

---

## What this increment concludes, stated before any number is shown

The scientific result of Increment 7 is mostly a **negative** one, and it is the honest one.

`sigma_v(z) = INTEGRAL rho(z) g dz` is only an *absolute* vertical stress when the density
column is known from the datum or the seabed down to the evaluation depth. For Boreas 1 and Poseidon 2, whose approved formation tops resolve the seabed,
bulk-density logging begins **kilometres below that seabed**. Poseidon North 1 and
Proteus 1ST2 have no approved formation-top file, so their seabed and shallow-column
thickness are **not determinable**. In every case the unmeasured shallow column is
unresolved; where its thickness is known, it carries **no density measurement at all**. No amount of
coverage *inside* the logged interval can supply it, and no defensible calibration exists in
this project to model it.

Increment 7 therefore does four things, in this order:

1. **Measures** what density each well actually has: sample counts, screening-bound
   failures, depth mapping, coverage, and every gap, in both measured and true vertical
   depth.
2. **Qualifies** that coverage against explicit, configuration-driven, separately auditable
   masks, and conditions only what the policy permits - short internal gaps, bridged
   linearly in true vertical depth, into a **separately named array** whose contribution to
   any stress is reported on its own.
3. **Integrates** the supported column, in **true vertical depth**, with the project's depth
   sign convention re-verified at run time against each well frame's own arrays rather than
   inferred from a variable name.
4. **Derives** - from coverage and QC evidence, never from a well name - what each well can
   support: an absolute overburden curve, a transparent screening bracket, a partial
   measured increment, or nothing at all.

Where the shallow column is unmeasured, this notebook reports a **partial measured
increment** and, where the missing thickness is at least *quantifiable*, a **low/base/high
screening bracket** whose assumed, conditioned and measured fractions are published on every row. It never presents a
single absolute `sigma_v` curve as measured truth, and it never fills a 3-4 km unmeasured
column with a convenient density trend.

### The four derived statuses

| status | meaning |
| --- | --- |
| `absolute_overburden_supported` | Eligible density reaches the seabed within the configured tolerance and no unresolved internal gap of any class interrupts the column. |
| `screening_sensitivity_only` | An eligible measured column exists and the seabed datum **is** resolved, so the unmeasured thickness is known and can be bracketed - but it is not measured. |
| `partial_measured_increment_only` | An eligible measured column exists but the seabed datum is **unresolved**, so the unmeasured thickness is not even quantifiable and no bracket can be built. |
| `not_eligible` | No integrable measured column exists. |

Each status is a function of measured coverage and QC results. No well name participates in
the derivation, and no configuration key names a well.
### Increment 7.0.4 corrective scope

This patch makes five end-to-end contracts explicit: the high scenario uses the P05 of eligible density samples; invalid public gap-threshold overrides fail closed; scenario fractions exhaustively partition assumed, conditioned and measured stress; locked seabed ingestion fails closed on missing, malformed, duplicate or non-finite matching data; and JSON output uses explicit UTF-8 LF bytes on every platform. The approved real-data values remain unchanged for the measured reasons printed later.


---
#### Step 1 - Access the project root

Runs in Google Colab against Google Drive, and equally in any environment where the project
root is the current working directory. Nothing is written before Step 2b confirms where we
are.

In [ ]:
import os
from pathlib import Path

COLAB_ROOT = "/content/drive/MyDrive/Poseidon_1D_MEM"
IN_COLAB = False
try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    IN_COLAB = True
except Exception:
    print("Not running in Google Colab; the project root must be the current "
          "working directory or an existing local Poseidon_1D_MEM tree.")

def _looks_like_project_root(path):
    p = Path(path)
    return (p / "pyproject.toml").is_file() and (p / "p2mem").is_dir()

_candidates = [COLAB_ROOT, os.getcwd()]
PROJECT_ROOT = next((c for c in _candidates if _looks_like_project_root(c)), None)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not locate the project root. Expected either "
        f"{COLAB_ROOT!r} (Colab/Drive) or the current working directory "
        f"({os.getcwd()!r}) to contain pyproject.toml and p2mem/."
    )
PROJECT_ROOT = str(Path(PROJECT_ROOT).resolve())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Running in Google Colab:", IN_COLAB)

#### Step 2 - Verify the LOCKED Increment 6.1.7 baseline is present

Increment 7 consumes the locked LAS ingestion, deviation survey, MD-TVD-TVDSS mapping,
formation-top output and well-frame layers. It never reconstructs them from memory.

In [ ]:
_required_locked_files = [
    "pyproject.toml",
    os.path.join("p2mem", "__init__.py"),
    os.path.join("p2mem", "units.py"),
    os.path.join("p2mem", "models.py"),
    os.path.join("p2mem", "deviation_models.py"),
    os.path.join("p2mem", "trajectory.py"),
    os.path.join("p2mem", "depth_mapping.py"),
    os.path.join("p2mem", "checkshot_models.py"),
    os.path.join("p2mem", "time_depth.py"),
    os.path.join("p2mem", "top_models.py"),
    os.path.join("p2mem", "wellframe_models.py"),
    os.path.join("p2mem", "wellframe.py"),
    os.path.join("p2mem", "petrophysics_models.py"),
    os.path.join("p2mem", "petrophysics.py"),
    os.path.join("p2mem", "method_eligibility.py"),
    os.path.join("p2mem", "io", "las.py"),
    os.path.join("p2mem", "io", "inventory.py"),
    os.path.join("p2mem", "io", "deviation.py"),
    os.path.join("p2mem", "io", "deviation_inventory.py"),
    os.path.join("p2mem", "io", "checkshot.py"),
    os.path.join("p2mem", "io", "checkshot_inventory.py"),
    os.path.join("p2mem", "io", "tops.py"),
    os.path.join("p2mem", "io", "tops_inventory.py"),
    os.path.join("p2mem", "io", "output_policy.py"),
    os.path.join("p2mem", "io", "petrophysics_inventory.py"),
    os.path.join("config", "las_curve_contracts.yml"),
    os.path.join("config", "deviation_survey_contracts.yml"),
    os.path.join("config", "checkshot_contracts.yml"),
    os.path.join("config", "formation_top_contracts.yml"),
    os.path.join("config", "petrophysics_eligibility.yml"),
    os.path.join("outputs", "05_formation_tops", "top_survey_corrected_markers.csv"),
]
_missing = [f for f in _required_locked_files
            if not os.path.exists(os.path.join(PROJECT_ROOT, f))]
if _missing:
    raise RuntimeError(
        "Locked Increment 6.1.7 baseline is missing file(s): " + ", ".join(_missing)
        + ". Run the Increment 6 notebook first; Increment 7 does not reconstruct the "
        "locked foundation from memory."
    )
print(f"Locked Increment 6.1.7 baseline confirmed present "
      f"({len(_required_locked_files)} required paths):")
for f in _required_locked_files:
    print("  -", f)

#### Step 2b - Enter the project root BEFORE any relative write

Every `%%writefile` below uses a path relative to the project root. Entering it first is what
makes those paths mean what they say.

In [ ]:
os.chdir(PROJECT_ROOT)

if Path.cwd().resolve() != Path(PROJECT_ROOT).resolve():
    raise RuntimeError(
        f"Failed to enter the project root. Expected {PROJECT_ROOT}, "
        f"actual working directory: {Path.cwd()}"
    )
if not Path("p2mem").is_dir():
    raise RuntimeError(
        f"Required p2mem directory is missing under {PROJECT_ROOT}. "
        "Extract the approved Increment package before running this notebook."
    )
print("Working directory confirmed:", Path.cwd())

#### Step 3 - Create the Increment 7 directory additions

In [ ]:
for d in ["outputs/07_density_overburden", "outputs/07_density_overburden/figures"]:
    os.makedirs(os.path.join(PROJECT_ROOT, d), exist_ok=True)
print("Increment 7 directories ready.")

#### Step 4 - Verify the approved LAS and deviation-survey filenames, and their SHA-256

The four approved LAS files and four approved deviation surveys are private project inputs.
They are read here and are **never** copied into the delivered package.

In [ ]:
import hashlib

LAS_DIR = os.path.join(PROJECT_ROOT, "data", "raw", "las")
DEV_DIR = os.path.join(PROJECT_ROOT, "data", "raw", "deviation")

LAS_FILES = {
    "Boreas_1": "Boreas_1_logs.las",
    "Poseidon_2": "Poseidon_2_logs.las",
    "Poseidon_North_1": "Poseidon_North_1_logs.las",
    "Proteus_1ST2": "Proteus_1ST2_logs.las",
}
DEV_FILES = {
    "Boreas_1": "Boreas 1_dev.txt",
    "Poseidon_2": "Poseidon 2_dev.txt",
    "Poseidon_North_1": "Poseidon North 1_dev.txt",
    "Proteus_1ST2": "Proteus 1ST2_dev.txt",
}
LAS_PATHS = {k: os.path.join(LAS_DIR, v) for k, v in LAS_FILES.items()}
SURVEY_PATHS = {k: os.path.join(DEV_DIR, v) for k, v in DEV_FILES.items()}

_missing_inputs = [p for p in list(LAS_PATHS.values()) + list(SURVEY_PATHS.values())
                   if not os.path.exists(p)]
if _missing_inputs:
    raise RuntimeError(
        "Missing required approved input file(s): "
        + ", ".join(os.path.basename(p) for p in _missing_inputs)
        + ". Increment 7 requires exactly these four LAS files and four deviation "
        "surveys, under these exact names - no other file is accepted as a substitute."
    )
print("All eight approved input files found. SHA-256:")
for key in sorted(LAS_PATHS):
    for p in (LAS_PATHS[key], SURVEY_PATHS[key]):
        print(f"  {key} ({os.path.basename(p)}): "
              f"{hashlib.sha256(open(p, 'rb').read()).hexdigest()}")

#### Step 5 - Install dependencies

Increment 7 adds **no** runtime dependency. `numpy` and `pyyaml` are already required by the
locked package; `pytest`, `pandas` and `matplotlib` are development/reporting tools only.

In [ ]:
!pip install -q numpy pyyaml pytest pandas matplotlib

---

## Source reproduction

Every cell below writes one packaged file verbatim. Three pre-existing files change
(`pyproject.toml`, `p2mem/__init__.py`, `README.md` - version and narrative only); every
other file here is new to Increment 7. **No locked module, config, test, fixture, notebook,
manifest, ledger, output or figure from Increment 6.1.7 is modified.**

#### `pyproject.toml` - package version 0.7.4 (no new dependency)

In [ ]:
%%writefile pyproject.toml
[build-system]
requires = ["setuptools>=68.0"]
build-backend = "setuptools.build_meta"

[project]
name = "p2mem"
version = "0.7.4"
description = "Screening-level 1D Mechanical Earth Model workflow for Poseidon 2 (Tier C, uncalibrated / educational)."
readme = "README.md"
requires-python = ">=3.9"
license = { text = "All Rights Reserved. Copyright (c) 2026 Mikael Elgo. This is a personal portfolio project; no license is granted for reuse, redistribution, or commercial use without the author's explicit written permission." }
authors = [
    { name = "Mikael Elgo" }
]
keywords = ["geomechanics", "mechanical-earth-model", "pore-pressure", "wellbore-stability", "portfolio-project"]
classifiers = [
    "Development Status :: 3 - Alpha",
    "Programming Language :: Python :: 3",
    "Intended Audience :: Science/Research",
    "Topic :: Scientific/Engineering",
    "License :: Other/Proprietary License",
]

# Runtime dependencies are deliberately minimal. No unit-handling libraries
# (e.g. Pint) are used: unit conversions are implemented explicitly in
# p2mem.units so that every conversion factor is visible, documented, and
# testable rather than delegated to a third-party unit registry. PyYAML is
# added in Increment 2 for exactly one purpose: parsing the human-authored,
# human-reviewable per-file LAS curve contracts in
# config/las_curve_contracts.yml - a plain-text, diffable format was judged
# preferable to a hand-rolled config parser or a hard-coded Python dict.
dependencies = [
    "numpy>=1.24",
    "pyyaml>=6.0",
]

[project.optional-dependencies]
dev = [
    "pytest>=7.4",
]

[tool.setuptools.packages.find]
include = ["p2mem*"]

[tool.pytest.ini_options]
testpaths = ["tests"]
python_files = ["test_*.py"]


#### `p2mem/__init__.py` - version 0.7.4 and the Increment 7 scientific narrative

In [ ]:
%%writefile p2mem/__init__.py
"""
p2mem - Poseidon 2 1D Mechanical Earth Model workflow package.

Project classification: Tier C - Screening-Level / Uncalibrated Educational
1D Mechanical Earth Model (see project design review, Rev 1). Nothing in
this package should be presented as a calibrated, operational, or
field-validated result unless an explicit independent calibration record
is attached to that specific output.

This package is under incremental, gated construction.

* Increment 1 / 1.1 delivered the project skeleton and the unit-control
  system (``p2mem.units``).
* Increment 2 added an auditable LAS-ingestion layer with explicit
  per-file curve contracts (``p2mem.io.las``, ``p2mem.io.inventory``,
  ``p2mem.models``) for the four approved wells (Poseidon 2, Boreas 1,
  Poseidon North 1, Proteus 1ST2). It performs LAS parsing, curve-identity
  resolution, NULL-sentinel handling, and factual inventory generation
  ONLY - no deviation-survey processing, MD-to-TVD/TVDSS transformation,
  checkshot processing, formation-top correction, petrophysical
  interpretation, or any later-phase geomechanical calculation.
* Increment 2.1 / 2.1.1 are corrective patches to Increment 2, applied
  after independent technical audits, WITHOUT changing scope or the
  underlying LAS-parsing/curve-resolution architecture (which both audits
  found sound). 2.1 corrected: canonical array naming (every array is now
  explicitly unit-suffixed, e.g. ``VP_m_s`` rather than ``DTCO``, so a
  name can never be mistaken for the wrong physical quantity or unit);
  the measured-depth curve is now located via an explicit contract role
  rather than by matching a canonical name spelled "DEPT"; several
  file-identity checks (filename, SHA-256, WELL, VERS, WRAP, NULL) that
  were not previously blocking now are; curve-coverage statistics now
  report raw AND canonical values with explicit units; and per-well
  batch failures are now typed (``p2mem.models.IngestionFailure``)
  instead of bare caught exceptions. 2.1.1 corrected a packaging-only gap
  (three notebook ``%%writefile`` cells that had drifted from their
  packaged source files). See ``INCREMENT_02_v2.1_MANIFEST.md`` and
  ``INCREMENT_02_v2.1.1_MANIFEST.md`` for the full audits and
  corrected-file checksums.
* Increment 3 adds Petrel deviation-survey ingestion with explicit
  per-file contracts (``p2mem.io.deviation``), a standard minimum-
  curvature trajectory engine with a numerically stable ratio-factor
  limit (``p2mem.trajectory``), an explicit MD-referenced/TVD-referenced/
  TVDSS depth-reference framework and MD-to-TVD/TVDSS interpolation with
  no silent extrapolation (``p2mem.depth_mapping``), and typed dataclasses
  for all of the above (``p2mem.deviation_models``) - for the same four
  approved wells. It independently reproduces the Petrel-supplied
  trajectory to millimetre scale for three of the four wells and
  discloses (rather than resolves) a real, larger trajectory-
  reconstruction discrepancy found in Proteus 1ST2's deeper section - see
  ``INCREMENT_03_MANIFEST.md``. It performs deviation-survey ingestion,
  trajectory validation, and depth mapping ONLY - no checkshot
  processing, formation-top correction, petrophysical interpretation, or
  any later-phase geomechanical calculation.
* Increment 3.1 is a corrective patch to Increment 3, applied after an
  independent technical audit, WITHOUT changing scope, equations, real
  well data, or the locked LAS/Increment-2.1.1 foundation. It corrected
  four defects: (1) the four deviation-survey source filenames are the
  exact, literal names as they exist in Google Drive, which contain
  spaces (e.g. ``"Poseidon 2_dev.txt"``) - Increment 3 had incorrectly
  substituted underscores in the contract keys, notebook mapping, and
  tests, which would have failed to resolve against the real files;
  internal well keys (e.g. ``Poseidon_2``) remain underscored and are
  unaffected; (2) every exported CSV/JSON/manifest field is now
  guaranteed to carry a basename only, never a full environment-dependent
  build path (runtime-only diagnostic objects may still retain one);
  (3) the previously undisclosed inference that the supplied ``DLS``
  column is normalized as degrees per 30 metres is now explicitly flagged
  with a new, independently per-file-verified
  ``DLS_NORMALIZATION_INFERRED_AS_DEG_PER_30M`` WARNING (mirroring the
  pre-existing MD-unit-inference warning); (4) the dogleg angle between
  successive stations is now computed with a numerically stable
  ``arctan2(||cross||, dot)`` vector formulation (``p2mem.trajectory``)
  instead of ``arccos``, which was ill-conditioned near a zero dogleg and
  previously reported a spurious ~1e-6-degree value for two stations with
  identical inclination/azimuth. The real four-well data, station counts,
  tolerances, and the unresolved Proteus 1ST2 trajectory discrepancy are
  all unchanged by this patch. See ``INCREMENT_03_1_MANIFEST.md`` for the
  full audit and re-verification record.

* Increment 4 adds checkshot (velocity survey) ingestion with explicit
  per-file contracts (``p2mem.io.checkshot``, ``config/checkshot_
  contracts.yml``), typed checkshot dataclasses (``p2mem.checkshot_
  models``), deterministic checkshot inventory/QC-table builders
  (``p2mem.io.checkshot_inventory``), and a numerical time-depth layer
  (``p2mem.time_depth``): duplicate-tie detection/conditioning, average/
  interval velocity diagnostics, checkshot-vs-locked-survey depth-
  reference comparison, forward/inverse piecewise-linear time-depth
  interpolation with explicit coverage masking (no extrapolation), LAS
  MD-to-checkshot-time mapping within validated checkshot coverage only,
  and a Poseidon-2-only sonic-checkshot drift diagnostic (trapezoidal
  integration of sonic slowness vs. the checkshot-interpolated OWT
  increment over the same MD/Depth interval). Three checkshot files are
  admitted: ``Poseidon2-Checkshot.txt`` (Poseidon 2 - the ONLY checkshot
  approved to define a primary time-depth relationship),
  ``Boreas1-Checkshot.txt`` and ``Proteus1-Checkshot.txt`` (Boreas 1 and
  Proteus 1ST2 - supporting QC data only, newly admitted in this
  increment, never transferred into Poseidon 2 as a substitute time-depth
  model). Proteus 1ST2's association with ``Proteus1-Checkshot.txt`` is
  explicitly disclosed as inferred/unverified (no embedded well
  identifier). Poseidon North 1 has no approved checkshot file
  (``checkshot_availability: NOT_AVAILABLE`` - a factual data gap, not an
  ingestion failure). This increment reuses the LOCKED Increment 1
  ``owt_to_twt``/``twt_to_owt`` unit functions and the LOCKED Increment
  3/3.1.1 ``petrel_source_trace`` survey trajectory unchanged; it performs
  checkshot QC and time-depth framework work ONLY - no formation-top
  correction, lithology interpretation, density modelling, pore-pressure
  prediction, elastic properties, rock strength, stress modelling, or
  wellbore-stability analysis. See ``INCREMENT_04_MANIFEST.md`` for the
  full technical detail and independently recomputed statistics. NOTE:
  ``INCREMENT_04_MANIFEST.md`` contained one identified defect, corrected
  by Increment 4.1 below - do not rely on its original, uncorrected
  statement that TVDSS is strictly increasing after Depth-tie conditioning
  for all three wells.

* Increment 4.1 is a narrowly scoped corrective patch to Increment 4,
  applied after an independent technical/numerical-method audit, WITHOUT
  beginning Increment 5 or any formation-top/petrophysics/pore-pressure/
  mechanical-properties/stress/wellbore-stability work. It corrected two
  defects and hardened two numerical contracts: (1) ``tvdss_to_owt``/
  ``owt_to_tvdss`` (and their TWT equivalents) previously resolved a
  repeated (tied) value on the axis being inverted by silently keeping
  whichever tied row appeared first in the Depth-conditioned table and
  discarding the other (``p2mem.time_depth._build_strictly_increasing_
  table``, REMOVED) - an ORDER-DEPENDENT tie-break with no audit trail
  beyond a bare count. This is replaced by ``build_axis_conditioned_
  lookup_table``/``build_axis_conditioned_tables_for_well``: an explicit,
  ORDER-INVARIANT policy that groups every tied value by exact equality
  regardless of parse order, registers every tied row in a new typed
  audit register (``AxisTimeDepthTieRegisterEntry`` /
  ``checkshot_time_axis_tie_register.csv`` - separate from, and never
  confused with, the pre-existing Depth-axis ``DuplicateTieRegisterEntry``
  register), and uses the tied group's dependent-value MEDIAN as the
  conditioned representative (order-invariant; disclosed as reducing to
  the arithmetic mean for the size-2 groups observed in this project's
  real data). A genuine reversal (not a tie) in the axis being inverted
  raises ``TimeDepthError`` rather than being sorted, discarded, or forced
  monotonic. (2) ``INCREMENT_04_MANIFEST.md``'s statement that TVDSS is
  strictly increasing after Depth-tie conditioning for all three wells was
  INCORRECT - independently reproduced counts (Poseidon 2: two TVDSS-axis
  and two OWT-axis ties; Boreas 1: one TVDSS-axis tie, zero OWT-axis ties;
  Proteus 1ST2: none of either) are now documented in
  ``INCREMENT_04_1_MANIFEST.md`` and reflected in this module's own
  docstrings. (3) ``trapezoidal_integrate`` and (4) ``compute_sonic_
  checkshot_drift`` are hardened to validate their numerical
  preconditions (finite, one-dimensional, equal-length, strictly
  increasing MD/x where required) rather than silently integrating
  invalid input - most notably, a decreasing or duplicate MD run can no
  longer silently produce a physically invalid NEGATIVE transit time; it
  now raises ``TimeDepthError``. None of this hardening changes the
  already-verified real Poseidon 2 sonic-drift result, which is
  bit-for-bit unchanged. See ``INCREMENT_04_1_MANIFEST.md`` for the full
  audit, corrected statistics, and re-verification record.

* Increment 4.1.1 is a narrowly scoped numerical-validation corrective
  patch to Increment 4.1, applied after an independent numerical-method/
  software-QA audit, WITHOUT beginning Increment 5 or any formation-top/
  petrophysics/pore-pressure/mechanical-properties/stress/wellbore-
  stability work. It corrected four blocking defects and one input-safety
  gap, none of which altered any previously verified REAL Poseidon
  2/Boreas 1/Proteus 1ST2 result: (1) ``build_axis_conditioned_lookup_
  table`` grouped ALL occurrences of an identical axis value together
  GLOBALLY before checking for a reversal, so a reversal that returned to
  an already-seen value (e.g. ``[100.0, 200.0, 100.0]``) was silently
  hidden rather than raising ``TimeDepthError`` - it now evaluates the
  ORIGINAL, ungrouped sequence's successive differences for negativity
  BEFORE any grouping is attempted, which is provably equivalent to the
  4.1 behavior for every legitimate adjacent tie and strictly stronger
  against a non-adjacent reversal. (2) ``compute_sonic_checkshot_drift``/
  ``find_longest_finite_positive_run`` validated MD monotonicity only
  within the selected finite-positive-VP run, so a decreasing or
  duplicate MD value outside that run (e.g. at a NaN-VP station) could
  pass silently - the COMPLETE canonical ``md_m`` array is now required
  finite and strictly increasing before run-selection (the real Poseidon
  2 MD array, 31,897 samples, was independently re-verified to already
  satisfy this). (3) ``compare_checkshot_to_survey`` reached an untyped
  NumPy ``ValueError`` ("zero-size array to reduction operation") if
  every checkshot Depth row fell outside the locked survey's own MD
  coverage - it now raises a typed ``TimeDepthError`` naming the well,
  the checkshot Depth range, and the survey MD coverage. (4)
  ``p2mem.io.checkshot.load_checkshot_surveys`` did not catch
  ``TimeDepthError`` raised during numerical conditioning, so a defect in
  one well's data could stop the entire batch - it is now caught per well
  (never via a blanket ``except Exception``) and recorded as a typed
  ``CheckshotIngestionFailure(error_type="numerical_conditioning_
  failure")``, isolated exactly like every other expected per-well
  failure. (5) ``seconds_to_milliseconds``/``milliseconds_to_seconds``
  coerced their input directly, unlike ``p2mem.units``'s Increment-1
  input-safety policy, so a boolean, numeric-looking string, or complex
  value would be silently reinterpreted rather than rejected - both now
  reject such input with ``TypeError`` via a local, documented copy of
  ``p2mem.units``'s identical private dtype check (``p2mem/units.py``
  itself remains LOCKED and unmodified). See
  ``INCREMENT_04_1_1_MANIFEST.md`` for the full audit, the regression-test
  list, and the re-verification record.

* Increment 5 adds contract-driven formation-top ingestion, HRS-versus-
  selected-readable source RECONCILIATION, and survey-corrected
  stratigraphic depth mapping (``p2mem.top_models``, ``p2mem.io.tops``,
  ``p2mem.io.tops_inventory``) for the two approved wells with formation-
  top data (Poseidon 2, Boreas 1). Each well has TWO independently
  supplied top files - an "HRS" file (``Top_Name``/``MDRT_m`` only, no
  well name in the file body) and a "selected readable" file
  (``TOP_NAME``/``MDRT_M``/``TVDSS_M``/``NOTE``, with ``#``-comment
  headers and a dashed separator line, deliberately parsed rather than
  treated as data) - and neither is silently preferred: markers are
  matched by exact/normalized name or an explicit human-authored alias
  contract (never fuzzy matching), and MDRT is cross-checked between the
  two sources within a documented 0.005 m tolerance; a marker on which
  the two sources disagree beyond that tolerance is excluded from depth
  mapping (``mapping_status == "not_mapped_mdrt_unresolved"``) rather
  than resolved by picking one file. Reconciled MDRT is mapped through
  the LOCKED Increment 3.1.1 ``petrel_source_trace`` survey trajectory
  using the existing, unmodified ``p2mem.depth_mapping.
  map_las_md_to_tvd_tvdss`` (per marker, so one out-of-coverage marker
  never blocks the rest of the well; extrapolation is never performed),
  producing ``TVD_survey_m``/``TVDSS_survey_corrected_m`` alongside an
  explicit, unambiguous ``TVDSS_residual_source_minus_survey_m =
  TVDSS_source_m - TVDSS_survey_corrected_m`` residual field. Both file
  representations for both wells are classified
  ``well_identity_evidence_status = "inferred_unverified"`` (filename-only
  or in-file-comment association, never independently content-verified -
  a more conservative classification than Increment 4's checkshot files).
  Poseidon North 1 and Proteus 1ST2 have no approved formation-top file
  and are recorded as ``FormationTopAvailabilityRecord(..., "NOT_
  AVAILABLE")`` - never substituted or depth-correlated from another
  well. Independently recomputing (never hardcoding) this increment's two
  required regression findings against the real approved files confirmed:
  Poseidon 2's "selected readable" file's supplied TVDSS equals
  ``MDRT - 21.8 m`` EXACTLY for every one of its 9 markers (21.8 m is the
  well's own rotary-table elevation) - a literal vertical-well-assumption
  depth-reference defect, corrected in the derived
  ``TVDSS_survey_corrected_m`` representation while the raw
  ``TVDSS_source_m`` column is preserved unmodified; survey-corrected
  residuals reproduce Sea Bed at ~0 m, Plover Fm (Top Reservoir) at
  +1.68 m, and TD at +2.49 m, exactly as the approved Rev 1 design
  anticipated. Boreas 1's supplied TVDSS does NOT follow that pattern and
  its maximum absolute source-versus-survey residual is 0.0416 m, below
  the approved 0.05 m tolerance - confirming it was generated from the
  well's real surveyed trajectory, not a vertical-well shortcut, and it is
  therefore NOT "corrected" the way Poseidon 2 is. See
  ``INCREMENT_05_MANIFEST.md`` for the full audit, the per-file contracts,
  and the complete reconciliation/mapping results.

* Increment 5.1 is a narrowly scoped corrective patch to Increment 5,
  applied after an independent technical/software-QA audit, WITHOUT
  beginning Increment 6 or any petrophysics/pore-pressure/mechanical-
  properties/stress/wellbore-stability work, and WITHOUT changing any
  previously verified real Poseidon 2/Boreas 1 formation-top result. It
  corrected three defects and one documentation-accuracy gap: (1)
  ``reconcile_formation_top_sources`` documented "zero canonical markers
  in common between the two sources" as a fatal ``NO_COMMON_MARKERS``
  ERROR, but actually tested the emptiness of the UNION of both sources'
  canonical names - so two entirely DISJOINT, non-empty marker sets (e.g.
  HRS names sharing nothing with the readable file's names) were silently
  accepted as one-sided ``NOT_COMPARABLE`` entries instead of being
  rejected; the check now explicitly evaluates the INTERSECTION of the two
  sources' canonical names and raises the documented ERROR (and
  ``load_formation_top_well`` consequently raises
  ``TopSourceReconciliationError``) whenever both sources are non-empty but
  share nothing, while a legitimate one-sided marker (with at least one
  marker genuinely shared) remains the pre-existing, non-fatal
  ``NOT_COMPARABLE`` case. (2) ``load_formation_top_surveys`` recorded only
  the HRS file's path in every ``TopIngestionFailure.source_path``,
  regardless of which file/stage actually failed - so a failure originating
  from the "selected readable" file (missing file, malformed content, or a
  contract mismatch) could leak that file's own absolute path, unsanitized,
  into exported issues/availability/manifest rows (the sanitizer only ever
  stripped the recorded, and in that case WRONG, HRS path).
  ``TopIngestionFailure`` now carries an explicit ``failure_origin``
  ("hrs"/"readable"/"reconciliation"/"mapping"/"unknown") and both
  ``hrs_path``/``readable_path`` fields, ``load_formation_top_well`` tags
  each raised exception with the stage that actually failed, and every
  exporting function in ``p2mem.io.tops_inventory`` now sanitizes BOTH
  candidate paths (literal substring replacement only, never a regex) and
  reports the correctly identified failing file's basename as context. (3)
  ``reconcile_formation_top_sources`` - the public in-memory API, as
  distinct from the file parsers, which already enforced this - did not
  validate its own documented contract before any numerical comparison:
  non-finite (NaN/Inf) or negative MDRT/TVDSS values, a shorter
  ``NOTE_source`` tuple (previously reaching an untyped ``IndexError``), a
  non-1-dimensional array, and an unvalidated ``mdrt_agreement_tolerance_m``
  keyword (previously accepting ``NaN``, a negative value, or a boolean,
  reaching an incidental ``TypeError`` only for a string) were all silently
  accepted or reached an undocumented incidental error. All of these are
  now rejected before any comparison, with deliberate, documented
  exceptions (``TopParsingError`` for a value/structural defect,
  ``TypeError`` for a type-class defect - matching this module's existing
  split), while valid Python ``int``/``float`` and NumPy integer/floating
  scalars (including 0-d/size-1 arrays) continue to work. (4)
  ``INCREMENT_05_MANIFEST.md`` stated that no ``/home/``, ``/root/``,
  ``/content/``, or absolute path exists ANYWHERE in the package - this was
  inaccurate, since existing synthetic tests and historical documentation
  intentionally contain fake absolute-path strings as test inputs; the
  precise, narrowly scoped claim ("no environment-dependent build path
  appears in exported CSV/JSON outputs") is now stated explicitly in
  ``INCREMENT_05_1_MANIFEST.md``, which also acknowledges the prior
  wording was overbroad. ``INCREMENT_05_MANIFEST.md`` itself is a locked
  historical record and is NOT rewritten. See ``INCREMENT_05_1_MANIFEST.md``
  for the full audit, the regression-test list, and the real-data
  non-regression verification record.

* Increment 5.1.1 is a further narrowly scoped corrective patch to
  Increment 5.1, applied after an independent technical/software-QA audit,
  WITHOUT beginning Increment 6 and WITHOUT changing any scientific result,
  tolerance, depth-mapping method, or formation-top contract. It corrected
  one remaining reconciliation edge case and three documentation-accuracy
  gaps. (1) Increment 5.1 correctly rejected two non-empty, disjoint
  marker sets as a fatal ``NO_COMMON_MARKERS`` ERROR, but its condition
  (``both sources non-empty AND intersection empty``, plus a separate
  both-empty case) still silently accepted the remaining zero-common-
  markers configuration: exactly ONE source entirely empty and the other
  non-empty - the intersection of an empty set with anything is itself
  empty, so this was still a zero-common-markers condition, but the
  ``hrs_by_canon and readable_by_canon`` non-empty guard skipped it.
  ``reconcile_formation_top_sources`` now uses the single, strictly
  correct check ``if not common_markers`` (the intersection of the two
  sources' canonical names), which is empty in every zero-common-markers
  configuration - both empty, either side alone empty, or both non-empty
  and disjoint - and is never empty whenever at least one canonical marker
  is genuinely shared, so a legitimate one-sided marker alongside at least
  one shared marker remains the pre-existing, non-fatal ``NOT_COMPARABLE``
  case. (2) ``INCREMENT_05_1_MANIFEST.md`` stated that the real Increment
  5 baseline ZIP's SHA-256 "matches" a governing-prompt-supplied hash that
  was, in fact, a two-character truncation of the real 64-character value
  - a mathematical impossibility that is now stated transparently in
  ``INCREMENT_05_1_1_MANIFEST.md`` as a documentation/input typo, never as
  baseline corruption or uncertainty. (3) ``INCREMENT_05_1_MANIFEST.md``
  Section 7 stated that ``outputs/`` is excluded from the delivered ZIP;
  this was false - the delivered Increment 5.1 ZIP packages the
  ``outputs/`` tree (including the byte-identical Increment 5 formation-
  top outputs/figures) exactly as every prior increment's ZIP has; only
  the dev-only regenerated-comparison directories and private raw source
  files are excluded, and this is now stated accurately. (4) the same
  manifest's clean-room section stated that no ``*.las`` file exists in
  the package; this was false because small, intentionally packaged
  synthetic LAS/checkshot/deviation/top fixtures exist under
  ``tests/fixtures/`` for portable testing - the corrected wording
  distinguishes these fictional fixtures (never leaked private data) from
  the genuinely excluded real/private project LAS, deviation, checkshot,
  and formation-top source files. See ``INCREMENT_05_1_1_MANIFEST.md`` for
  the full audit, the regression-test list, and the real-data
  non-regression verification record.

Increment 6 (this release, v0.6.0) adds the gamma-ray QC, shale-proxy
sensitivity, well-frame assembly, and method-eligibility framework, on top
of the LOCKED Increment 1-5.1.2 foundation (no locked module, config,
test, fixture, notebook, output, or figure is modified by it). It
contributes:

* ``p2mem.wellframe_models`` / ``p2mem.wellframe`` - a typed, auditable
  per-well assembly of the locked canonical LAS curve arrays alongside the
  locked MD->TVD/TVDSS mapping, with per-sample validity masks, per-curve
  provenance, QC flags, and evidence classification. Sample count and
  order are preserved exactly; invalidity is expressed only through masks;
  no depth is ever extrapolated (a LAS sample outside the survey's own MD
  coverage is recorded as depth-unmapped, never clamped or held).
* ``config/petrophysics_eligibility.yml`` - the human-authored, reviewable
  per-well gamma-ray-family disposition, endpoint-sensitivity policy, and
  eligibility rules. Boreas 1 is formally excluded from every
  lithology-dependent, GR-normalized, shale-proxy and NCT-candidate
  calculation under the machine-readable reason
  ``BOREAS_ECGR_SCALE_UNRESOLVED``; it is EXCLUDED, never corrected or
  rescaled, because no calibration evidence exists to support any
  correction.
* ``p2mem.petrophysics_models`` / ``p2mem.petrophysics`` - factual
  GR-family QC statistics for every well (including excluded ones),
  per-well low/base/high endpoint scenarios whose measured endpoint values
  are recorded, and the dimensionless GR index with clipped and unclipped
  results retained side by side. The only permitted shale-proxy transform
  is the linear identity of the clipped index, exported under the
  self-labelling name ``VSH_GR_linear_proxy_frac``; every nonlinear Vsh
  transform (Larionov, Clavier, Stieber, ...) is deliberately DEFERRED
  pending a retrieved, verified primary-source method record.
* ``p2mem.method_eligibility`` - three input-admissibility masks
  (``eligible_density_for_sv``, ``eligible_dynamic_elastic``,
  ``eligible_sonic_nct_candidate``) plus contiguous-interval registers on
  MD/TVD/TVDSS with explicit, configured, tested gap tolerances that
  distinguish sample-count continuity from physical-depth continuity.
  ELIGIBILITY IS NOT VALIDITY: none of the gated methods is implemented,
  fitted, or validated here.
* ``p2mem.io.petrophysics_inventory`` - deterministic summary/interval/
  manifest builders that never export a per-sample real-data array and
  never emit an absolute path.

Increment 6 assigns NO named lithology anywhere. Gamma-ray response is not
uniquely diagnostic of rock type and no independent lithological evidence
exists in this project, so every classification it produces describes DATA
AND PROXY CONFIDENCE ONLY, guarded by an enforced prohibited-vocabulary
check.

Increment 6.1 (v0.6.1) is a narrowly scoped corrective patch
to Increment 6, applied after an independent geomechanics/rock-physics and
software audit. It changes no architecture and starts no new phase. Four
findings were corrected:

* **Vp/Vs terminology and boundaries.** Increment 6 labelled every excluded
  Vp/Vs ratio "non-physical" and used a strictly exclusive sqrt(2) bound.
  Both were wrong. With r = Vp/Vs, nu = (r^2 - 2) / (2 (r^2 - 1)) and
  K = rho (Vp^2 - (4/3) Vs^2), so: r = sqrt(2) gives nu = 0 EXACTLY and must
  be ACCEPTED by a non-negative-nu policy (the bound is now INCLUSIVE);
  sqrt(4/3) < r < sqrt(2) gives POSITIVE bulk modulus with negative Poisson
  ratio - unusual and outside this project's conservative policy, but not
  physically impossible; only r <= sqrt(4/3) implies a non-positive bulk
  modulus and is genuinely outside the isotropic elastic model; and r > 4 is
  a CONFIGURED PLAUSIBILITY LIMIT, not a Poisson-domain boundary (nu ~= 0.467
  there). The condition is renamed a CONFIGURED NON-NEGATIVE-POISSON-RATIO
  APPLICABILITY SCREEN, the three config bounds are separately named, and
  exclusions are diagnosed by regime (`n_ratio_nonpositive_bulk_modulus`,
  `n_ratio_positive_bulk_but_negative_poisson`,
  `n_ratio_above_configured_plausibility_max`, `n_vp_not_greater_than_vs`) -
  never aggregated under a single "non-physical" count.
* **Poseidon North 1 depth-tied status.** Its machine-readable `use_status`
  was `screening_proxy_allowed` while it has no approved formation tops,
  contradicting both the status vocabulary and the prose limitation. It is
  now `screening_proxy_allowed_depth_tied`, and a config invariant makes any
  contradictory has-tops/use-status pairing fail config loading loudly.
* **Unsupported lithology claims and a circular gate.** Project-specific
  "clastic-dominated" assertions and the unsupported "regionally persistent"
  cross-well claim are removed (two of the three GR-eligible wells have no
  approved tops, so cross-well stratigraphic persistence cannot be
  established). `clastic` and related rock-class terms join the prohibited
  vocabulary, and `named_lithology_assigned` is now DERIVED from an actual
  validation pass over every persisted label, per-well note, mask name and
  manifest statement - injecting a prohibited term makes the validation, the
  manifest flag and the completion gate fail together. Validation is
  three-tier: LABELS admit no prohibited term at all, per-well PROSE may name
  the method category ("shale proxy") but not assert a rock, and EXPLANATORY
  text may use rock names as generic examples of gamma-ray non-uniqueness
  but never in a sentence naming one of this project's wells.
* **Gross versus net/strict interval thickness.** The reported "qualifying
  thickness" summed block endpoint spans that may contain explicitly bridged
  ineligible samples. Thickness is now exported only under qualified names -
  `gross_thickness_*_m` (endpoint span, bridging-inclusive) and
  `net_thickness_*_m` (bridged gaps removed) - every mask is additionally
  decomposed under a `strict_no_gap` contiguity policy for comparison, every
  sensitivity case reports its bridged-sample and interrupted-block counts,
  and Figure 4 now draws qualifying and rejected sub-threshold blocks
  distinctly with the counted population stated.

Increment 6.1.1 (v0.6.2) is a second, narrower corrective
patch, applied after an independent audit of Increment 6.1. It starts no
new phase, changes no architecture, and alters no configured tolerance,
threshold, or qualifying-block policy. Five findings were corrected:

* **Named-lithology validator bypass.** The method-term allowlist that lets
  per-well prose name a METHOD ("shale proxy") contained "shale gas", which
  is not a method-category phrase but can be a direct geological/hydrocarbon
  assertion; sentences such as "This interval contains shale gas." therefore
  passed with zero violations. The allowlist is reduced to the two phrases
  that are genuinely method/quantity names required to describe this
  project's boundaries - ``shale proxy`` and ``shale volume`` - and the
  allowance is now evaluated PER SENTENCE and suppressed entirely in any
  sentence carrying a geological-assertion cue (``contains``, ``comprises``,
  ``bearing``, ``facies``, ...) or where the phrase is immediately preceded
  by a quantity cue (``has a high shale volume``). Regression tests prove
  the allowlist can no longer hide a geological assertion and that injecting
  either bypass case into manifest content makes ``named_lithology_assigned``
  true, ``lithology_validation.n_violations`` non-zero, and the completion
  gate fail.
* **Stale Vp/Vs scientific description.** The active top-level docstring of
  ``p2mem.method_eligibility`` still described the screen as keeping samples
  within the Poisson domain under a strictly EXCLUSIVE sqrt(2) ratio bound,
  contradicting the corrected implementation. It now states the configured
  non-negative-Poisson-ratio applicability screen with an INCLUSIVE
  ``Vp/Vs >= sqrt(2)`` bound and says explicitly that this is neither a
  physical-possibility test nor a boundary of the mathematical Poisson
  domain. A regression test asserts the corrected wording against the live
  module documentation.
* **Ambiguous interruption counts.** ``n_interruptions`` was a boolean-like
  flag reported as a count (always 1 for every bridged block), and
  ``n_interrupted_subruns`` did not describe what it counted. Interval
  records now carry three separately named, independently meaningful
  quantities: ``n_bridged_samples`` (ineligible samples absorbed inside the
  gross block), ``n_bridged_gaps`` (distinct bridged runs) and
  ``n_eligible_subruns`` (strictly contiguous eligible sub-runs), with the
  identity ``n_bridged_gaps == n_eligible_subruns - 1`` enforced at
  construction. The ambiguous aliases are gone from all active exports.
* **Notebook output-count error.** The notebook creates and checks eight
  deterministic CSV/JSON outputs but its gate text and printed label said
  seven. Both now say eight, and the gate asserts both the declared count
  and the existence of all eight files. Four figures remain separate,
  giving twelve outputs/figures in total.
* **Incorrect Increment 6.1 delta arithmetic.** The Increment 6.1 completion
  record stated "Changed (12)"; a clean recursive comparison gives 18
  changed, 3 added and 0 removed (21 path differences). That statement is
  explicitly superseded - not silently rewritten - in the Increment 6.1.1
  manifest and completion record, which report both measured deltas file by
  file.

Increment 6.1.2 (v0.6.3) is a third corrective patch, applied
after an independent audit of Increment 6.1.1. It starts no new phase, changes
no architecture, alters no scientific threshold, tolerance, endpoint scenario,
contiguity policy, GR disposition, depth-mapping rule, or real-data
interpretation, and adds no runtime dependency (NumPy and PyYAML remain the
only two). Two residual findings were corrected:

* **The named-lithology validator was bidirectionally incorrect.** Increment
  6.1.1 decided the method-phrase allowance from a SENTENCE-WIDE assertion-cue
  list plus a fixed LOOK-BEHIND window. Both were structurally wrong. Because
  nothing to the RIGHT of an allowed phrase was ever inspected, and because
  the normalizer destroyed possessives, direct assertions such as "Poseidon
  2's shale volume is high.", "Poseidon 2's shale volume is 70 percent." and
  "The interval's shale volume exceeds 60 percent." passed with ZERO
  violations. Conversely, because a sentence-wide cue fires without asking
  what the cue word is predicated OF, legitimate method and boundary
  statements such as "This method contains a shale proxy calculation." and
  "The analysis shows no shale volume was computed." were reported as
  lithological assertions. The allowance is no longer a property of the
  sentence: it is decided for EACH OCCURRENCE of an allowed phrase from that
  occurrence's own local grammar - whether it is possessed by a geological
  entity (``the interval's shale volume``), modified by a magnitude (``a high
  shale volume``), predicated with an amount or a dominance (``is 70
  percent``, ``exceeds 60 percent``, ``dominates the interval``), or sits in a
  sentence whose composition verb takes a GEOLOGICAL subject (``the unit
  comprises ...``, but not ``this method contains ...``). Apostrophes and
  possessives are parsed rather than erased, sentence boundaries include line
  breaks, and genuinely ambiguous project-specific sentences fail closed.
  ``SCOPE_LABEL`` remains absolute: a label gets no latitude at all.

* **Interval-record invariants were only partially enforced.** Increment 6.1.1
  claimed the interruption-count identities were enforced at construction, but
  the only check was ``n_bridged_gaps == n_eligible_subruns - 1``, skipped
  whenever either value was ``None`` or ``n_eligible_subruns`` was 0. Records
  with missing counts, negative counts, zero sub-runs, boolean counts, or
  bridged samples without a bridged gap were all constructible. All three
  counts are now validated together as one coherent record: each is required,
  must be a true integer count (booleans, strings, complex values and
  fractional floats are rejected, never coerced), ``n_bridged_samples >= 0``,
  ``n_bridged_gaps >= 0``, ``n_eligible_subruns >= 1``, ``n_bridged_gaps ==
  n_eligible_subruns - 1``, ``n_bridged_samples == 0`` if and only if
  ``n_bridged_gaps == 0``, and ``n_bridged_samples >= n_bridged_gaps`` because
  every distinct gap holds at least one sample. Each violation raises
  ``PetrophysicsInputError`` naming the offending field or relationship, and
  the removed ``n_interruptions`` / ``n_interrupted_subruns`` field names are
  rejected outright rather than silently ignored.

Every scientific result is unchanged by this patch: the same eight
deterministic outputs and four figures are produced, byte for byte.

Increment 6.1.3 (v0.6.4) is the fourth and architectural
corrective patch, applied after an independent audit of Increment 6.1.2. It
starts no new phase, changes no scientific threshold, tolerance, endpoint
scenario, contiguity policy, GR disposition, depth-mapping rule, or real-data
interpretation, and adds no runtime dependency (NumPy and PyYAML remain the
only two).

**The validator no longer tries to understand English.** Increments 6.1.1 and
6.1.2 each attempted to decide, from grammar, whether a sentence containing a
rock name was NAMING A METHOD or ASSERTING GEOLOGY - 6.1.1 with a
sentence-wide cue list and a fixed look-behind window, 6.1.2 with
per-occurrence possessive/modifier/predicate/subject analysis. Both were
audited and both failed in BOTH directions. 6.1.2 still missed "The interval
has a shale volume.", "The shale volume in Poseidon 2 exceeds 60 percent." and
"Poseidon 2 shale volume was determined to be high.", while wrongly rejecting
"The well contains no shale volume estimate." Every one of those is a list gap
or a window edge. The lesson is not that the lists were too small: free-text
geological-assertion detection is unbounded, and no finite grammar closes it.

Increment 6.1.3 replaces judgement with membership. Every scope is now decided
by set membership or exact equality:

* ``SCOPE_LABEL`` - verdicts. Zero allowance.
* ``SCOPE_INTERPRETIVE`` - project-specific prose. **ZERO ALLOWANCE.** There is
  no exemption path at all, so there is nothing to bypass. The
  ``allow_method_phrases`` parameter is gone.
* ``SCOPE_METHOD`` (new) - method and limitation statements. The text must
  EQUAL a member of the closed, provenance-tagged ``METHOD_STATEMENTS``
  registry. Not matched, not scored - equal. The registry has five members,
  measured rather than guessed by scanning every string literal in active
  packaged source and every persisted field under a zero-allowance rule.
* ``SCOPE_EXPLANATORY`` - generic scientific text. A rock name is permitted
  only when the sentence refers to no project well AND no project rock body.
  6.1.2 checked well names alone, which is why an assertion about "the
  interval" behaved inconsistently between scopes.

The deletion is the correction: ``ALLOWED_METHOD_TERM_PHRASES``,
``GEOLOGICAL_ENTITY_TOKENS``, ``COMPOSITION_PREDICATES``,
``MAGNITUDE_PREDICATES``, ``MAGNITUDE_WORDS``, ``COPULAR_VERBS``,
``ATTRIBUTIVE_ROCK_TOKENS``, ``NEGATION_TOKENS``, the look-behind and
look-ahead windows, subject resolution and occurrence classification are all
GONE, and a regression test asserts none of them is importable. Correctness is
proved by CLOSURE - every prohibited term, in every position, inside carrier
prose built from the exact constructions that defeated both previous
implementations - not by a list of example sentences.

This patch also closes a gap neither audit reported: three strings this project
WRITES INTO PACKAGED EXPORTS had never been inside the validated scope in any
increment, including a ``calibration_status`` value literally containing
``not_a_shale_volume`` persisted to every row of
``gr_proxy_sensitivity_summary.csv``. All three are now scanned in
``SCOPE_METHOD``, which raises ``n_fields_checked`` from 138 to 143.

**Interval-record type gate.** Increment 6.1.2 enforced the relational rules
but let three type defects through: ``0.0 / 0.0 / 1.0`` was accepted although
the contract requires integers; ``NaN`` and ``Inf`` escaped as bare
``ValueError`` / ``OverflowError`` from ``int()``; and an unknown keyword such
as a misspelled count name was silently ignored, leaving the real count unset.
Keyword acceptance is now a WHITELIST over the declared slots, floats are
rejected outright including whole-valued ones, and non-finite values raise a
typed ``PetrophysicsInputError`` before any conversion is attempted.

All eleven scientific outputs and figures are byte-identical to Increment
6.1.2; only the manifest changes, by the single scalar named above.

Increment 6.1.4 (v0.6.5) completes the architecture Increment
6.1.3 began. It is deliberately narrow: one scope rule, no other change.

Increment 6.1.3 closed LABEL and INTERPRETIVE by removing every exemption and
closed METHOD by exact registry membership - but left EXPLANATORY decided by a
FINITE TOKEN LIST of project references. That is the same shape of rule that
failed in 6.1.1 and 6.1.2, and it had the same defect. Ordinary stratigraphic
and exploration nouns were absent from the list, so in explanatory scope

    "The member is shale."               passed
    "The group is limestone."            passed
    "The package is a clean sandstone."  passed
    "The play is shale-dominated."       passed
    "The prospect is carbonate."         passed
    "The target is sandstone."           passed

while "The upper member is a clean sandstone reservoir." failed only because
``reservoir`` happened to be listed - an accident, which is what a list-shaped
rule produces.

The rule is INVERTED and made identical in kind to SCOPE_METHOD: explanatory
text carrying a prohibited term is a violation UNLESS the text is a member of
the closed ``GENERIC_EXPLANATORY_STATEMENTS`` registry. It FAILS CLOSED, so no
vocabulary gap can admit anything. ``PROJECT_WELL_NAME_TOKENS``,
``PROJECT_ROCK_BODY_TOKENS`` and ``PROJECT_REFERENCE_TOKENS`` are deleted, and
a regression test asserts none of them is importable. Nothing in the validator
now enumerates what a project reference looks like.

The generic registry is **empty**, as a measured fact rather than an omission:
this project persists exactly one explanatory field,
``manifest.named_lithology_statement``, and it carries no prohibited term at
all. The mechanism is nevertheless live and tested, because later increments
explaining gamma-ray non-uniqueness may genuinely need to write "a clean
sandstone and a clean limestone read alike on GR" - and when they do, that is
a registered, reviewable statement rather than a sentence admitted by shape.

All four validation scopes are now closed by membership or exact equality, and
the closure proof covers all of them. All twelve outputs and figures are
byte-identical to Increment 6.1.3.

Increment 6.1.5 (v0.6.6) replaces blacklist-based acceptance
with POSITIVE AUTHORIZATION, and is the final Increment 6 corrective patch.

Increments 6.1 through 6.1.4 all asked the same question in different ways:
"does this text contain a geological assertion?" All four shared one
structural assumption - that content is ACCEPTABLE BY DEFAULT and becomes
unacceptable only when a recognizer fires - and all four were defeated,
finally by a word no recognizer had been given. In the 6.1.4 package these
all passed label, interpretive and explanatory scope:

    "The interval is chalk."         "The interval is chert."
    "The interval is halite."        "The interval is tuff."
    "The interval is gypsum."        "The interval is basalt."
    "The interval is conglomerate."  "The interval is dolostone."
    "The interval is lignite."       "The interval is calcareous."

and so did "The interval is qxzite.", a word that does not exist. They already
failed in METHOD scope, which was the only scope then requiring registration -
and that is the clue this patch acts on. A longer blacklist would have caught
the first ten and still missed the eleventh, so lengthening it is neither a
completion criterion nor the mechanism this package's assurance rests on.

The model is inverted. Nothing is acceptable by default:

* ``SCOPE_LABEL`` - the value must be a member of ``APPROVED_LABELS``, a
  registry of typed, enumerated label values each declaring its field kind,
  purpose and provenance. Arbitrary caller-supplied label text is rejected
  even when it contains no recognizable rock name at all.
* ``SCOPE_INTERPRETIVE`` - the text must resolve to a registered
  ``statement_id``, or to a reviewed ``template_id`` whose substitutions are
  strictly typed. Unregistered free text is rejected unconditionally.
* ``SCOPE_METHOD`` - the text must resolve to a registered ``statement_id``,
  and the id and the exact rendered text are validated TOGETHER, so neither a
  renamed id nor an edited sentence passes on the strength of the other.
* ``SCOPE_EXPLANATORY`` - identical, for every non-empty statement, whether or
  not any prohibited term is detected. The early-pass behaviour equivalent to
  "if no prohibited term is found: accept" is gone from every scope.

The registries were MEASURED from the actual persisted Increment 6 export, not
designed: 17 approved label values, 17 registered statements (11 interpretive,
5 method, 1 explanatory) and 1 controlled template covering the three per-well
confidence rationales, whose only variable parts are three decimal literals.
Every registered entry carries a stable id, its scope, its exact text or
controlled template, a scientific purpose, a provenance justification, and its
permitted typed substitutions. Duplicate ids fail at import.

``PROHIBITED_LITHOLOGY_TERMS`` survives ONLY as a supplementary diagnostic
linter. It authorizes nothing, and it is never cited as evidence that all
named lithologies have been detected - it cannot be: its 28 terms include
neither ``chalk`` nor ``chert`` nor ``qxzite``, and every rejection listed
above happens with that linter returning empty.

THE DEFENSIBLE ASSURANCE STATEMENT, which supersedes the wording of every
earlier Increment 6 manifest: every persisted project-specific classification
and interpretive statement is generated from an approved typed value, a
controlled template, or a registered statement, and arbitrary free text cannot
enter these controlled fields. This is NOT a claim that the software
understands or exhaustively recognizes natural-language lithology; it does
not, and no earlier version did. Free-form notebook narrative and
documentation lie outside these controlled fields and remain subject to
ordinary manual scientific review.

All twelve outputs and figures are byte-identical to Increment 6.1.4.

Increment 6.1.6 (previous corrective patch, v0.6.7) enforces authorization at the EMISSION
BOUNDARY. Increment 6.1.5's positive-authorization model was sound but was
applied to a scope the manifest builder RECONSTRUCTED from dispositions,
confidences and masks - a parallel object, not the records written to disk. A
`GrEndpointScenario` carrying `description="The interval is chalk."` was
persisted verbatim by `build_gr_endpoint_scenario_rows()` while never entering
that 143-field scope, so it could not move `named_lithology_assigned`. The unit
validator was closed; the export path was not.

Five findings, all reproduced first:

* **Exported endpoint description bypass** - closed. The row builder now
  authorizes the value it is about to place in the record and raises
  ``PetrophysicsInputError`` if it cannot.
* **Incomplete output coverage** - closed. ``p2mem.io.output_policy`` declares a
  category for EVERY string column and JSON path of all eight artifacts (105
  entries). Nothing is unclassified; unknown artifacts, columns, JSON paths and
  categories fail closed. Categories: structural enum, identifier, filename,
  typed label, registered statement, controlled template, structured diagnostic,
  sanitized diagnostic. There is no general category admitting arbitrary prose.
* **Field-kind label mismatch** - closed. ``APPROVED_LABELS`` is keyed by
  ``(field_kind, value)``; ``use_status="GR"`` and ``mask_name="measured"`` now
  fail, the field kind is supplied by the caller and never parsed from a context
  string, and duplicate pairs fail at import.
* **Stale exported derivation** - closed. The superseded prohibited-term wording
  is replaced by a registered statement describing the model actually
  implemented, so the assurance prose is itself authorized.
* **Duplicated manifest statement** - closed. The manifest takes
  ``REGISTERED_STATEMENTS["named_lithology_statement"].text``; no second literal
  exists to diverge from it.

Export is two-stage: build the exact pre-serialization records, authorize every
string occurrence, write, then RE-READ the written bytes and authorize again.
A value mutated after authorization is caught before delivery, and nothing is
written at all if any field fails.

The assurance metrics now say what they count. ``n_fields_checked=143`` is gone;
the manifest reports ``scope_object_fields_checked`` alongside an
``emitted_field_coverage`` block giving total emitted string-field occurrences,
controlled occurrences, structural occurrences, occurrences outside the
guarantee, unclassified fields, unauthorized fields and field-kind mismatches.
Machine diagnostics and the operator-facing issue text are counted separately
and are explicitly OUTSIDE the controlled-interpretation guarantee.

All seven CSV artifacts and all four figures are byte-identical to
Increment 6.1.5. ``petrophysics_eligibility_manifest.json`` changes, and only
inside ``lithology_validation``: the derivation is corrected and the coverage
metrics become truthful. No numerical, disposition, mask, threshold or depth
value moves.

Increment 6.1.7 (this release, v0.6.8) corrects the remaining export-boundary
assurance defect. Increment 6.1.6 discovered fields from non-empty string
values, so missing controlled columns, empty or numeric-looking controlled
strings, non-string substitutions and unknown fields with non-prose values
could escape collection. It also wrote into the official directory before the
post-write check and compared aggregate coverage counts, allowing an
authorized value to be changed into a different authorized value without
detection and leaving partial artifacts after a rejected write.

Increment 6.1.7 defines exact schemas for all eight artifacts and validates
artifact inventory, ordered CSV columns, JSON keys, requiredness, strict types
and finite numerics independently of prose authorization. Every schema-declared
string occurrence is then authorized, including empty and numeric-looking
strings. Candidate files are written only to an isolated sibling directory,
re-read and re-validated, and every typed field and row is compared against the
authorized pre-serialization record before failure-atomic publication. Unknown
well identifiers, stale output artifacts, serializer failures, field additions
or deletions, row reordering, type changes and authorized-to-authorized value
changes all fail closed while leaving the official destination unchanged.
This is failure-atomic for handled process errors; it is not a claim of
multi-file atomicity across power loss or operating-system failure.

Increment 6.1.7 is now LOCKED. Every module, config, test, fixture, notebook,
manifest, ledger, output and figure it delivered is byte-identical in this
release; the only pre-existing files that change are this module's version and
narrative, ``pyproject.toml``'s version, and ``README.md``.

Increment 7 / 7.0.1 / 7.0.2 / 7.0.3 / 7.0.4 (this release, v0.7.4) adds DENSITY QC,
DENSITY-COVERAGE QUALIFICATION, and a VERTICAL OVERBURDEN-STRESS FRAMEWORK.

Increment 7.0.1 is a narrowly scoped corrective patch. It makes profile
truncation disposition-driven so every unresolved internal gap stops the
integral, including a short gap left unresolved because bridging is disabled
or a bridge precondition fails. It also rejects coercible non-numeric scalar
inputs and malformed closed configuration vocabularies. The 10 m bridge
threshold is documented as a sensitivity-tested heuristic rather than a
mathematical error bound, and the shallow-column endpoints are labelled as
conditional/illustrative scenarios rather than physical bounds.

Increment 7.0.2 is an assurance-only corrective patch. It fixes the completion
gate's scenario-basis lookup so the gate reads the emitted
``assumed_density_basis`` field declared by the real CSV schema, and it makes
the public frozen ``OverburdenConfig`` constructor enforce the same strict
types, finite numerics, closed vocabularies and internal invariants as the YAML
loader. This closes bypasses through direct construction and
``dataclasses.replace`` without changing any scientific input, calculation,
threshold, eligibility result or output artifact.

Increment 7.0.3 closes one configuration-to-computation provenance defect.
The high shallow-column scenario is implemented from the explicitly stored
same-well P05 density statistic, so ``scenario_high_percentile`` is now
required to equal exactly 5.0 at both YAML loading and public constructor
entry points. Earlier releases accepted other percentile labels even though
the calculation continued to use P05. The packaged configuration already
uses 5.0, so no scientific input, calculation, threshold, eligibility result,
CSV/JSON artifact or figure changes in this patch.

Increment 7.0.4 is a final corrective patch following an independent audit of
the full data and publication paths. The high shallow-column endpoint now uses
the P05 of the eligible integration population, not all finite density values;
per-call gap thresholds and all threshold-bearing records reject boolean,
textual, complex, negative and non-finite values; scenario totals disclose
assumed, conditioned and measured fractions that sum to one; the locked seabed
reader fails closed on missing, malformed, duplicate or non-finite matching
rows; and the JSON exporter writes explicit UTF-8 LF bytes on every platform.
The four approved wells have identical finite and eligible RHOB populations,
zero bridged contribution in the two published scenario wells, and a valid
unique seabed table, so the measured scientific values remain unchanged. The
assurance JSON and scenario/QC schemas change to state these contracts
truthfully. Increment 8 has not been started.

What it does
------------
* ``config/overburden_stress.yml`` - the human-authored screening policy: the
  bulk-density plausibility band, the gap-conditioning threshold, standard
  gravity, the assumed seawater density, and the shallow-column scenarios. No
  well name appears anywhere in it.
* ``p2mem.overburden_models`` - typed records and the closed vocabularies for
  gap classes, gap dispositions, overburden statuses, limiting reasons, seabed
  bases and scenario names, with constructor-level invariants that refuse an
  incoherent record (a bridged shallow gap, a total reported while a component
  is unresolved, a gap count that disagrees with its sample count, an absolute
  status carrying a limiting reason).
* ``p2mem.density_qc`` - RHOB availability, unit and conversion confirmation,
  TWELVE explicit per-sample masks, factual QC statistics, gap classification
  into the five structurally distinct cases, and gap conditioning into a
  SEPARATELY NAMED array.
* ``p2mem.overburden`` - trapezoidal integration of ``rho*g*dz`` in TRUE
  VERTICAL DEPTH, run-time verification of the project's depth-sign
  convention against each frame's own arrays, evidence-derived method
  eligibility, transparent shallow-column scenarios, and the mandatory
  gap-threshold sensitivity.
* ``p2mem.io.overburden_policy`` / ``overburden_registry`` /
  ``overburden_inventory`` / ``overburden_workflow`` - the Increment 6.1.7
  output-policy architecture GENERALIZED so its registries arrive as an
  explicit policy bundle, plus Increment 7's own nine closed artifact schemas
  and its complete field classification. The locked Increment 6 module and
  export path are untouched; a test harness runs the generalized engine over
  the locked registries and the real packaged Increment 6 records and requires
  the same decision on every occurrence.

Scientific boundaries this increment does not cross
---------------------------------------------------
An absolute vertical overburden stress requires density coverage from the
relevant datum or seabed to the evaluation depth. Where that column is not
measured, Increment 7 reports a partial measured increment or a transparent
low/base/high sensitivity spread, and NEVER a single absolute curve presented
as measured truth. No density value is clipped, rescaled, smoothed, despiked,
replaced or extrapolated; invalid samples are masked, short internal gaps may
be linearly bridged in true vertical depth into a separate array whose
contribution is reported separately, and nothing else is filled. The water
column, the unresolved shallow column and standard gravity are ASSUMPTIONS,
and every reported stress is partitioned so that measured, conditioned and
assumed contributions stay separately visible.

Increment 7 assigns NO named lithology. The density screening band, the gap
policy, the shallow-column bracket and the eligibility ladder are expressed in
physical and coverage terms, and none of them is justified by an assumed rock
type.

Subsequent increments (pore pressure, elastic property calculation,
strength, stress, and wellbore-stability screening) are added one
validated phase at a time and are intentionally absent from this version -
importing them will fail until they exist. Increment 8 has NOT been started:
no pore-pressure prediction, NCT fitting, Eaton/Bowers/equivalent-depth/
drilling-exponent method, effective-stress calculation, elastic-property
modelling, rock-strength modelling, horizontal-stress calculation, stress
calibration, mud-window calculation, or wellbore-stability analysis exists in
this package.
"""

__version__ = "0.7.4"

# Fixed project-wide assurance tier. Referenced by later modules (reporting,
# plotting) so that every generated output can stamp its own classification
# without each module re-declaring the string. This value must not be
# changed without a documented calibration event (e.g. a verified RFT/MDT,
# LOT/XLOT, or core-calibrated log tie) recorded in the method-and-citation
# register.
ASSURANCE_TIER = "Tier C - Screening-Level / Uncalibrated Educational"

__all__ = ["__version__", "ASSURANCE_TIER"]


#### `README.md` - implementation status, directory tree, and the Increment 7 limitations

In [ ]:
%%writefile README.md
# Poseidon 2 — 1D Mechanical Earth Model

**Author:** Mikael Elgo

**Project classification:** Tier C — Screening-Level / Uncalibrated Educational 1D Mechanical Earth Model (MEM)

> **This project is screening-level, uncalibrated, and educational in nature. It is NOT validated against independent field measurements (no confirmed RFT/MDT pressure points, LOT/XLOT tests, or core-calibrated log ties are currently incorporated), and it must NOT be used for operational drilling, well-design, or any real-world decision-making. It exists to demonstrate a technically defensible, transparent, modular geomechanics workflow — not to produce field-ready predictions.**

---

## Purpose and technical scope

This repository implements a modular, reproducible 1D Mechanical Earth Model workflow for the Poseidon 2 well, built from well logs, deviation surveys, checkshot data, formation tops, and Vp/Vs data supplied for the project. The intended end-to-end scope (delivered incrementally, one validated phase at a time) covers:

- data quality control and depth alignment across LAS logs, deviation surveys, and checkshot data
- pore-pressure prediction (Eaton-family methods, contingent on a defensible normal compaction trend)
- elastic properties (dynamic Vp/Vs-derived Poisson's ratio, and density-dependent moduli where density coverage permits)
- rock-strength estimation
- vertical-stress (overburden) modelling
- horizontal-stress and wellbore-stability screening (Kirsch elastic wall-stress equations with Mohr–Coulomb/Mogi–Coulomb failure criteria)
- uncertainty treatment via deterministic low/base/high scenarios and one-at-a-time sensitivity (tornado) analysis, rather than unsupported probabilistic distributions

Every empirical or correlation-based relationship used anywhere in this project (Eaton, Bowers, Gardner, Castagna, etc.) is required to have a recorded source, stated units, applicability range, and calibration status in the project's method-and-citation register *before* it is implemented in code. Nothing is fabricated or assumed silently: missing measurements, missing calibration points, and unavailable data are always reported as unavailable rather than filled in.

This is a personal portfolio project intended to demonstrate scientific rigor, reproducibility, and honest handling of data limitations — not a commercial or operational deliverable.

## Current implementation status

**Increment 7 / 7.0.1 / 7.0.2 / 7.0.3 / 7.0.4 (this release, v0.7.4): Density QC, Density-Coverage Qualification, and the Vertical Overburden-Stress Framework.** Builds on the LOCKED Increment 6.1.7 baseline by adding bulk-density availability and coverage measurement, twelve explicit per-sample validity masks, gap classification and conditioning, trapezoidal integration of `rho*g*dz` in TRUE VERTICAL DEPTH, an evidence-derived eligibility ladder, and transparent shallow-column sensitivity scenarios. **No approved well supports an absolute vertical overburden-stress curve.** Boreas 1 and Poseidon 2 have quantified unmeasured seabed-to-log columns of 3,486.54 m and 3,567.14 m TVD respectively; Poseidon North 1 and Proteus 1ST2 have no approved formation tops, so their seabed, water-column and shallow-gap thicknesses are not determinable. Increment 7.0.1 corrects the fail-closed profile rule and hardens YAML-loaded configuration. Increment 7.0.2 corrects the notebook gate and closes direct-constructor configuration bypasses. Increment 7.0.3 closes the P05 configuration-label contract. Increment 7.0.4 closes five remaining end-to-end contracts: the high endpoint uses eligible-population P05, public threshold overrides and threshold-bearing records fail closed on invalid values, scenario fractions separately sum assumed + conditioned + measured to one, locked seabed ingestion rejects missing/malformed/duplicate/non-finite matching data, and JSON is serialized as explicit UTF-8 LF bytes on every platform. The four approved wells' measured values remain unchanged because their finite and eligible RHOB populations coincide, their published scenario wells have zero bridged contribution, and the locked seabed table is valid and unique. The 10 m bridging threshold remains a sensitivity-tested project heuristic—not a rigorous endpoint-derived error bound—and the shallow-column low/high values remain conditional illustrative scenarios, not physical bounds. See `INCREMENT_07_0_4_MANIFEST.md`; prior records remain locked historical records. **Increment 8 has not been started.**

**Increment 6 / 6.1 / … / 6.1.7 (this release, v0.6.8): Gamma-Ray QC, Shale-Proxy Sensitivity, Well-Frame Assembly, and Method-Eligibility Framework.** Increment 6.1.7 replaces value-driven output discovery with exact schemas for all eight artifacts. Artifact inventory, ordered CSV columns, JSON keys, requiredness, strict types and finite numeric values are validated independently of prose authorization; every declared string occurrence is then authorized, including empty and numeric-looking strings. Export writes a complete candidate set to an isolated staging directory, re-reads and re-validates it, and compares every typed field and row before failure-atomic publication. Missing/unknown fields, type substitution, authorized-to-authorized mutation, row reordering, stale artifacts, unknown well keys and serializer failures now fail closed without changing the official destination. This corrects assurance/export behavior only; no scientific calculation, threshold, disposition or result changes. See `INCREMENT_06_1_7_MANIFEST.md`.

**Increment 6.1.6 (previous corrective patch, v0.6.7): Gamma-Ray QC, Shale-Proxy Sensitivity, Well-Frame Assembly, and Method-Eligibility Framework.** Increment 6.1.6 enforces authorization at the **emission boundary** — see `INCREMENT_06_1_6_MANIFEST.md`. Increment 6.1.5's model was sound but validated a *reconstructed* scope rather than the records written to disk, so an endpoint `description` carrying an unauthorized sentence was persisted verbatim without ever entering that scope. Every string column and JSON path of all eight artifacts now carries a declared policy classification, labels are authorized by `(field_kind, value)` together, and export is two-stage: authorize the exact records, write, then re-authorize the written bytes. Eleven of twelve outputs are byte-identical to 6.1.5; the manifest changes only inside `lithology_validation`, where the derivation is corrected and the coverage metrics become truthful. Increment 6.1.5 is the preceding corrective patch — see `INCREMENT_06_1_5_MANIFEST.md`. It replaces blacklist-based acceptance with **positive authorization**: every persisted label must be an approved typed value, and every persisted interpretive, method and explanatory statement must resolve to a registered `statement_id` or a controlled `template_id` with strictly typed substitutions. Unregistered free text is rejected unconditionally, so an unknown lithology — or an invented word — cannot enter a controlled field. `PROHIBITED_LITHOLOGY_TERMS` is demoted to a supplementary diagnostic linter that authorizes nothing. All twelve outputs/figures are byte-identical to 6.1.4 and no scientific value changed. Increment 6.1.4 completes the architecture Increment 6.1.3 began — see `INCREMENT_06_1_4_MANIFEST.md`. 6.1.3 closed three of four validation scopes by membership or exact equality but left explanatory scope decided by a finite token list of project references, which omitted ordinary stratigraphic nouns (`member`, `group`, `package`, `play`, `prospect`, `target`) and therefore admitted assertions built on them. That rule is inverted and closed by registry, failing closed; the three token lists are deleted. All four scopes are now closed, all twelve outputs/figures are byte-identical to 6.1.3, and no scientific value changed. Increment 6.1.3 is the preceding architectural corrective patch, applied after an independent audit of Increment 6.1.2 — see `INCREMENT_06_1_3_MANIFEST.md` and the "Increment 6.1.3 update" bullet under Scientific limitations. It stops trying to detect geological assertions in free text at all: every validation scope is now decided by set membership or exact equality, project-specific prose has **zero allowance**, and method wording is admitted only by exact membership of a closed five-entry provenance registry. It also hardens the interval-record type gate and brings three previously unvalidated exported strings into scope. No scientific threshold, tolerance, policy, disposition or result changed; eleven of twelve outputs/figures are byte-identical to Increment 6.1.2 and the twelfth differs by one scalar. Increment 6.1.2 is the preceding corrective patch applied after an independent audit of Increment 6.1.1 — see `INCREMENT_06_1_2_MANIFEST.md` and the "Increment 6.1.2 update" bullet under Scientific limitations. It redesigns the named-lithology validator so the method-phrase allowance is decided per occurrence from local grammar rather than from a sentence-wide cue list and a fixed look-behind window (closing demonstrated false negatives AND demonstrated false positives), and enforces the interval-record interruption-count invariants completely at construction. No scientific threshold, tolerance, policy, disposition or result changed, and all twelve outputs/figures are byte-identical to Increment 6.1.1. Increment 6.1.1 is the preceding corrective patch applied after an independent audit of Increment 6.1 — see `INCREMENT_06_1_1_MANIFEST.md` and the "Increment 6.1.1 update" bullet under Scientific limitations. It closes a named-lithology validator bypass, corrects a stale Vp/Vs description in the active module documentation, replaces ambiguous interruption counts with three separately named quantities, corrects the notebook's output-file count, and supersedes an incorrect Increment 6.1 delta statement. Increment 6.1 is the preceding corrective patch applied after an independent geomechanics/rock-physics and software audit — see `INCREMENT_06_1_MANIFEST.md` and the "Increment 6.1 update" bullet under Scientific limitations. It corrects Vp/Vs domain terminology and boundaries, the Poseidon North 1 depth-tied status, unsupported lithology/correlation claims and a circular named-lithology gate, and gross-versus-net interval-thickness reporting. No architecture changed and no new phase was started. Builds on the LOCKED Increment 1-5.1.2 foundation (no locked module, config, test, fixture, notebook, output, or figure is modified) by adding an auditable per-well frame assembly, factual gamma-ray-family QC, an endpoint-sensitivity framework for a dimensionless screening proxy, and three method-eligibility masks with contiguous-interval registers. See `INCREMENT_06_MANIFEST.md` for the full technical design, the independently recomputed real-data findings, and the complete verification record; see "Increment 6" below for the scientific boundaries it deliberately does not cross. **Increment 7 has not been started.**

**Increment 5 / 5.1 / 5.1.1 / 5.1.2 (previous release, v0.5.2): Formation-Top Ingestion, Source Reconciliation, and Survey-Corrected Stratigraphic Depth Framework.** Builds on the LOCKED Increment 4.1.2 checkshot/time-depth layer, the LOCKED Increment 3.1.1 deviation-survey/depth-mapping layer, and the LOCKED Increment 2.1.1 LAS-ingestion layer (all unmodified) by adding contract-driven formation-top file ingestion for the two approved wells with formation-top data (Poseidon 2, Boreas 1), explicit reconciliation between each well's two independently supplied top-file representations, and mapping of reconciled marker depths through the locked survey trajectory to produce a corrected, auditable stratigraphic marker table. See `INCREMENT_05_MANIFEST.md` for the full technical design, the independently recomputed real-data findings, and the complete verification record; `INCREMENT_05_1_MANIFEST.md` for the Increment 5.1 corrective patch (a zero-common-marker reconciliation defect, readable-file absolute-path leakage, incomplete in-memory numerical validation, and a manifest-language correction — see "Increment 5.1 update" below); and `INCREMENT_05_1_1_MANIFEST.md` for the Increment 5.1.1 corrective patch (a remaining one-empty-source zero-common-marker edge case, plus three documentation-accuracy corrections — see "Increment 5.1.1 update" below). Formation-top ingestion/reconciliation/depth-correction only — no gamma-ray normalization, shale-volume calculation, named lithology classification, petrophysical interpretation, method-eligibility masks, shallow-density modelling, overburden-stress integration, NCT fitting, pore-pressure prediction, elastic properties, rock strength, horizontal stresses, or wellbore-stability analysis is performed in this increment.

New in Increment 6:
- `p2mem/wellframe_models.py` / `p2mem/wellframe.py` — a typed, auditable per-well assembly of the LOCKED canonical LAS curve arrays alongside the LOCKED MD→TVD/TVDSS mapping, carrying per-sample validity masks, per-curve provenance (source curve name, raw mnemonic, raw and canonical unit, conversion function, source filename), QC flags, and an evidence classification. Sample count and original file order are preserved exactly; an invalid sample is expressed only through a mask, never deleted, filled, interpolated, or reordered; every curve array is exposed read-only so a downstream consumer cannot mutate a locked loader's data through a frame. **No depth is ever extrapolated:** the locked `map_las_md_to_tvd_tvdss` is deliberately all-or-nothing, so this layer computes the in-coverage mask from the locked trajectory's own MD range and calls the locked mapper on the in-coverage subset only, leaving out-of-coverage samples as NaN with `depth_valid_mask == False` (`n_extrapolated` is 0 by construction, and the honest coverage gap is reported as `n_depth_unmapped` instead).
- `config/petrophysics_eligibility.yml` — the human-authored, human-reviewable per-well gamma-ray-family disposition, endpoint-sensitivity policy, physical-plausibility bounds, contiguity tolerances, and eligibility rules. The four wells carry four DISTINCT GR-family curves (`GR_api`, `GRD_api`, `ECGR_api`, and a second, independent `GR_api`) that are never merged, renamed, rescaled, or treated as geologically equivalent — Poseidon 2's and Proteus 1ST2's curves canonicalize to the same name but remain different tools in different wells with no cross-well calibration tie, so endpoints are always estimated per well from that well's own samples (a single universal cross-well endpoint pair is rejected at config-load time).
- `p2mem/petrophysics_models.py` / `p2mem/petrophysics.py` — factual GR-family QC statistics for EVERY well including excluded ones (the numbers justifying an exclusion must themselves be published), per-well low/base/high endpoint scenarios whose MEASURED endpoint values are recorded, and the dimensionless GR index `IGR = (GR − GR_low)/(GR_high − GR_low)` with clipped and unclipped results retained side by side so the amount of clipping — i.e. how far real data fell outside the assumed bracket — stays visible. Every endpoint carries `evidence_class = "assumed_configured"` and an explicit uncalibrated status; no code path can promote one to calibrated. The only permitted shale-proxy transform is the linear identity of the clipped index, exported under the self-labelling name `VSH_GR_linear_proxy_frac`.
- `p2mem/method_eligibility.py` — three input-admissibility masks (`eligible_density_for_sv`, `eligible_dynamic_elastic`, `eligible_sonic_nct_candidate`) with per-criterion pass counts and a named limiting criterion, plus contiguous-interval registers reported on MD, TVD and TVDSS. Gap bridging requires BOTH a small sample gap AND a small physical depth span, so sample-count continuity is never confused with physical-depth continuity; bridged samples are always disclosed separately from genuinely eligible ones.
- `p2mem/io/petrophysics_inventory.py` — deterministic summary/scenario/interval/manifest builders that never export a per-sample real-data array (which would effectively reproduce the private source logs) and never emit an absolute path, sanitizing every failure message against BOTH candidate source paths.
- **Key real-data findings (independently measured, not asserted):** Boreas 1's ECGR carries an unresolved scale/acquisition anomaly — median **8.40 API** against 36.06 / 36.37 / 41.41 API for the other three wells (4.3×–4.9× lower), a range of **[−0.0001, 519.18] API** including 4 negative and 42 exactly-zero samples, and **2,059 samples above the well's own declared seabed marker**. With no independent tool header, calibration record, or environmental-correction metadata available to adjudicate the cause, Boreas 1 is formally EXCLUDED (`BOREAS_ECGR_SCALE_UNRESOLVED`) from every lithology-dependent, GR-normalized, shale-proxy and NCT-candidate calculation — **excluded, never corrected**, since any shift, gain, or normalization would fabricate a calibration that does not exist and would silently propagate into every downstream result. Zero endpoint scenarios, zero proxies and zero lithology-dependent masks are computed for it; it remains available for factual raw-GR QC display and availability reporting only. Endpoint choice alone moves the screening-proxy median by up to 0.12 (dimensionless), and endpoint × threshold choice moves the qualifying sonic-NCT-CANDIDATE thickness by a factor of **8.0** in Poseidon 2 (144 m to 1,159 m), 3.5× in Poseidon North 1 and 2.3× in Proteus 1ST2 — quantifying exactly how much a later NCT result would depend on choices no data in this project constrains.
- **No named lithology is assigned anywhere in Increment 6, and the available data do not support assigning one.** Low-GR intervals occur independently in each of the three GR-eligible wells; cross-well stratigraphic persistence is NOT established, and cannot be, because Poseidon North 1 and Proteus 1ST2 have no approved formation tops to correlate within. Gamma-ray response is not uniquely diagnostic of rock type, and no core, cuttings description, image log, spectral GR, or calibrated multi-mineral solution exists in this project to adjudicate it. Every classification this increment produces (`GR_PROXY_HIGH` / `GR_PROXY_INTERMEDIATE` / `GR_PROXY_LOW` / `GR_NOT_AVAILABLE` / `GR_EXCLUDED_UNRESOLVED_SCALE`) describes DATA AND PROXY CONFIDENCE ONLY, and is machine-checked against an enforced prohibited-rock-name vocabulary.
- Still not implemented: named lithology interpretation, environmental GR correction, Boreas ECGR rescaling, neutron-density crossplot interpretation, any nonlinear Vsh transform (Larionov, Clavier, Stieber — all DEFERRED pending a retrieved, verified primary-source method record), shallow-density reconstruction, density extrapolation, vertical-stress integration, hydrostatic-pressure modelling, NCT fitting, Eaton/Bowers pore-pressure prediction, dynamic or static elastic-property calculation, rock-strength or friction-angle correlation, Shmin/SHmax modelling, stress-polygon construction, wellbore-stability analysis, and mud-weight recommendation.


New in Increment 5:
- `p2mem/top_models.py` — typed, frozen dataclasses for every formation-top header/contract/raw-row/reconciliation/corrected-marker result object, mirroring the checkshot/deviation layers' design philosophy. Two source representations exist per well — an "HRS" file (`Top_Name`/`MDRT_m` only, no well name in the file body) and a "selected readable" file (`TOP_NAME`/`MDRT_M`/`TVDSS_M`/`NOTE`, `#`-comment headers, a dashed separator line) — and every raw (`_source_`) value from both is preserved separately; nothing is ever silently overwritten, sorted, deduplicated, or repaired.
- `p2mem/io/tops.py` — an auditable formation-top parser, per-file contract resolver, and HRS-versus-readable source reconciler for the four approved files, keyed by their exact literal source filenames. Markers are matched between the two files by exact name, whitespace-normalized name, or an explicit human-authored alias contract (`config/formation_top_contracts.yml`'s `marker_name_aliases` — currently empty; no fuzzy/similarity matching of any kind is ever performed). MDRT is cross-checked between the two sources within a documented 0.005 m tolerance; a marker on which the two sources disagree beyond that tolerance is registered (`mdrt_status = "MISMATCH"`) and excluded from depth mapping (`mdrt_authority_basis = "disagreement_unresolved"`, `mapping_status = "not_mapped_mdrt_unresolved"`) rather than resolved by silently picking one file's value. Reconciled MDRT is mapped through the LOCKED Increment 3.1.1 `petrel_source_trace` survey trajectory using the existing, unmodified `p2mem.depth_mapping.map_las_md_to_tvd_tvdss` — reused unmodified, never reimplemented — called once per marker so that one out-of-coverage marker never blocks mapping of the rest of the well, and extrapolation is never performed (a marker outside survey MD coverage is reported as `mapping_status = "rejected_outside_coverage"`, never extrapolated).
- `p2mem/io/tops_inventory.py` — deterministic, metadata-only inventory/QC-table builders for the Increment 5 outputs (file inventory, marker/source register, HRS-versus-readable reconciliation table, survey-corrected marker table, ingestion issues, formation-top availability, JSON manifest) — never raw per-marker arrays beyond a single scalar per field, and never a full environment-dependent build path, only a basename.
- `config/formation_top_contracts.yml` — the human-authored, human-reviewable per-file formation-top contract for each of the four approved files, including each file's `well_identity_evidence_status` (both representations for both wells are `inferred_unverified` — the HRS files carry no well name in their body at all, association resting on the filename alone, and the readable files' well name appears only in a project-supplied `#` comment, not independently verified content; this is a deliberately MORE conservative classification than Increment 4's checkshot files, explicitly justified in `INCREMENT_05_MANIFEST.md`).
- **Key real-data findings (disclosed):** every one of Poseidon 2's 9 "selected readable" TVDSS values equals `MDRT_M - 21.8 m` EXACTLY (21.8 m is the well's own rotary-table elevation) — a literal vertical-well-assumption depth-reference defect, since a real TVDSS should differ from MD by more than a constant datum shift once a well deviates. Survey-corrected residuals (`TVDSS_source_m - TVDSS_survey_corrected_m`) reproduce Sea Bed at ≈0 m, Plover Fm (Top Reservoir) at ≈+1.68 m, and TD at ≈+2.49 m — confirming the anticipated defect and correcting it in the derived `TVDSS_survey_corrected_m` representation while `TVDSS_source_m` itself is preserved unmodified. Boreas 1's supplied TVDSS does NOT follow the `MDRT - 21.8` pattern and its maximum absolute source-versus-survey residual across all 10 markers is 0.0416 m, below the approved 0.05 m tolerance — confirming it was generated from the well's real surveyed trajectory, and it is therefore NOT "corrected" the way Poseidon 2 is; evidence is applied per file, per well, never by analogy. Poseidon North 1 and Proteus 1ST2 have no approved formation-top file and are recorded as `formation_top_availability: NOT_AVAILABLE`, a factual data gap, never substituted with another well's tops or correlated by depth alone.
- Still not implemented: gamma-ray normalization, shale-volume calculation, named lithology classification, petrophysical interpretation, method-eligibility masks, shallow-density modelling, overburden-stress integration, NCT fitting, pore-pressure prediction, elastic-property calculation, rock-strength estimation, horizontal stresses, or wellbore-stability calculations. Those remain explicitly out of scope for this increment.

New in the Increment 5.1 corrective patch (see "Increment 5.1 update" under Scientific limitations for the full defect list): `reconcile_formation_top_sources` now correctly rejects two disjoint, non-empty marker sets as a fatal `NO_COMMON_MARKERS` ERROR (previously silently accepted); `TopIngestionFailure` now carries an explicit `failure_origin` and both `hrs_path`/`readable_path`, so a failure originating from the "selected readable" file can no longer leak that file's own absolute path into any exported CSV/JSON field; and `reconcile_formation_top_sources` now fully validates its numeric-array and `mdrt_agreement_tolerance_m` inputs before any comparison, with deliberate typed exceptions rather than an incidental `IndexError`/`TypeError`. None of the real Poseidon 2 / Boreas 1 formation-top results above changed.

New in the Increment 5.1.1 corrective patch (see "Increment 5.1.1 update" under Scientific limitations for the full defect list): `reconcile_formation_top_sources` now rejects the one remaining zero-common-markers configuration Increment 5.1 missed — exactly one source entirely empty, the other non-empty — via a single, strictly correct intersection check (`if not common_markers`) that covers every zero-common-markers case at once, while a legitimate one-sided marker alongside at least one genuinely shared marker remains the pre-existing, non-fatal `NOT_COMPARABLE` case. `INCREMENT_05_1_MANIFEST.md`'s baseline-hash, packaged-outputs, and packaged-LAS-fixture wording is also corrected (documentation-only; see "Increment 5.1.1 update"). None of the real Poseidon 2 / Boreas 1 formation-top results above changed.

**Increment 4 / 4.1 / 4.1.1 / 4.1.2 (v0.4.1.1, LOCKED as of Increment 5): Checkshot (Velocity Survey) Ingestion, Duplicate-Tie Conditioning, and Time–Depth Framework.** Builds on the LOCKED Increment 3.1.1 deviation-survey/depth-mapping layer and the LOCKED Increment 2.1.1 LAS-ingestion layer (both unmodified - see below) by adding auditable checkshot parsing with explicit per-file contracts, raw-vs-conditioned duplicate-tie handling, average/interval velocity diagnostics, checkshot-vs-locked-survey depth-reference comparison, a coverage-masked forward/inverse piecewise-linear time-depth interpolation layer, LAS MD-to-checkshot-time mapping within validated coverage only, and a Poseidon-2-only sonic-checkshot drift diagnostic. See `INCREMENT_04_MANIFEST.md` for the full technical design and the independently recomputed real-data statistics, `INCREMENT_04_1_MANIFEST.md` for the Increment 4.1 corrective patch (order-invariant axis-tie conditioning for TVDSS↔OWT/TWT inversion, replacing an order-dependent defect; numerical-validation hardening — see "Increment 4.1 update" below), `INCREMENT_04_1_1_MANIFEST.md` for the Increment 4.1.1 numerical-validation corrective patch (a hidden-reversal grouping defect, incomplete full-MD validation, an untyped zero-coverage crash, missing batch isolation for numerical failures, and a unit-helper input-safety gap — see "Increment 4.1.1 update" below), and `INCREMENT_04_1_2_MANIFEST.md` for the Increment 4.1.2 packaging-only corrective patch (Colab line-ending reproducibility and a truthful notebook completion gate; no scientific, numerical, or version change). Checkshot data QC, duplicate-tie conditioning, and time-depth interpolation only — no formation-top correction, lithology interpretation, density modelling, pore-pressure prediction, elastic properties, rock strength, stress modelling, or wellbore-stability analysis is performed in this increment.

New in Increment 4:
- `p2mem/checkshot_models.py` — typed, frozen dataclasses for every checkshot header/contract/raw-row/duplicate-tie/velocity-diagnostic/depth-comparison/time-mapping/sonic-drift result object, mirroring the deviation-survey layer's design philosophy. Raw (`_source_`) values are always kept explicitly separate from conditioned (`_conditioned_`) values — never overwritten, never mixed.
- `p2mem/io/checkshot.py` — an auditable checkshot (velocity-survey) parser and per-file contract resolver for the three approved checkshot files (`Poseidon2-Checkshot.txt`, `Boreas1-Checkshot.txt`, `Proteus1-Checkshot.txt`), keyed by their exact literal source filenames. File-identity checks (filename, SHA-256, header/survey-statement text, column order, row width, numeric structure) are enforced as blocking `ERROR`s before any time-depth computation is attempted. Raises disclosure `WARNING`s including `DEPTH_BASIS_NOT_EXPLICITLY_DECLARED` (the source file's first column is labelled only `Depth`, never assumed to be MD without evidence) and, for Proteus 1ST2, `WELL_IDENTITY_INFERRED_UNVERIFIED` (the file carries no embedded well identifier tying it to Proteus 1ST2).
- `p2mem/time_depth.py` — the numerical time-depth layer: duplicate/repeated-tie detection and deterministic, disclosed median-based conditioning (raw rows always preserved and separately registered; ties never silently averaged, never force-monotonized with artificial epsilon increments); average velocity (`Vavg = TVDSS / OWT`) and interval velocity (`Vint = ΔTVDSS / ΔOWT`, NaN — never infinite or negative — for any invalid, zero, or non-increasing interval); checkshot-vs-locked-survey depth-reference comparison (residual = survey-interpolated TVDSS − checkshot-supplied TVDSS); forward/inverse piecewise-linear time-depth interpolation with explicit coverage masking (points outside validated checkshot coverage are reported as not-mapped, never extrapolated) via an order-invariant, median-based axis-tie-conditioned lookup table for TVDSS↔OWT/TWT inversion (Increment 4.1 — every tied value on the axis being inverted is grouped and registered, never resolved by an order-dependent "first wins" tie-break); LAS MD-to-checkshot-time mapping for Poseidon 2 within validated coverage only; and a Poseidon-2-only sonic-checkshot drift diagnostic (a local, version-independent trapezoidal integration of sonic slowness over the longest valid continuous MD interval, compared against the checkshot-interpolated OWT increment over the same interval — this diagnostic never modifies `VP_m_s`, `DTCO`, checkshot OWT, or the time-depth curve itself).
- `p2mem/io/checkshot_inventory.py` — deterministic, metadata-only inventory/QC-table builders for the Increment 4 / 4.1 outputs (file inventory, ingestion issues, Depth-axis duplicate-tie register, the Increment 4.1 TVDSS/OWT-axis tie register, depth-tie QC, velocity summary, sonic-checkshot drift summary, time-depth mapping summary, JSON manifest) — never raw per-sample checkshot or LAS arrays, and never a full environment-dependent build path, only a basename.
- `config/checkshot_contracts.yml` — the human-authored, human-reviewable per-file checkshot contract for each of the three approved files, including each well's `model_use_status` (`primary_model` for Poseidon 2 only; `qc_only` for Boreas 1 and Proteus 1ST2) and `identity_evidence_status` (`verified` for Poseidon 2 and Boreas 1; `inferred_unverified` for Proteus 1ST2).
- **Key real-data findings (disclosed):** Poseidon 2's raw checkshot file contains 5 repeated Depth ties, 6 non-increasing TVDSS steps, and 4 non-increasing OWT steps, all independently detected and conditioned (never silently smoothed); its checkshot-vs-survey TVDSS comparison shows a maximum absolute residual of ≈0.089 m with no systematic offset pattern. Boreas 1 shows 3 repeated Depth ties and 4 non-increasing TVDSS steps (OWT strictly increasing throughout) and a checkshot-vs-survey comparison with a near-constant offset of ≈−0.69 m, reported as an observed datum-like offset pattern — not a proven datum error. Proteus 1ST2's file is strictly increasing in all three columns with no repeated ties, and shows a near-constant offset of ≈+0.30 m against the locked survey trajectory; its association with Proteus 1ST2 remains `inferred_unverified` throughout every output. Poseidon 2's sonic-checkshot drift over its longest valid continuous sonic interval (≈MD 2449–4064 m) is ≈+16.7 ms (≈+4.4% of the checkshot-interpolated one-way transit time (OWT) increment over that interval — NOT two-way time; the diagnostic compares the integrated sonic transit time directly against the checkshot's OWT increment over the identical interval), reported as a diagnostic only. Poseidon North 1 has no approved checkshot file; this is recorded as `checkshot_availability: NOT_AVAILABLE`, a factual data gap, not an ingestion failure. Depth-tie conditioning alone does NOT guarantee TVDSS or OWT is itself strictly increasing (required for TVDSS↔OWT/TWT inversion) — see the Increment 4.1 update below for the corrected, order-invariant handling of this. See `INCREMENT_04_MANIFEST.md` for the full statistics, tables, and figures.
- Still not implemented: formation-top correction, lithology interpretation, density modelling, sonic/checkshot drift *correction*, synthetic extension of the time-depth relationship beyond measured checkshot coverage, pore-pressure prediction, elastic-property calculation, rock-strength estimation, overburden/horizontal stresses, or wellbore-stability calculations. Those remain explicitly out of scope for this increment.

**Increment 3 / 3.1 / 3.1.1: Deviation-Survey Ingestion, Minimum-Curvature Validation, and MD–TVD–TVDSS Depth Framework.** Builds on the LOCKED Increment 2.1.1 LAS-ingestion layer (unmodified - see below) by adding Petrel deviation-survey parsing, an explicit per-file survey contract, a standard minimum-curvature trajectory engine, an explicit depth-reference (MD/TVD/TVDSS) framework, and MD-to-TVD/TVDSS mapping of the existing LAS `MD_m` arrays. See `INCREMENT_03_MANIFEST.md` for the Increment 3 technical design and real four-well integration results, `INCREMENT_03_1_MANIFEST.md` for the Increment 3.1 corrective patch (four audit findings: source-filenames-with-spaces, absolute-path leakage, DLS-normalization disclosure, dogleg numerical stability — see "Increment 3.1 update" below), and `INCREMENT_03_1_1_MANIFEST.md` for the Increment 3.1.1 packaging-only corrective patch (notebook/source `%%writefile` synchronization; no scientific or numerical change). The Proteus 1ST2 trajectory-discrepancy finding is disclosed, not resolved, in any of these releases. This layer, and the Increment 2.1.1 LAS-ingestion layer beneath it, are LOCKED as of Increment 4 and reused unmodified.

Locked from Increment 2.1.1 (unmodified in Increment 3 unless a blocking defect is documented - none was found):
- `p2mem/units.py` — an explicit, NumPy-based unit-conversion layer (no external unit-registry dependency such as Pint) implementing 21 public conversion functions between oilfield and SI-internal units. Unchanged since Increment 1.1. See the module docstring and `tests/test_units.py`.
- `p2mem/models.py`, `p2mem/io/las.py`, `p2mem/io/inventory.py`, `config/las_curve_contracts.yml` — the auditable LAS 2.0 parser, per-file curve-contract resolver, and inventory builders for the four approved wells, corrected and independently re-verified through Increment 2.1.1 (158 tests passing, 4/4 real wells loading with zero ingestion errors). See `INCREMENT_02_v2.1.1_MANIFEST.md`.

New in Increment 3:
- `p2mem/deviation_models.py` — typed, frozen dataclasses for every deviation-survey/trajectory/depth-mapping result object (header info, per-file contract, raw station data, minimum-curvature result, trajectory-validation residuals, depth-basis selection, typed batch failures, LAS depth-mapping result), mirroring the LAS layer's design philosophy. Every source (`_source_`) array is kept explicitly separate from every independently computed (`_mc_`) array — never overwritten, never mixed.
- `p2mem/io/deviation.py` — an auditable Petrel deviation-survey (well-trace) parser and per-file contract resolver for the same four wells, keyed by their exact, literal source filenames (which contain spaces, e.g. `"Poseidon 2_dev.txt"` — corrected in Increment 3.1; see below). Extracts and preserves the full header block (well/survey identity, wellhead X/Y, datum and its MSL reference, coordinate-reference-system text, declared angle/depth/coordinate conventions) and the exact 11-column station table, with file-identity checks (filename, SHA-256, well/survey identifier, wellhead/datum values, coordinate system, column order, station count, MD coverage) all enforced as blocking `ERROR`s before any trajectory computation is attempted. Also raises two disclosure `WARNING`s for every successfully loaded file: the pre-existing `MD_UNIT_NOT_EXPLICITLY_DECLARED`, and the Increment 3.1 `DLS_NORMALIZATION_INFERRED_AS_DEG_PER_30M` (the supplied `DLS` column's degrees-per-30-m normalization is inferred, not header-declared, and is independently verified per file against a recomputation from that file's own inclination/azimuth).
- `p2mem/trajectory.py` — the standard minimum-curvature method (a numerically stable `arctan2(||cross||, dot)` dogleg-angle formulation — Increment 3.1 correction, see below — a ratio factor with an explicit Taylor-series limit as the dogleg approaches zero, TVD/northing/easting displacement, dogleg severity in degrees per 30 m), implemented explicitly and transparently with no third-party survey-computation library.
- `p2mem/depth_mapping.py` — MD-to-TVD/TVDSS interpolation of the locked LAS `MD_m` array against the explicitly selected depth-trajectory basis, using a documented, deterministic piecewise-linear station interpolation (never a per-sample minimum-curvature recomputation) that never extrapolates silently.
- `p2mem/io/deviation_inventory.py` — deterministic, metadata-only inventory-table builders for the Increment 3 outputs (file inventory, trajectory-validation summary, depth-reference register, LAS depth-mapping summary, ingestion issues, JSON manifest) — never raw per-sample station or LAS arrays, and (Increment 3.1 correction) never a full environment-dependent build path, only a basename.
- `config/deviation_survey_contracts.yml` — the human-authored, human-reviewable per-file deviation-survey contract for each of the four wells, keyed by the exact literal source filename (`"Poseidon 2_dev.txt"`, `"Boreas 1_dev.txt"`, `"Poseidon North 1_dev.txt"`, `"Proteus 1ST2_dev.txt"` — corrected in Increment 3.1), including an explicit, uniformly applied `depth_basis_policy` (`petrel_source_trace`, the conservative default given the Proteus 1ST2 finding below) and residual-comparison tolerances declared once and applied identically to every well (never tuned per well to force a pass/fail outcome).
- **Key real-data finding (disclosed, not resolved):** independent minimum-curvature reconstruction of Poseidon 2, Boreas 1, and Poseidon North 1 agrees with their Petrel-supplied TVD to approximately millimetre scale. Proteus 1ST2 shows a materially larger discrepancy (~0.19 m TVD, ~1.6 m easting at maximum) concentrated in its deeper section (below ~MD 4200 m), even though its own supplied dogleg-severity column is internally consistent with an independent recomputation from its own inclination/azimuth at every station. This is reported as a visible trajectory-validation `WARNING`, not corrected, hidden, or used to justify loosening every well's tolerance — see `INCREMENT_03_MANIFEST.md` Section 6 for the full investigation and the evidence pattern observed. Unaffected by the Increment 3.1 patch.
- Still not implemented (as of Increment 3/3.1/3.1.1): checkshot ingestion, time-depth conversion, formation-top correction, petrophysical interpretation, gamma-ray normalization, shale-volume calculation, lithology classification, normal-compaction-trend fitting, pore-pressure prediction, elastic-property calculation, rock-strength estimation, overburden/horizontal stresses, or wellbore-stability calculations. Those were explicitly out of scope for this increment. Checkshot ingestion and the time-depth framework were subsequently added in Increment 4 (see above); the remainder are added one gated increment at a time in later releases.

## Installation

Requires Python 3.9 or later.

```bash
# from the project root (the directory containing pyproject.toml)
pip install -e .
```

This installs the `p2mem` package in editable mode along with its runtime dependencies: NumPy (`numpy>=1.24`) and, as of Increment 2, PyYAML (`pyyaml>=6.0`) — used for parsing the human-authored curve contracts in `config/las_curve_contracts.yml`, `config/deviation_survey_contracts.yml` (Increment 3), `config/checkshot_contracts.yml` (Increment 4), and (new in Increment 5) `config/formation_top_contracts.yml`. No new runtime dependency was added in Increment 3, 4, or 5: the minimum-curvature engine, depth-mapping interpolation, duplicate-tie conditioning, velocity diagnostics, time-depth interpolation, and formation-top reconciliation/mapping all use only NumPy (including a local, version-independent trapezoidal-integration helper in `p2mem/time_depth.py`, added because `numpy.trapz`/`numpy.trapezoid` are not consistently available across supported NumPy versions). Matplotlib and pandas are used only for notebook display and QC-figure generation (`run_integration_03.py`/`run_integration_04.py`/`run_integration_05.py`, the Increment 3/4/5 notebooks) — never imported by the installable `p2mem` package itself. To also install the test dependency:

```bash
pip install -e ".[dev]"
```

## Running the tests

```bash
pytest -v
```

The suite in `tests/test_units.py` validates `p2mem/units.py` (unchanged since Increment 1.1) against analytical reference values, round-trip consistency, scalar/array inputs, NaN preservation, and rejection of invalid/nonphysical/ambiguous inputs. The suite in `tests/test_las.py` validates `p2mem/io/las.py` (locked since Increment 2.1.1) against small synthetic LAS fixtures. The suites in `tests/test_trajectory.py`, `tests/test_deviation.py`, and `tests/test_depth_mapping.py` (new in Increment 3; extended in Increment 3.1 with dogleg numerical-stability, DLS-normalization-disclosure, and real-filename-with-spaces regression/negative tests) validate the minimum-curvature engine, the Petrel deviation-survey parser/contract resolver, and the MD-to-TVD/TVDSS mapping respectively, against small synthetic fixtures under `tests/fixtures/` and in-memory synthetic data (this layer is locked, unmodified, as of Increment 4). `tests/test_deviation_inventory.py` (new in Increment 3.1) validates that no exported inventory/issues/manifest row, for a successful or a failed well, ever embeds a full environment-dependent build path. The suites in `tests/test_checkshot.py`, `tests/test_time_depth.py`, and `tests/test_checkshot_inventory.py` (new in Increment 4) validate the checkshot parser/contract resolver, the duplicate-tie conditioning/velocity-diagnostic/time-depth-interpolation/sonic-drift numerical layer, and the deterministic inventory/QC-table builders respectively, against small synthetic fixtures under `tests/fixtures/` (including a CRLF fixture used to verify line-ending detection) and in-memory synthetic/analytic data — including an analytic constant-velocity case used to independently verify the trapezoidal-integration helper. The suites in `tests/test_tops.py` and `tests/test_tops_inventory.py` (new in Increment 5) validate the formation-top parser/contract resolver/HRS-versus-readable reconciliation logic and the deterministic inventory/QC-table builders respectively, against small synthetic fixtures under `tests/fixtures/` and in-memory synthetic data, including synthetic analogs of both real regression findings (a vertical-assumption depth-reference defect with a growing residual, and a survey-consistent well with a near-zero residual). None of these suites require the private/raw project LAS, deviation, checkshot, or formation-top files, so the full suite runs the same way for anyone who clones this repository. Run the command above and read the reported pass/fail count directly — this document does not assert a fixed expected count, since that must always be read from the actual `pytest` output for the code currently on disk. Real integration validation (which DOES require the raw LAS/deviation/checkshot/formation-top files, not included in this repository) is a separate notebook run — see `02_LAS_Ingestion_and_Curve_Contracts.ipynb`, `03_Deviation_Survey_and_Depth_Framework.ipynb`, `04_Checkshot_QC_and_Time_Depth_Framework.ipynb`, and `05_Formation_Tops_and_Stratigraphic_Depth_Framework.ipynb`.

## Directory structure

```
Poseidon_1D_MEM/
├── README.md
├── pyproject.toml
├── p2mem/
│   ├── __init__.py
│   ├── models.py
│   ├── units.py
│   ├── deviation_models.py
│   ├── trajectory.py
│   ├── depth_mapping.py
│   ├── checkshot_models.py
│   ├── time_depth.py
│   ├── top_models.py
│   ├── wellframe_models.py
│   ├── wellframe.py
│   ├── petrophysics_models.py
│   ├── petrophysics.py
│   ├── method_eligibility.py
│   ├── overburden_models.py        (Increment 7)
│   ├── density_qc.py               (Increment 7)
│   ├── overburden.py               (Increment 7)
│   └── io/
│       ├── __init__.py
│       ├── las.py
│       ├── inventory.py
│       ├── deviation.py
│       ├── deviation_inventory.py
│       ├── checkshot.py
│       ├── checkshot_inventory.py
│       ├── tops.py
│       ├── tops_inventory.py
│       ├── output_policy.py
│       ├── petrophysics_inventory.py
│       ├── overburden_policy.py    (Increment 7)
│       ├── overburden_registry.py  (Increment 7)
│       ├── overburden_inventory.py (Increment 7)
│       └── overburden_workflow.py  (Increment 7)
├── tests/
│   ├── test_units.py
│   ├── test_las.py
│   ├── test_trajectory.py
│   ├── test_deviation.py
│   ├── test_depth_mapping.py
│   ├── test_deviation_inventory.py
│   ├── test_checkshot.py
│   ├── test_time_depth.py
│   ├── test_checkshot_inventory.py
│   ├── test_tops.py
│   ├── test_tops_inventory.py
│   ├── test_wellframe.py
│   ├── test_petrophysics.py
│   ├── test_method_eligibility.py
│   ├── synthetic_inc6.py
│   ├── synthetic_inc7.py           (Increment 7; fictional in-memory frames only)
│   ├── helpers_inc7.py             (Increment 7)
│   ├── test_density_qc.py          (Increment 7)
│   ├── test_overburden.py          (Increment 7)
│   ├── test_overburden_policy.py   (Increment 7)
│   ├── test_overburden_inventory.py (Increment 7)
│   └── fixtures/         (small synthetic LAS + deviation-survey + checkshot + formation-top files; no project raw data)
├── config/
│   ├── las_curve_contracts.yml
│   ├── deviation_survey_contracts.yml
│   ├── checkshot_contracts.yml
│   ├── formation_top_contracts.yml
│   ├── petrophysics_eligibility.yml
│   └── overburden_stress.yml       (Increment 7)
├── data/
│   └── raw/
│       ├── logs/         (the four raw LAS files - NOT included in this repository; immutable inputs)
│       ├── deviation/    (the four raw deviation-survey files, exact filenames contain spaces, e.g. "Poseidon 2_dev.txt" - NOT included in this repository; immutable inputs)
│       ├── checkshot/    (the three raw checkshot files, exact filenames e.g. "Poseidon2-Checkshot.txt" - NOT included in this repository; immutable inputs, never rewritten/renamed/"cleaned")
│       └── tops/         (the four raw formation-top files, e.g. "Poseidon_2_HRS_tops_no_wellname_MDRT.txt" - NOT included in this repository; immutable inputs)
├── notebooks/  (reserved for later increments)
└── outputs/
    ├── 02_las_inventory/            (Increment 2.1.1 real four-well run: CSV/JSON metadata only, no raw log samples)
    ├── 03_deviation_depth/          (Increment 3 real four-well run: CSV/JSON metadata + QC figures, no raw station/log samples)
    ├── 04_checkshot_time_depth/     (Increment 4 real three-file checkshot run: CSV/JSON metadata + QC figures, no raw per-sample checkshot/LAS arrays)
    ├── 05_formation_tops/           (Increment 5 real four-file formation-top run: CSV/JSON metadata + QC figures, no raw per-marker arrays beyond scalar fields)
    ├── 06_petrophysics_eligibility/ (Increment 6 real four-well run: CSV/JSON metadata + QC figures, no raw per-sample arrays)
    └── 07_density_overburden/       (Increment 7 real four-well run: 9 closed-schema CSV/JSON artifacts + 4 QC figures; the stress profile is a DECIMATED selection of existing samples, never a raw log dump)
```

`notebooks/` is created empty by the project-setup notebook cell and is not yet populated in-repo (the increment notebooks themselves are delivered as top-level files, e.g. `02_LAS_Ingestion_and_Curve_Contracts.ipynb`, `03_Deviation_Survey_and_Depth_Framework.ipynb`, `04_Checkshot_QC_and_Time_Depth_Framework.ipynb`, `05_Formation_Tops_and_Stratigraphic_Depth_Framework.ipynb`, `06_GR_QC_Shale_Proxy_and_Method_Eligibility.ipynb`, `07_Density_QC_and_Overburden_Stress_Framework.ipynb`, and are meant to be run from Google Drive per their own directory-setup cells; the Increment 7 notebook additionally runs unchanged from any working directory that is itself the project root).

## Scientific limitations

These limitations are specific to the Poseidon 2 dataset and this project's current increment, and are carried forward here so they are visible outside the conversation in which they were identified:

- **RHOB (bulk density) coverage in Poseidon 2 ends at approximately 5,296.85 m MD.** Sonic and other curves continue deeper, so Vp/Vs and dynamic Poisson's ratio remain computable below that depth, but density-dependent properties (Young's modulus, shear modulus, bulk modulus, acoustic impedance, shear impedance) are unavailable below it unless density is explicitly estimated and flagged as such — never silently substituted.
- **No reliable shale-based normal compaction trend (NCT) exists from Poseidon 2 alone.** A provisional, transferred candidate NCT identified in offset well Poseidon North 1 is a *candidate*, not a validated trend, and must not be presented as calibrated.
- **Independently measured Vp/Vs quality flags:** approximately 3.38% of Poseidon 2 Vp/Vs values fall below 1.5, and approximately 0.53% fall below the physical validity cutoff of √2 (≈1.4142) required for a non-negative dynamic Poisson's ratio.
- **No independent calibration data (RFT/MDT pressure points, LOT/XLOT tests, or core data) has been supplied or incorporated.** Any pore-pressure or stress output in later increments must be presented as a bounded or theoretical estimate, not a validated field prediction.
- **Empirical/correlation equations are not implemented until their governing equation, units, applicability range, and calibration status are recorded in the project's method-and-citation register.** Several candidate methods remain in "pending" status and are intentionally absent from the codebase for that reason, not because they were overlooked.
- Additional open items (offset-well GR/ECGR scale adjudication, missing formation tops for one offset well) are tracked in the project's design-review documentation and gate specific later phases (lithology and pore-pressure), not this increment.
- **Increment 2 update:** LAS ingestion independently reconfirms (does not newly discover, and does not act on) two previously-flagged anomalies from the Rev 1 design review: Boreas 1's ECGR curve (canonical name `ECGR_api`) ranges from approximately −0.0001 to 519.18 API (vs. roughly 5–205 API for the other three wells' GR-family curves) with 96.98% valid coverage; and Proteus 1ST2's LAS log file places its neutron-porosity curve (canonical name `NPHI_pct`) at column position 5 rather than the last position (8) used by the other three wells. Both are reported as ingestion facts (see `outputs/02_las_inventory/`); neither is rescaled, reinterpreted, or otherwise acted on by this increment.
- **Increment 2.1 / 2.1.1 update:** corrective patches addressing independent audits' naming, reporting, contract-validation, and notebook/source-synchronization findings — see `INCREMENT_02_v2.1_MANIFEST.md` and `INCREMENT_02_v2.1.1_MANIFEST.md`. No new scientific finding was made in either patch; the two anomalies above are unaffected and remain open items for a later, explicitly-scoped increment.
- **Increment 3 update:** deviation-survey ingestion independently reconfirms the Petrel-supplied trajectory for Poseidon 2, Boreas 1, and Poseidon North 1 to approximately millimetre scale via minimum curvature, and additionally DISCOVERS (not merely reconfirms) a real trajectory-reconstruction discrepancy in Proteus 1ST2's deeper section (~0.19 m TVD, ~1.6 m easting at maximum, concentrated below ~MD 4200 m) — see "Current implementation status" above and `INCREMENT_03_MANIFEST.md` Section 6 for the full investigation. This is disclosed as an open item, not corrected or hidden; downstream MD-to-TVD/TVDSS mapping for Proteus 1ST2 conservatively uses the Petrel-supplied source trajectory (not the disagreeing minimum-curvature trajectory) as a result.
- **Increment 3.1 update:** a corrective patch addressing an independent audit's findings on source-filename handling, output environment-independence, an undisclosed normalization inference, and dogleg-angle numerical conditioning — see `INCREMENT_03_1_MANIFEST.md` for the full audit and re-verification record. No new scientific finding was made in this patch; the Proteus 1ST2 discrepancy above is unaffected, remains disclosed exactly as before, and was neither corrected nor concealed. The only numerical changes are at the floating-point noise floor of the diagnostic `dogleg_deg`/`dls_deg_per_30m`/TVD-and-offset-residual fields (at most ~9×10⁻¹³ m for TVD, ~7×10⁻¹⁵ m for easting/northing, across all four real wells), with zero change to any well's PASS/WARNING status.
- **Increment 3.1.1 update:** a packaging-only corrective patch (three notebook `%%writefile` cells that had drifted from their packaged source files, mirroring the earlier Increment 2.1.1 finding). No scientific, numerical, or real-data change of any kind — see `INCREMENT_03_1_1_MANIFEST.md`.
- **Increment 4 update:** checkshot ingestion and the time-depth framework independently reproduce every raw-data anomaly the project design anticipated (Poseidon 2: 5 repeated Depth ties, 6 non-increasing TVDSS steps, 4 non-increasing OWT steps, and a ≈257 m gap in Depth coverage between 1313.1 m and 1570.1 m; Boreas 1: 3 repeated Depth ties and 4 non-increasing TVDSS steps with OWT strictly increasing; Proteus 1ST2: strictly increasing in all three raw columns) and DISCLOSES (not resolves) two further items: (1) Boreas 1's and Proteus 1ST2's checkshot-vs-locked-survey TVDSS comparisons each show a near-constant offset (≈−0.69 m and ≈+0.30 m respectively) — reported as an observed datum-like offset pattern, not a proven datum error, with both source references preserved unmodified; (2) `Proteus1-Checkshot.txt` carries no embedded well identifier, so its association with Proteus 1ST2 is recorded as `identity_status: inferred_unverified` and used for QC only, never as a substitute time-depth model for any other well. Poseidon 2's sonic-checkshot drift over its longest valid continuous sonic interval is a diagnostic finding only (≈+16.7 ms, ≈+4.4%) and does not trigger any correction of `VP_m_s`, `DTCO`, checkshot OWT, or the time-depth curve. Poseidon North 1 has no approved checkshot file and is recorded as a factual data gap (`checkshot_availability: NOT_AVAILABLE`), not substituted with another well's data. See `INCREMENT_04_MANIFEST.md` for the full statistics, tables, and figures. **NOTE:** `INCREMENT_04_MANIFEST.md` incorrectly stated TVDSS is strictly increasing after Depth-tie conditioning for all three wells; this was corrected by Increment 4.1 (see below) — do not rely on that original statement.
- **Increment 4.1 update:** a narrowly scoped corrective patch to Increment 4, applied after an independent numerical-method audit, that does NOT begin Increment 5 or any later-phase work. It corrected an order-dependent tie-break: `tvdss_to_owt`/`owt_to_tvdss` (and their TWT equivalents) previously resolved a repeated value on the axis being inverted by silently keeping whichever tied row was encountered first in the Depth-conditioned table and discarding the other, with no audit trail beyond a bare count. This is replaced by an explicit, order-invariant policy (`p2mem.time_depth.build_axis_conditioned_lookup_table`/`build_axis_conditioned_tables_for_well`): every tied value is grouped by exact equality regardless of parse order, every tied row is registered in a new audit register (`checkshot_time_axis_tie_register.csv` — separate from, and never confused with, the pre-existing Depth-axis `checkshot_duplicate_tie_register.csv`), and the group's dependent-value MEDIAN becomes the conditioned representative (order-invariant; a screening-level choice, not proof the original TVDSS↔OWT relationship was single-valued at that tied value). A genuine reversal (not a tie) raises a typed error rather than being sorted or forced monotonic. Independently reproduced real-data counts: Poseidon 2 has two TVDSS-axis and two OWT-axis tie groups after Depth-tie conditioning; Boreas 1 has one TVDSS-axis tie group and zero OWT-axis tie groups; Proteus 1ST2 has none of either — correcting `INCREMENT_04_MANIFEST.md`'s original, incorrect "strictly increasing for all three wells" statement. This patch also hardens `trapezoidal_integrate` and `compute_sonic_checkshot_drift` to validate their numerical preconditions (finite, one-dimensional, equal-length, strictly increasing MD/x where required) — most notably, a decreasing or duplicate MD run can no longer silently produce a physically invalid negative transit time; it now raises a typed error instead. None of this hardening changes the already-verified real Poseidon 2 sonic-drift result, which is bit-for-bit unchanged. See `INCREMENT_04_1_MANIFEST.md` for the full audit, corrected statistics, and re-verification record.

- **Increment 4.1.1 update:** a narrowly scoped numerical-validation corrective patch to Increment 4.1, applied after an independent numerical-method/software-QA audit, that does NOT begin Increment 5 or any later-phase work, and does NOT alter any previously verified real Poseidon 2/Boreas 1/Proteus 1ST2 result. It corrected four blocking defects and one input-safety gap. (1) `build_axis_conditioned_lookup_table` grouped ALL occurrences of an identical axis value together GLOBALLY before checking for a reversal, so a reversal that returned to an already-seen value (e.g. `[100.0, 200.0, 100.0]`) was silently hidden rather than raising a typed error; it now evaluates the ORIGINAL, ungrouped sequence's successive differences for negativity BEFORE any grouping is attempted — provably equivalent to the 4.1 behavior for every legitimate adjacent tie, and strictly stronger against a non-adjacent reversal. (2) `compute_sonic_checkshot_drift`/`find_longest_finite_positive_run` validated MD monotonicity only within the selected finite-positive-VP run, so a decreasing or duplicate MD value outside that run (e.g. at a NaN-VP station) could pass silently; the COMPLETE canonical `md_m` array is now required finite and strictly increasing before run-selection (the real Poseidon 2 MD array, 31,897 samples, was independently re-verified to already satisfy this — the real sonic-drift result is bit-for-bit unchanged). (3) `compare_checkshot_to_survey` reached an untyped NumPy `ValueError` ("zero-size array to reduction operation") if every checkshot Depth row fell outside the locked survey's own MD coverage; it now raises a typed error naming the well, the checkshot Depth range, and the survey MD coverage. (4) `p2mem.io.checkshot.load_checkshot_surveys` did not catch the typed numerical-conditioning error, so a defect in one well's data could stop the entire batch; it is now caught per well (never via a blanket exception handler) and recorded as a typed, isolated per-well failure, exactly like every other expected failure mode. (5) `seconds_to_milliseconds`/`milliseconds_to_seconds` coerced their input directly, unlike `p2mem.units`'s Increment-1 input-safety policy, so a boolean, numeric-looking string, or complex value would be silently reinterpreted rather than rejected; both now reject such input with a typed error via a local, documented copy of `p2mem.units`'s identical private dtype check (`p2mem/units.py` itself remains LOCKED and unmodified). See `INCREMENT_04_1_1_MANIFEST.md` for the full audit, the regression-test list, and the re-verification record.

- **Increment 5 update:** formation-top ingestion independently reproduces both required regression findings from the real approved files. Poseidon 2's "selected readable" file's supplied TVDSS equals `MDRT - 21.8 m` EXACTLY for every one of its 9 markers (21.8 m is the well's own rotary-table elevation) — a vertical-well-assumption depth-reference defect that is corrected in the derived `TVDSS_survey_corrected_m` representation, reproducing residuals of ≈0 m at Sea Bed, ≈+1.68 m at Plover Fm (Top Reservoir), and ≈+2.49 m at TD. Boreas 1's supplied TVDSS does not follow that pattern (maximum absolute residual 0.0416 m, below the 0.05 m tolerance) and is therefore NOT corrected — this is applied per file, per well, never by analogy from Poseidon 2's defect. Poseidon North 1 and Proteus 1ST2 have no approved formation-top file and are recorded as a factual data gap (`formation_top_availability: NOT_AVAILABLE`), never substituted with another well's tops or correlated by depth alone. Both formation-top file representations for both wells are classified `well_identity_evidence_status: inferred_unverified` — a deliberately more conservative classification than Increment 4's checkshot files, since neither the HRS files (no well name in the body) nor the readable files (well name only in a project-supplied comment) constitute independently verified file content; see `INCREMENT_05_MANIFEST.md` Section 2 for the full rationale. No lithology interpretation, petrophysical calculation, or geological correlation is performed on these markers — the Increment 5 marker-depth comparison figures are explicitly labelled as depth comparisons only.

- **Increment 5.1 update:** a narrowly scoped corrective patch to Increment 5, applied after an independent technical/software-QA audit, that does NOT begin Increment 6 or any later-phase work, and does NOT alter any previously verified real Poseidon 2/Boreas 1 formation-top result above. It corrected three defects and one documentation-accuracy gap. (1) `reconcile_formation_top_sources` documented "zero canonical markers in common between the two sources" as a fatal `NO_COMMON_MARKERS` ERROR, but actually tested the emptiness of the UNION of both sources' canonical names, so two entirely disjoint, non-empty marker sets were silently accepted as one-sided `NOT_COMPARABLE` entries instead of being rejected; the check now explicitly evaluates the INTERSECTION of the two sources' canonical names, and `load_formation_top_well` consequently raises `TopSourceReconciliationError` for this condition, while a legitimate one-sided marker (with at least one marker genuinely shared) remains the pre-existing, non-fatal `NOT_COMPARABLE` case. (2) `load_formation_top_surveys` recorded only the HRS file's path in every `TopIngestionFailure.source_path`, regardless of which file/stage actually failed, so a failure originating from the "selected readable" file (a missing file, malformed content, or a contract mismatch) could leak that file's own absolute path, unsanitized, into exported issues/availability/manifest rows; `TopIngestionFailure` now carries an explicit `failure_origin` and both `hrs_path`/`readable_path` fields, and every exporting function in `p2mem.io.tops_inventory` now sanitizes both candidate paths (literal substring replacement only, never a regex). (3) `reconcile_formation_top_sources` — the public in-memory API, as distinct from the file parsers, which already enforced this — did not validate its own documented contract before any numerical comparison: non-finite or negative MDRT/TVDSS values, a shorter `NOTE_source` tuple (previously reaching an untyped `IndexError`), a non-1-dimensional array, and an unvalidated `mdrt_agreement_tolerance_m` keyword (previously accepting NaN, a negative value, or a boolean) were all silently accepted or reached an undocumented incidental error; all are now rejected before any comparison, with deliberate, documented exceptions. (4) `INCREMENT_05_MANIFEST.md`'s statement that no absolute path exists ANYWHERE in the package was overbroad and inaccurate, since existing synthetic tests and historical documentation intentionally contain fake absolute-path strings as test inputs; the precise, narrowly scoped claim ("no environment-dependent build path appears in exported CSV/JSON outputs") is now stated explicitly in `INCREMENT_05_1_MANIFEST.md`, which also acknowledges the prior wording was overbroad — `INCREMENT_05_MANIFEST.md` itself is a locked historical record and is NOT rewritten. See `INCREMENT_05_1_MANIFEST.md` for the full audit, the regression-test list, and the real-data non-regression verification record.

- **Increment 5.1.1 update:** a further narrowly scoped corrective patch to Increment 5.1, applied after an independent technical/software-QA audit, that does NOT begin Increment 6 or any later-phase work, and does NOT alter any scientific result, tolerance, depth-mapping method, formation-top contract, or real-data output. It corrected one remaining reconciliation edge case and three documentation-accuracy gaps. (1) Increment 5.1 correctly rejected two non-empty, disjoint marker sets, but its condition ("both sources non-empty AND intersection empty", plus a separate both-empty case) still silently accepted the one remaining zero-common-markers configuration: exactly one source entirely empty and the other non-empty (the intersection of an empty set with anything is itself empty, so this was still a zero-common-markers condition). `reconcile_formation_top_sources` now uses the single, strictly correct check `if not common_markers` — empty in every zero-common-markers case (both empty, either side alone empty, or both non-empty and disjoint) and never empty whenever at least one canonical marker is genuinely shared, so a legitimate one-sided marker alongside at least one shared marker remains the pre-existing, non-fatal `NOT_COMPARABLE` case. (2) `INCREMENT_05_1_MANIFEST.md` stated that the real Increment 5 baseline ZIP's SHA-256 "matches" a governing-instruction-supplied hash that was, in fact, a two-character truncation of the real 64-character value — a mathematical impossibility, now stated transparently as a documentation/input typo, never as baseline corruption or uncertainty. (3) `INCREMENT_05_1_MANIFEST.md` Section 7 stated `outputs/` is excluded from the delivered ZIP; this was false — the delivered Increment 5.1 ZIP packages the `outputs/` tree (including the byte-identical Increment 5 formation-top outputs/figures), exactly as every prior increment's ZIP has; only dev-only regenerated-comparison directories and private raw source files are excluded, now stated accurately. (4) the same manifest's clean-room section stated no `*.las` file exists in the package; this was false because small, intentionally packaged synthetic LAS/checkshot/deviation/top fixtures exist under `tests/fixtures/` for portable testing — the corrected wording distinguishes these fictional fixtures from the genuinely excluded real/private project source files. See `INCREMENT_05_1_1_MANIFEST.md` for the full audit, the regression-test list, and the real-data non-regression verification record.

- **Increment 6 update:** the gamma-ray QC, screening-proxy sensitivity, well-frame and method-eligibility framework adds NO calibration and NO interpretation. Four specific limitations are newly quantified and disclosed. (1) **Boreas 1's ECGR is excluded, not corrected.** Its independently measured median (8.40 API) sits 4.3×–4.9× below the other three wells' GR medians, its range spans [−0.0001, 519.18] API including values at and below zero, and 2,059 of its samples lie above the well's own declared seabed marker. No tool header, calibration record, or environmental-correction metadata exists to adjudicate whether this is a scale, unit, tool-type, or acquisition problem, so the curve is formally excluded from every GR-derived calculation under the machine-readable reason `BOREAS_ECGR_SCALE_UNRESOLVED`. Rescaling it would fabricate a calibration this project does not have. (2) **The screening proxy is not a shale volume and is strongly endpoint-dependent.** `VSH_GR_linear_proxy_frac` is the linear identity of a clipped GR index computed under ASSUMED per-well percentile endpoints; across the three configured scenarios the proxy median moves by up to 0.12 dimensionless, and the qualifying sonic-NCT-candidate thickness moves by a factor of 8.0 in Poseidon 2 (144 m to 1,159 m), 3.5 in Poseidon North 1 and 2.3 in Proteus 1ST2. Any later NCT or pore-pressure result built on a single endpoint/threshold choice would inherit that full range as unquantified uncertainty. (3) **Eligibility is not validity.** The three masks record only whether a sample is technically admissible as INPUT to a later method; none of those methods is implemented, fitted, or validated here, and a sonic-NCT-candidate interval is emphatically not evidence that any interval is normally compacted, is a selected donor interval, or is any named lithology. Density and dynamic-elastic eligibility in all four wells begins only around 3,900–4,800 m TVDSS, so a later overburden integration cannot be supported from surface by these logs alone regardless of per-sample eligibility. (4) **Poseidon North 1 and Proteus 1ST2 have no approved formation tops**, so their above-seabed sample counts are reported as *not determinable* (never as 0) and all their results remain depth-tied and stratigraphically unvalidated. See `INCREMENT_06_MANIFEST.md` for the full measured statistics, sensitivity tables, and verification record.

- **Increment 6.1 update:** a narrowly scoped corrective patch to Increment 6, applied after an independent geomechanics/rock-physics and software audit. It does NOT start Increment 7 and implements no pore pressure, NCT fitting, elastic-property calculation, rock strength, stress, or wellbore-stability work. Four findings were corrected. (1) **Vp/Vs terminology and boundaries were scientifically inaccurate.** Increment 6 called every excluded Vp/Vs ratio "non-physical" and used a strictly exclusive √2 bound. With *r* = Vp/Vs, ν = (r²−2)/(2(r²−1)) and K = ρ(Vp²−(4/3)Vs²): *r* = √2 gives ν = 0 **exactly** and must be accepted by a non-negative-ν policy, so the bound is now **inclusive**; √(4/3) < *r* < √2 gives a **positive** bulk modulus with a negative Poisson's ratio — unusual and outside this project's conservative policy but *not* physically impossible; only *r* ≤ √(4/3) implies a non-positive bulk modulus and is genuinely outside the isotropic elastic model; and *r* > 4 is a **configured plausibility limit**, not a Poisson-domain boundary (ν ≈ 0.467 there). The condition is renamed a *configured non-negative-Poisson-ratio applicability screen*, the three bounds are separately named in config, and exclusions are diagnosed **by regime** rather than aggregated. Independently re-measured on the real data, the previously aggregated counts decompose as: Boreas 1 — 1 positive-K/negative-ν, 0 non-positive-K; Poseidon 2 — 22 and 0; Poseidon North 1 — **3 and 4** (the former single count of 7); Proteus 1ST2 — 3 and 0. No real sample sits exactly at √2, so the inclusive-bound correction changes no eligible count in this dataset — it corrects the policy, not the numbers. (2) **Poseidon North 1's machine-readable `use_status` contradicted its own data.** It read `screening_proxy_allowed` while the well has no approved formation tops; it is now `screening_proxy_allowed_depth_tied`, and a config invariant makes any contradictory has-tops/use-status pairing fail config loading loudly. (3) **Unsupported lithology and correlation claims were removed, and the named-lithology gate is no longer circular.** Project-specific "clastic-dominated" assertions are gone (a disclaimer does not undo an assertion), and the "regionally persistent" cross-well claim is replaced by the factual statement that low-GR intervals occur *independently* in each of the three GR-eligible wells with cross-well stratigraphic persistence **not established** — it cannot be, since two of those wells have no approved tops. `named_lithology_assigned` is now **derived** from an actual validation pass over every persisted label, per-well note, mask name and manifest statement rather than hardcoded; injecting a prohibited term makes the validation, the manifest flag and the completion gate fail together. (4) **Gross versus net interval thickness was ambiguous.** The reported "qualifying thickness" summed block endpoint spans that may contain explicitly bridged ineligible samples. Thickness is now exported only as `gross_thickness_*_m` (endpoint span, bridging-inclusive) or `net_thickness_*_m` (bridged gaps removed), every mask is additionally decomposed under a strict-no-gap policy, every sensitivity case reports its bridged-sample and interrupted-block counts, and Figure 4 distinguishes qualifying from rejected sub-threshold blocks with the counted population stated. Measured for Poseidon 2: configured **gross** 144.4–1,159.2 m (factor 8.03) versus **strict-no-gap** 140.0–1,110.2 m (factor 7.93). See `INCREMENT_06_1_MANIFEST.md` for the full audit and re-verification record.
- **Increment 6.1.1 update:** a second, narrower corrective patch, applied after an independent audit of Increment 6.1. It does NOT start Increment 7 and implements no pore pressure, NCT fitting, elastic-property calculation, rock strength, stress, or wellbore-stability work; it changes no configured tolerance, threshold, or qualifying-block policy. Five findings were corrected. (1) **The named-lithology validator could be bypassed.** The allowlist that lets per-well prose name a *method* contained `"shale gas"`, which is not a method-category phrase but can be a direct geological/hydrocarbon assertion — `"This interval contains shale gas."` passed with zero violations. The allowlist is reduced to `shale proxy` and `shale volume`, the only two phrases that are genuine method/quantity names required to state this project's boundaries, and the allowance is now evaluated **per sentence** and suppressed wherever the sentence carries a geological-assertion cue (`contains`, `comprises`, `bearing`, `facies`, …) or the phrase is immediately preceded by a quantity cue (`has a high shale volume`). Explanatory use of `shale proxy` / `shale volume` as a method or quantity name is preserved; use in any label remains prohibited outright. Regression tests prove the allowlist can no longer hide a geological assertion, and that injecting either bypass case into manifest content makes `named_lithology_assigned` true, `lithology_validation.n_violations` non-zero, and the completion gate fail. (2) **The active Vp/Vs description was stale.** The live top-level docstring of `p2mem.method_eligibility` still described the screen as keeping samples "inside the Poisson domain" under a strictly exclusive `Vp/Vs > √2`, contradicting the implementation corrected in 6.1. It now states the *configured non-negative-Poisson-ratio applicability screen* with an **inclusive** `Vp/Vs ≥ √2` bound and says explicitly that this is neither a physical-possibility test nor a boundary of the mathematical Poisson domain; a regression test asserts the corrected wording against the live module documentation. Superseded wording is retained only where it is clearly preserved as historical record. (3) **Interruption counts were ambiguous or wrong.** `n_interruptions` was a boolean-like flag reported as a count — all 327 bridged blocks reported exactly 1 — and `n_interrupted_subruns` did not describe what it counted. Interval records now carry `n_bridged_samples` (ineligible samples absorbed inside the gross block), `n_bridged_gaps` (distinct bridged runs) and `n_eligible_subruns` (strictly contiguous eligible sub-runs), with `n_bridged_gaps = n_eligible_subruns − 1` enforced at construction; a block with no gap reports 0/0/1 and a block with two separate bridged gaps reports 2 gaps and 3 sub-runs. The sensitivity summary distinguishes `n_bridged_samples_in_qualifying_blocks`, `n_bridged_gaps_in_qualifying_blocks` and `n_interrupted_qualifying_blocks`. No ambiguous alias survives in any active export. (4) **The notebook's output count was wrong.** It creates and checks eight deterministic CSV/JSON outputs while its gate text and printed label said seven; both now say eight and the gate asserts the declared count and the existence of all eight. Four figures remain separate — twelve outputs/figures in total. (5) **The Increment 6.1 delta arithmetic was incorrect.** That record stated "Changed (12)"; a clean recursive comparison gives **18 changed, 3 added, 0 removed (21 path differences)**. The locked 6.1 record is not rewritten; the statement is explicitly superseded, and both deltas are reported file by file, in `INCREMENT_06_1_1_MANIFEST.md`.
- **Increment 6.1.2 update:** a third, narrower corrective patch, applied after an independent audit of Increment 6.1.1. It does NOT start Increment 7, implements no pore pressure, NCT fitting, elastic-property calculation, rock strength, stress, or wellbore-stability work, changes no scientific threshold, tolerance, endpoint scenario, contiguity policy, GR disposition, depth-mapping rule, or real-data interpretation, and adds no runtime dependency. Two residual findings were corrected. (1) **The named-lithology validator was wrong in both directions.** Increment 6.1.1 decided the method-phrase allowance from a sentence-wide assertion-cue list plus a fixed look-behind window. Because nothing to the *right* of an allowed phrase was inspected, and because the normalizer erased possessives, `"Poseidon 2's shale volume is high."`, `"Poseidon 2's shale volume is 70 percent."` and `"The interval's shale volume exceeds 60 percent."` all passed with **zero violations** — the assertion lives in the predicate and the possessive, neither of which was ever read. Conversely, because a sentence-wide cue fires without asking what the cue word is predicated *of*, `"This method contains a shale proxy calculation."` and `"The analysis shows no shale volume was computed."` were reported as **false** lithological assertions. The allowance is now a property of each *occurrence*, decided from its own local grammar: a possessive geological entity, an adjacent magnitude modifier, a right-hand magnitude or dominance predicate, or a composition verb whose resolved subject head is a geological entity each withdraw it. A bare copula does not — `"the shale proxy is dimensionless"` describes a method, not an amount of rock. Apostrophes are parsed rather than erased, line breaks end sentences, ambiguous project-specific sentences fail closed, and labels keep zero latitude. A table-driven matrix of 59 discrimination rows across all three scopes, plus gate-path injections, pins the behaviour in both directions. (2) **Interval-record invariants were only partially enforced.** The 6.1.1 constructor checked only `n_bridged_gaps == n_eligible_subruns − 1`, and skipped even that whenever a value was `None` or `n_eligible_subruns` was 0 — so records with missing, negative, boolean or contradictory counts were constructible. All three counts are now validated together: required, strictly integral (booleans, strings, complex values and fractional floats rejected, never coerced), non-negative, `n_eligible_subruns ≥ 1`, `n_bridged_gaps = n_eligible_subruns − 1`, `n_bridged_samples = 0` **iff** `n_bridged_gaps = 0`, and `n_bridged_samples ≥ n_bridged_gaps`. Removed legacy field names are rejected outright rather than ignored. See `INCREMENT_06_1_2_MANIFEST.md` for the full audit and re-verification record.
- **Increment 6.1.3 update:** the fourth corrective patch, and an architectural one. It does NOT start Increment 7, changes no scientific threshold, tolerance, endpoint scenario, contiguity policy, GR disposition, depth-mapping rule, or real-data interpretation, and adds no runtime dependency. **(1) The validator no longer parses English.** Increments 6.1.1 and 6.1.2 each tried to decide from grammar whether a sentence containing a rock name named a *method* or asserted *geology*. Both were audited; both failed in both directions. 6.1.2 still missed `"The interval has a shale volume."`, `"The shale volume in Poseidon 2 exceeds 60 percent."` and `"Poseidon 2 shale volume was determined to be high."` — a missing predicate verb, a prepositional possessor, and a magnitude beyond the look-ahead window — while wrongly rejecting `"The well contains no shale volume estimate."` Each is a list gap or a window edge, and no finite grammar closes an unbounded space. Every scope is now decided by membership or equality: labels and project-specific prose have **zero allowance** (the `allow_method_phrases` parameter is deleted, so there is no exemption path left to bypass); method and limitation wording is admitted only when the text is **exactly equal** to a member of the closed, provenance-tagged `METHOD_STATEMENTS` registry, whose five members were *measured* by scanning every string literal in active packaged source under a zero-allowance rule; and explanatory text may use a rock name generically but never in a sentence referring to a project well **or a project rock body** (6.1.2 checked well names alone). All the grammar machinery — cue lists, magnitude vocabularies, look-behind and look-ahead windows, subject resolution, occurrence classification — is **deleted**, and a test asserts none of it is importable. Correctness is proved by **closure** over every prohibited term in every position inside carrier prose built from the exact constructions that defeated both previous implementations, not by a list of example sentences. **(2) Three exported strings had never been validated at all** — including a `calibration_status` value literally containing `not_a_shale_volume`, written into every row of `gr_proxy_sensitivity_summary.csv`. All three are now scanned, raising `n_fields_checked` from 138 to 143; that single scalar is the only difference in any output. **(3) The interval-record type gate is hardened**: keyword acceptance is a whitelist over declared slots (a misspelled count name no longer leaves the real count silently unset), floats are rejected outright including whole-valued ones, and `NaN`/`Inf` raise a typed `PetrophysicsInputError` before any conversion is attempted. See `INCREMENT_06_1_3_MANIFEST.md` for the full audit and re-verification record.
- **Increment 6.1.4 update:** a deliberately narrow completion of the Increment 6.1.3 architecture — one scope rule, no other change. It does NOT start Increment 7 and changes no scientific threshold, tolerance, policy, disposition or result. 6.1.3 closed labels and project-specific prose by removing every exemption, and closed method wording by exact registry membership — but left **explanatory** scope decided by a finite token list of project references. That is the same shape of rule that failed in 6.1.1 and 6.1.2, and it failed the same way: ordinary stratigraphic and exploration nouns were missing from the list, so `"The member is shale."`, `"The group is limestone."`, `"The package is a clean sandstone."`, `"The play is shale-dominated."`, `"The prospect is carbonate."` and `"The target is sandstone."` all passed — while `"The upper member is a clean sandstone reservoir."` failed only because `reservoir` happened to be listed, an accident that shows what a list-shaped rule produces. The rule is **inverted**: explanatory text carrying a rock term is a violation unless the text is a member of the closed `GENERIC_EXPLANATORY_STATEMENTS` registry. It **fails closed**, so no vocabulary gap can admit anything, and `PROJECT_WELL_NAME_TOKENS`, `PROJECT_ROCK_BODY_TOKENS` and `PROJECT_REFERENCE_TOKENS` are deleted. The generic registry is **empty** as a measured fact — this project persists exactly one explanatory field and it carries no rock name — but the mechanism is live and tested, because later increments explaining gamma-ray non-uniqueness will need it. **All four scopes are now closed by membership or exact equality**, and the closure proof covers every one of them. See `INCREMENT_06_1_4_MANIFEST.md` for the full record.
- **Increment 6.1.5 update:** the final Increment 6 corrective patch, and the one that changes the security model rather than the recognizer. Increments 6.1–6.1.4 all asked *"does this text contain a geological assertion?"* and all four shared one assumption — content is acceptable by default and becomes unacceptable only when a recognizer fires. All four were defeated, finally by a word no recognizer had been given: in the 6.1.4 package `"The interval is chalk."`, `"…halite."`, `"…gypsum."`, `"…conglomerate."`, `"…chert."`, `"…tuff."`, `"…basalt."`, `"…dolostone."`, `"…lignite."`, `"…calcareous."` all passed label, interpretive and explanatory scope, and so did `"The interval is qxzite."` — a word that does not exist. (They already failed in *method* scope, the one scope that then required registration; that contrast is what this patch generalises.) **A longer blacklist is not the fix and is not the mechanism the assurance claim rests on.** The model is inverted: labels must be members of a typed `APPROVED_LABELS` registry; interpretive, method and explanatory statements must resolve to a registered `statement_id`, or — interpretive only — a reviewed `template_id` whose substitutions are restricted to declared types. Id and exact rendered text are validated together, so unknown ids, id/text mismatches, case changes, near-misses, duplicate ids and undeclared template fields all fail. The registries were **measured** from the actual persisted export: 17 approved labels, 17 registered statements, 1 controlled template. `PROHIBITED_LITHOLOGY_TERMS` remains only as a diagnostic linter that authorizes nothing — every rejection listed above occurs with that linter returning **empty**. **The defensible assurance statement**, superseding the wording of every earlier Increment 6 manifest: *every persisted project-specific classification and interpretive statement is generated from an approved typed value, controlled template, or registered statement; arbitrary free text cannot enter these controlled fields.* This is **not** a claim that the software understands or exhaustively recognizes natural-language lithology — it does not, and no earlier version did. Free-form notebook narrative lies outside these controlled fields and remains subject to manual scientific review. See `INCREMENT_06_1_5_MANIFEST.md`.
- **Increment 6.1.6 update:** authorization is now enforced where records are actually EMITTED. 6.1.5 closed the unit validator but validated a scope the manifest builder *reconstructed*; a `GrEndpointScenario` with `description="The interval is chalk."` was persisted verbatim while absent from that 143-field scope, so it could not move `named_lithology_assigned`. Five findings, all reproduced first: the exported endpoint-description bypass; incomplete output coverage (endpoint `description`, `limitations`, `purpose`, `population_statement`, `limiting_criterion`, calibration/evidence labels, diagnostic text and manifest prose were not consistently validated); a field-kind mismatch letting `use_status="GR"` pass because `APPROVED_LABELS` was keyed by value alone; a stale exported `derivation` still describing prohibited-term validation; and a duplicated `named_lithology_statement` literal that could diverge from its registered copy while the gate stayed green. `p2mem.io.output_policy` now declares a category for **every** string column and JSON path of all eight artifacts (105 entries, zero unclassified) across eight closed categories, with **no** general category admitting arbitrary prose; labels are authorized by `(field_kind, value)`; the manifest statement comes from its registered id; and export authorizes the exact pre-serialization records, writes, then **re-authorizes the written bytes** so a post-authorization mutation is caught. Assurance metrics say what they count — `n_fields_checked=143` is replaced by `scope_object_fields_checked` plus an `emitted_field_coverage` block reporting total emitted occurrences, controlled occurrences, structural occurrences, occurrences outside the guarantee, unclassified fields, unauthorized fields and field-kind mismatches. Machine diagnostics and operator-facing issue text are counted separately and are explicitly outside the controlled-interpretation guarantee. See `INCREMENT_06_1_6_MANIFEST.md`.
- **Increment 6.1.7 update:** the 6.1.6 gate still inferred field presence from non-empty string values and wrote to the official directory before its post-write check. Exact schemas now validate all eight artifacts independently of content, including inventory, ordered CSV columns, JSON keys, requiredness, strict types, finite numerics and approved dynamic well identifiers. Every declared string occurrence is authorized even when empty or numeric-looking. The complete candidate set is written to an isolated sibling directory, re-read, re-validated and compared field-by-field and row-by-row using a typed canonical representation before publication. Missing/unknown fields, post-write type changes, row reordering, authorized-to-authorized substitutions, stale artifacts and serializer errors fail closed without altering the official destination. Publication is failure-atomic for handled process errors; no claim is made about multi-file atomicity across power loss or operating-system failure. No scientific calculation, threshold, disposition or result changed. See `INCREMENT_06_1_7_MANIFEST.md`.

- **Increment 7 / 7.0.1 / 7.0.2 / 7.0.3 update:** the density QC, coverage-qualification and vertical overburden-stress framework adds NO calibration. (1) **No approved well supports an absolute vertical overburden-stress curve.** All four wells carry a contract-resolved `RHOB_kg_m3` curve and are fully depth-mapped within survey coverage. Boreas 1 and Poseidon 2 have quantified unmeasured seabed-to-log columns of 3,486.54 m and 3,567.14 m TVD; Poseidon North 1 and Proteus 1ST2 have no determinable seabed or shallow-gap thickness because no approved formation-top file exists for either. (2) The low shallow-density endpoint is conditional on a fully saturated column, the high endpoint is an illustrative same-well P05 scenario motivated by monotonic-compaction reasoning, and the midpoint has no evidentiary support. None is a physical bound or best estimate, and 57–81% of the scenario totals is assumed. (3) The 10 m short-gap threshold is a reviewable heuristic evaluated through the complete 0/2/5/10/20/30 m sensitivity—not a rigorous error bound inferred from endpoint contrast. (4) **Increment 7.0.1 closes a fail-closed defect:** profile construction now truncates at every unresolved internal gap, including short or unmapped gaps; it no longer checks only the `long_internal_gap` class. The real approved workflow still bridges Boreas 1's 7.12 m gap, while its 15.64 m gap truncates the column and excludes 2,511 deeper eligible samples. (5) **Increment 7.0.2 closes two assurance defects:** the notebook gate now reads the schema-declared `assumed_density_basis` field, and the public frozen config constructor enforces loader-equivalent strict typing and invariants for direct construction and `dataclasses.replace()`. (6) **Increment 7.0.3 closes the remaining percentile-provenance defect:** the configured high percentile must be exactly 5.0 because the implemented statistic and scenario are explicitly P05; an inconsistent label now fails at both YAML loading and public construction. The packaged value was already 5.0, so no scientific output changes. See `INCREMENT_07_0_3_MANIFEST.md` for the corrective verification record.

- **Increment 7.0.4 update:** a final corrective patch after end-to-end audit. The high scenario's P05 is now calculated over `eligible_for_measured_integration`, matching its documented population; the distinct all-finite P05 remains available as factual QC. Every public threshold override and threshold-bearing result rejects boolean, textual, complex, negative, NaN and infinite values. Scenario accounting now exports three exhaustive fractions—assumed, conditioned and measured—whose constructor-enforced sum is one. The locked seabed reader fails closed on a missing table and malformed, duplicate or non-finite matching rows while still representing a valid header-only table as an explicit empty mapping. JSON output uses explicit UTF-8 bytes with LF newlines, making regeneration byte-identical on Windows and Linux/Colab. Approved real-data stress values, statuses, thresholds and figures remain unchanged; the QC/scenario CSV schemas and assurance JSON change additively to expose the corrected populations and fraction accounting. See `INCREMENT_07_0_4_MANIFEST.md`.

## Screening-level statement

**This 1D Mechanical Earth Model is a screening-level, uncalibrated, educational work product.** It has not been validated against independent field measurements and does not carry the assurance level required for drilling engineering, well design, casing/mud-weight selection, or any other operational decision. Any numerical result produced by this codebase should be read as illustrative of a defensible methodology applied to the available data, not as a certified or field-ready prediction.


#### `config/overburden_stress.yml` - the human-authored screening policy

Every number here is a **reviewable screening choice**, not a measurement: the bulk-density
plausibility band, the gap-bridging threshold, standard gravity, the assumed seawater
density, and the shallow-column bracket. **No well name appears anywhere in this file** -
overburden eligibility is derived from evidence, never configured per well.

In [ ]:
%%writefile config/overburden_stress.yml
# =============================================================================
# config/overburden_stress.yml
#
# Poseidon 2 1D MEM - Increment 7.0.1
# Human-authored density-QC screening policy, gap-conditioning policy, and
# vertical overburden-stress assumption register.
#
# Tier C - Screening-Level / Uncalibrated Educational.
#
# WHY THIS FILE IS HUMAN-AUTHORED
# -------------------------------
# Every number below is a REVIEWABLE SCREENING CHOICE, not a measurement and
# not a calibrated constant. A bulk-density plausibility band, a gap-bridging
# threshold, a seawater density, and a shallow-column bracket are all
# judgements about what this project is willing to accept for a screening-level
# result. Putting them in a plain-text, diffable, version-controlled file means
# an auditor can see and challenge each one without reading code, and means a
# result can be recomputed under a different judgement without editing code.
#
# WHAT THIS FILE MAY NOT DO
# -------------------------
# * It may not name a lithology. No key or value here asserts that any
#   interval is shale, sand, carbonate, or any other rock type, and no
#   shallow-column bracket below is justified by an assumed rock type.
# * It may not declare any bound, threshold, density, or scenario
#   "calibrated". There is no measured shallow-density control, no checkshot-
#   derived density, no core density, and no seawater measurement in this
#   project.
# * It may not repair a density log. There is no correction factor, no
#   rescaling, no smoothing, no despiking, and no extrapolation policy here.
#   An out-of-band or missing sample is MASKED, never modified.
# * It may not hardcode a per-well verdict. Overburden method eligibility is
#   DERIVED from measured coverage and QC results; no well name appears in
#   this file.
#
# UNITS
# -----
# Density is kg/m^3 (SI) throughout. The locked Increment 2.1.1 curve
# contracts already convert each file's recorded g/cm^3 RHOB column to
# kg/m^3 via the exact p2mem.units.gcc_to_kgm3 factor, so no unit decision is
# re-made here; this file only declares which canonical unit is ACCEPTED.
# Vertical depth is metres. Stress is pascals internally and is reported in
# both Pa and MPa. Gravity is m/s^2.
# =============================================================================

schema_version: "7.0.1"
increment: 7
assurance_tier: "Tier C - Screening-Level / Uncalibrated Educational"

# -----------------------------------------------------------------------------
# DENSITY SOURCE AND UNIT POLICY
# -----------------------------------------------------------------------------
density_source:

  # The ONLY canonical curve name Increment 7 will integrate. A well whose
  # locked curve contract does not resolve this exact canonical name has no
  # density for this increment; that is a factual data gap, reported as
  # 'rhob_not_available', never worked around by accepting a different curve.
  canonical_curve_name: "RHOB_kg_m3"

  # The ONLY canonical unit accepted. This is a CONFIRMATION check against the
  # locked loader's own recorded canonical unit, not a conversion instruction:
  # Increment 7 performs no unit conversion of its own. A curve arriving in
  # any other canonical unit is rejected with a typed error rather than
  # silently rescaled.
  accepted_canonical_unit: "kg/m3"

  # The conversion function the locked contract is expected to have applied.
  # Recorded for provenance and confirmed per well; a mismatch is reported as
  # a QC issue, never corrected here.
  expected_conversion_function: "gcc_to_kgm3"

  # Increment 7 never modifies the resolved RHOB array. Any conditioned
  # representation lives in a separately named array and carries its own mask.
  original_values_immutable: true

# -----------------------------------------------------------------------------
# SCREENING PLAUSIBILITY BAND
# -----------------------------------------------------------------------------
# These are SCREENING CRITERIA, not universal geological truth and not a
# statement about what rock exists in these wells.
#
# Lower bound 1000 kg/m^3: approximately the density of fresh water. It is a
# project screening threshold, not a universal physical floor. Interpreting it
# as a lower physical reference is conditional on a fully liquid-saturated
# porous medium; the project has no fluid-state measurement for every logged
# interval. Values below it are therefore flagged for review, not declared
# impossible geology.
#
# Upper bound 3500 kg/m^3: above the bulk density of any common sedimentary
# mineral assemblage at any porosity. This is a project credibility limit
# chosen to catch gross processing spikes, NOT a claim that a value just below
# it is physically reasonable or lithologically meaningful.
#
# BOTH bounds are INCLUSIVE. A sample exactly equal to a bound passes. This is
# stated explicitly because an inclusive/exclusive slip at a boundary is a
# silent scientific change, and it is tested at the exact boundary values.
screening_bounds:
  rhob_min_kg_m3: 1000.0
  rhob_max_kg_m3: 3500.0
  bounds_are_inclusive: true

# -----------------------------------------------------------------------------
# GAP-CONDITIONING POLICY
# -----------------------------------------------------------------------------
gap_conditioning:

  # Short internal gaps MAY be bridged. Bridging is linear interpolation of
  # density against TRUE VERTICAL DEPTH - the coordinate the stress integral
  # is actually taken in - between the last and first bracketing eligible
  # samples. It is never applied outside the bracketed interior of the
  # measured column.
  bridge_short_internal_gaps: true

  # The approved threshold, expressed in TVD metres (NOT measured depth and
  # NOT a sample count: a sample count would mean different things in
  # different logging runs, and an MD threshold would mean different things in
  # a deviated well).
  #
  # WHY 10 m. This is a transparent project heuristic, not a rigorous error
  # bound. Endpoint contrast alone cannot bound the unknown density inside a
  # missing interval without an additional assumption about its interior
  # behaviour. The full 0/2/5/10/20/30 m sensitivity is therefore the evidence
  # used to show the consequence of this choice. The conditioned contribution
  # is exported separately, and any internal gap left unresolved truncates the
  # integrable profile before the gap.
  short_gap_max_tvd_m: 10.0

  # Bridging is forbidden across the seabed, outside locked survey coverage,
  # and across any gap exceeding the threshold. These are hard invariants, not
  # tunable behaviour; they are declared here so the file states the policy
  # completely rather than leaving it implicit in code.
  bridge_across_seabed_allowed: false
  bridge_outside_survey_coverage_allowed: false
  bridge_long_gaps_allowed: false

  # The shallow seabed-to-first-valid-RHOB gap is NEVER treated as an internal
  # gap and is never bridged by the rule above, at any threshold. It is an
  # unmeasured column, not a dropout.
  shallow_gap_uses_internal_gap_rule: false

  # A shallow gap at or below this tolerance is treated as reaching the
  # seabed. Set to one median sample increment's order of magnitude; it exists
  # so that a log genuinely starting at the seabed is not failed by a
  # sub-metre offset. It is NOT a licence to ignore a real unmeasured column.
  shallow_gap_tolerance_tvd_m: 1.0

  # Thresholds re-run for the mandatory gap-threshold sensitivity report. The
  # approved threshold above must appear in this list.
  sensitivity_thresholds_tvd_m: [0.0, 2.0, 5.0, 10.0, 20.0, 30.0]

# -----------------------------------------------------------------------------
# INTEGRATION POLICY
# -----------------------------------------------------------------------------
integration:

  # sigma_v(z) = INTEGRAL rho(z) g dz, evaluated in TRUE VERTICAL DEPTH.
  # Trapezoidal quadrature between consecutive eligible samples: exact for a
  # piecewise-linear density profile, deterministic, and introducing no fitted
  # parameter. Measured-depth integration is structurally impossible here -
  # the integrator accepts a vertical-depth array and rejects a non-monotonic
  # one.
  method: "trapezoidal_in_true_vertical_depth"

  # CGPM standard gravity, exact by definition. This is an ASSUMED constant:
  # no local gravity survey exists for these wells. Any result stating an
  # absolute stress carries this assumption.
  gravity_m_s2: 9.80665

  # Vertical-coordinate handling, declared rather than left to chance:
  #  * a zero increment (repeated or equal vertical coordinate) contributes
  #    exactly zero and is counted and reported;
  #  * a negative increment is REJECTED with a typed error - it is either a
  #    sorting defect or an attempt to integrate the wrong coordinate, and
  #    silently sorting it away would hide both.
  zero_increment_contribution: "exactly_zero_counted_and_reported"
  negative_increment_policy: "reject_with_typed_error"

  # Reported profile decimation. The integral itself uses EVERY eligible
  # sample; this controls only how densely the cumulative profile is exported.
  # Nodes are SELECTED existing samples, never interpolated values.
  profile_report_step_tvdss_m: 10.0

# -----------------------------------------------------------------------------
# WATER COLUMN
# -----------------------------------------------------------------------------
water_column:

  # ASSUMED seawater density. This project holds no measured seawater density,
  # no salinity profile, and no temperature profile for these locations. This
  # value is a configured screening assumption and must never be described as
  # measured data.
  seawater_density_kg_m3: 1025.0

  # Bracket used for the seawater sensitivity rows. Chosen to span ordinary
  # open-ocean seawater density; it is a bracket on an assumption, not an
  # uncertainty derived from a measurement.
  seawater_density_low_kg_m3: 1020.0
  seawater_density_high_kg_m3: 1030.0

  # The water column is integrated from the sea surface (TVDSS = 0) to the
  # seabed marker's TVDSS, taken from the LOCKED Increment 5 survey-corrected
  # marker table. A well with no approved formation-top file has no seabed
  # marker and therefore no determinable water column; it is reported as
  # unresolved, never assumed to be zero and never borrowed from another well.
  seabed_source: "locked_increment5_survey_corrected_marker_table"
  seabed_marker_name: "Sea Bed"
  cross_well_seabed_transfer_allowed: false

# -----------------------------------------------------------------------------
# UNRESOLVED SHALLOW-COLUMN SCREENING SCENARIOS
# -----------------------------------------------------------------------------
# READ THIS BEFORE READING ANY NUMBER DERIVED FROM IT.
#
# Where the density log begins far below the seabed, the interval between the
# seabed and the first eligible density sample is UNMEASURED. No absolute
# vertical stress can be stated as measured truth for such a well.
#
# The scenarios below exist ONLY to quantify the magnitude of that ignorance.
# They are a transparent bracket, published low/base/high, never collapsed to
# a single number, and never presented as an estimate of the true stress.
#
# HOW THE BRACKET IS JUSTIFIED, WITHOUT A LITHOLOGY AND WITHOUT A TREND
# --------------------------------------------------------------------
# low  : the configured seawater density. This is a conditional low scenario
#        only if the unknown column is fully saturated. The project does not
#        establish its fluid state, so this is not asserted as a physical
#        floor.
# high : the low tail (a configured percentile) of the SAME WELL'S OWN
#        measured eligible density population. The unresolved column lies
#        entirely ABOVE the measured column, which motivates an illustrative
#        monotonic-compaction scenario. Without lithology, fluid-state,
#        overpressure or shallow-density control, the measured P05 is not a
#        physical ceiling and the spread is not a rigorous uncertainty bound.
#        No depth-dependent density function is constructed anywhere.
# base : the arithmetic midpoint of low and high. It has NO evidentiary
#        support whatsoever. It exists so that the bracket has a labelled
#        centre for plotting, and it must never be selected as a result.
#
# The sensitivity spread is deliberately wide. It demonstrates dependence on
# the assumed shallow density; it is not a calibrated confidence interval.
shallow_column_scenarios:
  enabled: true
  low_basis: "configured_seawater_density_conditional_low_scenario"
  high_basis: "own_well_measured_p05_illustrative_upper_scenario"
  high_percentile: 5.0
  base_basis: "arithmetic_midpoint_of_low_and_high_no_evidentiary_support"
  base_is_not_a_best_estimate: true
  scenario_names: ["low", "base", "high"]

  # A scenario result may never be published without its assumed-fraction
  # disclosure (what proportion of the reported total originates in the
  # unmeasured column). Enforced in code; declared here so the policy is
  # visible in the reviewable file.
  require_assumed_fraction_disclosure: true

# -----------------------------------------------------------------------------
# METHOD-ELIGIBILITY DERIVATION
# -----------------------------------------------------------------------------
# The status codes and the ORDER in which limiting reasons are evaluated. The
# status is DERIVED from measured coverage and QC results for each well; no
# well name appears anywhere in this file.
eligibility:
  statuses:
    - "absolute_overburden_supported"
    - "screening_sensitivity_only"
    - "partial_measured_increment_only"
    - "not_eligible"

  # absolute_overburden_supported additionally requires that no unresolved
  # internal gap of any class interrupts the column between the seabed and the
  # evaluation depth. A short gap is still unresolved when bridging is
  # disabled or a bridge precondition fails; it may never be integrated
  # across merely because it is short.
  absolute_requires_uninterrupted_column: true

  # Minimum number of eligible samples before any measured increment is
  # reported at all. Two samples is the arithmetic minimum for one trapezoid;
  # this is deliberately not a coverage-quality threshold in disguise.
  min_eligible_samples_for_increment: 2

# -----------------------------------------------------------------------------
# EXPLICITLY NOT IMPLEMENTED IN INCREMENT 7
# -----------------------------------------------------------------------------
# Declared so that the absence of each is a recorded decision rather than an
# omission, and so that a future reader cannot mistake this framework for a
# pore-pressure or stress-modelling deliverable.
not_implemented:
  - "pore_pressure_prediction"
  - "sonic_normal_compaction_trend_fitting"
  - "resistivity_normal_compaction_trend_fitting"
  - "eaton_method"
  - "bowers_method"
  - "equivalent_depth_method"
  - "drilling_exponent_method"
  - "effective_stress_calculation"
  - "dynamic_elastic_property_modelling"
  - "static_elastic_property_modelling"
  - "rock_strength_modelling"
  - "horizontal_stress_calculation"
  - "stress_calibration"
  - "mud_window_calculation"
  - "breakout_analysis"
  - "tensile_fracture_analysis"
  - "wellbore_stability_analysis"
  - "named_lithology_assignment"
  - "mineralogical_interpretation"
  - "shallow_density_reconstruction_from_a_fitted_trend"
  - "density_extrapolation_beyond_measured_coverage"


#### `p2mem/overburden_models.py` - typed records, closed vocabularies, constructor invariants

Holds data structures and vocabularies only. Its constructors refuse an incoherent record: a
bridged shallow gap, a total reported while a component is unresolved, a gap count that
disagrees with its sample count, an absolute status carrying a limiting reason, a
non-decreasing-violating cumulative series, or a profile claiming to be integrated in
measured depth.

In [ ]:
%%writefile p2mem/overburden_models.py
"""
p2mem.overburden_models - typed Increment 7 structures, controlled vocabularies,
and constructor-level invariants for density QC and vertical overburden stress.

Scope
-----
This module holds DATA STRUCTURES and VOCABULARIES only. It reads no file,
computes no statistic, and integrates nothing. It exists so that every
Increment 7 record is constructed through a type that refuses to hold an
incoherent state, and so that every enumerated code this increment can emit is
declared in exactly one place.

Why the vocabularies live here rather than in `p2mem.wellframe_models`
----------------------------------------------------------------------
`p2mem.wellframe_models` is LOCKED Increment 6.1.7 content. Its
`APPROVED_LABELS`, `REGISTERED_STATEMENTS` and `REGISTERED_TEMPLATES`
registries were enumerated from the ACTUAL persisted Increment 6 export, and
their provenance strings say exactly that. Appending Increment 7 vocabulary to
those registries would (a) modify a locked file and (b) falsify the provenance
of the registry as a whole. Increment 7 therefore declares its own registries
here, and the Increment 7 output policy carries them in its own policy bundle.
The locked registries and the locked export path are untouched and continue to
govern the Increment 6 artifacts exactly as before.

Sign convention (verified, not assumed)
---------------------------------------
The locked `p2mem.depth_mapping` module documents and independently tests the
project's depth convention:

    MD and TVD are zero at the well datum (rotary table) and increase
    DOWNWARD; the datum elevation is referenced to mean sea level, positive
    UPWARD; therefore TVDSS_m = TVD_m - DatumElevation_m, and TVDSS is
    positive DOWNWARD from mean sea level.

Both TVD and TVDSS therefore increase downward, and a downward interval has a
POSITIVE increment in either coordinate. `dTVD == dTVDSS` exactly, because they
differ by a per-well constant. Increment 7 integrates in this convention and
re-verifies it at run time against the locked well frame rather than inferring
it from a variable name: see `p2mem.overburden.verify_depth_sign_convention`.
"""

from __future__ import annotations

import math
from dataclasses import dataclass, field
from typing import Dict, Optional, Tuple

import numpy as np

__all__ = [
    "OverburdenConfigError",
    "OverburdenInputError",
    "OverburdenEligibilityError",
    "DENSITY_MASK_NAMES",
    "GAP_CLASS_ISOLATED_SAMPLE",
    "GAP_CLASS_SHORT_INTERNAL",
    "GAP_CLASS_LONG_INTERNAL",
    "GAP_CLASS_SHALLOW",
    "GAP_CLASS_TERMINAL",
    "GAP_CLASS_UNMAPPED_DEPTH",
    "VALID_GAP_CLASSES",
    "GAP_DISPOSITION_BRIDGED",
    "GAP_DISPOSITION_UNRESOLVED",
    "GAP_DISPOSITION_NOT_APPLICABLE",
    "VALID_GAP_DISPOSITIONS",
    "STATUS_ABSOLUTE",
    "STATUS_SENSITIVITY_ONLY",
    "STATUS_PARTIAL_ONLY",
    "STATUS_NOT_ELIGIBLE",
    "VALID_OVERBURDEN_STATUSES",
    "REASON_RHOB_NOT_AVAILABLE",
    "REASON_RHOB_ALL_INVALID",
    "REASON_SHALLOW_COLUMN_UNRESOLVED",
    "REASON_SEABED_DATUM_UNRESOLVED",
    "REASON_INTERNAL_GAP_EXCEEDS_LIMIT",
    "REASON_INTERNAL_GAP_UNRESOLVED",
    "REASON_TERMINAL_COLUMN_UNRESOLVED",
    "REASON_DEPTH_MAPPING_INCOMPLETE",
    "REASON_SCREENING_BOUND_FAILURES",
    "REASON_SURVEY_COVERAGE_INSUFFICIENT",
    "REASON_NON_MONOTONIC_VERTICAL_DEPTH",
    "REASON_INSUFFICIENT_ELIGIBLE_SAMPLES",
    "REASON_UNIT_NOT_RESOLVED",
    "VALID_LIMITING_REASONS",
    "SEABED_BASIS_LOCKED_MARKER",
    "SEABED_BASIS_NOT_DETERMINABLE",
    "VALID_SEABED_BASES",
    "SCENARIO_LOW",
    "SCENARIO_BASE",
    "SCENARIO_HIGH",
    "SCENARIO_BASE_SEAWATER_LOW",
    "SCENARIO_BASE_SEAWATER_HIGH",
    "VALID_SCENARIO_NAMES",
    "SCENARIO_BASIS_LOW",
    "SCENARIO_BASIS_BASE",
    "SCENARIO_BASIS_HIGH",
    "VALID_SCENARIO_BASES",
    "OverburdenConfig",
    "DensityMaskSet",
    "DensityQcStats",
    "DensityGapRecord",
    "GapConditioningResult",
    "StressPartition",
    "VerticalStressProfile",
    "ShallowColumnScenario",
    "GapThresholdSensitivity",
    "OverburdenEligibility",
    "OverburdenIssue",
    "readonly",
]


# ---------------------------------------------------------------------------
# Typed exceptions
# ---------------------------------------------------------------------------

class OverburdenConfigError(ValueError):
    """Raised when `config/overburden_stress.yml` is missing, malformed,
    internally inconsistent, or missing a required key. A configuration
    problem is never silently defaulted."""


class OverburdenInputError(ValueError):
    """Raised for a structural or value defect in caller-supplied in-memory
    data: wrong dimensionality, length mismatch, non-finite or mis-ordered
    vertical coordinates, negative vertical increments, or an incoherent
    record. Deliberately distinct from `TypeError`, which this increment
    reserves for type-class defects (boolean, string, complex, object)."""


class OverburdenEligibilityError(RuntimeError):
    """Raised when a stress calculation is attempted for a well whose derived
    eligibility status forbids it. A hard scientific boundary, enforced as an
    exception rather than a silent empty result so that a caller cannot
    mistake 'nothing was computed' for 'zero stress'."""


# ---------------------------------------------------------------------------
# Controlled vocabularies
# ---------------------------------------------------------------------------

#: The explicit, separately auditable per-sample masks Increment 7 constructs.
#: Every one of these is a full-length boolean array aligned sample-for-sample
#: with the well's canonical `MD_m`; none of them modifies a density value.
DENSITY_MASK_NAMES: Tuple[str, ...] = (
    "source_value_present",
    "finite_numeric_density",
    "unit_resolved",
    "screening_range_plausible",
    "below_seabed_sample",
    "depth_mapping_valid",
    "within_survey_coverage",
    "eligible_for_measured_integration",
    "bridged_short_gap",
    "unresolved_long_gap",
    "unresolved_shallow_column",
    "unresolved_terminal_column",
)

GAP_CLASS_ISOLATED_SAMPLE = "isolated_invalid_sample"
GAP_CLASS_SHORT_INTERNAL = "short_internal_gap"
GAP_CLASS_LONG_INTERNAL = "long_internal_gap"
GAP_CLASS_SHALLOW = "shallow_seabed_to_first_valid_gap"
GAP_CLASS_TERMINAL = "terminal_below_last_valid_gap"
GAP_CLASS_UNMAPPED_DEPTH = "gap_caused_by_missing_depth_mapping"
VALID_GAP_CLASSES: Tuple[str, ...] = (
    GAP_CLASS_ISOLATED_SAMPLE, GAP_CLASS_SHORT_INTERNAL, GAP_CLASS_LONG_INTERNAL,
    GAP_CLASS_SHALLOW, GAP_CLASS_TERMINAL, GAP_CLASS_UNMAPPED_DEPTH,
)

GAP_DISPOSITION_BRIDGED = "bridged_linear_in_tvd"
GAP_DISPOSITION_UNRESOLVED = "unresolved_not_bridged"
GAP_DISPOSITION_NOT_APPLICABLE = "not_applicable"
VALID_GAP_DISPOSITIONS: Tuple[str, ...] = (
    GAP_DISPOSITION_BRIDGED, GAP_DISPOSITION_UNRESOLVED, GAP_DISPOSITION_NOT_APPLICABLE,
)

STATUS_ABSOLUTE = "absolute_overburden_supported"
STATUS_SENSITIVITY_ONLY = "screening_sensitivity_only"
STATUS_PARTIAL_ONLY = "partial_measured_increment_only"
STATUS_NOT_ELIGIBLE = "not_eligible"
VALID_OVERBURDEN_STATUSES: Tuple[str, ...] = (
    STATUS_ABSOLUTE, STATUS_SENSITIVITY_ONLY, STATUS_PARTIAL_ONLY, STATUS_NOT_ELIGIBLE,
)

REASON_RHOB_NOT_AVAILABLE = "rhob_not_available"
REASON_RHOB_ALL_INVALID = "rhob_all_invalid"
REASON_SHALLOW_COLUMN_UNRESOLVED = "shallow_density_column_unresolved"
REASON_SEABED_DATUM_UNRESOLVED = "seabed_datum_unresolved"
REASON_INTERNAL_GAP_EXCEEDS_LIMIT = "internal_gap_exceeds_limit"
REASON_INTERNAL_GAP_UNRESOLVED = "internal_gap_unresolved"
REASON_TERMINAL_COLUMN_UNRESOLVED = "terminal_density_column_unresolved"
REASON_DEPTH_MAPPING_INCOMPLETE = "depth_mapping_incomplete"
REASON_SCREENING_BOUND_FAILURES = "screening_bound_failures"
REASON_SURVEY_COVERAGE_INSUFFICIENT = "survey_coverage_insufficient"
REASON_NON_MONOTONIC_VERTICAL_DEPTH = "non_monotonic_vertical_depth"
REASON_INSUFFICIENT_ELIGIBLE_SAMPLES = "insufficient_eligible_samples"
REASON_UNIT_NOT_RESOLVED = "density_unit_not_resolved"
VALID_LIMITING_REASONS: Tuple[str, ...] = (
    REASON_DEPTH_MAPPING_INCOMPLETE,
    REASON_INSUFFICIENT_ELIGIBLE_SAMPLES,
    REASON_INTERNAL_GAP_EXCEEDS_LIMIT,
    REASON_INTERNAL_GAP_UNRESOLVED,
    REASON_NON_MONOTONIC_VERTICAL_DEPTH,
    REASON_RHOB_ALL_INVALID,
    REASON_RHOB_NOT_AVAILABLE,
    REASON_SCREENING_BOUND_FAILURES,
    REASON_SEABED_DATUM_UNRESOLVED,
    REASON_SHALLOW_COLUMN_UNRESOLVED,
    REASON_SURVEY_COVERAGE_INSUFFICIENT,
    REASON_TERMINAL_COLUMN_UNRESOLVED,
    REASON_UNIT_NOT_RESOLVED,
)

SEABED_BASIS_LOCKED_MARKER = "locked_increment5_survey_corrected_sea_bed_marker"
SEABED_BASIS_NOT_DETERMINABLE = "not_determinable_no_approved_formation_tops"
VALID_SEABED_BASES: Tuple[str, ...] = (
    SEABED_BASIS_LOCKED_MARKER, SEABED_BASIS_NOT_DETERMINABLE,
)

SCENARIO_LOW = "low"
SCENARIO_BASE = "base"
SCENARIO_HIGH = "high"
SCENARIO_BASE_SEAWATER_LOW = "base_seawater_low"
SCENARIO_BASE_SEAWATER_HIGH = "base_seawater_high"
VALID_SCENARIO_NAMES: Tuple[str, ...] = (
    SCENARIO_LOW, SCENARIO_BASE, SCENARIO_HIGH,
    SCENARIO_BASE_SEAWATER_LOW, SCENARIO_BASE_SEAWATER_HIGH,
)

# These tokens deliberately say "scenario", not "floor", "ceiling" or
# "bound".  The shallow interval is unmeasured, so its two endpoint densities
# are transparent sensitivity assumptions rather than rigorous physical
# limits on the unknown depth-averaged density.
SCENARIO_BASIS_LOW = "configured_seawater_density_conditional_low_scenario"
SCENARIO_BASIS_BASE = "arithmetic_midpoint_of_low_and_high_no_evidentiary_support"
SCENARIO_BASIS_HIGH = "own_well_measured_p05_illustrative_upper_scenario"
VALID_SCENARIO_BASES: Tuple[str, ...] = (
    SCENARIO_BASIS_LOW, SCENARIO_BASIS_BASE, SCENARIO_BASIS_HIGH,
)


# ---------------------------------------------------------------------------
# Shared validation helpers
# ---------------------------------------------------------------------------

def readonly(arr: np.ndarray) -> np.ndarray:
    """Return a read-only view of `arr`.

    Every mask and derived array this increment publishes is handed out
    read-only so that one consumer cannot mutate a shared object and silently
    change another consumer's result. Mirrors the locked well-frame
    `_readonly` precedent; a ~4-line local copy is deliberately preferred over
    editing a locked module to export a private helper.
    """
    view = np.asarray(arr)
    view = view.view()
    view.setflags(write=False)
    return view


def _require_exact_int(value, context: str) -> int:
    """Accept only a genuine Python `int`.

    `True` is an `int` in Python and `1.0` compares equal to `1`; both are
    rejected. A count that arrived as a float is a defect in the caller, not
    something to round.
    """
    if isinstance(value, (bool, np.bool_)):
        raise OverburdenInputError(f"{context}: boolean is not an integer count.")
    if isinstance(value, (np.integer,)):
        return int(value)
    if type(value) is not int:
        raise OverburdenInputError(
            f"{context}: expected an int count, got {type(value).__name__} {value!r}.")
    return value


def _require_nonneg_int(value, context: str) -> int:
    out = _require_exact_int(value, context)
    if out < 0:
        raise OverburdenInputError(f"{context}: count must be >= 0, got {out}.")
    return out


def _require_finite_float(value, context: str, allow_none: bool = False) -> Optional[float]:
    if value is None:
        if allow_none:
            return None
        raise OverburdenInputError(f"{context}: a finite number is required, got None.")
    if isinstance(value, (bool, np.bool_)):
        raise OverburdenInputError(f"{context}: boolean is not a numeric quantity.")
    if isinstance(value, complex) or isinstance(value, np.complexfloating):
        raise OverburdenInputError(f"{context}: complex is not a numeric quantity.")
    if isinstance(value, str) or isinstance(value, bytes):
        raise OverburdenInputError(f"{context}: string/bytes is not a numeric quantity.")
    try:
        out = float(value)
    except (TypeError, ValueError) as exc:
        raise OverburdenInputError(
            f"{context}: value {value!r} is not convertible to a float.") from exc
    if not math.isfinite(out):
        raise OverburdenInputError(f"{context}: value must be finite, got {out!r}.")
    return out


def _require_in(value, allowed: Tuple[str, ...], context: str) -> str:
    if value not in allowed:
        raise OverburdenInputError(
            f"{context}: {value!r} is not one of {list(allowed)}.")
    return value


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class OverburdenConfig:
    """The validated contents of `config/overburden_stress.yml`.

    Frozen: a configuration value must not change between the moment a result
    is computed and the moment its assumption register is exported, or the
    exported register would not describe the run that produced the numbers.
    """

    source_filename: str
    schema_version: str
    increment: int
    assurance_tier: str

    canonical_curve_name: str
    accepted_canonical_unit: str
    expected_conversion_function: str

    rhob_min_kg_m3: float
    rhob_max_kg_m3: float
    bounds_are_inclusive: bool

    bridge_short_internal_gaps: bool
    short_gap_max_tvd_m: float
    shallow_gap_tolerance_tvd_m: float
    sensitivity_thresholds_tvd_m: Tuple[float, ...]

    integration_method: str
    gravity_m_s2: float
    profile_report_step_tvdss_m: float

    seawater_density_kg_m3: float
    seawater_density_low_kg_m3: float
    seawater_density_high_kg_m3: float
    seabed_marker_name: str

    scenarios_enabled: bool
    scenario_high_percentile: float
    scenario_names: Tuple[str, ...]

    min_eligible_samples_for_increment: int
    absolute_requires_uninterrupted_column: bool
    not_implemented: Tuple[str, ...]

    def __post_init__(self) -> None:
        """Validate the public constructor as strictly as the YAML loader.

        ``dataclasses.replace`` calls this constructor, and is the supported
        way tests and sensitivity runs derive one frozen configuration from
        another.  The constructor must therefore reject the same ambiguous
        types and malformed closed vocabularies as ``load_overburden_config``;
        validation only in the loader would leave a second, weaker public
        entry point into the numerical workflow.
        """
        def require_string(field_name: str, value: object) -> str:
            if not isinstance(value, str) or not value.strip():
                raise OverburdenConfigError(
                    f"{field_name} must be a non-empty string, got "
                    f"{type(value).__name__} {value!r}.")
            return value

        def require_bool(field_name: str, value: object) -> bool:
            if type(value) is not bool:
                raise OverburdenConfigError(
                    f"{field_name} must be a boolean, got "
                    f"{type(value).__name__} {value!r}.")
            return value

        def require_number(field_name: str, value: object) -> float:
            if (isinstance(value, (bool, np.bool_))
                    or not isinstance(value, (int, float, np.integer, np.floating))):
                raise OverburdenConfigError(
                    f"{field_name} must be a finite real number, got "
                    f"{type(value).__name__} {value!r}.")
            try:
                out = float(value)
            except (TypeError, ValueError, OverflowError) as exc:
                raise OverburdenConfigError(
                    f"{field_name} must be a finite real number, got "
                    f"{type(value).__name__} {value!r}.") from exc
            if not math.isfinite(out):
                raise OverburdenConfigError(
                    f"{field_name} must be finite, got {value!r}.")
            return out

        for field_name in (
            "source_filename", "schema_version", "assurance_tier",
            "canonical_curve_name", "accepted_canonical_unit",
            "expected_conversion_function", "integration_method",
            "seabed_marker_name",
        ):
            require_string(field_name, getattr(self, field_name))

        if self.schema_version != "7.0.1":
            raise OverburdenConfigError(
                f"schema_version must be exactly '7.0.1', got {self.schema_version!r}.")
        if type(self.increment) is not int or self.increment != 7:
            raise OverburdenConfigError(
                f"increment must be the integer 7, got {self.increment!r}.")
        if "vertical_depth" not in self.integration_method:
            raise OverburdenConfigError(
                "integration_method must declare a vertical-depth formulation.")

        for field_name in (
            "bounds_are_inclusive", "bridge_short_internal_gaps",
            "scenarios_enabled", "absolute_requires_uninterrupted_column",
        ):
            require_bool(field_name, getattr(self, field_name))

        for field_name in (
            "rhob_min_kg_m3", "rhob_max_kg_m3", "short_gap_max_tvd_m",
            "shallow_gap_tolerance_tvd_m", "gravity_m_s2",
            "profile_report_step_tvdss_m", "seawater_density_kg_m3",
            "seawater_density_low_kg_m3", "seawater_density_high_kg_m3",
            "scenario_high_percentile",
        ):
            object.__setattr__(
                self, field_name, require_number(field_name, getattr(self, field_name)))

        if type(self.sensitivity_thresholds_tvd_m) is not tuple:
            raise OverburdenConfigError(
                "sensitivity_thresholds_tvd_m must be a non-empty tuple of finite "
                "non-negative numbers in strictly increasing order.")
        if not self.sensitivity_thresholds_tvd_m:
            raise OverburdenConfigError(
                "sensitivity_thresholds_tvd_m must be a non-empty tuple.")
        thresholds = tuple(
            require_number("sensitivity_thresholds_tvd_m", value)
            for value in self.sensitivity_thresholds_tvd_m)
        if any(value < 0.0 for value in thresholds):
            raise OverburdenConfigError(
                "sensitivity_thresholds_tvd_m values must all be >= 0.")
        if any(right <= left for left, right in zip(thresholds, thresholds[1:])):
            raise OverburdenConfigError(
                "sensitivity_thresholds_tvd_m must be unique and strictly increasing.")
        object.__setattr__(self, "sensitivity_thresholds_tvd_m", thresholds)

        if type(self.scenario_names) is not tuple or self.scenario_names != (
                SCENARIO_LOW, SCENARIO_BASE, SCENARIO_HIGH):
            raise OverburdenConfigError(
                "scenario_names must be exactly ('low', 'base', 'high').")
        if (type(self.min_eligible_samples_for_increment) is not int
                or isinstance(self.min_eligible_samples_for_increment, bool)):
            raise OverburdenConfigError(
                "min_eligible_samples_for_increment must be a genuine integer, "
                f"got {type(self.min_eligible_samples_for_increment).__name__} "
                f"{self.min_eligible_samples_for_increment!r}.")
        if type(self.not_implemented) is not tuple or not self.not_implemented:
            raise OverburdenConfigError(
                "not_implemented must be a non-empty tuple of non-empty strings.")
        if any(not isinstance(value, str) or not value.strip()
               for value in self.not_implemented):
            raise OverburdenConfigError(
                "not_implemented entries must all be non-empty strings.")
        if len(set(self.not_implemented)) != len(self.not_implemented):
            raise OverburdenConfigError("not_implemented contains duplicate entries.")

        if self.rhob_min_kg_m3 >= self.rhob_max_kg_m3:
            raise OverburdenConfigError(
                f"screening_bounds: rhob_min_kg_m3 ({self.rhob_min_kg_m3}) must be strictly "
                f"less than rhob_max_kg_m3 ({self.rhob_max_kg_m3}).")
        if self.short_gap_max_tvd_m < 0.0:
            raise OverburdenConfigError(
                "gap_conditioning: short_gap_max_tvd_m must be >= 0.")
        if self.shallow_gap_tolerance_tvd_m < 0.0:
            raise OverburdenConfigError(
                "gap_conditioning: shallow_gap_tolerance_tvd_m must be >= 0.")
        if self.short_gap_max_tvd_m not in self.sensitivity_thresholds_tvd_m:
            raise OverburdenConfigError(
                f"gap_conditioning: the approved short_gap_max_tvd_m "
                f"({self.short_gap_max_tvd_m}) must appear in "
                f"sensitivity_thresholds_tvd_m {list(self.sensitivity_thresholds_tvd_m)}, "
                f"so that the approved case is always one of the reported cases.")
        if self.gravity_m_s2 <= 0.0:
            raise OverburdenConfigError("integration: gravity_m_s2 must be > 0.")
        if not (self.seawater_density_low_kg_m3
                <= self.seawater_density_kg_m3
                <= self.seawater_density_high_kg_m3):
            raise OverburdenConfigError(
                "water_column: seawater_density_kg_m3 must lie within "
                "[seawater_density_low_kg_m3, seawater_density_high_kg_m3].")
        if self.scenario_high_percentile != 5.0:
            raise OverburdenConfigError(
                "shallow_column_scenarios: high_percentile must be exactly 5.0, "
                "because Increment 7 stores and uses the measured P05 statistic; "
                "accepting another value would misstate which percentile supplied "
                "the high scenario.")
        if self.min_eligible_samples_for_increment < 2:
            raise OverburdenConfigError(
                "eligibility: min_eligible_samples_for_increment must be >= 2 (two samples "
                "is the arithmetic minimum for one trapezoid).")
        if self.profile_report_step_tvdss_m <= 0.0:
            raise OverburdenConfigError(
                "integration: profile_report_step_tvdss_m must be > 0.")


# ---------------------------------------------------------------------------
# Per-sample masks
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class DensityMaskSet:
    """The complete set of explicit, separately auditable per-sample masks.

    Every mask is a full-length read-only boolean array aligned
    sample-for-sample with the well's canonical `MD_m`, in original file
    order. NOTHING in this structure holds a density value: a mask records
    what is true about a sample, never a substitute for it.

    `below_seabed_sample` is `None` - not an all-False or all-True array -
    when the well has no approved formation-top file and therefore no seabed
    marker. `None` means NOT DETERMINABLE. An all-False array would mean
    "every sample is above the seabed", and an all-True array would mean
    "every sample is below it"; both would be assertions this project cannot
    make, and the eligibility conjunction would silently absorb either.
    """

    well_key: str
    n_samples: int
    seabed_resolved: bool

    source_value_present: np.ndarray
    finite_numeric_density: np.ndarray
    unit_resolved: np.ndarray
    screening_range_plausible: np.ndarray
    below_seabed_sample: Optional[np.ndarray]
    depth_mapping_valid: np.ndarray
    within_survey_coverage: np.ndarray
    eligible_for_measured_integration: np.ndarray
    bridged_short_gap: np.ndarray
    unresolved_long_gap: np.ndarray
    unresolved_shallow_column: np.ndarray
    unresolved_terminal_column: np.ndarray

    def __post_init__(self) -> None:
        n = _require_nonneg_int(self.n_samples, f"DensityMaskSet {self.well_key!r}: n_samples")
        for name in DENSITY_MASK_NAMES:
            arr = getattr(self, name)
            if arr is None:
                if name != "below_seabed_sample":
                    raise OverburdenInputError(
                        f"DensityMaskSet {self.well_key!r}: mask {name!r} may not be None.")
                if self.seabed_resolved:
                    raise OverburdenInputError(
                        f"DensityMaskSet {self.well_key!r}: below_seabed_sample is None but "
                        f"seabed_resolved is True; a resolved seabed must produce a mask.")
                continue
            if not isinstance(arr, np.ndarray) or arr.dtype != np.bool_:
                raise OverburdenInputError(
                    f"DensityMaskSet {self.well_key!r}: mask {name!r} must be a boolean "
                    f"numpy array, got {type(arr).__name__} dtype "
                    f"{getattr(arr, 'dtype', None)!r}.")
            if arr.ndim != 1 or arr.size != n:
                raise OverburdenInputError(
                    f"DensityMaskSet {self.well_key!r}: mask {name!r} has shape {arr.shape}, "
                    f"expected a 1-D array of {n} sample(s).")
            if arr.flags.writeable:
                raise OverburdenInputError(
                    f"DensityMaskSet {self.well_key!r}: mask {name!r} must be read-only.")
        if self.below_seabed_sample is None and self.seabed_resolved:
            raise OverburdenInputError(
                f"DensityMaskSet {self.well_key!r}: seabed_resolved/below_seabed_sample "
                f"disagree.")
        if self.below_seabed_sample is not None and not self.seabed_resolved:
            raise OverburdenInputError(
                f"DensityMaskSet {self.well_key!r}: below_seabed_sample is present but "
                f"seabed_resolved is False.")

        # The eligibility mask is a CONJUNCTION. It can never be true where a
        # constituent condition is false, and it can never be true for a
        # sample that is simultaneously declared unresolved.
        elig = self.eligible_for_measured_integration
        for name in ("finite_numeric_density", "unit_resolved",
                     "screening_range_plausible", "depth_mapping_valid",
                     "within_survey_coverage"):
            if bool(np.any(elig & ~getattr(self, name))):
                raise OverburdenInputError(
                    f"DensityMaskSet {self.well_key!r}: eligible_for_measured_integration is "
                    f"True where {name} is False; the eligibility mask must be a conjunction "
                    f"of its declared constituents.")
        if self.below_seabed_sample is not None and bool(
                np.any(elig & ~self.below_seabed_sample)):
            raise OverburdenInputError(
                f"DensityMaskSet {self.well_key!r}: eligible_for_measured_integration is True "
                f"above the resolved seabed.")
        for name in ("unresolved_long_gap", "unresolved_shallow_column",
                     "unresolved_terminal_column"):
            if bool(np.any(elig & getattr(self, name))):
                raise OverburdenInputError(
                    f"DensityMaskSet {self.well_key!r}: a sample is both eligible and "
                    f"{name}; these are mutually exclusive by construction.")
        if bool(np.any(self.bridged_short_gap & elig)):
            raise OverburdenInputError(
                f"DensityMaskSet {self.well_key!r}: a bridged sample is also marked as an "
                f"eligible MEASURED sample; a bridged value is conditioned, not measured.")
        if bool(np.any(self.bridged_short_gap & self.unresolved_long_gap)):
            raise OverburdenInputError(
                f"DensityMaskSet {self.well_key!r}: a sample is both bridged and inside an "
                f"unresolved long gap.")
        if bool(np.any(self.finite_numeric_density & ~self.source_value_present)):
            raise OverburdenInputError(
                f"DensityMaskSet {self.well_key!r}: a finite density exists where no source "
                f"value is present.")

    def counts(self) -> Dict[str, Optional[int]]:
        """Return `{mask_name: n_true}`, with `None` for a not-determinable mask."""
        out: Dict[str, Optional[int]] = {}
        for name in DENSITY_MASK_NAMES:
            arr = getattr(self, name)
            out[name] = None if arr is None else int(np.count_nonzero(arr))
        return out


# ---------------------------------------------------------------------------
# QC statistics
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class DensityQcStats:
    """Factual, descriptive density QC for ONE well.

    Every field is a MEASUREMENT about the curve as recorded, or a count of
    samples failing an explicitly configured screening criterion. Nothing here
    is an interpretation, and nothing here modifies the curve.
    """

    well_key: str
    source_las_filename: str
    curve_present: bool
    canonical_curve_name: Optional[str]
    source_curve_name: Optional[str]
    raw_mnemonic: Optional[str]
    raw_unit: Optional[str]
    canonical_unit: Optional[str]
    conversion_function: Optional[str]
    unit_resolved: bool
    conversion_confirmed: bool

    n_samples: int
    n_source_present: int
    n_finite: int
    n_non_finite: int
    n_non_positive: int
    n_below_screening_min: int
    n_above_screening_max: int
    n_screening_bound_failures: int
    n_in_screening_band: int
    n_depth_mapped: int
    n_depth_unmapped: int
    n_eligible: int

    rhob_min_kg_m3: Optional[float]
    rhob_max_kg_m3: Optional[float]
    rhob_median_kg_m3: Optional[float]
    rhob_p05_kg_m3: Optional[float]
    rhob_p95_kg_m3: Optional[float]
    rhob_eligible_p05_kg_m3: Optional[float]

    first_valid_md_m: Optional[float]
    last_valid_md_m: Optional[float]
    first_valid_tvd_m: Optional[float]
    last_valid_tvd_m: Optional[float]
    first_valid_tvdss_m: Optional[float]
    last_valid_tvdss_m: Optional[float]
    gross_coverage_md_m: Optional[float]
    gross_coverage_tvd_m: Optional[float]
    median_md_step_m: Optional[float]

    seabed_basis: str
    seabed_mdrt_m: Optional[float]
    seabed_tvd_m: Optional[float]
    seabed_tvdss_m: Optional[float]
    shallow_gap_md_m: Optional[float]
    shallow_gap_tvd_m: Optional[float]
    terminal_gap_md_m: Optional[float]
    terminal_gap_tvd_m: Optional[float]

    n_internal_gaps: int
    n_internal_gap_samples: int
    longest_internal_gap_md_m: Optional[float]
    longest_internal_gap_tvd_m: Optional[float]

    depth_basis_used: str
    depth_map_status: str
    survey_md_min_m: float
    survey_md_max_m: float
    n_samples_outside_survey_coverage: int

    def __post_init__(self) -> None:
        _require_in(self.seabed_basis, VALID_SEABED_BASES,
                    f"DensityQcStats {self.well_key!r}: seabed_basis")
        for name in ("n_samples", "n_source_present", "n_finite", "n_non_finite",
                     "n_non_positive", "n_below_screening_min", "n_above_screening_max",
                     "n_screening_bound_failures", "n_in_screening_band", "n_depth_mapped",
                     "n_depth_unmapped", "n_eligible", "n_internal_gaps",
                     "n_internal_gap_samples", "n_samples_outside_survey_coverage"):
            _require_nonneg_int(getattr(self, name),
                                f"DensityQcStats {self.well_key!r}: {name}")
        if self.n_finite + self.n_non_finite != self.n_samples:
            raise OverburdenInputError(
                f"DensityQcStats {self.well_key!r}: n_finite + n_non_finite "
                f"({self.n_finite} + {self.n_non_finite}) must equal n_samples "
                f"({self.n_samples}).")
        if self.n_depth_mapped + self.n_depth_unmapped != self.n_samples:
            raise OverburdenInputError(
                f"DensityQcStats {self.well_key!r}: n_depth_mapped + n_depth_unmapped must "
                f"equal n_samples.")
        expected_failures = (self.n_below_screening_min + self.n_above_screening_max
                             + self.n_non_positive)
        if self.n_screening_bound_failures != expected_failures:
            raise OverburdenInputError(
                f"DensityQcStats {self.well_key!r}: n_screening_bound_failures "
                f"({self.n_screening_bound_failures}) must equal n_below_screening_min + "
                f"n_above_screening_max + n_non_positive ({expected_failures}).")
        if self.n_in_screening_band + self.n_screening_bound_failures != self.n_finite:
            raise OverburdenInputError(
                f"DensityQcStats {self.well_key!r}: in-band and bound-failure counts must "
                f"partition the finite samples.")
        if self.n_eligible > self.n_in_screening_band:
            raise OverburdenInputError(
                f"DensityQcStats {self.well_key!r}: n_eligible ({self.n_eligible}) cannot "
                f"exceed n_in_screening_band ({self.n_in_screening_band}).")
        if self.curve_present and self.canonical_curve_name is None:
            raise OverburdenInputError(
                f"DensityQcStats {self.well_key!r}: curve_present is True but "
                f"canonical_curve_name is None.")
        if not self.curve_present and self.n_finite != 0:
            raise OverburdenInputError(
                f"DensityQcStats {self.well_key!r}: curve_present is False but "
                f"n_finite is {self.n_finite}.")
        finite_stats = (
            self.rhob_min_kg_m3, self.rhob_max_kg_m3, self.rhob_median_kg_m3,
            self.rhob_p05_kg_m3, self.rhob_p95_kg_m3,
        )
        if self.n_finite == 0 and any(v is not None for v in finite_stats):
            raise OverburdenInputError(
                f"DensityQcStats {self.well_key!r}: finite-population statistics must "
                f"all be None when n_finite is zero.")
        if self.n_finite > 0:
            for name in ("rhob_min_kg_m3", "rhob_max_kg_m3", "rhob_median_kg_m3",
                         "rhob_p05_kg_m3", "rhob_p95_kg_m3"):
                _require_finite_float(
                    getattr(self, name), f"DensityQcStats {self.well_key!r}: {name}")
        if self.n_eligible == 0 and self.rhob_eligible_p05_kg_m3 is not None:
            raise OverburdenInputError(
                f"DensityQcStats {self.well_key!r}: rhob_eligible_p05_kg_m3 must be "
                f"None when n_eligible is zero.")
        if self.n_eligible > 0:
            _require_finite_float(
                self.rhob_eligible_p05_kg_m3,
                f"DensityQcStats {self.well_key!r}: rhob_eligible_p05_kg_m3")
        if (self.seabed_basis == SEABED_BASIS_NOT_DETERMINABLE
                and self.seabed_mdrt_m is not None):
            raise OverburdenInputError(
                f"DensityQcStats {self.well_key!r}: seabed_basis says not determinable but a "
                f"seabed depth is present.")
        if (self.seabed_basis == SEABED_BASIS_LOCKED_MARKER
                and self.seabed_mdrt_m is None):
            raise OverburdenInputError(
                f"DensityQcStats {self.well_key!r}: seabed_basis cites the locked marker but "
                f"no seabed depth is present.")


# ---------------------------------------------------------------------------
# Gaps
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class DensityGapRecord:
    """ONE classified density gap in ONE well.

    A gap is a factual description of an interval with no eligible measured
    density. `gap_class` distinguishes the five structurally different cases
    the Increment 7 policy requires to be kept apart, plus the depth-mapping
    case; `disposition` records what was actually done about it.
    """

    well_key: str
    gap_index: int
    gap_class: str
    disposition: str
    n_samples: int
    start_index: Optional[int]
    end_index: Optional[int]
    md_start_m: Optional[float]
    md_end_m: Optional[float]
    tvd_start_m: Optional[float]
    tvd_end_m: Optional[float]
    thickness_md_m: Optional[float]
    thickness_tvd_m: Optional[float]
    threshold_tvd_m: Optional[float]
    bounding_density_above_kg_m3: Optional[float]
    bounding_density_below_kg_m3: Optional[float]

    def __post_init__(self) -> None:
        _require_in(self.gap_class, VALID_GAP_CLASSES,
                    f"DensityGapRecord {self.well_key!r}: gap_class")
        _require_in(self.disposition, VALID_GAP_DISPOSITIONS,
                    f"DensityGapRecord {self.well_key!r}: disposition")
        _require_nonneg_int(self.gap_index, f"DensityGapRecord {self.well_key!r}: gap_index")
        _require_nonneg_int(self.n_samples, f"DensityGapRecord {self.well_key!r}: n_samples")
        if self.threshold_tvd_m is not None:
            threshold = _require_finite_float(
                self.threshold_tvd_m,
                f"DensityGapRecord {self.well_key!r}: threshold_tvd_m")
            if threshold < 0.0:
                raise OverburdenInputError(
                    f"DensityGapRecord {self.well_key!r}: threshold_tvd_m must be >= 0, "
                    f"got {threshold}.")
        for name in ("thickness_md_m", "thickness_tvd_m"):
            value = getattr(self, name)
            if value is not None and value < 0.0:
                raise OverburdenInputError(
                    f"DensityGapRecord {self.well_key!r}: {name} must be >= 0, got {value}.")
        if self.disposition == GAP_DISPOSITION_BRIDGED:
            if self.gap_class != GAP_CLASS_SHORT_INTERNAL:
                raise OverburdenInputError(
                    f"DensityGapRecord {self.well_key!r}: only a "
                    f"{GAP_CLASS_SHORT_INTERNAL!r} may be bridged, not {self.gap_class!r}. "
                    f"Bridging a shallow, terminal, long or unmapped gap is prohibited.")
            if self.n_samples < 1:
                raise OverburdenInputError(
                    f"DensityGapRecord {self.well_key!r}: a bridged gap must contain at "
                    f"least one conditioned sample.")
            if (self.bounding_density_above_kg_m3 is None
                    or self.bounding_density_below_kg_m3 is None):
                raise OverburdenInputError(
                    f"DensityGapRecord {self.well_key!r}: a bridged gap must record both "
                    f"bracketing measured densities.")
        if (self.gap_class == GAP_CLASS_SHORT_INTERNAL
                and self.threshold_tvd_m is not None
                and self.thickness_tvd_m is not None
                and self.thickness_tvd_m > self.threshold_tvd_m):
            raise OverburdenInputError(
                f"DensityGapRecord {self.well_key!r}: a gap classified short "
                f"({self.thickness_tvd_m} m TVD) exceeds the threshold "
                f"({self.threshold_tvd_m} m TVD).")
        if (self.gap_class == GAP_CLASS_LONG_INTERNAL
                and self.threshold_tvd_m is not None
                and self.thickness_tvd_m is not None
                and self.thickness_tvd_m <= self.threshold_tvd_m):
            raise OverburdenInputError(
                f"DensityGapRecord {self.well_key!r}: a gap classified long "
                f"({self.thickness_tvd_m} m TVD) does not exceed the threshold "
                f"({self.threshold_tvd_m} m TVD).")


@dataclass(frozen=True)
class GapConditioningResult:
    """The outcome of applying the configured gap-conditioning policy to ONE well.

    `conditioned_density_kg_m3` is a SEPARATELY NAMED array. The original
    resolved RHOB array is never touched; a caller holding both can diff them
    and see exactly which samples were conditioned, and `bridged_mask` names
    those samples independently.
    """

    well_key: str
    threshold_tvd_m: float
    bridging_enabled: bool
    conditioned_density_kg_m3: np.ndarray
    bridged_mask: np.ndarray
    n_bridged_gaps: int
    n_bridged_samples: int
    bridged_thickness_md_m: float
    bridged_thickness_tvd_m: float
    n_long_gaps: int
    n_long_gap_samples: int
    long_gap_thickness_md_m: float
    long_gap_thickness_tvd_m: float
    gaps: Tuple[DensityGapRecord, ...] = field(default_factory=tuple)

    def __post_init__(self) -> None:
        threshold = _require_finite_float(
            self.threshold_tvd_m,
            f"GapConditioningResult {self.well_key!r}: threshold_tvd_m")
        if threshold < 0.0:
            raise OverburdenInputError(
                f"GapConditioningResult {self.well_key!r}: threshold_tvd_m must be >= 0, "
                f"got {threshold}.")
        if type(self.bridging_enabled) is not bool:
            raise OverburdenInputError(
                f"GapConditioningResult {self.well_key!r}: bridging_enabled must be a "
                f"boolean, got {type(self.bridging_enabled).__name__}.")
        for name in ("n_bridged_gaps", "n_bridged_samples", "n_long_gaps",
                     "n_long_gap_samples"):
            _require_nonneg_int(getattr(self, name),
                                f"GapConditioningResult {self.well_key!r}: {name}")
        if not isinstance(self.conditioned_density_kg_m3, np.ndarray):
            raise OverburdenInputError(
                f"GapConditioningResult {self.well_key!r}: conditioned density must be a "
                f"numpy array.")
        if self.conditioned_density_kg_m3.flags.writeable:
            raise OverburdenInputError(
                f"GapConditioningResult {self.well_key!r}: conditioned density array must be "
                f"read-only.")
        if self.bridged_mask.dtype != np.bool_:
            raise OverburdenInputError(
                f"GapConditioningResult {self.well_key!r}: bridged_mask must be boolean.")
        if self.bridged_mask.size != self.conditioned_density_kg_m3.size:
            raise OverburdenInputError(
                f"GapConditioningResult {self.well_key!r}: bridged_mask and conditioned "
                f"density lengths differ.")
        if int(np.count_nonzero(self.bridged_mask)) != self.n_bridged_samples:
            raise OverburdenInputError(
                f"GapConditioningResult {self.well_key!r}: n_bridged_samples "
                f"({self.n_bridged_samples}) disagrees with bridged_mask "
                f"({int(np.count_nonzero(self.bridged_mask))}).")
        # A gap count and a sample count must move together: n gaps with zero
        # samples, or zero gaps with n samples, is an incoherent record.
        if (self.n_bridged_gaps == 0) != (self.n_bridged_samples == 0):
            raise OverburdenInputError(
                f"GapConditioningResult {self.well_key!r}: n_bridged_gaps "
                f"({self.n_bridged_gaps}) and n_bridged_samples "
                f"({self.n_bridged_samples}) must both be zero or both be non-zero.")
        if self.n_bridged_gaps and self.n_bridged_samples < self.n_bridged_gaps:
            raise OverburdenInputError(
                f"GapConditioningResult {self.well_key!r}: {self.n_bridged_gaps} bridged "
                f"gap(s) cannot contain only {self.n_bridged_samples} sample(s).")
        if (self.n_long_gaps == 0) != (self.n_long_gap_samples == 0):
            raise OverburdenInputError(
                f"GapConditioningResult {self.well_key!r}: n_long_gaps and "
                f"n_long_gap_samples must both be zero or both be non-zero.")
        if self.n_long_gaps and self.n_long_gap_samples < self.n_long_gaps:
            raise OverburdenInputError(
                f"GapConditioningResult {self.well_key!r}: {self.n_long_gaps} long gap(s) "
                f"cannot contain only {self.n_long_gap_samples} sample(s).")
        if not self.bridging_enabled and self.n_bridged_gaps:
            raise OverburdenInputError(
                f"GapConditioningResult {self.well_key!r}: bridging is disabled but "
                f"{self.n_bridged_gaps} gap(s) were bridged.")
        for name in ("bridged_thickness_md_m", "bridged_thickness_tvd_m",
                     "long_gap_thickness_md_m", "long_gap_thickness_tvd_m"):
            value = _require_finite_float(
                getattr(self, name), f"GapConditioningResult {self.well_key!r}: {name}")
            if value < 0.0:
                raise OverburdenInputError(
                    f"GapConditioningResult {self.well_key!r}: {name} must be >= 0.")
        n_bridged_records = sum(1 for g in self.gaps
                                if g.disposition == GAP_DISPOSITION_BRIDGED)
        if n_bridged_records != self.n_bridged_gaps:
            raise OverburdenInputError(
                f"GapConditioningResult {self.well_key!r}: {n_bridged_records} bridged gap "
                f"record(s) but n_bridged_gaps is {self.n_bridged_gaps}.")


# ---------------------------------------------------------------------------
# Stress
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class StressPartition:
    """The separated contributions to a reported vertical stress, in pascals.

    The partition exists so that a reader can always see how much of a number
    is measurement and how much is assumption. `measured_formation_pa` is the
    only component derived exclusively from recorded density; every other
    component depends on a configured assumption, an interpolation, or both.
    """

    water_column_pa: Optional[float]
    measured_formation_pa: float
    bridged_gap_pa: float
    unresolved_shallow_pa: Optional[float]
    total_pa: Optional[float]

    def __post_init__(self) -> None:
        _require_finite_float(self.water_column_pa, "StressPartition: water_column_pa",
                              allow_none=True)
        _require_finite_float(self.measured_formation_pa,
                              "StressPartition: measured_formation_pa")
        _require_finite_float(self.bridged_gap_pa, "StressPartition: bridged_gap_pa")
        _require_finite_float(self.unresolved_shallow_pa,
                              "StressPartition: unresolved_shallow_pa", allow_none=True)
        _require_finite_float(self.total_pa, "StressPartition: total_pa", allow_none=True)
        for name in ("water_column_pa", "measured_formation_pa", "bridged_gap_pa",
                     "unresolved_shallow_pa", "total_pa"):
            value = getattr(self, name)
            if value is not None and value < 0.0:
                raise OverburdenInputError(
                    f"StressPartition: {name} must be >= 0 (a downward column cannot remove "
                    f"vertical stress), got {value}.")
        if self.total_pa is not None:
            parts = [self.water_column_pa, self.measured_formation_pa, self.bridged_gap_pa,
                     self.unresolved_shallow_pa]
            if any(p is None for p in parts):
                raise OverburdenInputError(
                    "StressPartition: a total may not be reported while any component is "
                    "unresolved; an unresolved component is not zero.")
            expected = sum(parts)  # type: ignore[arg-type]
            if not math.isclose(expected, self.total_pa, rel_tol=1e-12, abs_tol=1e-6):
                raise OverburdenInputError(
                    f"StressPartition: components sum to {expected} Pa but total_pa is "
                    f"{self.total_pa} Pa.")


@dataclass(frozen=True)
class VerticalStressProfile:
    """The measured vertical-stress increment profile for ONE well.

    `tvdss_m` is the vertical coordinate the integral was actually taken in.
    `cumulative_measured_increment_pa[i]` is the integral of rho*g*dz from the
    FIRST eligible sample down to sample `i`; it is an INCREMENT, not an
    absolute stress, and it is zero at the first node by definition.
    """

    well_key: str
    n_nodes: int
    integration_coordinate: str
    gravity_m_s2: float
    md_m: np.ndarray
    tvd_m: np.ndarray
    tvdss_m: np.ndarray
    density_kg_m3: np.ndarray
    bridged_mask: np.ndarray
    cumulative_measured_increment_pa: np.ndarray
    total_measured_increment_pa: float
    total_bridged_increment_pa: float
    n_zero_thickness_intervals: int
    n_intervals: int
    top_tvd_m: float
    base_tvd_m: float
    top_tvdss_m: float
    base_tvdss_m: float
    #: True when the integrable column was cut short at an unresolved long
    #: gap, so the profile's base is ABOVE the well's deepest eligible
    #: density sample. Increment 7 never integrates through an unresolved
    #: gap, and never silently joins two columns that are not adjacent.
    column_truncated_at_unresolved_gap: bool
    #: Eligible density samples that exist BELOW the truncation and were
    #: therefore excluded from this profile. Zero when nothing was truncated.
    n_eligible_samples_below_truncation: int

    def __post_init__(self) -> None:
        n = _require_nonneg_int(self.n_nodes,
                                f"VerticalStressProfile {self.well_key!r}: n_nodes")
        if n < 2:
            raise OverburdenInputError(
                f"VerticalStressProfile {self.well_key!r}: a profile requires at least two "
                f"nodes, got {n}.")
        for name in ("md_m", "tvd_m", "tvdss_m", "density_kg_m3", "bridged_mask",
                     "cumulative_measured_increment_pa"):
            arr = getattr(self, name)
            if not isinstance(arr, np.ndarray) or arr.ndim != 1 or arr.size != n:
                raise OverburdenInputError(
                    f"VerticalStressProfile {self.well_key!r}: {name} must be a 1-D array of "
                    f"{n} element(s).")
            if arr.flags.writeable:
                raise OverburdenInputError(
                    f"VerticalStressProfile {self.well_key!r}: {name} must be read-only.")
        if self.integration_coordinate != "tvdss_m":
            raise OverburdenInputError(
                f"VerticalStressProfile {self.well_key!r}: integration_coordinate must be "
                f"'tvdss_m'; measured-depth integration is not representable here.")
        if int(self.n_intervals) != n - 1:
            raise OverburdenInputError(
                f"VerticalStressProfile {self.well_key!r}: n_intervals must equal n_nodes-1.")
        _require_nonneg_int(self.n_zero_thickness_intervals,
                            f"VerticalStressProfile {self.well_key!r}: "
                            f"n_zero_thickness_intervals")
        if self.n_zero_thickness_intervals > self.n_intervals:
            raise OverburdenInputError(
                f"VerticalStressProfile {self.well_key!r}: more zero-thickness intervals "
                f"than intervals.")
        cum = self.cumulative_measured_increment_pa
        if float(cum[0]) != 0.0:
            raise OverburdenInputError(
                f"VerticalStressProfile {self.well_key!r}: the cumulative increment must be "
                f"exactly zero at the first node, got {float(cum[0])!r}.")
        if bool(np.any(np.diff(cum) < 0.0)):
            raise OverburdenInputError(
                f"VerticalStressProfile {self.well_key!r}: the cumulative increment must be "
                f"non-decreasing downward.")
        if not math.isclose(float(cum[-1]), float(self.total_measured_increment_pa),
                            rel_tol=1e-12, abs_tol=1e-6):
            raise OverburdenInputError(
                f"VerticalStressProfile {self.well_key!r}: total_measured_increment_pa does "
                f"not equal the final cumulative value.")
        if self.total_bridged_increment_pa < 0.0:
            raise OverburdenInputError(
                f"VerticalStressProfile {self.well_key!r}: total_bridged_increment_pa must "
                f"be >= 0.")
        if self.total_bridged_increment_pa > self.total_measured_increment_pa + 1e-6:
            raise OverburdenInputError(
                f"VerticalStressProfile {self.well_key!r}: the bridged contribution cannot "
                f"exceed the whole integrated increment.")
        _require_nonneg_int(
            self.n_eligible_samples_below_truncation,
            f"VerticalStressProfile {self.well_key!r}: n_eligible_samples_below_truncation")
        if not isinstance(self.column_truncated_at_unresolved_gap, bool):
            raise OverburdenInputError(
                f"VerticalStressProfile {self.well_key!r}: "
                f"column_truncated_at_unresolved_gap must be a bool.")
        if (self.n_eligible_samples_below_truncation > 0
                and not self.column_truncated_at_unresolved_gap):
            raise OverburdenInputError(
                f"VerticalStressProfile {self.well_key!r}: eligible samples are reported "
                f"below a truncation that is not declared.")


@dataclass(frozen=True)
class ShallowColumnScenario:
    """ONE transparent screening scenario for the UNMEASURED shallow column.

    This is not an estimate of the true stress. It is a statement of the form
    "if the depth-averaged bulk density of the unmeasured column were X, the
    total would be Y, of which fraction F originates in the assumption".
    """

    well_key: str
    scenario_name: str
    assumed_shallow_density_kg_m3: float
    assumed_density_basis: str
    seawater_density_kg_m3: float
    gravity_m_s2: float
    unresolved_thickness_tvd_m: float
    water_column_thickness_tvd_m: float
    measured_thickness_tvd_m: float
    partition: StressPartition
    assumed_fraction_of_total: float
    conditioned_fraction_of_total: float
    measured_fraction_of_total: float

    def __post_init__(self) -> None:
        _require_in(self.scenario_name, VALID_SCENARIO_NAMES,
                    f"ShallowColumnScenario {self.well_key!r}: scenario_name")
        for name in ("assumed_shallow_density_kg_m3", "seawater_density_kg_m3",
                     "gravity_m_s2"):
            value = _require_finite_float(
                getattr(self, name), f"ShallowColumnScenario {self.well_key!r}: {name}")
            if value <= 0.0:
                raise OverburdenInputError(
                    f"ShallowColumnScenario {self.well_key!r}: {name} must be > 0.")
        for name in ("unresolved_thickness_tvd_m", "water_column_thickness_tvd_m",
                     "measured_thickness_tvd_m"):
            value = _require_finite_float(
                getattr(self, name), f"ShallowColumnScenario {self.well_key!r}: {name}")
            if value < 0.0:
                raise OverburdenInputError(
                    f"ShallowColumnScenario {self.well_key!r}: {name} must be >= 0.")
        if self.partition.total_pa is None:
            raise OverburdenInputError(
                f"ShallowColumnScenario {self.well_key!r}: a scenario must produce a total; "
                f"if a component is unresolved the well is not scenario-eligible.")
        for name in ("assumed_fraction_of_total", "conditioned_fraction_of_total",
                     "measured_fraction_of_total"):
            value = _require_finite_float(
                getattr(self, name), f"ShallowColumnScenario {self.well_key!r}: {name}")
            if not (0.0 <= value <= 1.0):
                raise OverburdenInputError(
                    f"ShallowColumnScenario {self.well_key!r}: {name} must lie in [0, 1], "
                    f"got {value}.")
        fraction_sum = (self.assumed_fraction_of_total
                        + self.conditioned_fraction_of_total
                        + self.measured_fraction_of_total)
        if not math.isclose(fraction_sum, 1.0, rel_tol=1e-12, abs_tol=1e-12):
            raise OverburdenInputError(
                f"ShallowColumnScenario {self.well_key!r}: assumed, conditioned and "
                f"measured fractions must sum to 1, got {fraction_sum!r}.")


@dataclass(frozen=True)
class GapThresholdSensitivity:
    """What the approved gap threshold actually changed, for ONE well."""

    well_key: str
    threshold_tvd_m: float
    is_approved_threshold: bool
    n_bridged_gaps: int
    n_bridged_samples: int
    bridged_thickness_tvd_m: float
    n_long_gaps: int
    long_gap_thickness_tvd_m: float
    n_eligible_or_bridged_samples: int
    total_measured_increment_pa: Optional[float]
    bridged_increment_pa: Optional[float]
    derived_status: str

    def __post_init__(self) -> None:
        _require_in(self.derived_status, VALID_OVERBURDEN_STATUSES,
                    f"GapThresholdSensitivity {self.well_key!r}: derived_status")
        for name in ("n_bridged_gaps", "n_bridged_samples", "n_long_gaps",
                     "n_eligible_or_bridged_samples"):
            _require_nonneg_int(getattr(self, name),
                                f"GapThresholdSensitivity {self.well_key!r}: {name}")
        if (self.n_bridged_gaps == 0) != (self.n_bridged_samples == 0):
            raise OverburdenInputError(
                f"GapThresholdSensitivity {self.well_key!r}: bridged gap and sample counts "
                f"must both be zero or both be non-zero.")
        threshold = _require_finite_float(
            self.threshold_tvd_m,
            f"GapThresholdSensitivity {self.well_key!r}: threshold_tvd_m")
        if threshold < 0.0:
            raise OverburdenInputError(
                f"GapThresholdSensitivity {self.well_key!r}: threshold_tvd_m must be >= 0, "
                f"got {threshold}.")
        if type(self.is_approved_threshold) is not bool:
            raise OverburdenInputError(
                f"GapThresholdSensitivity {self.well_key!r}: is_approved_threshold must "
                f"be a boolean, got {type(self.is_approved_threshold).__name__}.")
        for name in ("bridged_thickness_tvd_m", "long_gap_thickness_tvd_m"):
            value = _require_finite_float(
                getattr(self, name), f"GapThresholdSensitivity {self.well_key!r}: {name}")
            if value < 0.0:
                raise OverburdenInputError(
                    f"GapThresholdSensitivity {self.well_key!r}: {name} must be >= 0, "
                    f"got {value}.")


@dataclass(frozen=True)
class OverburdenEligibility:
    """The DERIVED overburden method-eligibility verdict for ONE well.

    `status` is computed from measured coverage and QC results. No well name
    participates in the derivation, and no configuration key names a well.
    `limiting_reasons` is an ordered tuple of enumerated codes - never prose.
    """

    well_key: str
    status: str
    limiting_reasons: Tuple[str, ...]
    seabed_resolved: bool
    seabed_basis: str
    n_eligible_samples: int
    n_bridged_samples: int
    eligible_top_tvd_m: Optional[float]
    eligible_base_tvd_m: Optional[float]
    eligible_top_tvdss_m: Optional[float]
    eligible_base_tvdss_m: Optional[float]
    measured_thickness_tvd_m: Optional[float]
    shallow_unresolved_thickness_tvd_m: Optional[float]
    terminal_unresolved_thickness_tvd_m: Optional[float]
    water_column_thickness_tvd_m: Optional[float]
    n_unresolved_internal_gaps: int
    n_unresolved_long_gaps: int
    unresolved_long_gap_thickness_tvd_m: float
    column_uninterrupted: bool
    measured_increment_pa: Optional[float]
    bridged_increment_pa: Optional[float]
    absolute_stress_supported: bool

    def __post_init__(self) -> None:
        _require_in(self.status, VALID_OVERBURDEN_STATUSES,
                    f"OverburdenEligibility {self.well_key!r}: status")
        _require_in(self.seabed_basis, VALID_SEABED_BASES,
                    f"OverburdenEligibility {self.well_key!r}: seabed_basis")
        if not isinstance(self.limiting_reasons, tuple):
            raise OverburdenInputError(
                f"OverburdenEligibility {self.well_key!r}: limiting_reasons must be a tuple.")
        for reason in self.limiting_reasons:
            _require_in(reason, VALID_LIMITING_REASONS,
                        f"OverburdenEligibility {self.well_key!r}: limiting reason")
        if len(set(self.limiting_reasons)) != len(self.limiting_reasons):
            raise OverburdenInputError(
                f"OverburdenEligibility {self.well_key!r}: limiting_reasons contains "
                f"duplicates: {list(self.limiting_reasons)}.")
        if list(self.limiting_reasons) != sorted(self.limiting_reasons):
            raise OverburdenInputError(
                f"OverburdenEligibility {self.well_key!r}: limiting_reasons must be sorted "
                f"for determinism, got {list(self.limiting_reasons)}.")
        for name in ("n_eligible_samples", "n_bridged_samples",
                     "n_unresolved_internal_gaps", "n_unresolved_long_gaps"):
            _require_nonneg_int(getattr(self, name),
                                f"OverburdenEligibility {self.well_key!r}: {name}")

        if self.status == STATUS_ABSOLUTE:
            if self.limiting_reasons:
                raise OverburdenInputError(
                    f"OverburdenEligibility {self.well_key!r}: status {STATUS_ABSOLUTE!r} "
                    f"cannot carry limiting reason(s) {list(self.limiting_reasons)}.")
            if not self.absolute_stress_supported:
                raise OverburdenInputError(
                    f"OverburdenEligibility {self.well_key!r}: status {STATUS_ABSOLUTE!r} "
                    f"requires absolute_stress_supported to be True.")
            if not self.seabed_resolved or not self.column_uninterrupted:
                raise OverburdenInputError(
                    f"OverburdenEligibility {self.well_key!r}: an absolute status requires a "
                    f"resolved seabed and an uninterrupted column.")
        else:
            if self.absolute_stress_supported:
                raise OverburdenInputError(
                    f"OverburdenEligibility {self.well_key!r}: absolute_stress_supported is "
                    f"True but status is {self.status!r}.")
            if not self.limiting_reasons:
                raise OverburdenInputError(
                    f"OverburdenEligibility {self.well_key!r}: status {self.status!r} must "
                    f"state at least one enumerated limiting reason.")
        if self.status == STATUS_SENSITIVITY_ONLY and not self.seabed_resolved:
            raise OverburdenInputError(
                f"OverburdenEligibility {self.well_key!r}: {STATUS_SENSITIVITY_ONLY!r} "
                f"requires a resolved seabed, because the unresolved column's thickness must "
                f"be known before it can be bracketed.")
        # `not_eligible` is a statement about the INTEGRABLE column, not about
        # the raw eligible-sample count. A well can hold many eligible samples
        # and still support no measured increment - for example when an
        # unresolved long gap sits immediately below the shallowest eligible
        # sample, leaving fewer than two samples in the only column that could
        # ever be tied to the section above it. The coherent invariant is
        # therefore that no increment was produced.
        if self.status == STATUS_NOT_ELIGIBLE and self.measured_increment_pa is not None:
            raise OverburdenInputError(
                f"OverburdenEligibility {self.well_key!r}: {STATUS_NOT_ELIGIBLE!r} is "
                f"inconsistent with a reported measured increment of "
                f"{self.measured_increment_pa} Pa.")
        if self.status != STATUS_NOT_ELIGIBLE and self.measured_increment_pa is None:
            raise OverburdenInputError(
                f"OverburdenEligibility {self.well_key!r}: status {self.status!r} claims a "
                f"measured column but no increment was produced.")
        if self.seabed_resolved != (self.seabed_basis == SEABED_BASIS_LOCKED_MARKER):
            raise OverburdenInputError(
                f"OverburdenEligibility {self.well_key!r}: seabed_resolved and seabed_basis "
                f"disagree.")
        if self.n_unresolved_long_gaps > self.n_unresolved_internal_gaps:
            raise OverburdenInputError(
                f"OverburdenEligibility {self.well_key!r}: n_unresolved_long_gaps "
                f"({self.n_unresolved_long_gaps}) exceeds n_unresolved_internal_gaps "
                f"({self.n_unresolved_internal_gaps}).")
        if (self.n_unresolved_internal_gaps == 0) != self.column_uninterrupted:
            raise OverburdenInputError(
                f"OverburdenEligibility {self.well_key!r}: column_uninterrupted "
                f"({self.column_uninterrupted}) disagrees with "
                f"n_unresolved_internal_gaps ({self.n_unresolved_internal_gaps}).")


@dataclass(frozen=True)
class OverburdenIssue:
    """One operator-facing QC issue raised by the Increment 7 workflow."""

    severity: str
    code: str
    context: str
    message: str

    def __post_init__(self) -> None:
        _require_in(self.severity, ("INFO", "WARNING", "ERROR"),
                    "OverburdenIssue: severity")
        if not self.code or not self.code.replace("_", "").isalnum():
            raise OverburdenInputError(
                f"OverburdenIssue: code {self.code!r} must be a non-empty alphanumeric/"
                f"underscore token.")


#### `p2mem/density_qc.py` - availability, unit confirmation, twelve masks, gaps, conditioning

Never clips, rescales, smooths, despikes, replaces or extrapolates RHOB. The resolved density
array is read and never written; bridging writes only into a separately named array, only
inside the bracketed interior of the measured column, only across gaps at or below the
approved TVD threshold, and never across the seabed or outside locked survey coverage.

In [ ]:
%%writefile p2mem/density_qc.py
"""
p2mem.density_qc - RHOB availability, unit/provenance confirmation, explicit
validity masks, gap classification, and gap conditioning (Increment 7).

What this module does
---------------------
* Loads and validates `config/overburden_stress.yml`.
* Confirms, per well, that the locked Increment 2.1.1 curve contract resolved
  a density curve, under the accepted canonical unit, by the expected
  conversion function.
* Builds twelve explicit, separately auditable per-sample masks.
* Measures factual density QC statistics.
* Classifies every gap in the density column into the structurally distinct
  cases Increment 7 requires to be kept apart.
* Applies the configured gap-conditioning policy, producing a SEPARATELY
  NAMED conditioned array.

What this module never does
---------------------------
It never clips, rescales, smooths, despikes, replaces, or extrapolates RHOB.
The resolved density array handed in by the locked well frame is read and
never written. Bridging writes only into a new array, only inside the
bracketed interior of the measured column, only across gaps at or below the
configured TVD threshold, and never across the seabed or outside locked survey
coverage.
"""

from __future__ import annotations

from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import yaml

from p2mem.overburden_models import (
    GAP_CLASS_LONG_INTERNAL,
    GAP_CLASS_SHALLOW,
    GAP_CLASS_SHORT_INTERNAL,
    GAP_CLASS_TERMINAL,
    GAP_CLASS_UNMAPPED_DEPTH,
    GAP_DISPOSITION_BRIDGED,
    GAP_DISPOSITION_UNRESOLVED,
    SCENARIO_BASIS_BASE,
    SCENARIO_BASIS_HIGH,
    SCENARIO_BASIS_LOW,
    SEABED_BASIS_LOCKED_MARKER,
    SEABED_BASIS_NOT_DETERMINABLE,
    DensityGapRecord,
    DensityMaskSet,
    DensityQcStats,
    GapConditioningResult,
    OverburdenConfig,
    OverburdenConfigError,
    OverburdenInputError,
    VALID_OVERBURDEN_STATUSES,
    readonly,
)

__all__ = [
    "load_overburden_config",
    "resolve_density_slot",
    "build_density_masks",
    "compute_density_qc_stats",
    "classify_density_gaps",
    "condition_density_gaps",
    "DensityUnitError",
]


class DensityUnitError(TypeError):
    """Raised when a density curve's canonical unit is not the accepted unit.

    A `TypeError` subclass because an unexpected unit is a type-class defect
    in the quantity, not a value defect: Increment 7 performs no conversion of
    its own and will not integrate a quantity whose unit it cannot confirm.
    """


def _validated_gap_threshold(value, context: str) -> float:
    """Return one public gap-threshold override as a finite non-negative float.

    Configuration values already pass the frozen ``OverburdenConfig`` gate.
    Public per-call overrides require the same safety: no coercion from text,
    booleans or complex values, and no NaN/Inf value that could silently alter
    the short/long classification.
    """
    if (isinstance(value, (bool, np.bool_))
            or not isinstance(value, (int, float, np.integer, np.floating))):
        raise OverburdenInputError(
            f"{context}: threshold_tvd_m must be a finite non-negative real number, "
            f"got {type(value).__name__} {value!r}.")
    out = float(value)
    if not np.isfinite(out) or out < 0.0:
        raise OverburdenInputError(
            f"{context}: threshold_tvd_m must be finite and >= 0, got {out!r}.")
    return out


# ---------------------------------------------------------------------------
# Input validation helpers
# ---------------------------------------------------------------------------

def _reject_ambiguous_dtype(raw: np.ndarray, context: str) -> None:
    """Reject boolean, string/bytes, complex and otherwise non-numeric dtype
    input with `TypeError` before any numeric use.

    Documented local copy of the identical check established in the LOCKED
    `p2mem.units`, `p2mem.time_depth`, `p2mem.io.tops` and `p2mem.petrophysics`
    modules. Those modules remain unmodified; duplicating ~10 lines here is
    deliberately preferred over editing a locked module to export a private
    helper.
    """
    kind = raw.dtype.kind
    if kind == "b":
        raise TypeError(f"{context}: boolean input is not accepted as a numeric quantity.")
    if kind in ("U", "S"):
        raise TypeError(f"{context}: string/bytes input is not accepted as a numeric quantity.")
    if kind == "c":
        raise TypeError(f"{context}: complex input is not accepted as a numeric quantity.")
    if kind == "O":
        raise TypeError(f"{context}: object-dtype input is not accepted as a numeric quantity.")
    if kind not in ("i", "u", "f"):
        raise TypeError(f"{context}: unsupported array dtype {raw.dtype!r} for a numeric quantity.")


class _NoDuplicateKeySafeLoader(yaml.SafeLoader):
    """A SafeLoader that refuses duplicate mapping keys.

    Documented local copy of the loader the locked configuration modules
    already use. PyYAML's default silently keeps the last duplicate, which
    would let an edited config change a screening bound without any visible
    diff at the point of use.
    """


def _no_duplicate_keys(loader, node, deep=False):
    mapping = {}
    for key_node, value_node in node.value:
        key = loader.construct_object(key_node, deep=deep)
        if key in mapping:
            raise OverburdenConfigError(
                f"duplicate configuration key {key!r} at line "
                f"{key_node.start_mark.line + 1}; a duplicate key silently overrides an "
                f"earlier reviewed value.")
        mapping[key] = loader.construct_object(value_node, deep=deep)
    return mapping


_NoDuplicateKeySafeLoader.add_constructor(
    yaml.resolver.BaseResolver.DEFAULT_MAPPING_TAG, _no_duplicate_keys)


_REQUIRED_TOP_KEYS = (
    "schema_version", "increment", "assurance_tier", "density_source",
    "screening_bounds", "gap_conditioning", "integration", "water_column",
    "shallow_column_scenarios", "eligibility", "not_implemented",
)
_KNOWN_TOP_KEYS = frozenset(_REQUIRED_TOP_KEYS)

_SECTION_KEYS = {
    "density_source": ("canonical_curve_name", "accepted_canonical_unit",
                       "expected_conversion_function", "original_values_immutable"),
    "screening_bounds": ("rhob_min_kg_m3", "rhob_max_kg_m3", "bounds_are_inclusive"),
    "gap_conditioning": ("bridge_short_internal_gaps", "short_gap_max_tvd_m",
                         "bridge_across_seabed_allowed",
                         "bridge_outside_survey_coverage_allowed",
                         "bridge_long_gaps_allowed",
                         "shallow_gap_uses_internal_gap_rule",
                         "shallow_gap_tolerance_tvd_m",
                         "sensitivity_thresholds_tvd_m"),
    "integration": ("method", "gravity_m_s2", "zero_increment_contribution",
                    "negative_increment_policy", "profile_report_step_tvdss_m"),
    "water_column": ("seawater_density_kg_m3", "seawater_density_low_kg_m3",
                     "seawater_density_high_kg_m3", "seabed_source",
                     "seabed_marker_name", "cross_well_seabed_transfer_allowed"),
    "shallow_column_scenarios": ("enabled", "low_basis", "high_basis", "high_percentile",
                                 "base_basis", "base_is_not_a_best_estimate",
                                 "scenario_names",
                                 "require_assumed_fraction_disclosure"),
    "eligibility": ("statuses", "absolute_requires_uninterrupted_column",
                    "min_eligible_samples_for_increment"),
}


def _require_number(section: str, key: str, value, filename: str) -> float:
    if isinstance(value, bool):
        raise OverburdenConfigError(
            f"{filename}: {section}.{key} must be a number, not a boolean.")
    if not isinstance(value, (int, float)):
        raise OverburdenConfigError(
            f"{filename}: {section}.{key} must be a number, got "
            f"{type(value).__name__} {value!r}.")
    out = float(value)
    if out != out or out in (float("inf"), float("-inf")):
        raise OverburdenConfigError(
            f"{filename}: {section}.{key} must be finite, got {value!r}.")
    return out


def _require_bool(section: str, key: str, value, filename: str) -> bool:
    if not isinstance(value, bool):
        raise OverburdenConfigError(
            f"{filename}: {section}.{key} must be a boolean, got "
            f"{type(value).__name__} {value!r}.")
    return value


def _require_str(section: str, key: str, value, filename: str) -> str:
    if not isinstance(value, str) or not value:
        raise OverburdenConfigError(
            f"{filename}: {section}.{key} must be a non-empty string, got "
            f"{type(value).__name__} {value!r}.")
    return value


def _require_integer(section: str, key: str, value, filename: str) -> int:
    """Require a genuine integer; never truncate a floating-point policy value."""
    if not isinstance(value, int) or isinstance(value, bool):
        raise OverburdenConfigError(
            f"{filename}: {section}.{key} must be an integer, got "
            f"{type(value).__name__} {value!r}.")
    return value


def load_overburden_config(yaml_path: str) -> OverburdenConfig:
    """Load and validate `config/overburden_stress.yml`.

    Every required key is checked here so that a typo fails at load time
    rather than deep inside an integral, and every unknown top-level key or
    unknown key inside a known section is REJECTED rather than ignored - an
    ignored key is an unreviewed policy change.

    Four policy invariants are enforced as hard errors because violating any
    of them would silently cross an Increment 7 scientific boundary:

      * `bridge_across_seabed_allowed` must be false;
      * `bridge_outside_survey_coverage_allowed` must be false;
      * `bridge_long_gaps_allowed` must be false;
      * `shallow_gap_uses_internal_gap_rule` must be false - the unmeasured
        seabed-to-log column is not a dropout and must never be filled by the
        internal-gap rule.
    """
    path = Path(yaml_path)
    if not path.is_file():
        raise OverburdenConfigError(f"Overburden stress config not found: {yaml_path!r}.")
    with path.open("r", encoding="utf-8") as fh:
        raw = yaml.load(fh, Loader=_NoDuplicateKeySafeLoader)
    if not isinstance(raw, dict):
        raise OverburdenConfigError(f"{path.name}: top level must be a mapping.")
    if raw.get("schema_version") != "7.0.1":
        raise OverburdenConfigError(
            f"{path.name}: schema_version must be exactly '7.0.1', got "
            f"{raw.get('schema_version')!r}.")

    for key in _REQUIRED_TOP_KEYS:
        if key not in raw:
            raise OverburdenConfigError(
                f"{path.name}: required top-level key {key!r} is missing.")
    unknown_top = sorted(set(raw) - _KNOWN_TOP_KEYS)
    if unknown_top:
        raise OverburdenConfigError(
            f"{path.name}: unknown top-level key(s) {unknown_top}. An unrecognised key is "
            f"rejected rather than ignored, because an ignored key is an unreviewed policy "
            f"change.")

    for section, keys in _SECTION_KEYS.items():
        block = raw[section]
        if not isinstance(block, dict):
            raise OverburdenConfigError(f"{path.name}: {section!r} must be a mapping.")
        missing = [k for k in keys if k not in block]
        if missing:
            raise OverburdenConfigError(
                f"{path.name}: {section} is missing required key(s) {missing}.")
        unknown = sorted(set(block) - set(keys))
        if unknown:
            raise OverburdenConfigError(
                f"{path.name}: {section} contains unknown key(s) {unknown}.")

    ds = raw["density_source"]
    sb = raw["screening_bounds"]
    gc = raw["gap_conditioning"]
    ig = raw["integration"]
    wc = raw["water_column"]
    sc = raw["shallow_column_scenarios"]
    el = raw["eligibility"]

    if not _require_bool("density_source", "original_values_immutable",
                         ds["original_values_immutable"], path.name):
        raise OverburdenConfigError(
            f"{path.name}: density_source.original_values_immutable must be true. Increment 7 "
            f"never modifies a resolved density array.")
    for key in ("bridge_across_seabed_allowed", "bridge_outside_survey_coverage_allowed",
                "bridge_long_gaps_allowed", "shallow_gap_uses_internal_gap_rule"):
        if _require_bool("gap_conditioning", key, gc[key], path.name):
            raise OverburdenConfigError(
                f"{path.name}: gap_conditioning.{key} must be false. This is a hard "
                f"scientific invariant of Increment 7, not tunable behaviour.")
    if _require_bool("water_column", "cross_well_seabed_transfer_allowed",
                     wc["cross_well_seabed_transfer_allowed"], path.name):
        raise OverburdenConfigError(
            f"{path.name}: water_column.cross_well_seabed_transfer_allowed must be false. A "
            f"seabed marker belongs to the well it was picked in.")
    if not _require_bool("shallow_column_scenarios", "base_is_not_a_best_estimate",
                         sc["base_is_not_a_best_estimate"], path.name):
        raise OverburdenConfigError(
            f"{path.name}: shallow_column_scenarios.base_is_not_a_best_estimate must be true. "
            f"The base scenario is an arithmetic midpoint with no evidentiary support and "
            f"must never be declared a best estimate.")
    if not _require_bool("shallow_column_scenarios", "require_assumed_fraction_disclosure",
                         sc["require_assumed_fraction_disclosure"], path.name):
        raise OverburdenConfigError(
            f"{path.name}: shallow_column_scenarios.require_assumed_fraction_disclosure must "
            f"be true. A scenario total may not be published without stating how much of it "
            f"originates in the assumption.")

    method = _require_str("integration", "method", ig["method"], path.name)
    if "vertical_depth" not in method:
        raise OverburdenConfigError(
            f"{path.name}: integration.method {method!r} does not declare a vertical-depth "
            f"formulation. Increment 7 integrates in true vertical depth only.")

    thresholds = gc["sensitivity_thresholds_tvd_m"]
    if not isinstance(thresholds, list) or not thresholds:
        raise OverburdenConfigError(
            f"{path.name}: gap_conditioning.sensitivity_thresholds_tvd_m must be a non-empty "
            f"list.")
    threshold_values = tuple(
        _require_number("gap_conditioning", "sensitivity_thresholds_tvd_m", t, path.name)
        for t in thresholds)
    if any(t < 0.0 for t in threshold_values):
        raise OverburdenConfigError(
            f"{path.name}: gap_conditioning.sensitivity_thresholds_tvd_m must all be >= 0.")
    if len(set(threshold_values)) != len(threshold_values):
        raise OverburdenConfigError(
            f"{path.name}: gap_conditioning.sensitivity_thresholds_tvd_m contains duplicates.")

    statuses = el["statuses"]
    if not isinstance(statuses, list) or tuple(statuses) != VALID_OVERBURDEN_STATUSES:
        raise OverburdenConfigError(
            f"{path.name}: eligibility.statuses must equal the four declared status codes "
            f"in their required order: {list(VALID_OVERBURDEN_STATUSES)!r}.")
    scenario_names = sc["scenario_names"]
    if (not isinstance(scenario_names, list)
            or [str(s) for s in scenario_names] != ["low", "base", "high"]):
        raise OverburdenConfigError(
            f"{path.name}: shallow_column_scenarios.scenario_names must be exactly "
            f"['low', 'base', 'high']; low/base/high reporting is mandatory.")

    not_impl = raw["not_implemented"]
    if not isinstance(not_impl, list) or not not_impl:
        raise OverburdenConfigError(
            f"{path.name}: 'not_implemented' must be a non-empty list.")
    if any(not isinstance(item, str) or not item for item in not_impl):
        raise OverburdenConfigError(
            f"{path.name}: every 'not_implemented' entry must be a non-empty string; "
            f"numeric and other values are not coerced.")
    if len(set(not_impl)) != len(not_impl):
        raise OverburdenConfigError(
            f"{path.name}: 'not_implemented' contains duplicate entries.")

    expected_bases = {
        "low_basis": SCENARIO_BASIS_LOW,
        "base_basis": SCENARIO_BASIS_BASE,
        "high_basis": SCENARIO_BASIS_HIGH,
    }
    for key, expected in expected_bases.items():
        if sc[key] != expected:
            raise OverburdenConfigError(
                f"{path.name}: shallow_column_scenarios.{key} must be exactly "
                f"{expected!r}, got {sc[key]!r}.")

    increment = raw["increment"]
    if not isinstance(increment, int) or isinstance(increment, bool) or increment != 7:
        raise OverburdenConfigError(
            f"{path.name}: 'increment' must be the integer 7, got {increment!r}.")

    scenario_high_percentile = _require_number(
        "shallow_column_scenarios", "high_percentile",
        sc["high_percentile"], path.name)
    if scenario_high_percentile != 5.0:
        raise OverburdenConfigError(
            f"{path.name}: shallow_column_scenarios.high_percentile must be exactly "
            f"5.0 because Increment 7 stores and uses P05, got "
            f"{scenario_high_percentile!r}.")

    return OverburdenConfig(
        source_filename=path.name,
        schema_version=_require_str("", "schema_version", raw["schema_version"], path.name),
        increment=increment,
        assurance_tier=_require_str("", "assurance_tier", raw["assurance_tier"], path.name),
        canonical_curve_name=_require_str(
            "density_source", "canonical_curve_name", ds["canonical_curve_name"], path.name),
        accepted_canonical_unit=_require_str(
            "density_source", "accepted_canonical_unit", ds["accepted_canonical_unit"],
            path.name),
        expected_conversion_function=_require_str(
            "density_source", "expected_conversion_function",
            ds["expected_conversion_function"], path.name),
        rhob_min_kg_m3=_require_number("screening_bounds", "rhob_min_kg_m3",
                                       sb["rhob_min_kg_m3"], path.name),
        rhob_max_kg_m3=_require_number("screening_bounds", "rhob_max_kg_m3",
                                       sb["rhob_max_kg_m3"], path.name),
        bounds_are_inclusive=_require_bool("screening_bounds", "bounds_are_inclusive",
                                           sb["bounds_are_inclusive"], path.name),
        bridge_short_internal_gaps=_require_bool(
            "gap_conditioning", "bridge_short_internal_gaps",
            gc["bridge_short_internal_gaps"], path.name),
        short_gap_max_tvd_m=_require_number(
            "gap_conditioning", "short_gap_max_tvd_m", gc["short_gap_max_tvd_m"], path.name),
        shallow_gap_tolerance_tvd_m=_require_number(
            "gap_conditioning", "shallow_gap_tolerance_tvd_m",
            gc["shallow_gap_tolerance_tvd_m"], path.name),
        sensitivity_thresholds_tvd_m=tuple(sorted(threshold_values)),
        integration_method=method,
        gravity_m_s2=_require_number("integration", "gravity_m_s2", ig["gravity_m_s2"],
                                     path.name),
        profile_report_step_tvdss_m=_require_number(
            "integration", "profile_report_step_tvdss_m",
            ig["profile_report_step_tvdss_m"], path.name),
        seawater_density_kg_m3=_require_number(
            "water_column", "seawater_density_kg_m3", wc["seawater_density_kg_m3"],
            path.name),
        seawater_density_low_kg_m3=_require_number(
            "water_column", "seawater_density_low_kg_m3", wc["seawater_density_low_kg_m3"],
            path.name),
        seawater_density_high_kg_m3=_require_number(
            "water_column", "seawater_density_high_kg_m3", wc["seawater_density_high_kg_m3"],
            path.name),
        seabed_marker_name=_require_str(
            "water_column", "seabed_marker_name", wc["seabed_marker_name"], path.name),
        scenarios_enabled=_require_bool("shallow_column_scenarios", "enabled",
                                        sc["enabled"], path.name),
        scenario_high_percentile=scenario_high_percentile,
        scenario_names=tuple(str(s) for s in scenario_names),
        min_eligible_samples_for_increment=_require_integer(
            "eligibility", "min_eligible_samples_for_increment",
            el["min_eligible_samples_for_increment"], path.name),
        absolute_requires_uninterrupted_column=_require_bool(
            "eligibility", "absolute_requires_uninterrupted_column",
            el["absolute_requires_uninterrupted_column"], path.name),
        not_implemented=tuple(not_impl),
    )


# ---------------------------------------------------------------------------
# Curve resolution and unit confirmation
# ---------------------------------------------------------------------------

def resolve_density_slot(well_frame, config: OverburdenConfig):
    """Return `(slot, unit_resolved, conversion_confirmed)` for one well.

    `slot` is the locked well frame's own `CurveSlot` for the configured
    canonical density curve, or `None` when the well's locked contract
    resolved no such curve. A missing density curve is a factual data gap, not
    an error - the caller reports `rhob_not_available`.

    A curve that EXISTS but carries an unexpected canonical unit is a
    different matter: it is rejected with `DensityUnitError`, because
    Increment 7 performs no conversion of its own and must never integrate a
    quantity whose unit it cannot confirm.
    """
    slot = well_frame.curve(config.canonical_curve_name)
    if slot is None:
        return None, False, False
    if slot.canonical_unit != config.accepted_canonical_unit:
        raise DensityUnitError(
            f"{well_frame.well_key}: density curve "
            f"{config.canonical_curve_name!r} carries canonical unit "
            f"{slot.canonical_unit!r}, but Increment 7 accepts only "
            f"{config.accepted_canonical_unit!r}. Increment 7 performs no unit conversion of "
            f"its own; a curve whose unit cannot be confirmed is refused rather than "
            f"rescaled.")
    conversion_confirmed = (slot.conversion_function == config.expected_conversion_function)
    return slot, True, conversion_confirmed


# ---------------------------------------------------------------------------
# Masks
# ---------------------------------------------------------------------------

def _screening_mask(values: np.ndarray, config: OverburdenConfig) -> np.ndarray:
    finite = np.isfinite(values)
    if config.bounds_are_inclusive:
        band = (values >= config.rhob_min_kg_m3) & (values <= config.rhob_max_kg_m3)
    else:
        band = (values > config.rhob_min_kg_m3) & (values < config.rhob_max_kg_m3)
    return finite & band


def build_density_masks(
    well_frame,
    config: OverburdenConfig,
    *,
    seabed_mdrt_m: Optional[float] = None,
    gap_result: Optional[GapConditioningResult] = None,
) -> DensityMaskSet:
    """Construct the twelve explicit per-sample masks for ONE well.

    The masks are built in dependency order and every one is retained; none is
    collapsed into another. `eligible_for_measured_integration` is the explicit
    conjunction

        finite AND unit_resolved AND screening_range_plausible
        AND depth_mapping_valid AND within_survey_coverage
        [AND below_seabed_sample, when the seabed is resolved]

    and it is deliberately NOT extended with a bridged sample: a bridged value
    is conditioned, not measured, and travels in its own mask.

    `gap_result`, when supplied, contributes the three unresolved-region masks
    and the bridged mask. When it is omitted, those four masks are all-False -
    a state that means "gap conditioning has not been applied yet", and which
    the caller resolves by passing the result of `condition_density_gaps`.
    """
    n = int(well_frame.n_samples)
    slot, unit_ok, _ = resolve_density_slot(well_frame, config)

    if slot is None:
        values = np.full(n, np.nan, dtype=np.float64)
        present = np.zeros(n, dtype=bool)
    else:
        values = np.asarray(slot.values)
        _reject_ambiguous_dtype(values, f"{well_frame.well_key}: RHOB values")
        values = values.astype(np.float64, copy=True)
        # "Source value present" is a CURVE-LEVEL fact expressed per sample:
        # this well's locked contract resolved a density column, so a value
        # slot exists at every depth. It is deliberately NOT the same mask as
        # `finite_numeric_density`. The locked Increment 2.1.1 loader has
        # already substituted each file's NULL sentinel with NaN, so an
        # absent token and a recorded NULL are indistinguishable downstream;
        # claiming to separate them per sample would assert a distinction the
        # data no longer carry. The distinction that IS available - a well
        # with no density column at all - is exactly what this mask records.
        present = np.ones(n, dtype=bool)

    finite = np.isfinite(values)
    unit_resolved = np.full(n, bool(unit_ok), dtype=bool)
    screening = _screening_mask(values, config) & unit_resolved
    depth_valid = np.asarray(well_frame.depth_valid_mask, dtype=bool)

    md = np.asarray(well_frame.MD_m, dtype=np.float64)
    within_survey = (md >= float(well_frame.survey_md_min_m)) & (
        md <= float(well_frame.survey_md_max_m))

    if seabed_mdrt_m is None:
        below_seabed: Optional[np.ndarray] = None
        seabed_resolved = False
    else:
        seabed_resolved = True
        below_seabed = md >= float(seabed_mdrt_m)

    eligible = finite & unit_resolved & screening & depth_valid & within_survey
    if below_seabed is not None:
        eligible = eligible & below_seabed

    if gap_result is None:
        bridged = np.zeros(n, dtype=bool)
        long_gap = np.zeros(n, dtype=bool)
        shallow = np.zeros(n, dtype=bool)
        terminal = np.zeros(n, dtype=bool)
    else:
        bridged = np.asarray(gap_result.bridged_mask, dtype=bool).copy()
        long_gap = np.zeros(n, dtype=bool)
        shallow = np.zeros(n, dtype=bool)
        terminal = np.zeros(n, dtype=bool)
        for gap in gap_result.gaps:
            if gap.start_index is None or gap.end_index is None:
                continue
            lo, hi = int(gap.start_index), int(gap.end_index)
            if gap.gap_class == GAP_CLASS_LONG_INTERNAL:
                long_gap[lo:hi + 1] = True
            elif gap.gap_class == GAP_CLASS_SHALLOW:
                shallow[lo:hi + 1] = True
            elif gap.gap_class == GAP_CLASS_TERMINAL:
                terminal[lo:hi + 1] = True

    return DensityMaskSet(
        well_key=well_frame.well_key,
        n_samples=n,
        seabed_resolved=seabed_resolved,
        source_value_present=readonly(present),
        finite_numeric_density=readonly(finite),
        unit_resolved=readonly(unit_resolved),
        screening_range_plausible=readonly(screening),
        below_seabed_sample=None if below_seabed is None else readonly(below_seabed),
        depth_mapping_valid=readonly(depth_valid),
        within_survey_coverage=readonly(within_survey),
        eligible_for_measured_integration=readonly(eligible),
        bridged_short_gap=readonly(bridged),
        unresolved_long_gap=readonly(long_gap),
        unresolved_shallow_column=readonly(shallow),
        unresolved_terminal_column=readonly(terminal),
    )


# ---------------------------------------------------------------------------
# Gap classification
# ---------------------------------------------------------------------------

def _runs_of_false(mask: np.ndarray, lo: int, hi: int) -> List[Tuple[int, int]]:
    """Return inclusive `(start, end)` index pairs of False runs in `mask[lo:hi+1]`."""
    runs: List[Tuple[int, int]] = []
    i = lo
    while i <= hi:
        if not mask[i]:
            j = i
            while j <= hi and not mask[j]:
                j += 1
            runs.append((i, j - 1))
            i = j
        else:
            i += 1
    return runs


def classify_density_gaps(
    well_frame,
    masks: DensityMaskSet,
    config: OverburdenConfig,
    *,
    seabed_tvd_m: Optional[float] = None,
    threshold_tvd_m: Optional[float] = None,
) -> Tuple[DensityGapRecord, ...]:
    """Classify every gap in ONE well's density column.

    Five structurally distinct cases are kept apart, exactly as the Increment 7
    policy requires:

      * an internal gap at or below the threshold  -> short_internal_gap
      * an internal gap above the threshold        -> long_internal_gap
      * seabed to first eligible sample            -> shallow gap
      * below the last eligible sample             -> terminal gap
      * a run rendered ineligible only by missing depth mapping
                                                   -> unmapped-depth gap

    An isolated invalid sample is not a separate class in the emitted record:
    it is an internal gap of one sample, and its `n_samples` field says so.
    The distinction the policy asks for is preserved in the count, not
    duplicated as a third internal category that would have to be kept
    consistent with the other two.

    The shallow gap is classified here and is NEVER a candidate for bridging.
    Its disposition is always `unresolved_not_bridged`, at any threshold.
    """
    threshold = _validated_gap_threshold(
        config.short_gap_max_tvd_m if threshold_tvd_m is None else threshold_tvd_m,
        "classify_density_gaps")
    elig = np.asarray(masks.eligible_for_measured_integration, dtype=bool)
    md = np.asarray(well_frame.MD_m, dtype=np.float64)
    tvd = np.asarray(well_frame.TVD_m, dtype=np.float64)
    depth_valid = np.asarray(well_frame.depth_valid_mask, dtype=bool)
    values = None
    slot = well_frame.curve(config.canonical_curve_name)
    if slot is not None:
        values = np.asarray(slot.values, dtype=np.float64)

    records: List[DensityGapRecord] = []
    idx = np.flatnonzero(elig)

    def _depth(arr, i):
        v = float(arr[i])
        return v if np.isfinite(v) else None

    if idx.size == 0:
        # No eligible sample at all: the whole column below the seabed is an
        # unresolved shallow gap when the seabed is known, and otherwise there
        # is no determinable datum to measure a shallow gap from.
        if seabed_tvd_m is not None:
            below_seabed = masks.below_seabed_sample
            spanned = (np.flatnonzero(np.asarray(below_seabed, dtype=bool))
                       if below_seabed is not None else np.array([], dtype=int))
            records.append(DensityGapRecord(
                well_key=well_frame.well_key, gap_index=0, gap_class=GAP_CLASS_SHALLOW,
                disposition=GAP_DISPOSITION_UNRESOLVED, n_samples=int(spanned.size),
                start_index=(int(spanned[0]) if spanned.size else None),
                end_index=(int(spanned[-1]) if spanned.size else None),
                md_start_m=None, md_end_m=None,
                tvd_start_m=float(seabed_tvd_m), tvd_end_m=None,
                thickness_md_m=None, thickness_tvd_m=None, threshold_tvd_m=threshold,
                bounding_density_above_kg_m3=None, bounding_density_below_kg_m3=None))
        return tuple(records)

    first, last = int(idx[0]), int(idx[-1])
    gap_index = 0

    if seabed_tvd_m is not None:
        top_tvd = _depth(tvd, first)
        thickness = (None if top_tvd is None else max(0.0, top_tvd - float(seabed_tvd_m)))
        # The shallow unresolved column runs from the seabed down to the first
        # eligible sample. Its THICKNESS is measured between those two depths,
        # which is the quantity that matters. Its LAS sample range covers only
        # the part of that column the log actually spans: a log starting below
        # the seabed leaves the interval above its own first sample with no
        # LAS row at all, and no mask can represent a sample that does not
        # exist. Both facts are recorded, and they are deliberately not
        # conflated.
        below_seabed = masks.below_seabed_sample
        if below_seabed is not None:
            in_column = np.zeros(int(md.size), dtype=bool)
            in_column[:first] = True
            in_column &= np.asarray(below_seabed, dtype=bool)
            spanned = np.flatnonzero(in_column)
        else:  # pragma: no cover - a seabed TVD without a seabed mask
            spanned = np.array([], dtype=int)
        records.append(DensityGapRecord(
            well_key=well_frame.well_key, gap_index=gap_index, gap_class=GAP_CLASS_SHALLOW,
            disposition=GAP_DISPOSITION_UNRESOLVED, n_samples=int(spanned.size),
            start_index=(int(spanned[0]) if spanned.size else None),
            end_index=(int(spanned[-1]) if spanned.size else None),
            md_start_m=None, md_end_m=float(md[first]),
            tvd_start_m=float(seabed_tvd_m), tvd_end_m=top_tvd,
            thickness_md_m=None, thickness_tvd_m=thickness, threshold_tvd_m=threshold,
            bounding_density_above_kg_m3=None, bounding_density_below_kg_m3=None))
        gap_index += 1

    for lo, hi in _runs_of_false(elig, first, last):
        above, below = lo - 1, hi + 1
        t_above, t_below = _depth(tvd, above), _depth(tvd, below)
        thickness_tvd = (None if (t_above is None or t_below is None)
                         else max(0.0, t_below - t_above))
        thickness_md = float(md[below] - md[above])
        # A run that is ineligible ONLY because its depth mapping is missing
        # is a different physical situation from a missing density reading,
        # and it is never bridgeable: there is no defensible vertical
        # coordinate to interpolate against.
        run_unmapped = bool(np.all(~depth_valid[lo:hi + 1]))
        if run_unmapped:
            gap_class = GAP_CLASS_UNMAPPED_DEPTH
            disposition = GAP_DISPOSITION_UNRESOLVED
        elif thickness_tvd is not None and thickness_tvd <= threshold:
            gap_class = GAP_CLASS_SHORT_INTERNAL
            disposition = (GAP_DISPOSITION_BRIDGED if config.bridge_short_internal_gaps
                           else GAP_DISPOSITION_UNRESOLVED)
        else:
            gap_class = GAP_CLASS_LONG_INTERNAL
            disposition = GAP_DISPOSITION_UNRESOLVED
        records.append(DensityGapRecord(
            well_key=well_frame.well_key, gap_index=gap_index, gap_class=gap_class,
            disposition=disposition, n_samples=hi - lo + 1,
            start_index=lo, end_index=hi,
            md_start_m=float(md[above]), md_end_m=float(md[below]),
            tvd_start_m=t_above, tvd_end_m=t_below,
            thickness_md_m=thickness_md, thickness_tvd_m=thickness_tvd,
            threshold_tvd_m=threshold,
            bounding_density_above_kg_m3=(
                None if values is None else float(values[above])),
            bounding_density_below_kg_m3=(
                None if values is None else float(values[below]))))
        gap_index += 1

    n_terminal = int(md.size) - last - 1
    if n_terminal > 0:
        t_last = _depth(tvd, last)
        t_end = _depth(tvd, int(md.size) - 1)
        records.append(DensityGapRecord(
            well_key=well_frame.well_key, gap_index=gap_index, gap_class=GAP_CLASS_TERMINAL,
            disposition=GAP_DISPOSITION_UNRESOLVED, n_samples=n_terminal,
            start_index=last + 1, end_index=int(md.size) - 1,
            md_start_m=float(md[last]), md_end_m=float(md[-1]),
            tvd_start_m=t_last, tvd_end_m=t_end,
            thickness_md_m=float(md[-1] - md[last]),
            thickness_tvd_m=(None if (t_last is None or t_end is None)
                             else max(0.0, t_end - t_last)),
            threshold_tvd_m=threshold,
            bounding_density_above_kg_m3=(None if values is None else float(values[last])),
            bounding_density_below_kg_m3=None))

    return tuple(records)


# ---------------------------------------------------------------------------
# Gap conditioning
# ---------------------------------------------------------------------------

def condition_density_gaps(
    well_frame,
    masks: DensityMaskSet,
    config: OverburdenConfig,
    *,
    seabed_tvd_m: Optional[float] = None,
    threshold_tvd_m: Optional[float] = None,
) -> GapConditioningResult:
    """Apply the configured gap-conditioning policy to ONE well.

    Bridging is linear interpolation of density against TRUE VERTICAL DEPTH -
    the coordinate the stress integral is taken in - between the two
    bracketing eligible samples. It writes ONLY into a new array; the well
    frame's own resolved density array is never touched.

    A gap is bridged only if ALL of the following hold:
      * bridging is enabled in configuration;
      * the gap is an internal gap, strictly inside the bracketed measured
        column (so it can never reach the seabed or the terminal column);
      * its TVD thickness is at or below the approved threshold;
      * every sample in it has a valid locked depth mapping and lies within
        locked survey coverage;
      * both bracketing samples carry finite, in-band measured density.
    """
    threshold = _validated_gap_threshold(
        config.short_gap_max_tvd_m if threshold_tvd_m is None else threshold_tvd_m,
        "condition_density_gaps")
    gaps = classify_density_gaps(
        well_frame, masks, config, seabed_tvd_m=seabed_tvd_m, threshold_tvd_m=threshold)

    slot = well_frame.curve(config.canonical_curve_name)
    n = int(well_frame.n_samples)
    if slot is None:
        conditioned = np.full(n, np.nan, dtype=np.float64)
    else:
        # An explicit copy. The locked CurveSlot array is read-only by
        # construction; this makes the separation of the conditioned
        # representation from the original an observable property of the code,
        # not a consequence of someone else's flag.
        conditioned = np.array(slot.values, dtype=np.float64, copy=True)

    tvd = np.asarray(well_frame.TVD_m, dtype=np.float64)
    md = np.asarray(well_frame.MD_m, dtype=np.float64)
    within_survey = np.asarray(masks.within_survey_coverage, dtype=bool)
    depth_valid = np.asarray(masks.depth_mapping_valid, dtype=bool)

    bridged = np.zeros(n, dtype=bool)
    n_bridged_gaps = 0
    n_bridged_samples = 0
    bridged_md = 0.0
    bridged_tvd = 0.0
    n_long_gaps = 0
    n_long_samples = 0
    long_md = 0.0
    long_tvd = 0.0
    emitted: List[DensityGapRecord] = []

    for gap in gaps:
        if gap.gap_class == GAP_CLASS_LONG_INTERNAL:
            n_long_gaps += 1
            n_long_samples += gap.n_samples
            long_md += float(gap.thickness_md_m or 0.0)
            long_tvd += float(gap.thickness_tvd_m or 0.0)
            emitted.append(gap)
            continue
        if gap.gap_class != GAP_CLASS_SHORT_INTERNAL:
            emitted.append(gap)
            continue
        lo, hi = int(gap.start_index), int(gap.end_index)
        above, below = lo - 1, hi + 1
        can_bridge = (
            config.bridge_short_internal_gaps
            and bool(np.all(depth_valid[lo:hi + 1]))
            and bool(np.all(within_survey[lo:hi + 1]))
            and np.isfinite(tvd[above]) and np.isfinite(tvd[below])
            and np.isfinite(conditioned[above]) and np.isfinite(conditioned[below])
            and float(tvd[below]) > float(tvd[above])
        )
        if not can_bridge:
            emitted.append(DensityGapRecord(
                well_key=gap.well_key, gap_index=gap.gap_index,
                gap_class=gap.gap_class, disposition=GAP_DISPOSITION_UNRESOLVED,
                n_samples=gap.n_samples, start_index=gap.start_index,
                end_index=gap.end_index, md_start_m=gap.md_start_m,
                md_end_m=gap.md_end_m, tvd_start_m=gap.tvd_start_m,
                tvd_end_m=gap.tvd_end_m, thickness_md_m=gap.thickness_md_m,
                thickness_tvd_m=gap.thickness_tvd_m, threshold_tvd_m=gap.threshold_tvd_m,
                bounding_density_above_kg_m3=gap.bounding_density_above_kg_m3,
                bounding_density_below_kg_m3=gap.bounding_density_below_kg_m3))
            continue
        z0, z1 = float(tvd[above]), float(tvd[below])
        r0, r1 = float(conditioned[above]), float(conditioned[below])
        z = tvd[lo:hi + 1]
        conditioned[lo:hi + 1] = r0 + (r1 - r0) * (z - z0) / (z1 - z0)
        bridged[lo:hi + 1] = True
        n_bridged_gaps += 1
        n_bridged_samples += gap.n_samples
        bridged_md += float(md[below] - md[above])
        bridged_tvd += (z1 - z0)
        emitted.append(DensityGapRecord(
            well_key=gap.well_key, gap_index=gap.gap_index, gap_class=gap.gap_class,
            disposition=GAP_DISPOSITION_BRIDGED, n_samples=gap.n_samples,
            start_index=gap.start_index, end_index=gap.end_index,
            md_start_m=gap.md_start_m, md_end_m=gap.md_end_m,
            tvd_start_m=gap.tvd_start_m, tvd_end_m=gap.tvd_end_m,
            thickness_md_m=gap.thickness_md_m, thickness_tvd_m=gap.thickness_tvd_m,
            threshold_tvd_m=gap.threshold_tvd_m,
            bounding_density_above_kg_m3=r0, bounding_density_below_kg_m3=r1))

    return GapConditioningResult(
        well_key=well_frame.well_key,
        threshold_tvd_m=threshold,
        bridging_enabled=bool(config.bridge_short_internal_gaps),
        conditioned_density_kg_m3=readonly(conditioned),
        bridged_mask=readonly(bridged),
        n_bridged_gaps=n_bridged_gaps,
        n_bridged_samples=n_bridged_samples,
        bridged_thickness_md_m=bridged_md,
        bridged_thickness_tvd_m=bridged_tvd,
        n_long_gaps=n_long_gaps,
        n_long_gap_samples=n_long_samples,
        long_gap_thickness_md_m=long_md,
        long_gap_thickness_tvd_m=long_tvd,
        gaps=tuple(emitted),
    )


# ---------------------------------------------------------------------------
# QC statistics
# ---------------------------------------------------------------------------

def compute_density_qc_stats(
    well_frame,
    masks: DensityMaskSet,
    config: OverburdenConfig,
    *,
    seabed_mdrt_m: Optional[float] = None,
    seabed_tvd_m: Optional[float] = None,
    seabed_tvdss_m: Optional[float] = None,
    gaps: Tuple[DensityGapRecord, ...] = (),
) -> DensityQcStats:
    """Measure factual density QC statistics for ONE well.

    Every number returned is a measurement about the curve as recorded or a
    count of samples failing an explicitly configured screening criterion.
    Nothing here modifies, corrects, or interprets the curve.
    """
    n = int(well_frame.n_samples)
    slot, unit_ok, conversion_confirmed = resolve_density_slot(well_frame, config)

    md = np.asarray(well_frame.MD_m, dtype=np.float64)
    tvd = np.asarray(well_frame.TVD_m, dtype=np.float64)
    tvdss = np.asarray(well_frame.TVDSS_m, dtype=np.float64)

    if slot is None:
        values = np.full(n, np.nan, dtype=np.float64)
    else:
        values = np.asarray(slot.values, dtype=np.float64)

    finite = np.asarray(masks.finite_numeric_density, dtype=bool)
    in_band = np.asarray(masks.screening_range_plausible, dtype=bool)
    elig = np.asarray(masks.eligible_for_measured_integration, dtype=bool)
    depth_valid = np.asarray(masks.depth_mapping_valid, dtype=bool)
    within_survey = np.asarray(masks.within_survey_coverage, dtype=bool)

    n_non_positive = int(np.count_nonzero(finite & (values <= 0.0)))
    below_min = finite & (values > 0.0) & (
        values < config.rhob_min_kg_m3 if config.bounds_are_inclusive
        else values <= config.rhob_min_kg_m3)
    above_max = finite & (
        values > config.rhob_max_kg_m3 if config.bounds_are_inclusive
        else values >= config.rhob_max_kg_m3)
    n_below = int(np.count_nonzero(below_min))
    n_above = int(np.count_nonzero(above_max))

    fv = values[finite]
    eligible_values = values[elig]
    idx = np.flatnonzero(elig)

    def _at(arr, i):
        v = float(arr[i])
        return v if np.isfinite(v) else None

    if idx.size:
        first, last = int(idx[0]), int(idx[-1])
        first_md, last_md = float(md[first]), float(md[last])
        first_tvd, last_tvd = _at(tvd, first), _at(tvd, last)
        first_tvdss, last_tvdss = _at(tvdss, first), _at(tvdss, last)
        gross_md = last_md - first_md
        gross_tvd = (None if (first_tvd is None or last_tvd is None)
                     else last_tvd - first_tvd)
    else:
        first_md = last_md = first_tvd = last_tvd = None
        first_tvdss = last_tvdss = gross_md = gross_tvd = None

    internal = [g for g in gaps if g.gap_class in (
        GAP_CLASS_SHORT_INTERNAL, GAP_CLASS_LONG_INTERNAL, GAP_CLASS_UNMAPPED_DEPTH)]
    shallow = next((g for g in gaps if g.gap_class == GAP_CLASS_SHALLOW), None)
    terminal = next((g for g in gaps if g.gap_class == GAP_CLASS_TERMINAL), None)

    return DensityQcStats(
        well_key=well_frame.well_key,
        source_las_filename=well_frame.source_las_filename,
        curve_present=slot is not None,
        canonical_curve_name=(None if slot is None else slot.canonical_name),
        source_curve_name=(None if slot is None else slot.source_curve_name),
        raw_mnemonic=(None if slot is None else slot.raw_mnemonic),
        raw_unit=(None if slot is None else slot.raw_unit),
        canonical_unit=(None if slot is None else slot.canonical_unit),
        conversion_function=(None if slot is None else slot.conversion_function),
        unit_resolved=bool(unit_ok),
        conversion_confirmed=bool(conversion_confirmed),
        n_samples=n,
        n_source_present=int(np.count_nonzero(masks.source_value_present)),
        n_finite=int(np.count_nonzero(finite)),
        n_non_finite=int(n - np.count_nonzero(finite)),
        n_non_positive=n_non_positive,
        n_below_screening_min=n_below,
        n_above_screening_max=n_above,
        n_screening_bound_failures=n_non_positive + n_below + n_above,
        n_in_screening_band=int(np.count_nonzero(in_band)),
        n_depth_mapped=int(np.count_nonzero(depth_valid)),
        n_depth_unmapped=int(n - np.count_nonzero(depth_valid)),
        n_eligible=int(np.count_nonzero(elig)),
        rhob_min_kg_m3=(float(np.min(fv)) if fv.size else None),
        rhob_max_kg_m3=(float(np.max(fv)) if fv.size else None),
        rhob_median_kg_m3=(float(np.median(fv)) if fv.size else None),
        rhob_p05_kg_m3=(float(np.percentile(fv, 5.0)) if fv.size else None),
        rhob_p95_kg_m3=(float(np.percentile(fv, 95.0)) if fv.size else None),
        rhob_eligible_p05_kg_m3=(
            float(np.percentile(eligible_values, config.scenario_high_percentile))
            if eligible_values.size else None),
        first_valid_md_m=first_md,
        last_valid_md_m=last_md,
        first_valid_tvd_m=first_tvd,
        last_valid_tvd_m=last_tvd,
        first_valid_tvdss_m=first_tvdss,
        last_valid_tvdss_m=last_tvdss,
        gross_coverage_md_m=gross_md,
        gross_coverage_tvd_m=gross_tvd,
        median_md_step_m=(float(np.median(np.diff(md))) if md.size > 1 else None),
        seabed_basis=(SEABED_BASIS_LOCKED_MARKER if seabed_mdrt_m is not None
                      else SEABED_BASIS_NOT_DETERMINABLE),
        seabed_mdrt_m=(None if seabed_mdrt_m is None else float(seabed_mdrt_m)),
        seabed_tvd_m=(None if seabed_tvd_m is None else float(seabed_tvd_m)),
        seabed_tvdss_m=(None if seabed_tvdss_m is None else float(seabed_tvdss_m)),
        shallow_gap_md_m=(None if shallow is None or first_md is None or seabed_mdrt_m is None
                          else float(first_md - float(seabed_mdrt_m))),
        shallow_gap_tvd_m=(None if shallow is None else shallow.thickness_tvd_m),
        terminal_gap_md_m=(None if terminal is None else terminal.thickness_md_m),
        terminal_gap_tvd_m=(None if terminal is None else terminal.thickness_tvd_m),
        n_internal_gaps=len(internal),
        n_internal_gap_samples=sum(g.n_samples for g in internal),
        longest_internal_gap_md_m=(
            max((g.thickness_md_m for g in internal if g.thickness_md_m is not None),
                default=None)),
        longest_internal_gap_tvd_m=(
            max((g.thickness_tvd_m for g in internal if g.thickness_tvd_m is not None),
                default=None)),
        depth_basis_used=well_frame.depth_basis_used,
        depth_map_status=well_frame.depth_map_status,
        survey_md_min_m=float(well_frame.survey_md_min_m),
        survey_md_max_m=float(well_frame.survey_md_max_m),
        n_samples_outside_survey_coverage=int(n - np.count_nonzero(within_survey)),
    )


#### `p2mem/overburden.py` - vertical-stress integration and evidence-derived eligibility

Integrates in **true vertical depth**, rejects a non-monotonic vertical coordinate rather than
sorting it, counts and discloses zero-thickness intervals, and truncates at the first
unresolved long gap rather than silently joining two columns that are not adjacent. The
project's depth sign convention is re-verified against each frame's own arrays at run time.

In [ ]:
%%writefile p2mem/overburden.py
"""
p2mem.overburden - vertical overburden-stress framework (Increment 7).

The integral
------------
    sigma_v(z) = INTEGRAL_0^z rho(z') g dz'

evaluated in TRUE VERTICAL DEPTH. For a deviated well this is NOT the same as
integrating against measured depth: the measured-depth span of an interval
always equals or exceeds its true-vertical span, and integrating in MD would
inflate the result by exactly that ratio. This module accepts a VERTICAL-depth
array and rejects a non-monotonic one, so measured-depth integration is not
merely discouraged - it is structurally unavailable.

No well name appears anywhere in this module, in `p2mem.density_qc`, or in
`config/overburden_stress.yml`. Method eligibility is derived from measured
coverage and QC evidence; there is no identifier for a branch to read.

Sign convention - VERIFIED, not inferred from a variable name
--------------------------------------------------------------
The locked `p2mem.depth_mapping` module documents and independently tests:

    TVD is zero at the well datum and increases DOWNWARD; datum elevation is
    referenced to mean sea level, positive UPWARD; TVDSS = TVD - datum
    elevation, so TVDSS is positive DOWNWARD from mean sea level.

`verify_depth_sign_convention` re-derives this from the LOCKED well frame's
own arrays at run time - it checks that TVD increases with MD, that
TVDSS == TVD - datum_elevation to floating-point tolerance, and that the
constant offset is exactly the datum elevation - and raises rather than
proceeding if any of those fails. Nothing in this module reads a sign from a
name.

Separation of measurement from assumption
-----------------------------------------
Every reported stress is partitioned into:

  * water column          - ASSUMED seawater density, configured
  * measured formation    - recorded RHOB only
  * conditioned short gap - linearly bridged density, reported separately
  * unresolved shallow    - model-dependent, only ever a labelled scenario

A total is reported only when no component is unresolved. An unresolved
component is never treated as zero.
"""

from __future__ import annotations

import math
from numbers import Real
from typing import Dict, List, Optional, Tuple

import numpy as np

from p2mem.overburden_models import (
    GAP_CLASS_LONG_INTERNAL,
    GAP_CLASS_SHALLOW,
    GAP_CLASS_TERMINAL,
    GAP_DISPOSITION_UNRESOLVED,
    GAP_DISPOSITION_BRIDGED,
    REASON_DEPTH_MAPPING_INCOMPLETE,
    REASON_INSUFFICIENT_ELIGIBLE_SAMPLES,
    REASON_INTERNAL_GAP_EXCEEDS_LIMIT,
    REASON_INTERNAL_GAP_UNRESOLVED,
    REASON_RHOB_ALL_INVALID,
    REASON_RHOB_NOT_AVAILABLE,
    REASON_SCREENING_BOUND_FAILURES,
    REASON_SEABED_DATUM_UNRESOLVED,
    REASON_SHALLOW_COLUMN_UNRESOLVED,
    REASON_SURVEY_COVERAGE_INSUFFICIENT,
    REASON_TERMINAL_COLUMN_UNRESOLVED,
    REASON_UNIT_NOT_RESOLVED,
    SCENARIO_BASE,
    SCENARIO_BASE_SEAWATER_HIGH,
    SCENARIO_BASE_SEAWATER_LOW,
    SCENARIO_HIGH,
    SCENARIO_LOW,
    SCENARIO_BASIS_BASE,
    SCENARIO_BASIS_HIGH,
    SCENARIO_BASIS_LOW,
    SEABED_BASIS_LOCKED_MARKER,
    SEABED_BASIS_NOT_DETERMINABLE,
    STATUS_ABSOLUTE,
    STATUS_NOT_ELIGIBLE,
    STATUS_PARTIAL_ONLY,
    STATUS_SENSITIVITY_ONLY,
    DensityQcStats,
    GapConditioningResult,
    OverburdenConfig,
    OverburdenEligibility,
    OverburdenInputError,
    ShallowColumnScenario,
    StressPartition,
    VerticalStressProfile,
    readonly,
)

__all__ = [
    "DEPTH_CONVENTION_STATEMENT",
    "SIGN_CONVENTION_ID",
    "verify_depth_sign_convention",
    "integrate_vertical_stress",
    "water_column_stress_pa",
    "uniform_column_stress_pa",
    "build_stress_profile",
    "build_gap_threshold_sensitivity",
    "derive_overburden_eligibility",
    "build_shallow_column_scenarios",
    "pa_to_mpa",
]

#: The verified, machine-checkable identity of the depth convention this
#: module integrates in. Emitted with every result so that a reader never has
#: to reconstruct it from a variable name.
SIGN_CONVENTION_ID = "tvd_and_tvdss_increase_downward_tvdss_zero_at_msl"

DEPTH_CONVENTION_STATEMENT = (
    "TVD is zero at the well datum and increases downward; the datum elevation is "
    "referenced to mean sea level, positive upward; TVDSS = TVD - datum elevation and is "
    "positive downward from mean sea level. A downward interval therefore has a positive "
    "increment in both coordinates, and dTVD equals dTVDSS exactly. This convention is "
    "documented and independently tested in the locked depth-mapping module, and is "
    "re-verified against each well frame's own arrays at run time rather than inferred "
    "from a variable name."
)


def pa_to_mpa(value: Optional[float]) -> Optional[float]:
    """Convert pascals to megapascals. Exact decimal factor, no rounding."""
    return None if value is None else float(value) / 1.0e6


# ---------------------------------------------------------------------------
# Sign-convention verification
# ---------------------------------------------------------------------------

def verify_depth_sign_convention(well_frame, *, tolerance_m: float = 1e-6) -> Dict[str, object]:
    """Re-derive and confirm the project's depth-sign convention for ONE well.

    Three independent checks against the LOCKED well frame's own arrays:

      1. TVD is non-decreasing with increasing MD over the depth-mapped
         samples (a well trajectory cannot rise as measured depth increases);
      2. `TVDSS == TVD - datum_elevation_m` to within `tolerance_m`;
      3. the constant offset between TVD and TVDSS equals the frame's own
         recorded datum elevation, to within `tolerance_m`.

    Raises `OverburdenInputError` naming the failed check rather than
    proceeding. Returns the measured evidence so it can be exported.
    """
    tvd = np.asarray(well_frame.TVD_m, dtype=np.float64)
    tvdss = np.asarray(well_frame.TVDSS_m, dtype=np.float64)
    md = np.asarray(well_frame.MD_m, dtype=np.float64)
    valid = np.asarray(well_frame.depth_valid_mask, dtype=bool)
    datum = float(well_frame.datum_elevation_m)

    if int(np.count_nonzero(valid)) < 2:
        raise OverburdenInputError(
            f"{well_frame.well_key}: fewer than two depth-mapped samples; the depth-sign "
            f"convention cannot be verified, and Increment 7 does not assume it.")

    t, s, m = tvd[valid], tvdss[valid], md[valid]
    order = np.argsort(m, kind="stable")
    t_ord = t[order]
    d_tvd = np.diff(t_ord)
    n_decreasing = int(np.count_nonzero(d_tvd < -tolerance_m))
    if n_decreasing:
        raise OverburdenInputError(
            f"{well_frame.well_key}: TVD decreases with increasing MD at {n_decreasing} "
            f"sample interval(s) (most negative {float(np.min(d_tvd)):.9f} m). The project "
            f"depth convention requires TVD to increase downward; Increment 7 refuses to "
            f"integrate against a coordinate whose direction it cannot confirm.")

    offset = t - s
    max_offset_dev = float(np.max(np.abs(offset - datum)))
    if max_offset_dev > tolerance_m:
        raise OverburdenInputError(
            f"{well_frame.well_key}: TVD - TVDSS deviates from the recorded datum elevation "
            f"({datum} m) by up to {max_offset_dev:.9f} m, exceeding the {tolerance_m} m "
            f"tolerance. The relationship TVDSS = TVD - datum_elevation is the project's "
            f"stated convention and is not assumed when it cannot be confirmed.")

    d_tvdss = np.diff(s[order])
    max_increment_dev = float(np.max(np.abs(d_tvd - d_tvdss))) if d_tvd.size else 0.0
    if max_increment_dev > tolerance_m:
        raise OverburdenInputError(
            f"{well_frame.well_key}: dTVD and dTVDSS differ by up to {max_increment_dev:.9f} "
            f"m; they must be identical because the two coordinates differ by a per-well "
            f"constant.")

    return {
        "sign_convention_id": SIGN_CONVENTION_ID,
        "verified": True,
        "n_depth_mapped_samples": int(np.count_nonzero(valid)),
        "datum_elevation_m": datum,
        "max_tvd_minus_tvdss_deviation_m": max_offset_dev,
        "max_dtvd_minus_dtvdss_deviation_m": max_increment_dev,
        "min_tvd_increment_m": float(np.min(d_tvd)) if d_tvd.size else 0.0,
        "tvd_increases_downward": True,
    }


# ---------------------------------------------------------------------------
# Core integration
# ---------------------------------------------------------------------------

def _reject_ambiguous_dtype(raw: np.ndarray, context: str) -> None:
    """Reject boolean, string/bytes, complex and object dtype with `TypeError`.

    Documented local copy of the check established in the locked
    `p2mem.units` / `p2mem.time_depth` / `p2mem.io.tops` modules.
    """
    kind = raw.dtype.kind
    if kind == "b":
        raise TypeError(f"{context}: boolean input is not accepted as a numeric quantity.")
    if kind in ("U", "S"):
        raise TypeError(f"{context}: string/bytes input is not accepted as a numeric quantity.")
    if kind == "c":
        raise TypeError(f"{context}: complex input is not accepted as a numeric quantity.")
    if kind == "O":
        raise TypeError(f"{context}: object-dtype input is not accepted as a numeric quantity.")
    if kind not in ("i", "u", "f"):
        raise TypeError(f"{context}: unsupported array dtype {raw.dtype!r} for a numeric quantity.")


def _require_real_scalar(value, name: str, context: str, *, positive: bool = False,
                         non_negative: bool = False) -> float:
    """Validate one finite real scalar without coercing strings or complex values."""
    if isinstance(value, bool) or not isinstance(value, Real):
        raise TypeError(
            f"{context}: {name} must be a real numeric scalar, got "
            f"{type(value).__name__} {value!r}.")
    out = float(value)
    if not math.isfinite(out):
        raise OverburdenInputError(f"{context}: {name} must be finite, got {value!r}.")
    if positive and out <= 0.0:
        raise OverburdenInputError(
            f"{context}: {name} must be strictly positive, got {value!r}.")
    if non_negative and out < 0.0:
        raise OverburdenInputError(f"{context}: {name} must be >= 0, got {value!r}.")
    return out


def integrate_vertical_stress(
    density_kg_m3,
    vertical_depth_m,
    gravity_m_s2: float,
    *,
    context: str = "integrate_vertical_stress",
) -> Tuple[np.ndarray, int]:
    """Trapezoidal integration of rho*g*dz against a VERTICAL depth coordinate.

    Returns `(cumulative_pa, n_zero_thickness_intervals)`, where
    `cumulative_pa[0]` is exactly 0.0 and `cumulative_pa[i]` is the integral
    from the first node down to node `i`.

    Preconditions, checked in order and never silently repaired:

      * both inputs are 1-D numeric arrays of equal length, at least 2 long;
      * neither carries boolean, string, complex, or object dtype;
      * every density is finite and strictly positive;
      * every vertical depth is finite;
      * every vertical increment is non-negative. A ZERO increment (a repeated
        or equal vertical coordinate) contributes exactly zero, is counted,
        and is returned to the caller for disclosure. A NEGATIVE increment is
        REJECTED with `OverburdenInputError` - it is either a sorting defect
        or an attempt to integrate the wrong coordinate, and sorting it away
        silently would hide both.

    The inputs are never mutated; the returned array is a new read-only array.
    """
    rho_raw = np.asarray(density_kg_m3)
    z_raw = np.asarray(vertical_depth_m)
    _reject_ambiguous_dtype(rho_raw, f"{context}: density")
    _reject_ambiguous_dtype(z_raw, f"{context}: vertical depth")

    rho = rho_raw.astype(np.float64, copy=True)
    z = z_raw.astype(np.float64, copy=True)
    if rho.ndim != 1 or z.ndim != 1:
        raise OverburdenInputError(
            f"{context}: density and vertical depth must both be 1-D arrays, got shapes "
            f"{rho.shape} and {z.shape}.")
    if rho.size != z.size:
        raise OverburdenInputError(
            f"{context}: density has {rho.size} sample(s) but vertical depth has {z.size}; "
            f"a stress integral never truncates or pads to reconcile a length disagreement.")
    if rho.size < 2:
        raise OverburdenInputError(
            f"{context}: at least two samples are required to form one trapezoid, got "
            f"{rho.size}.")

    g = _require_real_scalar(
        gravity_m_s2, "gravity_m_s2", context, positive=True)

    if not np.all(np.isfinite(rho)):
        n_bad = int(np.count_nonzero(~np.isfinite(rho)))
        raise OverburdenInputError(
            f"{context}: {n_bad} non-finite density value(s) reached the integrator. A NaN or "
            f"infinity is never treated as zero and is never interpolated away here; it must "
            f"be excluded by an explicit mask upstream.")
    if not np.all(rho > 0.0):
        n_bad = int(np.count_nonzero(rho <= 0.0))
        raise OverburdenInputError(
            f"{context}: {n_bad} non-positive density value(s) reached the integrator; a "
            f"non-positive bulk density is not a formation property.")
    if not np.all(np.isfinite(z)):
        n_bad = int(np.count_nonzero(~np.isfinite(z)))
        raise OverburdenInputError(
            f"{context}: {n_bad} non-finite vertical-depth value(s) reached the integrator.")

    dz = np.diff(z)
    if np.any(dz < 0.0):
        n_bad = int(np.count_nonzero(dz < 0.0))
        worst = float(np.min(dz))
        raise OverburdenInputError(
            f"{context}: {n_bad} negative vertical increment(s) (most negative {worst:.9f} m). "
            f"The integrator refuses a non-monotonic vertical coordinate rather than sorting "
            f"it: a decreasing vertical depth is either a data-ordering defect or the wrong "
            f"coordinate (for example measured depth in a deviated well), and silently "
            f"sorting would hide both.")

    n_zero = int(np.count_nonzero(dz == 0.0))
    segment = 0.5 * (rho[:-1] + rho[1:]) * g * dz
    cumulative = np.empty(rho.size, dtype=np.float64)
    cumulative[0] = 0.0
    np.cumsum(segment, out=cumulative[1:])
    return readonly(cumulative), n_zero


def water_column_stress_pa(
    seabed_tvdss_m: float, seawater_density_kg_m3: float, gravity_m_s2: float
) -> float:
    """Vertical stress at the seabed from the ASSUMED seawater column.

    The seawater density is a CONFIGURED ASSUMPTION. This project holds no
    measured seawater density, salinity, or temperature profile for these
    locations, and this value must never be described as measured data.
    """
    return uniform_column_stress_pa(
        seabed_tvdss_m, seawater_density_kg_m3, gravity_m_s2,
        context="water_column_stress_pa")


def uniform_column_stress_pa(
    thickness_m: float, density_kg_m3: float, gravity_m_s2: float,
    *, context: str = "uniform_column_stress_pa",
) -> float:
    """`rho * g * h` for a column of uniform assumed density.

    Used for the water column and for the unresolved shallow-column scenarios.
    It is deliberately a single explicit product rather than a call into the
    array integrator: a uniform assumed density is not a measurement, and
    routing it through the measured-data path would blur that distinction.
    """
    h = _require_real_scalar(
        thickness_m, "thickness_m", context, non_negative=True)
    rho = _require_real_scalar(
        density_kg_m3, "density_kg_m3", context, positive=True)
    g = _require_real_scalar(
        gravity_m_s2, "gravity_m_s2", context, positive=True)
    return rho * g * h


# ---------------------------------------------------------------------------
# Per-well stress profile
# ---------------------------------------------------------------------------

def build_stress_profile(
    well_frame,
    masks,
    gap_result: GapConditioningResult,
    config: OverburdenConfig,
) -> Optional[VerticalStressProfile]:
    """Build ONE well's measured vertical-stress increment profile.

    The profile spans the contiguous integrable column: the eligible measured
    samples, plus any sample the configured policy actually bridged. It is
    TRUNCATED at the first unresolved internal gap - Increment 7 never integrates
    through an unresolved gap, because the density there is unknown and
    treating it as absent would silently join two columns that are not
    adjacent.

    Why the FIRST segment, rather than the longest one
    ---------------------------------------------------
    Vertical stress accumulates downward, so a segment is only ever useful if
    it can be tied to the column above it. The segment beginning at the
    SHALLOWEST eligible sample is the only one that could ever be connected to
    the shallow column and therefore to an absolute stress; a deeper segment
    sitting below an unresolved gap is stress-disconnected from everything
    above it and cannot contribute to sigma_v at all until that gap is
    resolved. Selecting the longest segment instead would report a larger
    number that is no more connectable, which is exactly the kind of
    flattering-but-useless result this increment is built to avoid.

    Returns `None` when that first segment holds fewer than the configured
    minimum number of samples - which is a genuine "no measured increment",
    even when the well holds many eligible samples further down.
    """
    elig = np.asarray(masks.eligible_for_measured_integration, dtype=bool)
    bridged = np.asarray(gap_result.bridged_mask, dtype=bool)
    usable = elig | bridged

    idx = np.flatnonzero(elig)
    if idx.size < config.min_eligible_samples_for_increment:
        return None

    first = int(idx[0])
    # Truncate at the first unresolved INTERNAL gap below the first eligible
    # sample.  This is disposition-driven rather than class-driven: a short
    # gap can remain unresolved when bridging is disabled or when a bridge
    # precondition fails, and an unmapped-depth gap is equally non-integrable.
    # Shallow and terminal gaps lie outside the bracketed measured column.
    unresolved_starts = sorted(
        int(g.start_index) for g in gap_result.gaps
        if g.disposition == GAP_DISPOSITION_UNRESOLVED
        and g.gap_class not in (GAP_CLASS_SHALLOW, GAP_CLASS_TERMINAL)
        and g.start_index is not None
        and int(g.start_index) > first)
    stop = unresolved_starts[0] if unresolved_starts else int(usable.size)

    window = np.zeros(usable.size, dtype=bool)
    window[first:stop] = True
    take = np.flatnonzero(usable & window)
    if take.size < config.min_eligible_samples_for_increment:
        return None

    density = np.asarray(gap_result.conditioned_density_kg_m3, dtype=np.float64)[take]
    md_nodes = np.asarray(well_frame.MD_m, dtype=np.float64)[take]
    tvd = np.asarray(well_frame.TVD_m, dtype=np.float64)[take]
    tvdss = np.asarray(well_frame.TVDSS_m, dtype=np.float64)[take]
    bridged_take = bridged[take]

    cumulative, n_zero = integrate_vertical_stress(
        density, tvdss, config.gravity_m_s2,
        context=f"{well_frame.well_key}: measured vertical-stress increment")

    # The bridged contribution is the part of the same integral whose
    # trapezoid touches at least one conditioned sample. Attributing a
    # trapezoid with one measured and one bridged endpoint to the bridged
    # side is the conservative choice: it can only overstate how much of the
    # result depends on interpolation, never understate it.
    dz = np.diff(tvdss)
    seg = 0.5 * (density[:-1] + density[1:]) * float(config.gravity_m_s2) * dz
    touches_bridge = bridged_take[:-1] | bridged_take[1:]
    bridged_pa = float(np.sum(seg[touches_bridge])) if seg.size else 0.0

    return VerticalStressProfile(
        well_key=well_frame.well_key,
        n_nodes=int(take.size),
        integration_coordinate="tvdss_m",
        gravity_m_s2=float(config.gravity_m_s2),
        md_m=readonly(md_nodes.copy()),
        tvd_m=readonly(tvd.copy()),
        tvdss_m=readonly(tvdss.copy()),
        density_kg_m3=readonly(density.copy()),
        bridged_mask=readonly(bridged_take.copy()),
        cumulative_measured_increment_pa=cumulative,
        total_measured_increment_pa=float(cumulative[-1]),
        total_bridged_increment_pa=bridged_pa,
        n_zero_thickness_intervals=n_zero,
        n_intervals=int(take.size) - 1,
        top_tvd_m=float(tvd[0]),
        base_tvd_m=float(tvd[-1]),
        top_tvdss_m=float(tvdss[0]),
        base_tvdss_m=float(tvdss[-1]),
        column_truncated_at_unresolved_gap=bool(unresolved_starts),
        n_eligible_samples_below_truncation=int(np.count_nonzero(elig[stop:])),
    )


# ---------------------------------------------------------------------------
# Eligibility derivation
# ---------------------------------------------------------------------------

def derive_overburden_eligibility(
    stats: DensityQcStats,
    masks,
    gap_result: GapConditioningResult,
    profile: Optional[VerticalStressProfile],
    config: OverburdenConfig,
) -> OverburdenEligibility:
    """DERIVE ONE well's overburden method-eligibility status from evidence.

    No well name participates. The status is a function of measured coverage
    and QC results only:

      not_eligible                    - no density curve, or fewer than the
                                        configured minimum eligible samples
      partial_measured_increment_only - an eligible measured column exists,
                                        but the seabed datum is UNRESOLVED, so
                                        the unmeasured shallow column's
                                        thickness is not even quantifiable and
                                        no bracket can be constructed
      screening_sensitivity_only      - an eligible measured column exists and
                                        the seabed datum IS resolved, so the
                                        unmeasured column's thickness is known
                                        and can be transparently bracketed -
                                        but it is not measured
      absolute_overburden_supported   - the density column reaches the seabed
                                        within the configured tolerance and no
                                        unresolved internal gap of any class
                                        interrupts it

    The ladder is deliberately ordered by what the EVIDENCE can support, not
    by how complete an answer each rung produces.
    """
    reasons: List[str] = []
    seabed_resolved = stats.seabed_basis == SEABED_BASIS_LOCKED_MARKER

    if not stats.curve_present:
        reasons.append(REASON_RHOB_NOT_AVAILABLE)
    if not stats.unit_resolved and stats.curve_present:
        reasons.append(REASON_UNIT_NOT_RESOLVED)
    if stats.curve_present and stats.n_finite == 0:
        reasons.append(REASON_RHOB_ALL_INVALID)
    if stats.n_screening_bound_failures > 0:
        reasons.append(REASON_SCREENING_BOUND_FAILURES)
    if stats.n_depth_unmapped > 0:
        reasons.append(REASON_DEPTH_MAPPING_INCOMPLETE)
    if stats.n_samples_outside_survey_coverage > 0:
        reasons.append(REASON_SURVEY_COVERAGE_INSUFFICIENT)
    if not seabed_resolved:
        reasons.append(REASON_SEABED_DATUM_UNRESOLVED)

    n_long = int(gap_result.n_long_gaps)
    unresolved_internal = tuple(
        g for g in gap_result.gaps
        if g.disposition == GAP_DISPOSITION_UNRESOLVED
        and g.gap_class not in (GAP_CLASS_SHALLOW, GAP_CLASS_TERMINAL))
    has_unresolved_internal = bool(unresolved_internal)
    if n_long > 0:
        reasons.append(REASON_INTERNAL_GAP_EXCEEDS_LIMIT)
    if any(g.gap_class != GAP_CLASS_LONG_INTERNAL for g in unresolved_internal):
        reasons.append(REASON_INTERNAL_GAP_UNRESOLVED)

    terminal = next((g for g in gap_result.gaps if g.gap_class == GAP_CLASS_TERMINAL), None)
    if terminal is not None and (terminal.thickness_tvd_m or 0.0) > 0.0:
        reasons.append(REASON_TERMINAL_COLUMN_UNRESOLVED)

    shallow = next((g for g in gap_result.gaps if g.gap_class == GAP_CLASS_SHALLOW), None)
    shallow_thickness = None if shallow is None else shallow.thickness_tvd_m
    shallow_unresolved = (
        seabed_resolved and shallow_thickness is not None
        and shallow_thickness > config.shallow_gap_tolerance_tvd_m)
    if shallow_unresolved:
        reasons.append(REASON_SHALLOW_COLUMN_UNRESOLVED)

    n_eligible = int(stats.n_eligible)
    if n_eligible < config.min_eligible_samples_for_increment or profile is None:
        reasons.append(REASON_INSUFFICIENT_ELIGIBLE_SAMPLES)
        status = STATUS_NOT_ELIGIBLE
    elif not seabed_resolved:
        status = STATUS_PARTIAL_ONLY
    elif shallow_unresolved or (
            has_unresolved_internal and config.absolute_requires_uninterrupted_column):
        status = STATUS_SENSITIVITY_ONLY
    else:
        status = STATUS_ABSOLUTE

    if status == STATUS_ABSOLUTE:
        # An absolute status admits no limiting reason. Any reason that
        # survived above but does not actually restrict the absolute integral
        # (a screening-bound failure among excluded samples, for instance)
        # would still be misleading here, so the absolute rung is granted only
        # when the reason list is genuinely empty.
        blocking = [r for r in reasons if r != REASON_TERMINAL_COLUMN_UNRESOLVED]
        if blocking:
            status = STATUS_SENSITIVITY_ONLY
        else:
            reasons = []

    water_thickness = (None if stats.seabed_tvdss_m is None
                       else max(0.0, float(stats.seabed_tvdss_m)))

    return OverburdenEligibility(
        well_key=stats.well_key,
        status=status,
        limiting_reasons=tuple(sorted(set(reasons))),
        seabed_resolved=seabed_resolved,
        seabed_basis=stats.seabed_basis,
        n_eligible_samples=n_eligible,
        n_bridged_samples=int(gap_result.n_bridged_samples),
        eligible_top_tvd_m=(None if profile is None else profile.top_tvd_m),
        eligible_base_tvd_m=(None if profile is None else profile.base_tvd_m),
        eligible_top_tvdss_m=(None if profile is None else profile.top_tvdss_m),
        eligible_base_tvdss_m=(None if profile is None else profile.base_tvdss_m),
        measured_thickness_tvd_m=(
            None if profile is None else profile.base_tvd_m - profile.top_tvd_m),
        shallow_unresolved_thickness_tvd_m=shallow_thickness,
        terminal_unresolved_thickness_tvd_m=(
            None if terminal is None else terminal.thickness_tvd_m),
        water_column_thickness_tvd_m=water_thickness,
        n_unresolved_internal_gaps=len(unresolved_internal),
        n_unresolved_long_gaps=n_long,
        unresolved_long_gap_thickness_tvd_m=float(gap_result.long_gap_thickness_tvd_m),
        column_uninterrupted=not has_unresolved_internal,
        measured_increment_pa=(
            None if profile is None else profile.total_measured_increment_pa),
        bridged_increment_pa=(
            None if profile is None else profile.total_bridged_increment_pa),
        absolute_stress_supported=(status == STATUS_ABSOLUTE),
    )


# ---------------------------------------------------------------------------
# Shallow-column screening scenarios
# ---------------------------------------------------------------------------

def build_shallow_column_scenarios(
    stats: DensityQcStats,
    eligibility: OverburdenEligibility,
    profile: Optional[VerticalStressProfile],
    config: OverburdenConfig,
) -> Tuple[ShallowColumnScenario, ...]:
    """Build the transparent low/base/high screening scenarios for ONE well.

    Returns an EMPTY tuple unless the well's derived status is
    `screening_sensitivity_only` - the only status for which the unmeasured
    column's thickness is known and can therefore be bracketed at all. A well
    whose seabed is unresolved produces no scenario, because bracketing an
    interval of unknown thickness would be arithmetic without content.

    Scenario endpoints, restated here because they govern what the numbers mean:

      low  = the configured seawater density. This is a conditional lower
             scenario only under a fully saturated-column assumption; the
             project does not establish the unknown interval's fluid state.
      high = the configured low percentile of the SAME WELL'S OWN measured
             eligible density population. This is an illustrative upper
             scenario motivated by monotonic-compaction reasoning, not a
             physical ceiling or rigorous uncertainty bound: no lithology,
             fluid-state or depth-dependent density model constrains it.
      base = the arithmetic midpoint. NO evidentiary support. It exists only
             so the bracket has a labelled centre, and it is never a result.

    Two further rows re-run the base case at the configured seawater-density
    bracket, so the reader can see how little of the spread comes from the
    seawater assumption and how much from the unmeasured sediment column.
    """
    if not config.scenarios_enabled:
        return ()
    if eligibility.status != STATUS_SENSITIVITY_ONLY:
        return ()
    if profile is None or stats.seabed_tvdss_m is None:
        return ()
    unresolved_thickness = eligibility.shallow_unresolved_thickness_tvd_m
    if unresolved_thickness is None or unresolved_thickness <= 0.0:
        return ()
    if stats.rhob_eligible_p05_kg_m3 is None:
        return ()

    g = float(config.gravity_m_s2)
    water_thickness = float(stats.seabed_tvdss_m)
    measured_thickness = float(profile.base_tvd_m - profile.top_tvd_m)
    measured_pa = float(profile.total_measured_increment_pa)
    bridged_pa = float(profile.total_bridged_increment_pa)
    # The measured formation contribution reported in the partition excludes
    # the conditioned part, which travels in its own component.
    measured_only_pa = max(0.0, measured_pa - bridged_pa)

    low_density = float(config.seawater_density_kg_m3)
    high_density = float(stats.rhob_eligible_p05_kg_m3)
    if high_density <= low_density:
        # The evidence does not support a bracket with a positive width: the
        # two selected scenario endpoints are inverted or degenerate. No
        # scenario is published rather than an inverted or degenerate one.
        return ()
    base_density = 0.5 * (low_density + high_density)

    plan: List[Tuple[str, float, str, float]] = [
        (SCENARIO_LOW, low_density, config_low_basis(), float(config.seawater_density_kg_m3)),
        (SCENARIO_BASE, base_density, config_base_basis(),
         float(config.seawater_density_kg_m3)),
        (SCENARIO_HIGH, high_density, config_high_basis(),
         float(config.seawater_density_kg_m3)),
        (SCENARIO_BASE_SEAWATER_LOW, base_density, config_base_basis(),
         float(config.seawater_density_low_kg_m3)),
        (SCENARIO_BASE_SEAWATER_HIGH, base_density, config_base_basis(),
         float(config.seawater_density_high_kg_m3)),
    ]

    out: List[ShallowColumnScenario] = []
    for name, density, basis, seawater in plan:
        water_pa = water_column_stress_pa(water_thickness, seawater, g)
        shallow_pa = uniform_column_stress_pa(unresolved_thickness, density, g,
                                              context=f"{stats.well_key}: shallow scenario")
        total = water_pa + measured_only_pa + bridged_pa + shallow_pa
        partition = StressPartition(
            water_column_pa=water_pa,
            measured_formation_pa=measured_only_pa,
            bridged_gap_pa=bridged_pa,
            unresolved_shallow_pa=shallow_pa,
            total_pa=total,
        )
        assumed = (water_pa + shallow_pa) / total if total > 0.0 else 0.0
        conditioned_fraction = bridged_pa / total if total > 0.0 else 0.0
        measured_fraction = measured_only_pa / total if total > 0.0 else 0.0
        out.append(ShallowColumnScenario(
            well_key=stats.well_key,
            scenario_name=name,
            assumed_shallow_density_kg_m3=density,
            assumed_density_basis=basis,
            seawater_density_kg_m3=seawater,
            gravity_m_s2=g,
            unresolved_thickness_tvd_m=float(unresolved_thickness),
            water_column_thickness_tvd_m=water_thickness,
            measured_thickness_tvd_m=measured_thickness,
            partition=partition,
            assumed_fraction_of_total=assumed,
            conditioned_fraction_of_total=conditioned_fraction,
            measured_fraction_of_total=measured_fraction,
        ))
    return tuple(out)


# ---------------------------------------------------------------------------
# Gap-threshold sensitivity
# ---------------------------------------------------------------------------

def build_gap_threshold_sensitivity(
    well_frame,
    masks,
    stats: DensityQcStats,
    config: OverburdenConfig,
    *,
    seabed_tvd_m: Optional[float] = None,
) -> Tuple["GapThresholdSensitivity", ...]:
    """Re-run gap conditioning, integration and status derivation at EVERY
    configured threshold, for ONE well.

    This is the mandatory sensitivity to the approved gap threshold. It exists
    so that a reader can see exactly what the approved threshold changed - how
    many gaps it bridged, how much stress that contributed, and whether it
    moved the derived status - rather than taking a single configured number
    on trust. The approved threshold is flagged in its own row; nothing here
    selects a preferred value.

    The import is local to avoid a module-level cycle: `density_qc` imports
    the shared models, and this function is the only place `overburden` needs
    the conditioning entry point.
    """
    from p2mem.density_qc import condition_density_gaps
    from p2mem.overburden_models import GapThresholdSensitivity

    out: List[GapThresholdSensitivity] = []
    for threshold in config.sensitivity_thresholds_tvd_m:
        gap_result = condition_density_gaps(
            well_frame, masks, config, seabed_tvd_m=seabed_tvd_m,
            threshold_tvd_m=float(threshold))
        profile = build_stress_profile(well_frame, masks, gap_result, config)
        eligibility = derive_overburden_eligibility(
            stats, masks, gap_result, profile, config)
        usable = int(np.count_nonzero(
            np.asarray(masks.eligible_for_measured_integration, dtype=bool)
            | np.asarray(gap_result.bridged_mask, dtype=bool)))
        out.append(GapThresholdSensitivity(
            well_key=well_frame.well_key,
            threshold_tvd_m=float(threshold),
            is_approved_threshold=bool(
                float(threshold) == float(config.short_gap_max_tvd_m)),
            n_bridged_gaps=int(gap_result.n_bridged_gaps),
            n_bridged_samples=int(gap_result.n_bridged_samples),
            bridged_thickness_tvd_m=float(gap_result.bridged_thickness_tvd_m),
            n_long_gaps=int(gap_result.n_long_gaps),
            long_gap_thickness_tvd_m=float(gap_result.long_gap_thickness_tvd_m),
            n_eligible_or_bridged_samples=usable,
            total_measured_increment_pa=(
                None if profile is None else profile.total_measured_increment_pa),
            bridged_increment_pa=(
                None if profile is None else profile.total_bridged_increment_pa),
            derived_status=eligibility.status,
        ))
    return tuple(out)


#: The three bracket bases, as enumerated tokens rather than prose. They are
#: emitted verbatim and are members of the Increment 7 controlled vocabulary.
def config_low_basis() -> str:
    return SCENARIO_BASIS_LOW


def config_base_basis() -> str:
    return SCENARIO_BASIS_BASE


def config_high_basis() -> str:
    return SCENARIO_BASIS_HIGH


#### `p2mem/io/overburden_policy.py` - the Increment 6.1.7 output-policy architecture, generalized

The same closed-schema-first algorithm, lifted so its registries arrive as an explicit policy
bundle instead of module globals - because the locked Increment 6 module cannot be modified,
and because adding Increment 7's artifacts to its global registries would make every
Increment 6 export fail closed for not presenting them. One category is added:
`enumerated_code_list`, which is strictly stronger than a structured diagnostic because every
element is checked against a closed vocabulary rather than a grammar.

In [ ]:
%%writefile p2mem/io/overburden_policy.py
"""
Increment 7 - SCHEMA-DRIVEN, FAILURE-ATOMIC OUTPUT AUTHORIZATION, generalized.

Relationship to the LOCKED Increment 6.1.7 policy
--------------------------------------------------
`p2mem.io.output_policy` is locked. Its registries (`CSV_SCHEMAS`,
`OUTPUT_FIELD_POLICY`, `OUTPUT_STATEMENTS`, `JSON_OBJECT_KEYS`, ...) are
MODULE GLOBALS, and its `validate_emitted_records` requires the COMPLETE
declared artifact set to be presented. Adding Increment 7's nine artifacts to
those globals would (a) modify a locked file and (b) make every Increment 6
export fail closed for not presenting Increment 7 artifacts.

This module therefore does what the Increment 7 brief asks - it EXTENDS the
architecture rather than reimplementing a weaker one. The engine here is the
same algorithm, lifted so that its registries arrive as an explicit
`PolicyBundle` argument instead of being read from module globals:

  * the same closed-schema-first, authorize-second ordering;
  * the same eight categories, plus ONE addition (`enumerated_code_list`,
    below), all reusing the locked category constants;
  * the same collect-from-the-ACTUAL-records rule - field discovery is
    schema-driven, never inferred from a value;
  * the same isolated-staging, re-read, re-validate, canonical-compare,
    failure-atomic publication.

`tests/test_overburden_policy.py` runs this generalized engine against the
LOCKED Increment 6 registries and requires it to reproduce the locked
engine's accept/reject decision on every occurrence of the real, packaged
Increment 6 output records. That is the evidence that the generalization is
faithful rather than merely similar.

The one added category
----------------------
`enumerated_code_list` carries a ';'-joined list of CONTROLLED ENUMERATED
CODES - the `limiting_reasons` field is the only user. Each element must be a
member of the field's declared vocabulary; the list must be sorted and free of
duplicates. This is strictly STRONGER than emitting the same information as a
`structured_diagnostic`, because every element is checked against a closed
vocabulary rather than against a grammar, so it counts as CONTROLLED.

SCOPE OF THE GUARANTEE is unchanged from the locked module:
`structured_diagnostic` and `sanitized_diagnostic` remain outside the
controlled-interpretation guarantee and are counted separately.
"""

from typing import Dict, Optional, Tuple

from p2mem.io.output_policy import (
    CATEGORIES as LOCKED_CATEGORIES,
    CONTROLLED_CATEGORIES as LOCKED_CONTROLLED_CATEGORIES,
    STRUCTURAL_CATEGORIES,
    UNGUARANTEED_CATEGORIES,
    SANITIZED_DIAGNOSTIC_CHARSET,
    SANITIZED_DIAGNOSTIC_MAX_LEN,
    CoverageReport,
    CsvArtifactSchema,
    FieldOccurrence,
    FieldPolicy as LockedFieldPolicy,
)
from p2mem.wellframe_models import (
    ApprovedLabel,
    FIELD_TYPES,
    RegisteredStatement,
    RegisteredTemplate,
    find_prohibited_lithology_terms,
)
from p2mem.overburden_models import (
    DENSITY_MASK_NAMES,
    VALID_GAP_CLASSES,
    VALID_GAP_DISPOSITIONS,
    VALID_LIMITING_REASONS,
    VALID_OVERBURDEN_STATUSES,
    VALID_SCENARIO_NAMES,
    VALID_SEABED_BASES,
)

__all__ = [
    "CATEGORY_ENUMERATED_CODE_LIST", "CATEGORIES", "CONTROLLED_CATEGORIES",
    "PolicyBundle", "FieldPolicy", "OVERBURDEN_BUNDLE",
    "OVERBURDEN_ARTIFACTS", "OVERBURDEN_CSV_SCHEMAS", "OVERBURDEN_MANIFEST_ARTIFACT",
    "OVERBURDEN_STATEMENTS", "OVERBURDEN_TEMPLATES", "OVERBURDEN_LABELS",
    "APPROVED_WELL_KEYS", "SCOPE_OUTPUT_INC7",
    "collect_string_fields", "authorize_occurrence", "validate_artifact_schema",
    "validate_emitted_records", "canonicalize_emitted_records",
    "export_authorized_outputs", "OutputAuthorizationError",
    "bundle_from_locked_increment6",
]

SCOPE_OUTPUT_INC7 = "output_increment7"

CATEGORY_ENUMERATED_CODE_LIST = "enumerated_code_list"
CATEGORIES: Tuple[str, ...] = LOCKED_CATEGORIES + (CATEGORY_ENUMERATED_CODE_LIST,)
CONTROLLED_CATEGORIES: Tuple[str, ...] = (
    LOCKED_CONTROLLED_CATEGORIES + (CATEGORY_ENUMERATED_CODE_LIST,))


class FieldPolicy(LockedFieldPolicy):
    """The declared classification of one emitted string field.

    Subclasses the locked `FieldPolicy` so that the two policies cannot drift
    apart structurally, and widens only the accepted category set.
    """

    def __init__(self, artifact, field, category, field_kind=None,
                 allowed_values=(), statement_ids=(), template_id=None,
                 allow_empty=False):
        if category == CATEGORY_ENUMERATED_CODE_LIST:
            # Bypass the locked constructor's category check for the one added
            # category, then set every attribute the locked class declares.
            if not allowed_values:
                raise ValueError(
                    f"{artifact}:{field}: an enumerated_code_list must declare its closed "
                    f"vocabulary in allowed_values")
            self.artifact = artifact
            self.field = field
            self.category = category
            self.field_kind = field_kind
            self.allowed_values = tuple(allowed_values)
            self.statement_ids = tuple(statement_ids)
            self.template_id = template_id
            self.allow_empty = bool(allow_empty)
            return
        super().__init__(artifact, field, category, field_kind=field_kind,
                         allowed_values=allowed_values, statement_ids=statement_ids,
                         template_id=template_id, allow_empty=allow_empty)


# ---------------------------------------------------------------------------
# The policy bundle: every registry the engine consults, in one object
# ---------------------------------------------------------------------------

class PolicyBundle:
    """All registries required to validate and authorize one artifact set.

    Everything the locked engine read from module globals is a field here, so
    the same code can govern Increment 6's artifacts, Increment 7's, or a
    synthetic set constructed by a test, without any of them being able to
    weaken another.
    """

    __slots__ = ("name", "field_policy", "statements", "templates", "labels",
                 "csv_schemas", "manifest_artifact", "json_object_keys",
                 "json_list_item_kinds", "json_boolean_paths", "json_integer_paths",
                 "json_number_paths", "json_nullable_paths", "approved_well_keys",
                 "artifacts", "field_types", "extra_destination_names")

    def __init__(self, name, field_policies, statements, templates, labels,
                 csv_schemas, manifest_artifact, json_object_keys,
                 json_list_item_kinds, json_boolean_paths, json_integer_paths,
                 json_number_paths, json_nullable_paths, approved_well_keys,
                 field_types, extra_destination_names=("figures",)):
        self.name = name
        self.field_policy: Dict[Tuple[str, str], FieldPolicy] = {}
        for pol in field_policies:
            key = (pol.artifact, pol.field)
            if key in self.field_policy:
                raise ValueError(f"{name}: duplicate output field policy {key}")
            self.field_policy[key] = pol
        self.statements = {s.statement_id: s for s in statements}
        self.templates = {t.template_id: t for t in templates}
        self.labels: Dict[str, Dict[str, ApprovedLabel]] = {}
        for lab in labels:
            bucket = self.labels.setdefault(lab.field_kind, {})
            if lab.value in bucket:
                raise ValueError(
                    f"{name}: duplicate label {(lab.field_kind, lab.value)}")
            bucket[lab.value] = lab
        self.csv_schemas = dict(csv_schemas)
        self.manifest_artifact = manifest_artifact
        self.json_object_keys = dict(json_object_keys)
        self.json_list_item_kinds = dict(json_list_item_kinds)
        self.json_boolean_paths = frozenset(json_boolean_paths)
        self.json_integer_paths = frozenset(json_integer_paths)
        self.json_number_paths = frozenset(json_number_paths)
        self.json_nullable_paths = frozenset(json_nullable_paths)
        self.approved_well_keys = frozenset(approved_well_keys)
        self.field_types = dict(field_types)
        self.artifacts: Tuple[str, ...] = tuple(sorted(
            tuple(self.csv_schemas) + ((manifest_artifact,) if manifest_artifact else ())))
        self.extra_destination_names = frozenset(extra_destination_names)
        self._assert_schema_policy_agreement()

    def _assert_schema_policy_agreement(self):
        """Every schema-declared CSV string field must be classified exactly once.

        The locked module performs this as an import-time assertion. Doing it
        in the constructor gives the same guarantee for every bundle, including
        one a test builds, and prevents a future schema/policy drift from
        weakening the gate silently.
        """
        by_artifact: Dict[str, set] = {}
        for (artifact, field) in self.field_policy:
            if artifact in self.csv_schemas:
                by_artifact.setdefault(artifact, set()).add(field)
        for artifact, schema in self.csv_schemas.items():
            declared = set(schema.string_fields)
            classified = by_artifact.get(artifact, set())
            if declared != classified:
                raise ValueError(
                    f"{self.name}/{artifact}: CSV schema string fields and authorization "
                    f"policy differ: schema_only={sorted(declared - classified)}, "
                    f"policy_only={sorted(classified - declared)}")

    def approved_label(self, field_kind, value):
        return self.labels.get(field_kind, {}).get(value)


# ---------------------------------------------------------------------------
# Type predicates
# ---------------------------------------------------------------------------

def _is_integer_literal(value):
    if isinstance(value, bool) or not isinstance(value, str) or not value:
        return False
    body = value[1:] if value[0] in "+-" else value
    return bool(body) and body.isdigit()


OVERBURDEN_FIELD_TYPES = dict(FIELD_TYPES)
OVERBURDEN_FIELD_TYPES["integer"] = _is_integer_literal


def _is_number_with_optional_unit(token, field_types):
    if not isinstance(token, str) or not token:
        return False
    i = len(token)
    while i > 0 and token[i - 1].isalpha():
        i -= 1
    return field_types["decimal"](token[:i]) and token[i:].isalpha() or (
        i == len(token) and field_types["decimal"](token))


# ---------------------------------------------------------------------------
# Schema validation
# ---------------------------------------------------------------------------

def _schema_violation(artifact, field, location, reason):
    return {
        "artifact": artifact, "field": field, "location": location,
        "reason": "SCHEMA: " + reason, "linter_terms": [],
    }


def _csv_stage(payload, schema, requested):
    if requested in ("pre", "post"):
        return requested
    for row in payload if isinstance(payload, list) else ():
        if not isinstance(row, dict):
            continue
        for field in schema.integer_fields | schema.number_fields | schema.boolean_fields:
            if field in row and row[field] is not None and not isinstance(row[field], str):
                return "pre"
    return "post"


def _valid_decimal_string(value):
    import decimal
    if not isinstance(value, str) or not value:
        return False
    try:
        return decimal.Decimal(value).is_finite()
    except decimal.InvalidOperation:
        return False


def _validate_csv_schema(bundle, artifact, payload, serialization_stage="auto"):
    """Validate keys, order, requiredness and types without inspecting prose."""
    schema = bundle.csv_schemas[artifact]
    out = []
    if not isinstance(payload, list):
        return [_schema_violation(
            artifact, None, None, "CSV payload must be a list of row dictionaries")]
    if len(payload) < schema.min_rows:
        out.append(_schema_violation(
            artifact, None, None,
            f"CSV requires at least {schema.min_rows} row(s), found {len(payload)}"))
    stage = _csv_stage(payload, schema, serialization_stage)
    for i, row in enumerate(payload):
        location = f"row[{i}]"
        if not isinstance(row, dict):
            out.append(_schema_violation(
                artifact, None, location, "CSV row must be a dictionary"))
            continue
        actual = tuple(row.keys())
        if actual != schema.columns:
            missing = [c for c in schema.columns if c not in row]
            unknown = [c for c in actual if c not in schema.columns]
            out.append(_schema_violation(
                artifact, None, location,
                f"ordered columns differ; missing={missing}, unknown={unknown}, "
                f"expected={list(schema.columns)}, actual={list(actual)}"))
        for field in schema.columns:
            if field not in row:
                continue
            value = row[field]
            nullable = field in schema.nullable_fields
            kind = schema.kind(field)
            if kind != "string" and (
                    (stage == "pre" and value is None)
                    or (stage == "post" and value == "")):
                if nullable:
                    continue
                out.append(_schema_violation(
                    artifact, field, f"{location}.{field}",
                    "required value is null/empty after serialization"))
                continue
            valid = False
            if kind == "string":
                valid = isinstance(value, str)
            elif kind == "integer":
                valid = ((type(value) is int) if stage == "pre"
                         else isinstance(value, str) and _is_integer_literal(value))
            elif kind == "number":
                if stage == "pre":
                    import math
                    valid = (type(value) in (int, float) and math.isfinite(value))
                else:
                    valid = _valid_decimal_string(value)
            elif kind == "boolean":
                valid = ((type(value) is bool) if stage == "pre"
                         else value in ("True", "False"))
            if not valid:
                out.append(_schema_violation(
                    artifact, field, f"{location}.{field}",
                    f"expected {kind} at {stage}-serialization stage, got "
                    f"{type(value).__name__} {value!r}"))
    return out


def _normalize_json_path(path, well_keys):
    for w in sorted(well_keys, key=len, reverse=True):
        path = path.replace("/" + w + "/", "/*/")
        if path.endswith("/" + w):
            path = path[: -len(w)] + "*"
    out, i = [], 0
    while i < len(path):
        if path[i] == "[":
            j = path.index("]", i)
            out.append("[]")
            i = j + 1
        else:
            out.append(path[i])
            i += 1
    return "".join(out)


def _expected_json_primitive_kind(bundle, path):
    if path in bundle.json_boolean_paths:
        return "boolean"
    if path in bundle.json_integer_paths:
        return "integer"
    if path in bundle.json_number_paths:
        return "number"
    if (bundle.manifest_artifact, path) in bundle.field_policy:
        return "string"
    return None


def _validate_json_schema(bundle, artifact, payload, well_keys=()):
    """Validate the complete manifest tree, including empty/non-string leaves."""
    out = []
    supplied_wells = frozenset(well_keys)
    unknown_wells = supplied_wells - bundle.approved_well_keys
    if unknown_wells:
        out.append(_schema_violation(
            artifact, "/wells", "/wells",
            f"well_keys contains unapproved identifier(s): {sorted(unknown_wells)}"))

    def walk(node, path):
        norm = _normalize_json_path(path, well_keys)
        if isinstance(node, dict):
            expected = bundle.json_object_keys.get(norm, "__missing__")
            if expected == "__missing__":
                out.append(_schema_violation(
                    artifact, norm, path, "object path is not declared"))
                return
            if expected is None:
                expected = supplied_wells
            actual = frozenset(node)
            if actual != expected:
                out.append(_schema_violation(
                    artifact, norm, path,
                    f"object keys differ; missing={sorted(expected - actual)}, "
                    f"unknown={sorted(actual - expected)}"))
            for key in sorted(actual & expected):
                walk(node[key], f"{path}/{key}")
            return
        if isinstance(node, list):
            expected_kind = bundle.json_list_item_kinds.get(norm)
            if expected_kind is None:
                out.append(_schema_violation(
                    artifact, norm, path, "list path is not declared"))
                return
            for i, value in enumerate(node):
                if expected_kind == "object" and not isinstance(value, dict):
                    out.append(_schema_violation(
                        artifact, norm, f"{path}[{i}]", "list item must be an object"))
                elif expected_kind == "string" and not isinstance(value, str):
                    out.append(_schema_violation(
                        artifact, norm, f"{path}[{i}]", "list item must be a string"))
                elif expected_kind == "number" and not (
                        type(value) in (int, float)
                        and __import__("math").isfinite(value)):
                    out.append(_schema_violation(
                        artifact, norm, f"{path}[{i}]", "list item must be a finite number"))
                else:
                    walk(value, f"{path}[{i}]")
            return
        kind = _expected_json_primitive_kind(bundle, norm)
        if kind is None:
            out.append(_schema_violation(
                artifact, norm, path, "primitive path is not declared"))
            return
        if node is None:
            if norm not in bundle.json_nullable_paths:
                out.append(_schema_violation(
                    artifact, norm, path, "required value is null"))
            return
        if kind == "string":
            valid = isinstance(node, str)
        elif kind == "boolean":
            valid = type(node) is bool
        elif kind == "integer":
            valid = type(node) is int
        else:
            import math
            valid = type(node) in (int, float) and math.isfinite(node)
        if not valid:
            out.append(_schema_violation(
                artifact, norm, path,
                f"expected {kind}, got {type(node).__name__} {node!r}"))

    if not isinstance(payload, dict):
        return [_schema_violation(
            artifact, None, None, "JSON manifest payload must be an object")]
    walk(payload, "")
    return out


def validate_artifact_schema(bundle, artifact, payload, well_keys=(),
                             serialization_stage="auto"):
    if artifact in bundle.csv_schemas:
        return _validate_csv_schema(bundle, artifact, payload, serialization_stage)
    if artifact == bundle.manifest_artifact:
        return _validate_json_schema(bundle, artifact, payload, well_keys)
    return [_schema_violation(
        artifact, None, None, "artifact has no declared schema")]


# ---------------------------------------------------------------------------
# Collection: the ACTUAL string fields of the ACTUAL records
# ---------------------------------------------------------------------------

def collect_string_fields(bundle, artifact, payload, well_keys=()):
    """Collect every schema-declared string occurrence, including ``""``.

    Field discovery is schema-driven. Numeric-looking text in a declared prose
    field is therefore still prose and must authorize; unknown or missing keys
    are handled independently by `validate_artifact_schema`.
    """
    out = []
    if artifact in bundle.csv_schemas and isinstance(payload, list):
        schema = bundle.csv_schemas[artifact]
        for i, row in enumerate(payload):
            if not isinstance(row, dict):
                continue
            for column in schema.columns:
                value = row.get(column)
                if column in schema.string_fields and isinstance(value, str):
                    out.append(FieldOccurrence(
                        artifact, column, value, f"row[{i}].{column}"))
        return out
    if artifact == bundle.manifest_artifact and isinstance(payload, dict):
        def walk(node, path):
            if isinstance(node, dict):
                for key, value in node.items():
                    walk(value, f"{path}/{key}")
            elif isinstance(node, list):
                for i, value in enumerate(node):
                    walk(value, f"{path}[{i}]")
            elif isinstance(node, str):
                norm = _normalize_json_path(path, well_keys)
                if _expected_json_primitive_kind(bundle, norm) == "string":
                    out.append(FieldOccurrence(artifact, norm, node, path))
        walk(payload, "")
    return out


# ---------------------------------------------------------------------------
# Authorization of one occurrence
# ---------------------------------------------------------------------------

_FORBIDDEN_IN_DIAGNOSTIC = ("\n", "\r", "\t", "\\", "://")


def _authorize_structured(value, field_types):
    for item in value.split(";"):
        item = item.strip()
        if not item:
            return False
        if "=" in item:
            name, _, num = item.partition("=")
            if not (name.replace("_", "").isalnum() and _is_integer_literal(num)):
                return False
        elif item.endswith(")") and "(" in item:
            name, _, rest = item.partition("(")
            lo, sep, hi = rest[:-1].partition("<")
            if not (name.replace("_", "").isalnum() and sep
                    and _is_number_with_optional_unit(lo, field_types)
                    and _is_number_with_optional_unit(hi, field_types)):
                return False
        elif ":" in item:
            code, _, token = item.partition(":")
            if not (code.replace("_", "").isalnum() and code.isupper()
                    and token.replace("_", "").replace(".", "").replace("-", "").isalnum()):
                return False
        else:
            return False
    return True


def _authorize_sanitized(value):
    if len(value) > SANITIZED_DIAGNOSTIC_MAX_LEN:
        return False, "exceeds the declared diagnostic length bound"
    if any(bad in value for bad in _FORBIDDEN_IN_DIAGNOSTIC):
        return False, "contains a forbidden control or path-like sequence"
    if not set(value) <= SANITIZED_DIAGNOSTIC_CHARSET:
        offending = sorted(set(value) - SANITIZED_DIAGNOSTIC_CHARSET)
        return False, f"contains characters outside the declared charset: {offending}"
    return True, None


def _authorize_code_list(value, policy):
    """Authorize a ';'-joined list of controlled enumerated codes."""
    items = value.split(";")
    if any(not item for item in items):
        return False, "an enumerated code list may not contain an empty element"
    unknown = [i for i in items if i not in policy.allowed_values]
    if unknown:
        return False, (
            f"code(s) {unknown} are not in the closed vocabulary declared for "
            f"{policy.artifact}:{policy.field!r}")
    if len(set(items)) != len(items):
        return False, "an enumerated code list may not repeat a code"
    if items != sorted(items):
        return False, (
            "an enumerated code list must be sorted, so that the same evidence always "
            "produces the same bytes")
    return True, None


def _parse_template(template, text):
    names, literals, buf, rest = [], [], "", template
    while "{" in rest:
        head, _, rest = rest.partition("{")
        name, _, rest = rest.partition("}")
        literals.append(buf + head)
        names.append(name)
        buf = ""
    literals.append(buf + rest)
    if not text.startswith(literals[0]):
        return None
    remainder, values = text[len(literals[0]):], {}
    for name, nxt in zip(names, literals[1:]):
        if nxt:
            value, sep, remainder = remainder.partition(nxt)
            if not sep:
                return None
        else:
            value, remainder = remainder, ""
        values[name] = value
    return None if remainder else values


def authorize_occurrence(bundle, occ):
    """Authorize ONE emitted occurrence against ONE bundle.

    Returns `(ok, authorization, reason)`. `authorization` is the credential
    that travels with the value until serialization, so the emitted bytes can
    be re-checked against what was authorized.
    """
    policy = bundle.field_policy.get((occ.artifact, occ.field))
    if policy is None:
        return False, None, (
            f"UNCLASSIFIED output field {occ.artifact}:{occ.field!r}. Every emitted "
            f"string column and JSON path must carry a declared policy classification; "
            f"an unknown artifact, column or JSON path fails closed.")
    if occ.value == "":
        if policy.allow_empty:
            return True, ("declared_empty", policy.category), None
        return False, None, (
            f"empty string is not allowed for required field "
            f"{occ.artifact}:{occ.field!r}")
    cat = policy.category
    if cat in ("structural_enum", "identifier"):
        if occ.value not in policy.allowed_values:
            return False, None, (
                f"value is not in the enumerated {cat} vocabulary declared for "
                f"{occ.artifact}:{occ.field!r}")
        return True, (cat, occ.value), None
    if cat == "filename":
        if ("/" in occ.value or "\\" in occ.value or occ.value.startswith(".")
                or not occ.value.strip()):
            return False, None, "filename must be a bare basename with no path separator"
        return True, ("filename", occ.value), None
    if cat == "typed_label":
        lab = bundle.approved_label(policy.field_kind, occ.value)
        if lab is None:
            other = sorted(k for k in bundle.labels if occ.value in bundle.labels[k])
            extra = f" (it is approved only as {other})" if other else ""
            return False, None, (
                f"value is not approved for field_kind {policy.field_kind!r}{extra}")
        return True, ("typed_label", policy.field_kind, occ.value), None
    if cat == CATEGORY_ENUMERATED_CODE_LIST:
        ok, why = _authorize_code_list(occ.value, policy)
        return ((True, ("enumerated_code_list", tuple(occ.value.split(";"))), None)
                if ok else (False, None, why))
    if cat == "registered_statement":
        for sid in policy.statement_ids:
            st = bundle.statements.get(sid)
            if st is not None and st.text == occ.value:
                return True, ("statement", sid), None
        return False, None, (
            f"text does not exactly match any statement registered for "
            f"{occ.artifact}:{occ.field!r} ({list(policy.statement_ids)}). Registered "
            f"prose is authorized by id and exact text; near-misses are rejected.")
    if cat == "controlled_template":
        if occ.value in policy.allowed_values:
            return True, ("template_sentinel", occ.value), None
        for sid in policy.statement_ids:
            st = bundle.statements.get(sid)
            if st is not None and st.text == occ.value:
                return True, ("statement", sid), None
        tpl = bundle.templates.get(policy.template_id)
        if tpl is None:
            return False, None, f"unknown template_id {policy.template_id!r}"
        values = _parse_template(tpl.template, occ.value)
        if values is None:
            return False, None, (
                f"text does not match template {tpl.template_id!r}; the fixed prose must "
                f"match character for character")
        if set(values) != set(tpl.fields):
            return False, None, f"template {tpl.template_id!r} substitution set mismatch"
        bad = sorted(n for n, v in values.items()
                     if not bundle.field_types[tpl.fields[n]](v))
        if bad:
            return False, None, (
                f"template {tpl.template_id!r} substitution(s) {bad} do not satisfy their "
                f"declared type, so the template cannot carry prose")
        return True, ("template", tpl.template_id, values), None
    if cat == "structured_diagnostic":
        if occ.value in policy.allowed_values:
            return True, ("structured_sentinel", occ.value), None
        if not _authorize_structured(occ.value, bundle.field_types):
            return False, None, (
                "value does not satisfy the declared machine-diagnostic grammar "
                "(name=integer / name(number<number) / CODE:token, joined by ';')")
        return True, ("structured_diagnostic", None), None
    if cat == "sanitized_diagnostic":
        ok, why = _authorize_sanitized(occ.value)
        return (True, ("sanitized_diagnostic", None), None) if ok else (False, None, why)
    return False, None, f"unhandled category {cat!r}"  # pragma: no cover


# ---------------------------------------------------------------------------
# Whole-set validation
# ---------------------------------------------------------------------------

def validate_emitted_records(bundle, payloads, well_keys=(), expected_artifacts=None,
                             serialization_stage="auto"):
    """Validate schema, then authorize every declared string occurrence.

    `payloads` maps artifact filename -> payload. Every declared artifact must
    be present; exact keys/order, requiredness and types are checked
    independently of values. Only after that closed schema pass are string
    values authorized.
    """
    violations, authorizations = [], []
    counts = dict(controlled=0, structural=0, unguaranteed=0,
                  unclassified=0, unauthorized=0, kind_mismatch=0, schema=0)
    stmts, tpls, labels = set(), set(), set()
    declared = set(bundle.artifacts if expected_artifacts is None else expected_artifacts)
    for name in sorted(set(payloads) - declared):
        violations.append(_schema_violation(
            name, None, None, "artifact has no declared output schema"))
        counts["unclassified"] += 1
        counts["schema"] += 1
    for name in sorted(declared - set(payloads)):
        violations.append(_schema_violation(
            name, None, None, "declared artifact was not presented for validation"))
        counts["unclassified"] += 1
        counts["schema"] += 1
    total = 0
    for artifact in sorted(set(payloads) & declared):
        schema_bad = validate_artifact_schema(
            bundle, artifact, payloads[artifact], well_keys,
            serialization_stage=serialization_stage)
        violations.extend(schema_bad)
        counts["schema"] += len(schema_bad)
        for occ in collect_string_fields(bundle, artifact, payloads[artifact], well_keys):
            total += 1
            ok, auth, reason = authorize_occurrence(bundle, occ)
            policy = bundle.field_policy.get((artifact, occ.field))
            cat = policy.category if policy is not None else None
            if cat in CONTROLLED_CATEGORIES:
                counts["controlled"] += 1
            elif cat in STRUCTURAL_CATEGORIES:
                counts["structural"] += 1
            elif cat in UNGUARANTEED_CATEGORIES:
                counts["unguaranteed"] += 1
            if ok:
                authorizations.append((occ, auth))
                if auth[0] == "statement":
                    stmts.add(auth[1])
                elif auth[0] == "template":
                    tpls.add(auth[1])
                elif auth[0] == "typed_label":
                    labels.add((auth[1], auth[2]))
                continue
            if policy is None:
                counts["unclassified"] += 1
            elif cat == "typed_label":
                counts["kind_mismatch"] += 1
                counts["unauthorized"] += 1
            elif cat in CONTROLLED_CATEGORIES:
                counts["unauthorized"] += 1
            violations.append({
                "artifact": artifact, "field": occ.field, "location": occ.location,
                "reason": reason,
                "linter_terms": sorted(set(find_prohibited_lithology_terms(occ.value))),
            })
    return CoverageReport(
        n_string_field_occurrences=total,
        n_controlled_occurrences=counts["controlled"],
        n_structural_occurrences=counts["structural"],
        n_unguaranteed_occurrences=counts["unguaranteed"],
        n_unclassified_fields=counts["unclassified"],
        n_unauthorized_controlled_fields=counts["unauthorized"],
        n_field_kind_mismatches=counts["kind_mismatch"],
        n_schema_violations=counts["schema"],
        violations=tuple(violations), authorizations=tuple(authorizations),
        artifacts_inspected=tuple(sorted(set(payloads) & declared)),
        distinct_statements_used=len(stmts), distinct_templates_used=len(tpls),
        distinct_labels_used=len(labels),
    )


# ---------------------------------------------------------------------------
# The export gate
# ---------------------------------------------------------------------------

class OutputAuthorizationError(ValueError):
    """Raised when an artifact would persist an unauthorized controlled field."""


def _canonical_csv_value(value, kind, nullable, stage):
    import decimal
    if (stage == "pre" and value is None) or (stage == "post" and value == ""):
        if nullable:
            return ("null", None)
    if kind == "string":
        return ("string", value)
    if kind == "integer":
        return ("integer", int(value))
    if kind == "number":
        return ("number", decimal.Decimal(str(value)).normalize().as_tuple())
    if kind == "boolean":
        return ("boolean", value if stage == "pre" else value == "True")
    raise AssertionError(kind)  # pragma: no cover


def canonicalize_emitted_records(bundle, payloads, serialization_stage="auto"):
    """Return a complete, schema-aware semantic representation.

    CSV values are restored to their declared semantic types so that harmless
    serialization representation differences neither mask nor manufacture a
    mutation. Row order and every declared field value remain part of the
    comparison.
    """
    import json as _json
    canonical = []
    for artifact in sorted(payloads):
        payload = payloads[artifact]
        if artifact in bundle.csv_schemas:
            schema = bundle.csv_schemas[artifact]
            stage = _csv_stage(payload, schema, serialization_stage)
            rows = tuple(
                tuple(_canonical_csv_value(
                    row[field], schema.kind(field), field in schema.nullable_fields, stage)
                    for field in schema.columns)
                for row in payload
            )
            canonical.append((artifact, schema.columns, rows))
        else:
            canonical.append((artifact, _json.dumps(
                payload, sort_keys=True, ensure_ascii=False, allow_nan=False,
                separators=(",", ":"))))
    return tuple(canonical)


def export_authorized_outputs(bundle, out_dir, payloads, well_keys=(), writer=None):
    """Schema-driven, transactional two-stage export.

    STAGE 1 validates exact artifact/field schemas, requiredness, types and
    content authorization at the PRE-serialization stage. STAGE 2 writes only
    into a temporary sibling directory, re-reads the bytes, re-validates them
    at the POST-serialization stage, and compares every canonical record and
    value. The official destination is touched only after both stages pass.

    Returns `(report_before, report_after)`. Raises `OutputAuthorizationError`
    on any validation or serializer failure, leaving the destination
    unchanged. Publication uses per-file atomic replacement with rollback:
    failure-atomic for handled process errors, not a claim of multi-file
    atomicity across power loss or operating-system failure.
    """
    import csv as _csv
    import json as _json
    import os as _os
    import pathlib as _pathlib
    import shutil as _shutil
    import tempfile as _tempfile

    out_dir = _pathlib.Path(out_dir)
    before = validate_emitted_records(
        bundle, payloads, well_keys=well_keys, serialization_stage="pre")
    if not before.ok:
        raise OutputAuthorizationError(
            f"refusing to write: {len(before.violations)} schema/authorization "
            f"output field(s); first: "
            f"{before.violations[0] if before.violations else None}")
    canonical_before = canonicalize_emitted_records(
        bundle, payloads, serialization_stage="pre")

    allowed_destination_names = set(bundle.artifacts) | set(bundle.extra_destination_names)
    if out_dir.exists():
        if not out_dir.is_dir():
            raise OutputAuthorizationError(
                "output destination exists and is not a directory")
        stale = sorted(p.name for p in out_dir.iterdir()
                       if p.name not in allowed_destination_names)
        if stale:
            raise OutputAuthorizationError(
                f"output destination contains undeclared stale artifact(s): {stale}")

    parent = out_dir.parent
    parent.mkdir(parents=True, exist_ok=True)
    stage_dir = _pathlib.Path(_tempfile.mkdtemp(
        prefix=f".{out_dir.name}.stage-", dir=str(parent)))
    try:
        try:
            for name, payload in sorted(payloads.items()):
                target = stage_dir / name
                if writer is not None:
                    writer(target, payload)
                elif name.endswith(".json"):
                    # Encode explicitly so the published bytes are identical on
                    # Windows and Linux/Colab. Path.write_text() uses platform
                    # newline translation and previously emitted CRLF on Windows
                    # but LF on the delivery platform.
                    target.write_bytes(
                        (_json.dumps(payload, indent=2, sort_keys=True) + "\n")
                        .encode("utf-8"))
                else:
                    schema = bundle.csv_schemas[name]
                    with open(target, "w", encoding="utf-8", newline="") as fh:
                        w = _csv.DictWriter(fh, fieldnames=list(schema.columns))
                        w.writeheader()
                        w.writerows(payload)
        except OutputAuthorizationError:
            raise
        except Exception as exc:
            raise OutputAuthorizationError(
                f"serializer failed before publication: {exc}") from exc

        staged_names = {p.name for p in stage_dir.iterdir()}
        expected_names = set(payloads)
        if (staged_names != expected_names
                or any(p.is_symlink() or not p.is_file() for p in stage_dir.iterdir())):
            raise OutputAuthorizationError(
                f"serializer produced wrong artifact inventory: "
                f"missing={sorted(expected_names - staged_names)}, "
                f"unknown={sorted(staged_names - expected_names)}")

        reread = {}
        for name in payloads:
            target = stage_dir / name
            try:
                if name.endswith(".json"):
                    reread[name] = _json.loads(target.read_text(encoding="utf-8"))
                else:
                    with open(target, newline="", encoding="utf-8") as fh:
                        reread[name] = list(_csv.DictReader(fh))
            except Exception as exc:
                raise OutputAuthorizationError(
                    f"could not re-read staged artifact {name!r}: {exc}") from exc
        after = validate_emitted_records(
            bundle, reread, well_keys=well_keys, serialization_stage="post")
        if not after.ok:
            raise OutputAuthorizationError(
                f"post-serialization re-check failed: {len(after.violations)} field(s) "
                f"changed or became unauthorized after authorization; first: "
                f"{after.violations[0] if after.violations else None}")
        canonical_after = canonicalize_emitted_records(
            bundle, reread, serialization_stage="post")
        if canonical_after != canonical_before:
            mismatch = next((i for i, pair in enumerate(zip(
                canonical_before, canonical_after)) if pair[0] != pair[1]), None)
            raise OutputAuthorizationError(
                f"post-serialization canonical record mismatch at artifact index "
                f"{mismatch}; an authorized value, field, row order or type changed")

        out_dir.mkdir(parents=True, exist_ok=True)
        backup_dir = _pathlib.Path(_tempfile.mkdtemp(
            prefix=f".{out_dir.name}.backup-", dir=str(parent)))
        replaced, created = [], []
        try:
            for name in sorted(payloads):
                target = out_dir / name
                if target.exists():
                    _shutil.copy2(target, backup_dir / name)
                    replaced.append(name)
                else:
                    created.append(name)
                _os.replace(stage_dir / name, target)
        except Exception:
            for name in created:
                target = out_dir / name
                if target.exists():
                    target.unlink()
            for name in replaced:
                backup = backup_dir / name
                if backup.exists():
                    _os.replace(backup, out_dir / name)
            raise
        finally:
            _shutil.rmtree(backup_dir, ignore_errors=True)
        return before, after
    finally:
        _shutil.rmtree(stage_dir, ignore_errors=True)


# ---------------------------------------------------------------------------
# Faithfulness harness: the LOCKED Increment 6 registries, as a bundle
# ---------------------------------------------------------------------------

def bundle_from_locked_increment6():
    """Build a `PolicyBundle` from the LOCKED Increment 6.1.7 registries.

    Used by `tests/test_overburden_policy.py` to demonstrate that this
    generalized engine reproduces the locked engine's decision on every
    occurrence of the real, packaged Increment 6 records. It reads the locked
    module; it never writes to it.
    """
    from p2mem.io import output_policy as locked
    from p2mem.wellframe_models import APPROVED_LABELS

    labels = [lab for bucket in APPROVED_LABELS.values() for lab in bucket.values()]
    return PolicyBundle(
        name="locked_increment6",
        field_policies=tuple(locked.OUTPUT_FIELD_POLICY.values()),
        statements=tuple(locked.OUTPUT_STATEMENTS.values()),
        templates=tuple(locked.OUTPUT_TEMPLATES.values()),
        labels=tuple(labels),
        csv_schemas=locked.CSV_SCHEMAS,
        manifest_artifact=locked.MANIFEST_ARTIFACT,
        json_object_keys=locked.JSON_OBJECT_KEYS,
        json_list_item_kinds=locked.JSON_LIST_ITEM_KINDS,
        json_boolean_paths=locked.JSON_BOOLEAN_PATHS,
        json_integer_paths=locked.JSON_INTEGER_PATHS,
        json_number_paths=locked.JSON_NUMBER_PATHS,
        json_nullable_paths=locked.JSON_NULLABLE_PATHS,
        approved_well_keys=locked.APPROVED_WELL_KEYS,
        field_types=locked.OUTPUT_FIELD_TYPES,
    )


#### `p2mem/io/overburden_registry.py` - the CONTRACT: nine exact closed schemas and the complete field classification

Every statement, template and label here is enumerated from the records the export path
actually emits. An emitted string with no entry here fails closed at the export boundary.

In [ ]:
%%writefile p2mem/io/overburden_registry.py
"""
The Increment 7 output-policy REGISTRIES: exact closed schemas, the complete
field classification, and every statement, template and label this increment
is permitted to persist.

Kept in its own module so that `p2mem.io.overburden_policy` stays a readable
ENGINE and this file stays a readable CONTRACT. Nothing here executes logic;
everything here is a declaration that the engine enforces.

Provenance rule, inherited from the locked Increment 6.1.7 design: every entry
below is enumerated from the ACTUAL records the Increment 7 export path emits.
A statement that no artifact emits does not belong here, and an emitted string
with no entry here fails closed at the export boundary.
"""

from p2mem.io.output_policy import CsvArtifactSchema
from p2mem.wellframe_models import ApprovedLabel, RegisteredStatement, RegisteredTemplate
from p2mem.io.overburden_policy import (
    CATEGORY_ENUMERATED_CODE_LIST, FieldPolicy, OVERBURDEN_FIELD_TYPES, PolicyBundle,
    SCOPE_OUTPUT_INC7,
)
from p2mem.overburden import SIGN_CONVENTION_ID
from p2mem.overburden_models import (
    SCENARIO_BASIS_BASE, SCENARIO_BASIS_HIGH, SCENARIO_BASIS_LOW,
    VALID_GAP_CLASSES, VALID_GAP_DISPOSITIONS, VALID_LIMITING_REASONS,
    VALID_OVERBURDEN_STATUSES, VALID_SCENARIO_NAMES, VALID_SEABED_BASES,
)

__all__ = [
    "OVERBURDEN_ARTIFACTS", "OVERBURDEN_CSV_SCHEMAS", "OVERBURDEN_MANIFEST_ARTIFACT",
    "OVERBURDEN_STATEMENTS", "OVERBURDEN_TEMPLATES", "OVERBURDEN_LABELS",
    "APPROVED_WELL_KEYS", "OVERBURDEN_BUNDLE", "ISSUE_CODES", "NOT_IMPLEMENTED_TOKENS",
    "ASSURANCE_TIER_VALUE", "INCREMENT_TITLE",
]

ASSURANCE_TIER_VALUE = "Tier C - Screening-Level / Uncalibrated Educational"
INCREMENT_TITLE = "Density QC, Density-Coverage Qualification, and Vertical Overburden-Stress Framework"

OVERBURDEN_MANIFEST_ARTIFACT = "density_overburden_manifest.json"

APPROVED_WELL_KEYS = frozenset((
    "Boreas_1", "Poseidon_2", "Poseidon_North_1", "Proteus_1ST2",
))

#: Sentinel emitted in a curve-identity column when a well has no density curve.
NA_DENSITY = "not_applicable_no_density_curve"

ISSUE_CODES = (
    "DENSITY_CURVE_ABSENT",
    "DENSITY_CONVERSION_FUNCTION_UNEXPECTED",
    "SCREENING_BOUND_FAILURES_PRESENT",
    "MEASURED_MAXIMUM_NEAR_SCREENING_BOUND",
    "SHALLOW_DENSITY_COLUMN_UNRESOLVED",
    "SEABED_DATUM_UNRESOLVED",
    "LONG_INTERNAL_GAP_PRESENT",
    "TERMINAL_DENSITY_COLUMN_UNRESOLVED",
    "STRESS_COLUMN_TRUNCATED_AT_UNRESOLVED_GAP",
    "ABSOLUTE_OVERBURDEN_NOT_SUPPORTED",
    "WELL_FRAME_ASSEMBLY_FAILED",
)

NOT_IMPLEMENTED_TOKENS = (
    "pore_pressure_prediction",
    "sonic_normal_compaction_trend_fitting",
    "resistivity_normal_compaction_trend_fitting",
    "eaton_method",
    "bowers_method",
    "equivalent_depth_method",
    "drilling_exponent_method",
    "effective_stress_calculation",
    "dynamic_elastic_property_modelling",
    "static_elastic_property_modelling",
    "rock_strength_modelling",
    "horizontal_stress_calculation",
    "stress_calibration",
    "mud_window_calculation",
    "breakout_analysis",
    "tensile_fracture_analysis",
    "wellbore_stability_analysis",
    "named_lithology_assignment",
    "mineralogical_interpretation",
    "shallow_density_reconstruction_from_a_fitted_trend",
    "density_extrapolation_beyond_measured_coverage",
)

_UNIT_VALUES = (
    "kg/m3; m; Pa; MPa",
    "m",
    "kg/m3",
    "Pa; MPa",
    "count",
    "dimensionless",
)

_DEPTH_BASIS_VALUES = ("petrel_source_trace", "minimum_curvature_computed")
_DEPTH_MAP_STATUS_VALUES = (
    "fully_mapped_within_survey_coverage",
    "partially_mapped_survey_coverage_gap",
    "not_mapped_no_survey",
)
_INTEGRATION_METHOD_VALUES = ("trapezoidal_in_true_vertical_depth",)
_INTEGRATION_COORDINATE_VALUES = ("tvdss_m",)
_CURVE_NAME_VALUES = ("RHOB_kg_m3", NA_DENSITY)
_SOURCE_CURVE_VALUES = ("RHOB", NA_DENSITY)
_RAW_UNIT_VALUES = ("g/cc", "g/cm3", "G/C3", NA_DENSITY)
_CANONICAL_UNIT_VALUES = ("kg/m3", NA_DENSITY)
_CONVERSION_VALUES = ("gcc_to_kgm3", "identity", NA_DENSITY)


# ---------------------------------------------------------------------------
# Registered statements - the exact prose this increment may persist
# ---------------------------------------------------------------------------

_S = SCOPE_OUTPUT_INC7

_LIM_DATA = (
    "Screening-level, uncalibrated. This project holds no core density, no measured "
    "seawater density, no local gravity survey, and no shallow-density control, so no "
    "number derived here is calibrated. Asserts no named lithology.")

_STATEMENT_LIST = (
    RegisteredStatement(
        statement_id="availability_statistics_basis",
        scope=_S,
        text="Counts and depths measured over this well's OWN contract-resolved density curve, in the canonical unit the LOCKED Increment 2.1.1 loader recorded for it. No environmental correction, rescaling, smoothing, despiking, replacement, or extrapolation has been applied, and no value has been modified. These numbers describe the curve as recorded and imply no lithology.",
        purpose="Persisted as the 'statistics_basis' field of density_availability_inventory.csv.",
        provenance="Enumerated from the ACTUAL emitted density_availability_inventory.csv record."),
    RegisteredStatement(
        statement_id="availability_limitations",
        scope=_S,
        text="AVAILABILITY AND COVERAGE ONLY. Coverage is not fitness: a density curve can be complete over its own logged interval and still be unable to support an absolute overburden integral, because the integral needs the column above it as well. " + _LIM_DATA,
        purpose="Persisted as the 'limitations' field of density_availability_inventory.csv.",
        provenance="Enumerated from the ACTUAL emitted density_availability_inventory.csv record."),
    RegisteredStatement(
        statement_id="qc_statistics_basis",
        scope=_S,
        text="Mask counts are counts of samples satisfying explicitly declared, configuration-driven criteria; the screening plausibility band is a SCREENING CRITERION chosen to catch tool, hole-condition and processing artefacts, and is not a statement of universal geological truth or of what rock exists in this well. A sample failing a criterion is MASKED, never modified, clipped, rescaled or replaced.",
        purpose="Persisted as the 'statistics_basis' field of density_qc_summary.csv.",
        provenance="Enumerated from the ACTUAL emitted density_qc_summary.csv record."),
    RegisteredStatement(
        statement_id="qc_limitations",
        scope=_S,
        text="QC MASKS ONLY. Passing every mask makes a sample admissible to an integral; it does not make it correct, and it assigns no lithology. The original resolved density array is unchanged: any conditioned representation lives in a separately named array with its own mask. " + _LIM_DATA,
        purpose="Persisted as the 'limitations' field of density_qc_summary.csv.",
        provenance="Enumerated from the ACTUAL emitted density_qc_summary.csv record."),
    RegisteredStatement(
        statement_id="gap_inventory_limitations",
        scope=_S,
        text="FACTUAL GAP EXTENTS ONLY. A shallow gap between the seabed and the first valid density sample is an UNMEASURED COLUMN, not a dropout, and is never bridged at any threshold. A bridged short internal gap carries linearly interpolated density in true vertical depth; its samples are conditioned, not measured, and its contribution to any stress is reported separately. A long internal gap is never bridged and truncates the integrable column. " + _LIM_DATA,
        purpose="Persisted as the 'limitations' field of density_gap_inventory.csv.",
        provenance="Enumerated from the ACTUAL emitted density_gap_inventory.csv record."),
    RegisteredStatement(
        statement_id="eligibility_purpose",
        scope=_S,
        text="States what the MEASURED density coverage of this well can and cannot support for a vertical overburden-stress calculation. The status is DERIVED from measured coverage and QC results; no well name participates in the derivation and no configuration key names a well.",
        purpose="Persisted as the 'purpose' field of overburden_eligibility_summary.csv.",
        provenance="Enumerated from the ACTUAL emitted overburden_eligibility_summary.csv record."),
    RegisteredStatement(
        statement_id="eligibility_limitations",
        scope=_S,
        text="ELIGIBILITY IS NOT VALIDITY. A measured increment is the integral of recorded density over the supported interval ONLY; it is not an absolute vertical stress and must never be read as one. An absolute stress additionally requires the density column from the datum or seabed down to the evaluation depth, which no amount of coverage inside the logged interval can supply. Where that column is unmeasured, the deficit is reported as a thickness and bracketed as a transparent scenario, never filled. " + _LIM_DATA,
        purpose="Persisted as the 'limitations' field of overburden_eligibility_summary.csv.",
        provenance="Enumerated from the ACTUAL emitted overburden_eligibility_summary.csv record."),
    RegisteredStatement(
        statement_id="profile_limitations",
        scope=_S,
        text="MEASURED INCREMENT, NOT ABSOLUTE STRESS. The cumulative value is the integral of rho*g*dz from this well's FIRST eligible density sample down to the node, taken in true vertical depth by trapezoidal quadrature. It is zero at the first node by definition and excludes every column above it, including the water column and any unmeasured shallow section. Reported nodes are SELECTED existing samples at the configured step; no value here is interpolated for reporting, and the integral itself uses every eligible sample. " + _LIM_DATA,
        purpose="Persisted as the 'limitations' field of vertical_stress_profile.csv.",
        provenance="Enumerated from the ACTUAL emitted vertical_stress_profile.csv record."),
    RegisteredStatement(
        statement_id="scenario_limitations",
        scope=_S,
        text="TRANSPARENT SCENARIO, NOT A RESULT. Every total in this table depends on an ASSUMED depth-averaged bulk density for an unmeasured column and an ASSUMED seawater density. The low endpoint is conditional on a fully saturated column; the project does not establish the unknown interval's fluid state. The high endpoint is the same well's measured P05 over samples eligible for measured integration, used only as an illustrative upper scenario motivated by monotonic-compaction reasoning; it is not a physical ceiling or rigorous uncertainty bound. The distinct all-finite P05 remains a factual QC statistic and does not drive the scenario. The base case is the arithmetic midpoint and has NO evidentiary support. No depth-dependent density function is fitted anywhere, no lithology is assumed, and the base case is never a best estimate. Read the full sensitivity spread, never a single row. " + _LIM_DATA,
        purpose="Persisted as the 'limitations' field of shallow_column_scenarios.csv.",
        provenance="Enumerated from the ACTUAL emitted shallow_column_scenarios.csv record."),
    RegisteredStatement(
        statement_id="gap_sensitivity_limitations",
        scope=_S,
        text="SENSITIVITY TO A CONFIGURED CHOICE. Each row re-runs gap conditioning, integration and status derivation at one candidate threshold, so that the effect of the approved threshold is visible rather than assumed. The approved row is flagged; nothing in this table selects a preferred value, and a threshold that bridges more gaps is not thereby better. " + _LIM_DATA,
        purpose="Persisted as the 'limitations' field of gap_threshold_sensitivity.csv.",
        provenance="Enumerated from the ACTUAL emitted gap_threshold_sensitivity.csv record."),
    RegisteredStatement(
        statement_id="manifest_depth_convention_statement",
        scope=_S,
        text="TVD is zero at the well datum and increases downward; the datum elevation is referenced to mean sea level, positive upward; TVDSS = TVD - datum elevation and is positive downward from mean sea level. A downward interval therefore has a positive increment in both coordinates, and dTVD equals dTVDSS exactly. This convention is documented and independently tested in the locked depth-mapping module, and is re-verified against each well frame's own arrays at run time rather than inferred from a variable name.",
        purpose="Persisted as the 'depth_convention_statement' key of the Increment 7 manifest.",
        provenance="Enumerated from the ACTUAL emitted density_overburden_manifest.json record."),
    RegisteredStatement(
        statement_id="manifest_gravity_basis",
        scope=_S,
        text="CGPM standard gravity, exact by definition at 9.80665 m/s2. This is a CONFIGURED ASSUMPTION: no local gravity survey exists for these wells, and every absolute or scenario stress reported here carries it.",
        purpose="Persisted as the 'gravity_basis' key of the Increment 7 manifest.",
        provenance="Enumerated from the ACTUAL emitted density_overburden_manifest.json record."),
    RegisteredStatement(
        statement_id="manifest_assumption_statement",
        scope=_S,
        text="Every value in this register is a CONFIGURED SCREENING ASSUMPTION reviewed in config/overburden_stress.yml, not a measurement. The seawater density is assumed, not measured; the screening plausibility band is a project criterion, not a geological limit; the gap threshold is a heuristic whose effect is published as a sensitivity; and the shallow-column endpoints are illustrative scenarios, not physical bounds or a model.",
        purpose="Persisted as the 'statement' key of the manifest assumption register.",
        provenance="Enumerated from the ACTUAL emitted density_overburden_manifest.json record."),
    RegisteredStatement(
        statement_id="manifest_calibration_statement",
        scope=_S,
        text="No calibration data of any kind are available to this project: no RFT/MDT/DST pressure, no LOT/XLOT/DFIT stress fit, no core or log-calibrated density control, no measured seawater density, and no local gravity survey. Nothing in Increment 7 is calibrated, and no result here may be described as calibrated truth.",
        purpose="Persisted as the 'statement' key of the manifest calibration block.",
        provenance="Enumerated from the ACTUAL emitted density_overburden_manifest.json record."),
    RegisteredStatement(
        statement_id="manifest_named_lithology_statement",
        scope=_S,
        text="No named lithology is assigned anywhere in Increment 7. The density screening band, the gap policy, the shallow-column bracket and the eligibility ladder are all expressed in physical and coverage terms, and none of them is justified by an assumed rock type.",
        purpose="Persisted as the 'named_lithology_statement' key of the Increment 7 manifest.",
        provenance="Enumerated from the ACTUAL emitted density_overburden_manifest.json record."),
    RegisteredStatement(
        statement_id="manifest_lithology_model",
        scope=_S,
        text="positive_authorization_of_every_emitted_controlled_field",
        purpose="Persisted as the 'model' key of the manifest lithology-validation block.",
        provenance="Enumerated from the ACTUAL emitted density_overburden_manifest.json record."),
    RegisteredStatement(
        statement_id="manifest_lithology_derivation",
        scope=_S,
        text="Derived from the ACTUAL emitted records, not from a reconstructed parallel scope: every schema-declared string occurrence in every artifact this run serializes is collected by schema, classified, and authorized against a closed registry, before and after serialization.",
        purpose="Persisted as the 'derivation' key of the manifest lithology-validation block.",
        provenance="Enumerated from the ACTUAL emitted density_overburden_manifest.json record."),
    RegisteredStatement(
        statement_id="limitation_no_absolute_without_full_column",
        scope=_S,
        text="An absolute vertical overburden stress requires density coverage from the relevant datum or seabed to the evaluation depth. Where that column is not measured, Increment 7 reports a partial measured increment or a transparent screening bracket, and never a single absolute curve presented as measured truth.",
        purpose="Persisted as one entry of the manifest 'limitations' list.",
        provenance="Enumerated from the ACTUAL emitted density_overburden_manifest.json record."),
    RegisteredStatement(
        statement_id="limitation_no_density_repair",
        scope=_S,
        text="No density value is clipped, rescaled, smoothed, despiked, replaced or extrapolated anywhere in Increment 7. Invalid samples are masked; short internal gaps may be linearly bridged in true vertical depth into a separately named array whose contribution is reported separately; nothing else is filled.",
        purpose="Persisted as one entry of the manifest 'limitations' list.",
        provenance="Enumerated from the ACTUAL emitted density_overburden_manifest.json record."),
    RegisteredStatement(
        statement_id="limitation_assumed_components",
        scope=_S,
        text="The water column, the unresolved shallow column and standard gravity are ASSUMPTIONS, not measurements. Every reported stress is partitioned so that the measured formation contribution, the conditioned bridged contribution and the assumed contributions are separately visible, and no total is reported while any component is unresolved.",
        purpose="Persisted as one entry of the manifest 'limitations' list.",
        provenance="Enumerated from the ACTUAL emitted density_overburden_manifest.json record."),
    RegisteredStatement(
        statement_id="limitation_no_pore_pressure",
        scope=_S,
        text="Increment 7 computes no pore pressure, no effective stress, no normal compaction trend, no elastic property, no rock strength, no horizontal stress, no mud window and no wellbore-stability result. Those methods are deferred to later increments and are absent from this version; importing them will fail until they exist.",
        purpose="Persisted as one entry of the manifest 'limitations' list.",
        provenance="Enumerated from the ACTUAL emitted density_overburden_manifest.json record."),
    RegisteredStatement(
        statement_id="limitation_seabed_provenance",
        scope=_S,
        text="A seabed datum is read only from the LOCKED Increment 5 survey-corrected marker table for the well it was picked in. A well with no approved formation-top file has no determinable seabed, no determinable water column and no determinable shallow-gap thickness; those are reported as unresolved, never as zero and never borrowed from another well.",
        purpose="Persisted as one entry of the manifest 'limitations' list.",
        provenance="Enumerated from the ACTUAL emitted density_overburden_manifest.json record."),
)

OVERBURDEN_STATEMENTS = {s.statement_id: s for s in _STATEMENT_LIST}

_TEMPLATE_LIST = (
    RegisteredTemplate(
        template_id="scenario_basis",
        scope=_S,
        template=(
            "Screening scenario. The depth-averaged bulk density of the unmeasured column is "
            "ASSUMED at {assumed_density_kg_m3} kg/m3 over {unresolved_thickness_m} m of true "
            "vertical depth. Of the reported total, {assumed_fraction_pct} percent comes "
            "from configured column assumptions, {conditioned_fraction_pct} percent from "
            "explicitly bridged density, and {measured_fraction_pct} percent from strictly "
            "measured density. This is not a calibrated value and is not an estimate of "
            "the true vertical stress."),
        fields={"assumed_density_kg_m3": "decimal",
                "unresolved_thickness_m": "decimal",
                "assumed_fraction_pct": "decimal",
                "conditioned_fraction_pct": "decimal",
                "measured_fraction_pct": "decimal"},
        purpose="Persisted as the 'scenario_basis' field of shallow_column_scenarios.csv, "
                "carrying the measured three-component fraction disclosure the policy "
                "requires.",
        provenance="Enumerated from the ACTUAL emitted shallow_column_scenarios.csv record."),
    RegisteredTemplate(
        template_id="empty_mnemonic_ordinal",
        scope=_S,
        template="(empty mnemonic, ordinal {ordinal})",
        fields={"ordinal": "integer"},
        purpose="Persisted as the 'raw_mnemonic' field of "
                "density_availability_inventory.csv for a density column whose LAS ~C "
                "line carries an EMPTY mnemonic. The locked Increment 2.1.1 loader "
                "synthesises exactly this descriptor from the column's ordinal position; "
                "Increment 7 reproduces the loader's own identity string rather than "
                "inventing a mnemonic the file does not contain.",
        provenance="Enumerated from the ACTUAL emitted "
                   "density_availability_inventory.csv record."),
)

OVERBURDEN_TEMPLATES = {t.template_id: t for t in _TEMPLATE_LIST}


# ---------------------------------------------------------------------------
# Approved labels - verdicts a well is stamped with
# ---------------------------------------------------------------------------

def _labels(field_kind, values, purpose):
    return tuple(ApprovedLabel(
        value=v, field_kind=field_kind, purpose=purpose,
        provenance="Enumerated from the ACTUAL persisted Increment 7 export; every label "
                   "this increment writes is a member of this registry.")
        for v in values)


OVERBURDEN_LABELS = (
    _labels("overburden_status", VALID_OVERBURDEN_STATUSES,
            "Derived verdict on what this well's measured density coverage can support.")
    + _labels("seabed_basis", VALID_SEABED_BASES,
              "Where this well's seabed datum came from, or that it is not determinable.")
    + _labels("gap_class", VALID_GAP_CLASSES,
              "Structural classification of one gap in the density column.")
    + _labels("gap_disposition", VALID_GAP_DISPOSITIONS,
              "What the configured policy actually did about one gap.")
    + _labels("assumed_density_basis",
              (SCENARIO_BASIS_LOW, SCENARIO_BASIS_BASE, SCENARIO_BASIS_HIGH),
              "How one shallow-column scenario's assumed density was bounded.")
    + _labels("density_source", ("measured_rhob", "bridged_linear_in_tvd"),
              "Whether one profile node's density was recorded or conditioned.")
    + _labels("evidence_class", ("measured", "assumed_configured",
                                 "derived_locked_prior_increment"),
              "The evidential standing of the quantity in this record.")
    + _labels("calibration_status", ("uncalibrated_screening_only",),
              "Calibration standing; this project holds no calibration data of any kind.")
)


# ---------------------------------------------------------------------------
# Closed CSV schemas
# ---------------------------------------------------------------------------

AVAILABILITY = "density_availability_inventory.csv"
QC = "density_qc_summary.csv"
GAPS = "density_gap_inventory.csv"
ELIGIBILITY = "overburden_eligibility_summary.csv"
PROFILE = "vertical_stress_profile.csv"
SCENARIOS = "shallow_column_scenarios.csv"
SENSITIVITY = "gap_threshold_sensitivity.csv"
ISSUES = "density_overburden_issues.csv"

OVERBURDEN_CSV_SCHEMAS = {
    AVAILABILITY: CsvArtifactSchema(
        columns=(
            "well_key", "source_las_filename", "source_survey_filename",
            "density_curve_present", "canonical_curve_name", "source_curve_name",
            "raw_mnemonic", "raw_unit", "canonical_unit", "conversion_function",
            "unit_resolved", "conversion_confirmed", "evidence_class",
            "n_samples", "n_finite_rhob", "n_non_finite_rhob", "n_non_positive_rhob",
            "n_below_screening_min", "n_above_screening_max",
            "n_screening_bound_failures", "n_in_screening_band", "n_depth_mapped",
            "n_depth_unmapped", "n_outside_survey_coverage", "n_eligible",
            "first_valid_md_m", "last_valid_md_m", "first_valid_tvd_m",
            "last_valid_tvd_m", "first_valid_tvdss_m", "last_valid_tvdss_m",
            "gross_coverage_md_m", "gross_coverage_tvd_m", "median_md_step_m",
            "depth_basis_used", "depth_map_status", "survey_md_min_m", "survey_md_max_m",
            "unit", "assurance_tier", "statistics_basis", "limitations",
        ),
        integer_fields=("n_samples", "n_finite_rhob", "n_non_finite_rhob",
                        "n_non_positive_rhob", "n_below_screening_min",
                        "n_above_screening_max", "n_screening_bound_failures",
                        "n_in_screening_band", "n_depth_mapped", "n_depth_unmapped",
                        "n_outside_survey_coverage", "n_eligible"),
        number_fields=("first_valid_md_m", "last_valid_md_m", "first_valid_tvd_m",
                       "last_valid_tvd_m", "first_valid_tvdss_m", "last_valid_tvdss_m",
                       "gross_coverage_md_m", "gross_coverage_tvd_m", "median_md_step_m",
                       "survey_md_min_m", "survey_md_max_m"),
        boolean_fields=("density_curve_present", "unit_resolved", "conversion_confirmed"),
        nullable_fields=("first_valid_md_m", "last_valid_md_m", "first_valid_tvd_m",
                         "last_valid_tvd_m", "first_valid_tvdss_m", "last_valid_tvdss_m",
                         "gross_coverage_md_m", "gross_coverage_tvd_m",
                         "median_md_step_m")),
    QC: CsvArtifactSchema(
        columns=(
            "well_key", "canonical_curve_name", "screening_min_kg_m3",
            "screening_max_kg_m3", "bounds_are_inclusive", "n_source_value_present",
            "n_finite_numeric_density", "n_unit_resolved", "n_screening_range_plausible",
            "n_below_seabed_sample", "n_depth_mapping_valid", "n_within_survey_coverage",
            "n_eligible_for_measured_integration", "n_bridged_short_gap",
            "n_unresolved_long_gap", "n_unresolved_shallow_column",
            "n_unresolved_terminal_column", "rhob_min_kg_m3", "rhob_p05_kg_m3",
            "rhob_median_kg_m3", "rhob_p95_kg_m3", "rhob_max_kg_m3",
            "rhob_eligible_p05_kg_m3", "seabed_basis",
            "seabed_mdrt_m", "seabed_tvd_m", "seabed_tvdss_m", "shallow_gap_md_m",
            "shallow_gap_tvd_m", "terminal_gap_md_m", "terminal_gap_tvd_m",
            "n_internal_gaps", "n_internal_gap_samples", "longest_internal_gap_md_m",
            "longest_internal_gap_tvd_m", "mask_counts", "unit", "assurance_tier",
            "statistics_basis", "limitations",
        ),
        integer_fields=("n_source_value_present", "n_finite_numeric_density",
                        "n_unit_resolved", "n_screening_range_plausible",
                        "n_below_seabed_sample", "n_depth_mapping_valid",
                        "n_within_survey_coverage",
                        "n_eligible_for_measured_integration", "n_bridged_short_gap",
                        "n_unresolved_long_gap", "n_unresolved_shallow_column",
                        "n_unresolved_terminal_column", "n_internal_gaps",
                        "n_internal_gap_samples"),
        number_fields=("screening_min_kg_m3", "screening_max_kg_m3", "rhob_min_kg_m3",
                       "rhob_p05_kg_m3", "rhob_median_kg_m3", "rhob_p95_kg_m3",
                       "rhob_max_kg_m3", "rhob_eligible_p05_kg_m3",
                       "seabed_mdrt_m", "seabed_tvd_m",
                       "seabed_tvdss_m", "shallow_gap_md_m", "shallow_gap_tvd_m",
                       "terminal_gap_md_m", "terminal_gap_tvd_m",
                       "longest_internal_gap_md_m", "longest_internal_gap_tvd_m"),
        boolean_fields=("bounds_are_inclusive",),
        nullable_fields=("n_below_seabed_sample", "rhob_min_kg_m3", "rhob_p05_kg_m3",
                         "rhob_median_kg_m3", "rhob_p95_kg_m3", "rhob_max_kg_m3",
                         "rhob_eligible_p05_kg_m3",
                         "seabed_mdrt_m", "seabed_tvd_m", "seabed_tvdss_m",
                         "shallow_gap_md_m", "shallow_gap_tvd_m", "terminal_gap_md_m",
                         "terminal_gap_tvd_m", "longest_internal_gap_md_m",
                         "longest_internal_gap_tvd_m")),
    GAPS: CsvArtifactSchema(
        columns=(
            "well_key", "gap_index", "gap_class", "disposition", "n_samples",
            "start_index", "end_index", "md_start_m", "md_end_m", "thickness_md_m",
            "tvd_start_m", "tvd_end_m", "thickness_tvd_m", "threshold_tvd_m",
            "bounding_density_above_kg_m3", "bounding_density_below_kg_m3",
            "unit", "assurance_tier", "limitations",
        ),
        integer_fields=("gap_index", "n_samples", "start_index", "end_index"),
        number_fields=("md_start_m", "md_end_m", "thickness_md_m", "tvd_start_m",
                       "tvd_end_m", "thickness_tvd_m", "threshold_tvd_m",
                       "bounding_density_above_kg_m3", "bounding_density_below_kg_m3"),
        nullable_fields=("start_index", "end_index", "md_start_m", "md_end_m",
                         "thickness_md_m", "tvd_start_m", "tvd_end_m",
                         "thickness_tvd_m", "threshold_tvd_m",
                         "bounding_density_above_kg_m3", "bounding_density_below_kg_m3"),
        min_rows=0),
    ELIGIBILITY: CsvArtifactSchema(
        columns=(
            "well_key", "overburden_status", "limiting_reasons", "seabed_resolved",
            "seabed_basis", "n_eligible_samples", "n_bridged_samples",
            "n_unresolved_internal_gaps", "n_unresolved_long_gaps",
            "column_uninterrupted",
            "column_truncated_at_unresolved_gap", "n_eligible_samples_below_truncation",
            "eligible_top_tvd_m", "eligible_base_tvd_m", "eligible_top_tvdss_m",
            "eligible_base_tvdss_m", "measured_thickness_tvd_m",
            "water_column_thickness_tvd_m", "shallow_unresolved_thickness_tvd_m",
            "terminal_unresolved_thickness_tvd_m",
            "unresolved_long_gap_thickness_tvd_m", "measured_increment_pa",
            "measured_increment_mpa", "bridged_increment_pa", "bridged_increment_mpa",
            "absolute_stress_supported", "gravity_m_s2", "integration_method",
            "integration_coordinate", "depth_convention", "evidence_class",
            "calibration_status", "unit", "assurance_tier", "purpose", "limitations",
        ),
        integer_fields=("n_eligible_samples", "n_bridged_samples",
                        "n_unresolved_internal_gaps", "n_unresolved_long_gaps",
                        "n_eligible_samples_below_truncation"),
        number_fields=("eligible_top_tvd_m", "eligible_base_tvd_m",
                       "eligible_top_tvdss_m", "eligible_base_tvdss_m",
                       "measured_thickness_tvd_m", "water_column_thickness_tvd_m",
                       "shallow_unresolved_thickness_tvd_m",
                       "terminal_unresolved_thickness_tvd_m",
                       "unresolved_long_gap_thickness_tvd_m", "measured_increment_pa",
                       "measured_increment_mpa", "bridged_increment_pa",
                       "bridged_increment_mpa", "gravity_m_s2"),
        boolean_fields=("seabed_resolved", "column_uninterrupted",
                        "column_truncated_at_unresolved_gap",
                        "absolute_stress_supported"),
        nullable_fields=("eligible_top_tvd_m", "eligible_base_tvd_m",
                         "eligible_top_tvdss_m", "eligible_base_tvdss_m",
                         "measured_thickness_tvd_m", "water_column_thickness_tvd_m",
                         "shallow_unresolved_thickness_tvd_m",
                         "terminal_unresolved_thickness_tvd_m",
                         "measured_increment_pa", "measured_increment_mpa",
                         "bridged_increment_pa", "bridged_increment_mpa")),
    PROFILE: CsvArtifactSchema(
        columns=(
            "well_key", "node_index", "md_m", "tvd_m", "tvdss_m", "rhob_kg_m3",
            "density_source", "cumulative_measured_increment_pa",
            "cumulative_measured_increment_mpa", "gravity_m_s2",
            "integration_coordinate", "evidence_class", "calibration_status",
            "unit", "assurance_tier", "limitations",
        ),
        integer_fields=("node_index",),
        number_fields=("md_m", "tvd_m", "tvdss_m", "rhob_kg_m3",
                       "cumulative_measured_increment_pa",
                       "cumulative_measured_increment_mpa", "gravity_m_s2"),
        min_rows=0),
    SCENARIOS: CsvArtifactSchema(
        columns=(
            "well_key", "scenario_name", "assumed_shallow_density_kg_m3",
            "assumed_density_basis", "seawater_density_kg_m3", "gravity_m_s2",
            "water_column_thickness_tvd_m", "unresolved_thickness_tvd_m",
            "measured_thickness_tvd_m", "water_column_stress_pa",
            "unresolved_shallow_stress_pa", "measured_formation_stress_pa",
            "bridged_gap_stress_pa", "total_stress_pa", "total_stress_mpa",
            "assumed_fraction_of_total", "conditioned_fraction_of_total",
            "measured_fraction_of_total",
            "evidence_class", "calibration_status", "unit", "assurance_tier",
            "scenario_basis", "limitations",
        ),
        number_fields=("assumed_shallow_density_kg_m3", "seawater_density_kg_m3",
                       "gravity_m_s2", "water_column_thickness_tvd_m",
                       "unresolved_thickness_tvd_m", "measured_thickness_tvd_m",
                       "water_column_stress_pa", "unresolved_shallow_stress_pa",
                       "measured_formation_stress_pa", "bridged_gap_stress_pa",
                       "total_stress_pa", "total_stress_mpa",
                       "assumed_fraction_of_total", "conditioned_fraction_of_total",
                       "measured_fraction_of_total"),
        min_rows=0),
    SENSITIVITY: CsvArtifactSchema(
        columns=(
            "well_key", "threshold_tvd_m", "is_approved_threshold", "n_bridged_gaps",
            "n_bridged_samples", "bridged_thickness_tvd_m", "n_long_gaps",
            "long_gap_thickness_tvd_m", "n_eligible_or_bridged_samples",
            "total_measured_increment_pa", "total_measured_increment_mpa",
            "bridged_increment_pa", "derived_status", "unit", "assurance_tier",
            "limitations",
        ),
        integer_fields=("n_bridged_gaps", "n_bridged_samples", "n_long_gaps",
                        "n_eligible_or_bridged_samples"),
        number_fields=("threshold_tvd_m", "bridged_thickness_tvd_m",
                       "long_gap_thickness_tvd_m", "total_measured_increment_pa",
                       "total_measured_increment_mpa", "bridged_increment_pa"),
        boolean_fields=("is_approved_threshold",),
        nullable_fields=("total_measured_increment_pa", "total_measured_increment_mpa",
                         "bridged_increment_pa")),
    ISSUES: CsvArtifactSchema(
        columns=("severity", "code", "context", "message", "assurance_tier"),
        min_rows=0),
}


# ---------------------------------------------------------------------------
# Field classification
# ---------------------------------------------------------------------------

def _enum(artifact, field, values):
    return FieldPolicy(artifact, field, "structural_enum", allowed_values=values)


def _label(artifact, field, kind):
    return FieldPolicy(artifact, field, "typed_label", field_kind=kind)


def _stmt(artifact, field, *sids):
    return FieldPolicy(artifact, field, "registered_statement", statement_ids=sids)


_TIER = (ASSURANCE_TIER_VALUE,)
_WELLS = tuple(sorted(APPROVED_WELL_KEYS))

_FIELD_POLICIES = (
    # --- density_availability_inventory.csv -------------------------------
    FieldPolicy(AVAILABILITY, "well_key", "identifier", allowed_values=_WELLS),
    FieldPolicy(AVAILABILITY, "source_las_filename", "filename"),
    FieldPolicy(AVAILABILITY, "source_survey_filename", "filename"),
    _enum(AVAILABILITY, "canonical_curve_name", _CURVE_NAME_VALUES),
    _enum(AVAILABILITY, "source_curve_name", _SOURCE_CURVE_VALUES),
    FieldPolicy(AVAILABILITY, "raw_mnemonic", "controlled_template",
                allowed_values=_SOURCE_CURVE_VALUES,
                template_id="empty_mnemonic_ordinal"),
    _enum(AVAILABILITY, "raw_unit", _RAW_UNIT_VALUES),
    _enum(AVAILABILITY, "canonical_unit", _CANONICAL_UNIT_VALUES),
    _enum(AVAILABILITY, "conversion_function", _CONVERSION_VALUES),
    _label(AVAILABILITY, "evidence_class", "evidence_class"),
    _enum(AVAILABILITY, "depth_basis_used", _DEPTH_BASIS_VALUES),
    _enum(AVAILABILITY, "depth_map_status", _DEPTH_MAP_STATUS_VALUES),
    _enum(AVAILABILITY, "unit", _UNIT_VALUES),
    _enum(AVAILABILITY, "assurance_tier", _TIER),
    _stmt(AVAILABILITY, "statistics_basis", "availability_statistics_basis"),
    _stmt(AVAILABILITY, "limitations", "availability_limitations"),

    # --- density_qc_summary.csv -------------------------------------------
    FieldPolicy(QC, "well_key", "identifier", allowed_values=_WELLS),
    _enum(QC, "canonical_curve_name", _CURVE_NAME_VALUES),
    _label(QC, "seabed_basis", "seabed_basis"),
    FieldPolicy(QC, "mask_counts", "structured_diagnostic"),
    _enum(QC, "unit", _UNIT_VALUES),
    _enum(QC, "assurance_tier", _TIER),
    _stmt(QC, "statistics_basis", "qc_statistics_basis"),
    _stmt(QC, "limitations", "qc_limitations"),

    # --- density_gap_inventory.csv ----------------------------------------
    FieldPolicy(GAPS, "well_key", "identifier", allowed_values=_WELLS),
    _label(GAPS, "gap_class", "gap_class"),
    _label(GAPS, "disposition", "gap_disposition"),
    _enum(GAPS, "unit", _UNIT_VALUES),
    _enum(GAPS, "assurance_tier", _TIER),
    _stmt(GAPS, "limitations", "gap_inventory_limitations"),

    # --- overburden_eligibility_summary.csv -------------------------------
    FieldPolicy(ELIGIBILITY, "well_key", "identifier", allowed_values=_WELLS),
    _label(ELIGIBILITY, "overburden_status", "overburden_status"),
    FieldPolicy(ELIGIBILITY, "limiting_reasons", CATEGORY_ENUMERATED_CODE_LIST,
                allowed_values=tuple(sorted(VALID_LIMITING_REASONS + ("none",)))),
    _label(ELIGIBILITY, "seabed_basis", "seabed_basis"),
    _enum(ELIGIBILITY, "integration_method", _INTEGRATION_METHOD_VALUES),
    _enum(ELIGIBILITY, "integration_coordinate", _INTEGRATION_COORDINATE_VALUES),
    _enum(ELIGIBILITY, "depth_convention", (SIGN_CONVENTION_ID,)),
    _label(ELIGIBILITY, "evidence_class", "evidence_class"),
    _label(ELIGIBILITY, "calibration_status", "calibration_status"),
    _enum(ELIGIBILITY, "unit", _UNIT_VALUES),
    _enum(ELIGIBILITY, "assurance_tier", _TIER),
    _stmt(ELIGIBILITY, "purpose", "eligibility_purpose"),
    _stmt(ELIGIBILITY, "limitations", "eligibility_limitations"),

    # --- vertical_stress_profile.csv --------------------------------------
    FieldPolicy(PROFILE, "well_key", "identifier", allowed_values=_WELLS),
    _label(PROFILE, "density_source", "density_source"),
    _enum(PROFILE, "integration_coordinate", _INTEGRATION_COORDINATE_VALUES),
    _label(PROFILE, "evidence_class", "evidence_class"),
    _label(PROFILE, "calibration_status", "calibration_status"),
    _enum(PROFILE, "unit", _UNIT_VALUES),
    _enum(PROFILE, "assurance_tier", _TIER),
    _stmt(PROFILE, "limitations", "profile_limitations"),

    # --- shallow_column_scenarios.csv -------------------------------------
    FieldPolicy(SCENARIOS, "well_key", "identifier", allowed_values=_WELLS),
    _enum(SCENARIOS, "scenario_name", VALID_SCENARIO_NAMES),
    _label(SCENARIOS, "assumed_density_basis", "assumed_density_basis"),
    _label(SCENARIOS, "evidence_class", "evidence_class"),
    _label(SCENARIOS, "calibration_status", "calibration_status"),
    _enum(SCENARIOS, "unit", _UNIT_VALUES),
    _enum(SCENARIOS, "assurance_tier", _TIER),
    FieldPolicy(SCENARIOS, "scenario_basis", "controlled_template",
                template_id="scenario_basis"),
    _stmt(SCENARIOS, "limitations", "scenario_limitations"),

    # --- gap_threshold_sensitivity.csv ------------------------------------
    FieldPolicy(SENSITIVITY, "well_key", "identifier", allowed_values=_WELLS),
    _label(SENSITIVITY, "derived_status", "overburden_status"),
    _enum(SENSITIVITY, "unit", _UNIT_VALUES),
    _enum(SENSITIVITY, "assurance_tier", _TIER),
    _stmt(SENSITIVITY, "limitations", "gap_sensitivity_limitations"),

    # --- density_overburden_issues.csv ------------------------------------
    _enum(ISSUES, "severity", ("INFO", "WARNING", "ERROR")),
    _enum(ISSUES, "code", ISSUE_CODES),
    FieldPolicy(ISSUES, "context", "sanitized_diagnostic"),
    FieldPolicy(ISSUES, "message", "sanitized_diagnostic"),
    _enum(ISSUES, "assurance_tier", _TIER),
)

M = OVERBURDEN_MANIFEST_ARTIFACT

_MANIFEST_POLICIES = (
    _enum(M, "/assurance_tier", _TIER),
    _enum(M, "/increment_title", (INCREMENT_TITLE,)),
    FieldPolicy(M, "/config_filename", "filename"),
    _enum(M, "/config_schema_version", ("7.0.1",)),
    _enum(M, "/depth_convention", (SIGN_CONVENTION_ID,)),
    _stmt(M, "/depth_convention_statement", "manifest_depth_convention_statement"),
    _enum(M, "/integration_method", _INTEGRATION_METHOD_VALUES),
    _enum(M, "/integration_coordinate", _INTEGRATION_COORDINATE_VALUES),
    _stmt(M, "/gravity_basis", "manifest_gravity_basis"),
    _stmt(M, "/assumption_register/statement", "manifest_assumption_statement"),
    _stmt(M, "/calibration_data_available/statement", "manifest_calibration_statement"),
    _stmt(M, "/named_lithology_statement", "manifest_named_lithology_statement"),
    _stmt(M, "/lithology_validation/model", "manifest_lithology_model"),
    _stmt(M, "/lithology_validation/derivation", "manifest_lithology_derivation"),
    FieldPolicy(M, "/lithology_validation/violations[]/artifact", "sanitized_diagnostic"),
    FieldPolicy(M, "/lithology_validation/violations[]/field", "sanitized_diagnostic"),
    FieldPolicy(M, "/lithology_validation/violations[]/location", "sanitized_diagnostic"),
    FieldPolicy(M, "/lithology_validation/violations[]/reason", "sanitized_diagnostic"),
    FieldPolicy(M, "/lithology_validation/violations[]/linter_terms[]", "sanitized_diagnostic"),
    FieldPolicy(M, "/lithology_validation/emitted_field_coverage/artifacts_inspected[]",
                "filename"),
    _enum(M, "/methods_not_implemented[]", NOT_IMPLEMENTED_TOKENS),
    _stmt(M, "/limitations[]",
          "limitation_no_absolute_without_full_column", "limitation_no_density_repair",
          "limitation_assumed_components", "limitation_no_pore_pressure",
          "limitation_seabed_provenance"),
    _enum(M, "/issues[]/severity", ("INFO", "WARNING", "ERROR")),
    _enum(M, "/issues[]/code", ISSUE_CODES),
    FieldPolicy(M, "/issues[]/context", "sanitized_diagnostic"),
    FieldPolicy(M, "/issues[]/message", "sanitized_diagnostic"),
    FieldPolicy(M, "/wells/*/source_las_filename", "filename"),
    FieldPolicy(M, "/wells/*/source_survey_filename", "filename"),
    _enum(M, "/wells/*/canonical_curve_name", _CURVE_NAME_VALUES),
    _enum(M, "/wells/*/canonical_unit", _CANONICAL_UNIT_VALUES),
    _enum(M, "/wells/*/conversion_function", _CONVERSION_VALUES),
    _enum(M, "/wells/*/depth_basis_used", _DEPTH_BASIS_VALUES),
    _enum(M, "/wells/*/depth_map_status", _DEPTH_MAP_STATUS_VALUES),
    _label(M, "/wells/*/seabed_basis", "seabed_basis"),
    _label(M, "/wells/*/overburden_status", "overburden_status"),
    _enum(M, "/wells/*/limiting_reasons[]", tuple(sorted(VALID_LIMITING_REASONS))),
    _enum(M, "/wells/*/shallow_column_scenarios[]/scenario_name", VALID_SCENARIO_NAMES),
    _label(M, "/wells/*/shallow_column_scenarios[]/assumed_density_basis",
           "assumed_density_basis"),
    _label(M, "/wells/*/gap_threshold_sensitivity[]/derived_status", "overburden_status"),
)

JSON_OBJECT_KEYS = {
    "": frozenset((
        "assurance_tier", "assumption_register", "calibration_data_available",
        "config_filename", "config_schema_version", "depth_convention",
        "depth_convention_statement", "depth_convention_verified", "gravity_basis",
        "gravity_m_s2", "increment", "increment_title", "integration_coordinate",
        "integration_method", "issues", "limitations", "lithology_validation",
        "methods_not_implemented", "n_wells_absolute_supported", "n_wells_evaluated",
        "n_wells_not_eligible", "n_wells_partial_measured_only",
        "n_wells_screening_sensitivity_only", "named_lithology_assigned",
        "named_lithology_statement", "total_bridged_density_samples",
        "total_eligible_density_samples", "wells",
    )),
    "/assumption_register": frozenset((
        "bounds_are_inclusive", "profile_report_step_tvdss_m", "rhob_max_kg_m3",
        "rhob_min_kg_m3", "scenario_high_percentile", "seawater_density_high_kg_m3",
        "seawater_density_kg_m3", "seawater_density_low_kg_m3",
        "shallow_gap_tolerance_tvd_m", "short_gap_max_tvd_m", "statement",
    )),
    "/calibration_data_available": frozenset((
        "core_or_log_calibrated_density_control", "local_gravity_survey",
        "measured_seawater_density", "pressure_rft_mdt_dst",
        "stress_fit_lot_xlot_dfit", "statement",
    )),
    "/lithology_validation": frozenset((
        "model", "derivation", "emitted_field_coverage", "violations",
    )),
    "/lithology_validation/emitted_field_coverage": frozenset((
        "n_string_field_occurrences", "n_controlled_occurrences",
        "n_structural_occurrences", "n_unguaranteed_occurrences",
        "n_unclassified_fields", "n_unauthorized_controlled_fields",
        "n_field_kind_mismatches", "n_schema_violations", "violations",
        "artifacts_inspected", "distinct_statements_used", "distinct_templates_used",
        "distinct_labels_used",
    )),
    "/lithology_validation/violations[]": frozenset((
        "artifact", "field", "location", "reason", "linter_terms")),
    "/issues[]": frozenset(("severity", "code", "context", "message")),
    "/wells": None,
    "/wells/*": frozenset((
        "absolute_stress_supported", "bridged_increment_pa", "canonical_curve_name",
        "canonical_unit", "column_truncated_at_unresolved_gap", "column_uninterrupted",
        "conversion_function", "datum_elevation_m", "density_curve_present",
        "depth_basis_used", "depth_map_status", "gap_threshold_sensitivity",
        "limiting_reasons", "measured_increment_mpa", "measured_increment_pa",
        "n_bridged_samples", "n_depth_unmapped", "n_eligible", "n_finite_rhob",
        "n_samples", "n_screening_bound_failures", "n_unresolved_internal_gaps",
        "n_unresolved_long_gaps", "rhob_eligible_p05_kg_m3",
        "overburden_status", "seabed_basis", "seabed_resolved", "seabed_tvdss_m",
        "shallow_column_scenarios", "shallow_unresolved_thickness_tvd_m",
        "source_las_filename", "source_survey_filename",
        "terminal_unresolved_thickness_tvd_m", "unit_resolved",
    )),
    "/wells/*/shallow_column_scenarios[]": frozenset((
        "scenario_name", "assumed_density_basis", "assumed_shallow_density_kg_m3",
        "total_stress_pa", "total_stress_mpa", "assumed_fraction_of_total",
        "conditioned_fraction_of_total", "measured_fraction_of_total",
    )),
    "/wells/*/gap_threshold_sensitivity[]": frozenset((
        "threshold_tvd_m", "is_approved_threshold", "n_bridged_gaps", "n_long_gaps",
        "derived_status",
    )),
}

JSON_LIST_ITEM_KINDS = {
    "/issues": "object",
    "/limitations": "string",
    "/lithology_validation/emitted_field_coverage/artifacts_inspected": "string",
    "/lithology_validation/violations": "object",
    "/lithology_validation/violations[]/linter_terms": "string",
    "/methods_not_implemented": "string",
    "/wells/*/limiting_reasons": "string",
    "/wells/*/shallow_column_scenarios": "object",
    "/wells/*/gap_threshold_sensitivity": "object",
}

JSON_BOOLEAN_PATHS = frozenset((
    "/depth_convention_verified", "/named_lithology_assigned",
    "/assumption_register/bounds_are_inclusive",
    "/calibration_data_available/core_or_log_calibrated_density_control",
    "/calibration_data_available/local_gravity_survey",
    "/calibration_data_available/measured_seawater_density",
    "/calibration_data_available/pressure_rft_mdt_dst",
    "/calibration_data_available/stress_fit_lot_xlot_dfit",
    "/wells/*/absolute_stress_supported",
    "/wells/*/column_truncated_at_unresolved_gap",
    "/wells/*/column_uninterrupted", "/wells/*/density_curve_present",
    "/wells/*/seabed_resolved", "/wells/*/unit_resolved",
    "/wells/*/gap_threshold_sensitivity[]/is_approved_threshold",
))

JSON_INTEGER_PATHS = frozenset((
    "/increment", "/n_wells_absolute_supported", "/n_wells_evaluated",
    "/n_wells_not_eligible", "/n_wells_partial_measured_only",
    "/n_wells_screening_sensitivity_only", "/total_bridged_density_samples",
    "/total_eligible_density_samples",
    "/lithology_validation/emitted_field_coverage/distinct_labels_used",
    "/lithology_validation/emitted_field_coverage/distinct_statements_used",
    "/lithology_validation/emitted_field_coverage/distinct_templates_used",
    "/lithology_validation/emitted_field_coverage/n_controlled_occurrences",
    "/lithology_validation/emitted_field_coverage/n_field_kind_mismatches",
    "/lithology_validation/emitted_field_coverage/n_schema_violations",
    "/lithology_validation/emitted_field_coverage/n_string_field_occurrences",
    "/lithology_validation/emitted_field_coverage/n_structural_occurrences",
    "/lithology_validation/emitted_field_coverage/n_unauthorized_controlled_fields",
    "/lithology_validation/emitted_field_coverage/n_unclassified_fields",
    "/lithology_validation/emitted_field_coverage/n_unguaranteed_occurrences",
    "/lithology_validation/emitted_field_coverage/violations",
    "/wells/*/n_bridged_samples", "/wells/*/n_depth_unmapped", "/wells/*/n_eligible",
    "/wells/*/n_finite_rhob", "/wells/*/n_samples",
    "/wells/*/n_screening_bound_failures", "/wells/*/n_unresolved_internal_gaps",
    "/wells/*/n_unresolved_long_gaps",
    "/wells/*/gap_threshold_sensitivity[]/n_bridged_gaps",
    "/wells/*/gap_threshold_sensitivity[]/n_long_gaps",
))

JSON_NUMBER_PATHS = frozenset((
    "/gravity_m_s2", "/assumption_register/profile_report_step_tvdss_m",
    "/assumption_register/rhob_max_kg_m3", "/assumption_register/rhob_min_kg_m3",
    "/assumption_register/scenario_high_percentile",
    "/assumption_register/seawater_density_high_kg_m3",
    "/assumption_register/seawater_density_kg_m3",
    "/assumption_register/seawater_density_low_kg_m3",
    "/assumption_register/shallow_gap_tolerance_tvd_m",
    "/assumption_register/short_gap_max_tvd_m",
    "/wells/*/bridged_increment_pa", "/wells/*/datum_elevation_m",
    "/wells/*/measured_increment_mpa", "/wells/*/measured_increment_pa",
    "/wells/*/seabed_tvdss_m", "/wells/*/shallow_unresolved_thickness_tvd_m",
    "/wells/*/terminal_unresolved_thickness_tvd_m",
    "/wells/*/rhob_eligible_p05_kg_m3",
    "/wells/*/shallow_column_scenarios[]/assumed_fraction_of_total",
    "/wells/*/shallow_column_scenarios[]/conditioned_fraction_of_total",
    "/wells/*/shallow_column_scenarios[]/measured_fraction_of_total",
    "/wells/*/shallow_column_scenarios[]/assumed_shallow_density_kg_m3",
    "/wells/*/shallow_column_scenarios[]/total_stress_mpa",
    "/wells/*/shallow_column_scenarios[]/total_stress_pa",
    "/wells/*/gap_threshold_sensitivity[]/threshold_tvd_m",
))

JSON_NULLABLE_PATHS = frozenset((
    "/lithology_validation/violations[]/field",
    "/lithology_validation/violations[]/location",
    "/wells/*/bridged_increment_pa", "/wells/*/measured_increment_mpa",
    "/wells/*/measured_increment_pa", "/wells/*/seabed_tvdss_m",
    "/wells/*/rhob_eligible_p05_kg_m3",
    "/wells/*/shallow_unresolved_thickness_tvd_m",
    "/wells/*/terminal_unresolved_thickness_tvd_m",
))

OVERBURDEN_BUNDLE = PolicyBundle(
    name="increment7_density_overburden",
    field_policies=_FIELD_POLICIES + _MANIFEST_POLICIES,
    statements=_STATEMENT_LIST,
    templates=_TEMPLATE_LIST,
    labels=OVERBURDEN_LABELS,
    csv_schemas=OVERBURDEN_CSV_SCHEMAS,
    manifest_artifact=OVERBURDEN_MANIFEST_ARTIFACT,
    json_object_keys=JSON_OBJECT_KEYS,
    json_list_item_kinds=JSON_LIST_ITEM_KINDS,
    json_boolean_paths=JSON_BOOLEAN_PATHS,
    json_integer_paths=JSON_INTEGER_PATHS,
    json_number_paths=JSON_NUMBER_PATHS,
    json_nullable_paths=JSON_NULLABLE_PATHS,
    approved_well_keys=APPROVED_WELL_KEYS,
    field_types=OVERBURDEN_FIELD_TYPES,
)

#: The nine deterministic Increment 7 artifacts this policy governs.
OVERBURDEN_ARTIFACTS = OVERBURDEN_BUNDLE.artifacts


#### `p2mem/io/overburden_inventory.py` - deterministic export builders and the manifest

Row order is fixed by explicit sorts on stable keys, never by dictionary iteration order. The
manifest's coverage block is computed from the CSV payloads this module has already built -
the records that are actually serialized, not a reconstruction of them.

In [ ]:
%%writefile p2mem/io/overburden_inventory.py
"""
p2mem.io.overburden_inventory - deterministic Increment 7 export builders.

Every function here turns typed Increment 7 results into the EXACT records the
Increment 7 output policy governs. The records these builders return are the
records that are serialized: nothing downstream reconstructs a parallel scope,
and the manifest's own coverage block is computed from these very payloads.

Determinism
-----------
Row order is fixed by an explicit sort on stable keys (well key, then index or
enumerated position), never by dictionary iteration order. Column order is the
schema's own declared order. Two runs from two independent roots produce
byte-identical artifacts.
"""

from __future__ import annotations

from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np

from p2mem import ASSURANCE_TIER
from p2mem.overburden import (
    DEPTH_CONVENTION_STATEMENT, SIGN_CONVENTION_ID, pa_to_mpa,
)
from p2mem.overburden_models import (
    SEABED_BASIS_LOCKED_MARKER,
    STATUS_ABSOLUTE,
    STATUS_NOT_ELIGIBLE,
    STATUS_PARTIAL_ONLY,
    STATUS_SENSITIVITY_ONLY,
    DENSITY_MASK_NAMES,
    DensityGapRecord,
    DensityQcStats,
    GapConditioningResult,
    GapThresholdSensitivity,
    OverburdenConfig,
    OverburdenEligibility,
    OverburdenIssue,
    ShallowColumnScenario,
    VerticalStressProfile,
)
from p2mem.io.overburden_policy import validate_emitted_records
from p2mem.io.overburden_registry import (
    ASSURANCE_TIER_VALUE, AVAILABILITY, ELIGIBILITY, GAPS, INCREMENT_TITLE, ISSUES,
    NA_DENSITY, NOT_IMPLEMENTED_TOKENS, OVERBURDEN_BUNDLE, OVERBURDEN_STATEMENTS,
    OVERBURDEN_TEMPLATES, PROFILE, QC, SCENARIOS, SENSITIVITY,
)

__all__ = [
    "OVERBURDEN_OUTPUT_DIR",
    "build_density_availability_rows",
    "build_density_qc_rows",
    "build_density_gap_rows",
    "build_overburden_eligibility_rows",
    "build_vertical_stress_profile_rows",
    "build_shallow_column_scenario_rows",
    "build_gap_threshold_sensitivity_rows",
    "build_overburden_issue_rows",
    "build_overburden_manifest",
    "derive_overburden_issues",
]

OVERBURDEN_OUTPUT_DIR = "outputs/07_density_overburden"

_TIER = ASSURANCE_TIER_VALUE
_UNIT_ALL = "kg/m3; m; Pa; MPa"
_UNIT_M = "m"
_EVIDENCE_MEASURED = "measured"
_EVIDENCE_ASSUMED = "assumed_configured"
_CALIBRATION = "uncalibrated_screening_only"

# Sanity: the tier string this module stamps on every record must be the
# project-wide tier declared by the locked package, not a second copy that
# could drift from it.
if _TIER != ASSURANCE_TIER:  # pragma: no cover - fails at import
    raise ValueError(
        f"Increment 7 assurance tier {_TIER!r} differs from the locked package tier "
        f"{ASSURANCE_TIER!r}.")


def _f(value) -> Optional[float]:
    """Return a plain float, or None. Never a numpy scalar: the schema's
    pre-serialization type check requires `type(value) in (int, float)`, and a
    `numpy.float64` is neither."""
    return None if value is None else float(value)


def _i(value) -> Optional[int]:
    return None if value is None else int(value)


def _dec(value: float, places: int = 4) -> str:
    """Format a number as a decimal literal for a controlled-template slot."""
    return f"{float(value):.{places}f}"


def _sanitize(text: str, limit: int = 480) -> str:
    """Reduce an operator-facing string to the sanitized-diagnostic contract.

    Path separators, newlines and any character outside the declared charset
    are removed rather than escaped, because a diagnostic must never become a
    channel for a path or for arbitrary text.
    """
    from p2mem.io.output_policy import SANITIZED_DIAGNOSTIC_CHARSET
    cleaned = "".join(ch if ch in SANITIZED_DIAGNOSTIC_CHARSET else " " for ch in text)
    cleaned = " ".join(cleaned.split())
    return cleaned[:limit]


# ---------------------------------------------------------------------------
# CSV row builders
# ---------------------------------------------------------------------------

def build_density_availability_rows(
    stats_by_well: Dict[str, DensityQcStats],
    frames: Dict[str, object],
) -> List[dict]:
    """One row per well: what density exists, in what unit, over what interval."""
    rows = []
    for wk in sorted(stats_by_well):
        s = stats_by_well[wk]
        rows.append({
            "well_key": wk,
            "source_las_filename": s.source_las_filename,
            "source_survey_filename": frames[wk].source_survey_filename,
            "density_curve_present": bool(s.curve_present),
            "canonical_curve_name": s.canonical_curve_name or NA_DENSITY,
            "source_curve_name": s.source_curve_name or NA_DENSITY,
            "raw_mnemonic": s.raw_mnemonic or NA_DENSITY,
            "raw_unit": s.raw_unit or NA_DENSITY,
            "canonical_unit": s.canonical_unit or NA_DENSITY,
            "conversion_function": s.conversion_function or NA_DENSITY,
            "unit_resolved": bool(s.unit_resolved),
            "conversion_confirmed": bool(s.conversion_confirmed),
            "evidence_class": _EVIDENCE_MEASURED,
            "n_samples": _i(s.n_samples),
            "n_finite_rhob": _i(s.n_finite),
            "n_non_finite_rhob": _i(s.n_non_finite),
            "n_non_positive_rhob": _i(s.n_non_positive),
            "n_below_screening_min": _i(s.n_below_screening_min),
            "n_above_screening_max": _i(s.n_above_screening_max),
            "n_screening_bound_failures": _i(s.n_screening_bound_failures),
            "n_in_screening_band": _i(s.n_in_screening_band),
            "n_depth_mapped": _i(s.n_depth_mapped),
            "n_depth_unmapped": _i(s.n_depth_unmapped),
            "n_outside_survey_coverage": _i(s.n_samples_outside_survey_coverage),
            "n_eligible": _i(s.n_eligible),
            "first_valid_md_m": _f(s.first_valid_md_m),
            "last_valid_md_m": _f(s.last_valid_md_m),
            "first_valid_tvd_m": _f(s.first_valid_tvd_m),
            "last_valid_tvd_m": _f(s.last_valid_tvd_m),
            "first_valid_tvdss_m": _f(s.first_valid_tvdss_m),
            "last_valid_tvdss_m": _f(s.last_valid_tvdss_m),
            "gross_coverage_md_m": _f(s.gross_coverage_md_m),
            "gross_coverage_tvd_m": _f(s.gross_coverage_tvd_m),
            "median_md_step_m": _f(s.median_md_step_m),
            "depth_basis_used": s.depth_basis_used,
            "depth_map_status": s.depth_map_status,
            "survey_md_min_m": _f(s.survey_md_min_m),
            "survey_md_max_m": _f(s.survey_md_max_m),
            "unit": _UNIT_ALL,
            "assurance_tier": _TIER,
            "statistics_basis": OVERBURDEN_STATEMENTS["availability_statistics_basis"].text,
            "limitations": OVERBURDEN_STATEMENTS["availability_limitations"].text,
        })
    return rows


def build_density_qc_rows(
    stats_by_well: Dict[str, DensityQcStats],
    masks_by_well: Dict[str, object],
    config: OverburdenConfig,
) -> List[dict]:
    """One row per well: every mask count and the QC statistics behind it."""
    rows = []
    for wk in sorted(stats_by_well):
        s = stats_by_well[wk]
        counts = masks_by_well[wk].counts()
        # A machine-diagnostic rendering of the SAME counts, in the declared
        # `name=integer` grammar, so a reader diffing two runs sees one field
        # change rather than twelve.
        diag = ";".join(
            f"{name}={counts[name]}" for name in DENSITY_MASK_NAMES
            if counts[name] is not None)
        rows.append({
            "well_key": wk,
            "canonical_curve_name": s.canonical_curve_name or NA_DENSITY,
            "screening_min_kg_m3": _f(config.rhob_min_kg_m3),
            "screening_max_kg_m3": _f(config.rhob_max_kg_m3),
            "bounds_are_inclusive": bool(config.bounds_are_inclusive),
            "n_source_value_present": _i(counts["source_value_present"]),
            "n_finite_numeric_density": _i(counts["finite_numeric_density"]),
            "n_unit_resolved": _i(counts["unit_resolved"]),
            "n_screening_range_plausible": _i(counts["screening_range_plausible"]),
            "n_below_seabed_sample": _i(counts["below_seabed_sample"]),
            "n_depth_mapping_valid": _i(counts["depth_mapping_valid"]),
            "n_within_survey_coverage": _i(counts["within_survey_coverage"]),
            "n_eligible_for_measured_integration": _i(
                counts["eligible_for_measured_integration"]),
            "n_bridged_short_gap": _i(counts["bridged_short_gap"]),
            "n_unresolved_long_gap": _i(counts["unresolved_long_gap"]),
            "n_unresolved_shallow_column": _i(counts["unresolved_shallow_column"]),
            "n_unresolved_terminal_column": _i(counts["unresolved_terminal_column"]),
            "rhob_min_kg_m3": _f(s.rhob_min_kg_m3),
            "rhob_p05_kg_m3": _f(s.rhob_p05_kg_m3),
            "rhob_median_kg_m3": _f(s.rhob_median_kg_m3),
            "rhob_p95_kg_m3": _f(s.rhob_p95_kg_m3),
            "rhob_max_kg_m3": _f(s.rhob_max_kg_m3),
            "rhob_eligible_p05_kg_m3": _f(s.rhob_eligible_p05_kg_m3),
            "seabed_basis": s.seabed_basis,
            "seabed_mdrt_m": _f(s.seabed_mdrt_m),
            "seabed_tvd_m": _f(s.seabed_tvd_m),
            "seabed_tvdss_m": _f(s.seabed_tvdss_m),
            "shallow_gap_md_m": _f(s.shallow_gap_md_m),
            "shallow_gap_tvd_m": _f(s.shallow_gap_tvd_m),
            "terminal_gap_md_m": _f(s.terminal_gap_md_m),
            "terminal_gap_tvd_m": _f(s.terminal_gap_tvd_m),
            "n_internal_gaps": _i(s.n_internal_gaps),
            "n_internal_gap_samples": _i(s.n_internal_gap_samples),
            "longest_internal_gap_md_m": _f(s.longest_internal_gap_md_m),
            "longest_internal_gap_tvd_m": _f(s.longest_internal_gap_tvd_m),
            "mask_counts": diag,
            "unit": _UNIT_ALL,
            "assurance_tier": _TIER,
            "statistics_basis": OVERBURDEN_STATEMENTS["qc_statistics_basis"].text,
            "limitations": OVERBURDEN_STATEMENTS["qc_limitations"].text,
        })
    return rows


def build_density_gap_rows(
    gaps_by_well: Dict[str, Sequence[DensityGapRecord]],
) -> List[dict]:
    """One row per classified gap, ordered by well then by gap index."""
    rows = []
    for wk in sorted(gaps_by_well):
        for gap in sorted(gaps_by_well[wk], key=lambda g: g.gap_index):
            rows.append({
                "well_key": wk,
                "gap_index": _i(gap.gap_index),
                "gap_class": gap.gap_class,
                "disposition": gap.disposition,
                "n_samples": _i(gap.n_samples),
                "start_index": _i(gap.start_index),
                "end_index": _i(gap.end_index),
                "md_start_m": _f(gap.md_start_m),
                "md_end_m": _f(gap.md_end_m),
                "thickness_md_m": _f(gap.thickness_md_m),
                "tvd_start_m": _f(gap.tvd_start_m),
                "tvd_end_m": _f(gap.tvd_end_m),
                "thickness_tvd_m": _f(gap.thickness_tvd_m),
                "threshold_tvd_m": _f(gap.threshold_tvd_m),
                "bounding_density_above_kg_m3": _f(gap.bounding_density_above_kg_m3),
                "bounding_density_below_kg_m3": _f(gap.bounding_density_below_kg_m3),
                "unit": _UNIT_ALL,
                "assurance_tier": _TIER,
                "limitations": OVERBURDEN_STATEMENTS["gap_inventory_limitations"].text,
            })
    return rows


def build_overburden_eligibility_rows(
    eligibility_by_well: Dict[str, OverburdenEligibility],
    profiles_by_well: Dict[str, Optional[VerticalStressProfile]],
    config: OverburdenConfig,
) -> List[dict]:
    """One row per well: the derived verdict and the evidence behind it."""
    rows = []
    for wk in sorted(eligibility_by_well):
        e = eligibility_by_well[wk]
        p = profiles_by_well.get(wk)
        reasons = ";".join(e.limiting_reasons) if e.limiting_reasons else "none"
        rows.append({
            "well_key": wk,
            "overburden_status": e.status,
            "limiting_reasons": reasons,
            "seabed_resolved": bool(e.seabed_resolved),
            "seabed_basis": e.seabed_basis,
            "n_eligible_samples": _i(e.n_eligible_samples),
            "n_bridged_samples": _i(e.n_bridged_samples),
            "n_unresolved_internal_gaps": _i(e.n_unresolved_internal_gaps),
            "n_unresolved_long_gaps": _i(e.n_unresolved_long_gaps),
            "column_uninterrupted": bool(e.column_uninterrupted),
            "column_truncated_at_unresolved_gap": bool(
                False if p is None else p.column_truncated_at_unresolved_gap),
            "n_eligible_samples_below_truncation": _i(
                0 if p is None else p.n_eligible_samples_below_truncation),
            "eligible_top_tvd_m": _f(e.eligible_top_tvd_m),
            "eligible_base_tvd_m": _f(e.eligible_base_tvd_m),
            "eligible_top_tvdss_m": _f(e.eligible_top_tvdss_m),
            "eligible_base_tvdss_m": _f(e.eligible_base_tvdss_m),
            "measured_thickness_tvd_m": _f(e.measured_thickness_tvd_m),
            "water_column_thickness_tvd_m": _f(e.water_column_thickness_tvd_m),
            "shallow_unresolved_thickness_tvd_m": _f(
                e.shallow_unresolved_thickness_tvd_m),
            "terminal_unresolved_thickness_tvd_m": _f(
                e.terminal_unresolved_thickness_tvd_m),
            "unresolved_long_gap_thickness_tvd_m": _f(
                e.unresolved_long_gap_thickness_tvd_m),
            "measured_increment_pa": _f(e.measured_increment_pa),
            "measured_increment_mpa": _f(pa_to_mpa(e.measured_increment_pa)),
            "bridged_increment_pa": _f(e.bridged_increment_pa),
            "bridged_increment_mpa": _f(pa_to_mpa(e.bridged_increment_pa)),
            "absolute_stress_supported": bool(e.absolute_stress_supported),
            "gravity_m_s2": _f(config.gravity_m_s2),
            "integration_method": config.integration_method,
            "integration_coordinate": "tvdss_m",
            "depth_convention": SIGN_CONVENTION_ID,
            "evidence_class": _EVIDENCE_MEASURED,
            "calibration_status": _CALIBRATION,
            "unit": _UNIT_ALL,
            "assurance_tier": _TIER,
            "purpose": OVERBURDEN_STATEMENTS["eligibility_purpose"].text,
            "limitations": OVERBURDEN_STATEMENTS["eligibility_limitations"].text,
        })
    return rows


def _profile_node_indices(profile: VerticalStressProfile, step_m: float) -> List[int]:
    """Select existing profile nodes at approximately `step_m` in TVDSS.

    Nodes are SELECTED, never interpolated: every exported depth, density and
    cumulative value is one this well actually has. The first and last nodes
    are always included so the reported span equals the integrated span.
    """
    tvdss = np.asarray(profile.tvdss_m, dtype=np.float64)
    picked = [0]
    last = float(tvdss[0])
    for i in range(1, tvdss.size - 1):
        if float(tvdss[i]) - last >= step_m:
            picked.append(i)
            last = float(tvdss[i])
    if tvdss.size > 1:
        picked.append(int(tvdss.size) - 1)
    return picked


def build_vertical_stress_profile_rows(
    profiles_by_well: Dict[str, Optional[VerticalStressProfile]],
    config: OverburdenConfig,
) -> List[dict]:
    """The decimated measured-increment profile, ordered by well then depth."""
    rows = []
    for wk in sorted(profiles_by_well):
        p = profiles_by_well[wk]
        if p is None:
            continue
        md = np.asarray(p.md_m, dtype=np.float64)
        tvd = np.asarray(p.tvd_m, dtype=np.float64)
        tvdss = np.asarray(p.tvdss_m, dtype=np.float64)
        rho = np.asarray(p.density_kg_m3, dtype=np.float64)
        bridged = np.asarray(p.bridged_mask, dtype=bool)
        cum = np.asarray(p.cumulative_measured_increment_pa, dtype=np.float64)
        for node, i in enumerate(_profile_node_indices(
                p, float(config.profile_report_step_tvdss_m))):
            rows.append({
                "well_key": wk,
                "node_index": int(node),
                "md_m": float(md[i]),
                "tvd_m": float(tvd[i]),
                "tvdss_m": float(tvdss[i]),
                "rhob_kg_m3": float(rho[i]),
                "density_source": ("bridged_linear_in_tvd" if bool(bridged[i])
                                   else "measured_rhob"),
                "cumulative_measured_increment_pa": float(cum[i]),
                "cumulative_measured_increment_mpa": float(pa_to_mpa(float(cum[i]))),
                "gravity_m_s2": _f(config.gravity_m_s2),
                "integration_coordinate": "tvdss_m",
                "evidence_class": ("assumed_configured" if bool(bridged[i])
                                   else _EVIDENCE_MEASURED),
                "calibration_status": _CALIBRATION,
                "unit": _UNIT_ALL,
                "assurance_tier": _TIER,
                "limitations": OVERBURDEN_STATEMENTS["profile_limitations"].text,
            })
    return rows


def build_shallow_column_scenario_rows(
    scenarios_by_well: Dict[str, Sequence[ShallowColumnScenario]],
) -> List[dict]:
    """The transparent low/base/high scenarios, in declared scenario order."""
    from p2mem.overburden_models import VALID_SCENARIO_NAMES
    order = {name: i for i, name in enumerate(VALID_SCENARIO_NAMES)}
    tpl = OVERBURDEN_TEMPLATES["scenario_basis"]
    rows = []
    for wk in sorted(scenarios_by_well):
        for sc in sorted(scenarios_by_well[wk], key=lambda s: order[s.scenario_name]):
            part = sc.partition
            rows.append({
                "well_key": wk,
                "scenario_name": sc.scenario_name,
                "assumed_shallow_density_kg_m3": _f(sc.assumed_shallow_density_kg_m3),
                "assumed_density_basis": sc.assumed_density_basis,
                "seawater_density_kg_m3": _f(sc.seawater_density_kg_m3),
                "gravity_m_s2": _f(sc.gravity_m_s2),
                "water_column_thickness_tvd_m": _f(sc.water_column_thickness_tvd_m),
                "unresolved_thickness_tvd_m": _f(sc.unresolved_thickness_tvd_m),
                "measured_thickness_tvd_m": _f(sc.measured_thickness_tvd_m),
                "water_column_stress_pa": _f(part.water_column_pa),
                "unresolved_shallow_stress_pa": _f(part.unresolved_shallow_pa),
                "measured_formation_stress_pa": _f(part.measured_formation_pa),
                "bridged_gap_stress_pa": _f(part.bridged_gap_pa),
                "total_stress_pa": _f(part.total_pa),
                "total_stress_mpa": _f(pa_to_mpa(part.total_pa)),
                "assumed_fraction_of_total": _f(sc.assumed_fraction_of_total),
                "conditioned_fraction_of_total": _f(
                    sc.conditioned_fraction_of_total),
                "measured_fraction_of_total": _f(sc.measured_fraction_of_total),
                "evidence_class": _EVIDENCE_ASSUMED,
                "calibration_status": _CALIBRATION,
                "unit": _UNIT_ALL,
                "assurance_tier": _TIER,
                "scenario_basis": tpl.render({
                    "assumed_density_kg_m3": _dec(sc.assumed_shallow_density_kg_m3),
                    "unresolved_thickness_m": _dec(sc.unresolved_thickness_tvd_m),
                    "assumed_fraction_pct": _dec(
                        100.0 * sc.assumed_fraction_of_total, 2),
                    "conditioned_fraction_pct": _dec(
                        100.0 * sc.conditioned_fraction_of_total, 2),
                    "measured_fraction_pct": _dec(
                        100.0 * sc.measured_fraction_of_total, 2),
                }),
                "limitations": OVERBURDEN_STATEMENTS["scenario_limitations"].text,
            })
    return rows


def build_gap_threshold_sensitivity_rows(
    sensitivity_by_well: Dict[str, Sequence[GapThresholdSensitivity]],
) -> List[dict]:
    """One row per (well, candidate threshold), ordered by well then threshold."""
    rows = []
    for wk in sorted(sensitivity_by_well):
        for t in sorted(sensitivity_by_well[wk], key=lambda x: x.threshold_tvd_m):
            rows.append({
                "well_key": wk,
                "threshold_tvd_m": _f(t.threshold_tvd_m),
                "is_approved_threshold": bool(t.is_approved_threshold),
                "n_bridged_gaps": _i(t.n_bridged_gaps),
                "n_bridged_samples": _i(t.n_bridged_samples),
                "bridged_thickness_tvd_m": _f(t.bridged_thickness_tvd_m),
                "n_long_gaps": _i(t.n_long_gaps),
                "long_gap_thickness_tvd_m": _f(t.long_gap_thickness_tvd_m),
                "n_eligible_or_bridged_samples": _i(t.n_eligible_or_bridged_samples),
                "total_measured_increment_pa": _f(t.total_measured_increment_pa),
                "total_measured_increment_mpa": _f(
                    pa_to_mpa(t.total_measured_increment_pa)),
                "bridged_increment_pa": _f(t.bridged_increment_pa),
                "derived_status": t.derived_status,
                "unit": _UNIT_ALL,
                "assurance_tier": _TIER,
                "limitations": OVERBURDEN_STATEMENTS["gap_sensitivity_limitations"].text,
            })
    return rows


def build_overburden_issue_rows(issues: Sequence[OverburdenIssue]) -> List[dict]:
    """The operator-facing issue table, sorted for determinism."""
    rows = []
    for issue in sorted(issues, key=lambda i: (i.severity, i.code, i.context)):
        rows.append({
            "severity": issue.severity,
            "code": issue.code,
            "context": _sanitize(issue.context),
            "message": _sanitize(issue.message),
            "assurance_tier": _TIER,
        })
    return rows


# ---------------------------------------------------------------------------
# Issue derivation
# ---------------------------------------------------------------------------

def derive_overburden_issues(
    stats_by_well: Dict[str, DensityQcStats],
    eligibility_by_well: Dict[str, OverburdenEligibility],
    gap_results: Dict[str, GapConditioningResult],
    profiles_by_well: Dict[str, Optional[VerticalStressProfile]],
    config: OverburdenConfig,
    *,
    near_bound_tolerance_kg_m3: float = 25.0,
) -> List[OverburdenIssue]:
    """Derive the run's QC issues from MEASURED results, never from a well name."""
    issues: List[OverburdenIssue] = []
    for wk in sorted(stats_by_well):
        s = stats_by_well[wk]
        e = eligibility_by_well[wk]
        gr = gap_results[wk]
        p = profiles_by_well.get(wk)
        if not s.curve_present:
            issues.append(OverburdenIssue(
                "ERROR", "DENSITY_CURVE_ABSENT", wk,
                "No contract-resolved density curve; this well cannot contribute a "
                "measured vertical-stress increment."))
            continue
        if not s.conversion_confirmed:
            issues.append(OverburdenIssue(
                "WARNING", "DENSITY_CONVERSION_FUNCTION_UNEXPECTED", wk,
                f"Locked contract applied conversion {s.conversion_function} where the "
                f"Increment 7 configuration expects "
                f"{config.expected_conversion_function}; reported, not corrected."))
        if s.n_screening_bound_failures:
            issues.append(OverburdenIssue(
                "WARNING", "SCREENING_BOUND_FAILURES_PRESENT", wk,
                f"{s.n_screening_bound_failures} sample(s) fall outside the configured "
                f"screening plausibility band; they are masked out, never modified."))
        if (s.rhob_max_kg_m3 is not None
                and 0.0 <= config.rhob_max_kg_m3 - s.rhob_max_kg_m3
                <= near_bound_tolerance_kg_m3):
            issues.append(OverburdenIssue(
                "INFO", "MEASURED_MAXIMUM_NEAR_SCREENING_BOUND", wk,
                f"The maximum recorded density lies within "
                f"{config.rhob_max_kg_m3 - s.rhob_max_kg_m3:.4f} kg/m3 of the configured "
                f"upper screening bound; the band admits it, and a small change to the "
                f"bound would not."))
        if not e.seabed_resolved:
            issues.append(OverburdenIssue(
                "WARNING", "SEABED_DATUM_UNRESOLVED", wk,
                "No approved formation-top file, so no seabed marker exists in the locked "
                "Increment 5 output; water column and shallow-gap thickness are not "
                "determinable and are reported as unresolved, never as zero."))
        elif e.shallow_unresolved_thickness_tvd_m:
            issues.append(OverburdenIssue(
                "WARNING", "SHALLOW_DENSITY_COLUMN_UNRESOLVED", wk,
                f"{e.shallow_unresolved_thickness_tvd_m:.2f} m of true vertical depth "
                f"between the seabed and the first eligible density sample carries no "
                f"measured density; no absolute vertical stress is defensible for this "
                f"well."))
        if gr.n_long_gaps:
            issues.append(OverburdenIssue(
                "WARNING", "LONG_INTERNAL_GAP_PRESENT", wk,
                f"{gr.n_long_gaps} internal gap(s) totalling "
                f"{gr.long_gap_thickness_tvd_m:.2f} m of true vertical depth exceed the "
                f"approved bridging threshold and are not bridged."))
        if e.terminal_unresolved_thickness_tvd_m:
            issues.append(OverburdenIssue(
                "INFO", "TERMINAL_DENSITY_COLUMN_UNRESOLVED", wk,
                f"{e.terminal_unresolved_thickness_tvd_m:.2f} m of true vertical depth "
                f"below the last eligible density sample carries no measured density."))
        if p is not None and p.column_truncated_at_unresolved_gap:
            issues.append(OverburdenIssue(
                "WARNING", "STRESS_COLUMN_TRUNCATED_AT_UNRESOLVED_GAP", wk,
                f"The integrable column stops at the first unresolved long gap; "
                f"{p.n_eligible_samples_below_truncation} eligible sample(s) below it are "
                f"excluded rather than joined across unknown density."))
        if not e.absolute_stress_supported:
            issues.append(OverburdenIssue(
                "INFO", "ABSOLUTE_OVERBURDEN_NOT_SUPPORTED", wk,
                f"Derived status {e.status}; the measured density coverage does not "
                f"support an absolute vertical overburden-stress curve."))
    return issues


# ---------------------------------------------------------------------------
# Manifest
# ---------------------------------------------------------------------------

def build_overburden_manifest(
    stats_by_well: Dict[str, DensityQcStats],
    eligibility_by_well: Dict[str, OverburdenEligibility],
    gap_results: Dict[str, GapConditioningResult],
    profiles_by_well: Dict[str, Optional[VerticalStressProfile]],
    scenarios_by_well: Dict[str, Sequence[ShallowColumnScenario]],
    sensitivity_by_well: Dict[str, Sequence[GapThresholdSensitivity]],
    issues: Sequence[OverburdenIssue],
    frames: Dict[str, object],
    config: OverburdenConfig,
    *,
    depth_convention_verified: bool,
    output_payloads: Dict[str, list],
) -> dict:
    """Build the Increment 7 manifest, including its DERIVED coverage block.

    The coverage block is computed by running the Increment 7 validator over
    `output_payloads` - the ACTUAL CSV records this run will serialize. The
    manifest cannot present itself (it is still being built), so the CSV set is
    passed as the expected artifact set for that pass; the export gate then
    re-validates the COMPLETE set, manifest included, before and after
    serialization.
    """
    coverage = validate_emitted_records(
        OVERBURDEN_BUNDLE, output_payloads, well_keys=set(frames),
        expected_artifacts=set(output_payloads), serialization_stage="pre")
    cov = coverage.as_dict()
    # `emitted_field_coverage` is a COUNT block: its own `violations` key is the
    # number of violations, and the violation records themselves live once, in
    # the sibling `violations` list. Mirrors the locked Increment 6 manifest
    # shape so an auditor reads the two manifests the same way.
    cov["violations"] = len(coverage.violations)

    wells: Dict[str, dict] = {}
    for wk in sorted(stats_by_well):
        s = stats_by_well[wk]
        e = eligibility_by_well[wk]
        p = profiles_by_well.get(wk)
        fr = frames[wk]
        wells[wk] = {
            "source_las_filename": s.source_las_filename,
            "source_survey_filename": fr.source_survey_filename,
            "n_samples": _i(s.n_samples),
            "density_curve_present": bool(s.curve_present),
            "canonical_curve_name": s.canonical_curve_name or NA_DENSITY,
            "canonical_unit": s.canonical_unit or NA_DENSITY,
            "conversion_function": s.conversion_function or NA_DENSITY,
            "unit_resolved": bool(s.unit_resolved),
            "n_finite_rhob": _i(s.n_finite),
            "n_eligible": _i(s.n_eligible),
            "rhob_eligible_p05_kg_m3": _f(s.rhob_eligible_p05_kg_m3),
            "n_bridged_samples": _i(e.n_bridged_samples),
            "n_unresolved_internal_gaps": _i(e.n_unresolved_internal_gaps),
            "n_screening_bound_failures": _i(s.n_screening_bound_failures),
            "n_unresolved_long_gaps": _i(e.n_unresolved_long_gaps),
            "depth_basis_used": s.depth_basis_used,
            "depth_map_status": s.depth_map_status,
            "n_depth_unmapped": _i(s.n_depth_unmapped),
            "datum_elevation_m": _f(fr.datum_elevation_m),
            "seabed_resolved": bool(e.seabed_resolved),
            "seabed_basis": e.seabed_basis,
            "seabed_tvdss_m": _f(s.seabed_tvdss_m),
            "overburden_status": e.status,
            "limiting_reasons": list(e.limiting_reasons),
            "measured_increment_pa": _f(e.measured_increment_pa),
            "measured_increment_mpa": _f(pa_to_mpa(e.measured_increment_pa)),
            "bridged_increment_pa": _f(e.bridged_increment_pa),
            "shallow_unresolved_thickness_tvd_m": _f(
                e.shallow_unresolved_thickness_tvd_m),
            "terminal_unresolved_thickness_tvd_m": _f(
                e.terminal_unresolved_thickness_tvd_m),
            "column_uninterrupted": bool(e.column_uninterrupted),
            "column_truncated_at_unresolved_gap": bool(
                False if p is None else p.column_truncated_at_unresolved_gap),
            "absolute_stress_supported": bool(e.absolute_stress_supported),
            "shallow_column_scenarios": [
                {
                    "scenario_name": sc.scenario_name,
                    "assumed_density_basis": sc.assumed_density_basis,
                    "assumed_shallow_density_kg_m3": _f(
                        sc.assumed_shallow_density_kg_m3),
                    "total_stress_pa": _f(sc.partition.total_pa),
                    "total_stress_mpa": _f(pa_to_mpa(sc.partition.total_pa)),
                    "assumed_fraction_of_total": _f(sc.assumed_fraction_of_total),
                    "conditioned_fraction_of_total": _f(
                        sc.conditioned_fraction_of_total),
                    "measured_fraction_of_total": _f(sc.measured_fraction_of_total),
                }
                for sc in scenarios_by_well.get(wk, ())
            ],
            "gap_threshold_sensitivity": [
                {
                    "threshold_tvd_m": _f(t.threshold_tvd_m),
                    "is_approved_threshold": bool(t.is_approved_threshold),
                    "n_bridged_gaps": _i(t.n_bridged_gaps),
                    "n_long_gaps": _i(t.n_long_gaps),
                    "derived_status": t.derived_status,
                }
                for t in sorted(sensitivity_by_well.get(wk, ()),
                                key=lambda x: x.threshold_tvd_m)
            ],
        }

    statuses = [e.status for e in eligibility_by_well.values()]

    return {
        "assurance_tier": _TIER,
        "increment": 7,
        "increment_title": INCREMENT_TITLE,
        "config_filename": config.source_filename,
        "config_schema_version": config.schema_version,
        "depth_convention": SIGN_CONVENTION_ID,
        "depth_convention_statement": OVERBURDEN_STATEMENTS[
            "manifest_depth_convention_statement"].text,
        "depth_convention_verified": bool(depth_convention_verified),
        "integration_method": config.integration_method,
        "integration_coordinate": "tvdss_m",
        "gravity_m_s2": _f(config.gravity_m_s2),
        "gravity_basis": OVERBURDEN_STATEMENTS["manifest_gravity_basis"].text,
        "assumption_register": {
            "rhob_min_kg_m3": _f(config.rhob_min_kg_m3),
            "rhob_max_kg_m3": _f(config.rhob_max_kg_m3),
            "bounds_are_inclusive": bool(config.bounds_are_inclusive),
            "short_gap_max_tvd_m": _f(config.short_gap_max_tvd_m),
            "shallow_gap_tolerance_tvd_m": _f(config.shallow_gap_tolerance_tvd_m),
            "profile_report_step_tvdss_m": _f(config.profile_report_step_tvdss_m),
            "seawater_density_kg_m3": _f(config.seawater_density_kg_m3),
            "seawater_density_low_kg_m3": _f(config.seawater_density_low_kg_m3),
            "seawater_density_high_kg_m3": _f(config.seawater_density_high_kg_m3),
            "scenario_high_percentile": _f(config.scenario_high_percentile),
            "statement": OVERBURDEN_STATEMENTS["manifest_assumption_statement"].text,
        },
        "calibration_data_available": {
            "pressure_rft_mdt_dst": False,
            "stress_fit_lot_xlot_dfit": False,
            "core_or_log_calibrated_density_control": False,
            "measured_seawater_density": False,
            "local_gravity_survey": False,
            "statement": OVERBURDEN_STATEMENTS["manifest_calibration_statement"].text,
        },
        "n_wells_evaluated": len(stats_by_well),
        "n_wells_absolute_supported": sum(1 for s in statuses if s == STATUS_ABSOLUTE),
        "n_wells_screening_sensitivity_only": sum(
            1 for s in statuses if s == STATUS_SENSITIVITY_ONLY),
        "n_wells_partial_measured_only": sum(
            1 for s in statuses if s == STATUS_PARTIAL_ONLY),
        "n_wells_not_eligible": sum(1 for s in statuses if s == STATUS_NOT_ELIGIBLE),
        "total_eligible_density_samples": sum(
            int(s.n_eligible) for s in stats_by_well.values()),
        "total_bridged_density_samples": sum(
            int(g.n_bridged_samples) for g in gap_results.values()),
        # DERIVED, never hardcoded: a violation means an emitted controlled
        # field escaped its closed registry, which is precisely the condition
        # under which a named lithology could have reached an artifact.
        "named_lithology_assigned": bool(coverage.violations),
        "named_lithology_statement": OVERBURDEN_STATEMENTS[
            "manifest_named_lithology_statement"].text,
        "lithology_validation": {
            "model": OVERBURDEN_STATEMENTS["manifest_lithology_model"].text,
            "derivation": OVERBURDEN_STATEMENTS["manifest_lithology_derivation"].text,
            "emitted_field_coverage": cov,
            "violations": [dict(v) for v in coverage.violations],
        },
        "methods_not_implemented": list(NOT_IMPLEMENTED_TOKENS),
        "limitations": [
            OVERBURDEN_STATEMENTS["limitation_no_absolute_without_full_column"].text,
            OVERBURDEN_STATEMENTS["limitation_no_density_repair"].text,
            OVERBURDEN_STATEMENTS["limitation_assumed_components"].text,
            OVERBURDEN_STATEMENTS["limitation_seabed_provenance"].text,
            OVERBURDEN_STATEMENTS["limitation_no_pore_pressure"].text,
        ],
        "issues": [
            {"severity": r["severity"], "code": r["code"],
             "context": r["context"], "message": r["message"]}
            for r in build_overburden_issue_rows(issues)
        ],
        "wells": wells,
    }


#### `p2mem/io/overburden_workflow.py` - the ONE real-data workflow

The notebook, the completion gate, the determinism check and the integration tests all
execute this same code path. A workflow the notebook re-implements inline is a workflow whose
tests prove nothing about the notebook.

In [ ]:
%%writefile p2mem/io/overburden_workflow.py
"""
p2mem.io.overburden_workflow - the ONE Increment 7 real-data workflow.

Exists so that the notebook, the completion gate, the determinism check and
the integration tests all execute the SAME code path. A workflow that the
notebook re-implements inline is a workflow whose tests prove nothing about
the notebook.

This module reads approved input FILES through the LOCKED loaders and the
LOCKED Increment 5 marker table. It contains no science of its own: every
number it returns comes from `p2mem.density_qc` or `p2mem.overburden`.
"""

from __future__ import annotations

import csv
from dataclasses import dataclass
import math
import os
from typing import Dict

from p2mem.density_qc import (
    build_density_masks, compute_density_qc_stats, condition_density_gaps,
)
from p2mem.overburden import (
    build_gap_threshold_sensitivity, build_shallow_column_scenarios,
    build_stress_profile, derive_overburden_eligibility, verify_depth_sign_convention,
)
from p2mem.overburden_models import OverburdenConfig, OverburdenInputError

__all__ = [
    "LOCKED_MARKER_TABLE", "SeabedMarker", "SeabedMarkerError",
    "read_locked_seabed_markers",
    "OverburdenRun", "run_overburden_workflow", "build_overburden_payloads",
]

LOCKED_MARKER_TABLE = os.path.join(
    "outputs", "05_formation_tops", "top_survey_corrected_markers.csv")


class SeabedMarkerError(OverburdenInputError):
    """The locked seabed table is missing, malformed, ambiguous or non-finite."""


@dataclass(frozen=True)
class SeabedMarker:
    """One well's seabed datum, read from the LOCKED Increment 5 output only."""

    well_key: str
    mdrt_m: float
    tvd_m: float
    tvdss_m: float

    def __post_init__(self) -> None:
        if not isinstance(self.well_key, str) or not self.well_key.strip():
            raise SeabedMarkerError("SeabedMarker: well_key must be a non-empty string.")
        for name in ("mdrt_m", "tvd_m", "tvdss_m"):
            value = getattr(self, name)
            if isinstance(value, bool) or not isinstance(value, (int, float)):
                raise SeabedMarkerError(
                    f"SeabedMarker {self.well_key!r}: {name} must be a finite real number.")
            if not math.isfinite(float(value)) or float(value) < 0.0:
                raise SeabedMarkerError(
                    f"SeabedMarker {self.well_key!r}: {name} must be finite and >= 0, "
                    f"got {value!r}.")


def read_locked_seabed_markers(
    marker_table_path: str, marker_name: str
) -> Dict[str, SeabedMarker]:
    """Read the seabed marker for each well from the LOCKED Increment 5 table.

    A well absent from that table has NO seabed datum in this project. It is
    omitted from the returned mapping - never defaulted to zero, never
    inferred from the log start, and never borrowed from another well.
    """
    if not isinstance(marker_name, str) or not marker_name.strip():
        raise SeabedMarkerError("marker_name must be a non-empty string.")
    if not os.path.isfile(marker_table_path):
        raise SeabedMarkerError(
            f"Locked seabed marker table is missing or not a file: "
            f"{os.path.basename(str(marker_table_path))!r}.")
    out: Dict[str, SeabedMarker] = {}
    wanted = marker_name.strip().lower()
    with open(marker_table_path, newline="", encoding="utf-8") as fh:
        reader = csv.DictReader(fh)
        required = (
            "well_key", "canonical_marker_name", "MDRT_reconciled_m",
            "TVD_survey_m", "TVDSS_survey_corrected_m",
        )
        fields = reader.fieldnames or []
        if len(fields) != len(set(fields)):
            raise SeabedMarkerError("Locked seabed marker table has duplicate columns.")
        missing = [name for name in required if name not in fields]
        if missing:
            raise SeabedMarkerError(
                f"Locked seabed marker table is missing required column(s): {missing}.")
        for row_number, row in enumerate(reader, start=2):
            if row.get("canonical_marker_name", "").strip().lower() != wanted:
                continue
            try:
                well_key = row["well_key"].strip()
                marker = SeabedMarker(
                    well_key=well_key,
                    mdrt_m=float(row["MDRT_reconciled_m"]),
                    tvd_m=float(row["TVD_survey_m"]),
                    tvdss_m=float(row["TVDSS_survey_corrected_m"]),
                )
            except (KeyError, AttributeError, TypeError, ValueError,
                    SeabedMarkerError) as exc:
                raise SeabedMarkerError(
                    f"Locked seabed marker row {row_number} is malformed for the "
                    f"requested marker {marker_name!r}.") from exc
            if well_key in out:
                raise SeabedMarkerError(
                    f"Locked seabed marker table contains duplicate {marker_name!r} "
                    f"rows for well {well_key!r}.")
            out[well_key] = marker
    return out


class OverburdenRun:
    """Everything one Increment 7 run produced, per well and in aggregate."""

    __slots__ = ("config", "frames", "seabed", "sign_convention", "masks", "gaps",
                 "gap_results", "stats", "profiles", "eligibility", "scenarios",
                 "sensitivity", "issues")

    def __init__(self, **kw):
        for slot in self.__slots__:
            setattr(self, slot, kw.get(slot))

    @property
    def depth_convention_verified(self) -> bool:
        return bool(self.sign_convention) and all(
            ev.get("verified") is True for ev in self.sign_convention.values())


def run_overburden_workflow(
    frames: Dict[str, object],
    config: OverburdenConfig,
    *,
    marker_table_path: str = LOCKED_MARKER_TABLE,
) -> OverburdenRun:
    """Execute the complete Increment 7 workflow for a set of well frames.

    Deterministic: wells are processed in sorted key order, and every result
    is a function of the frames, the configuration, and the locked marker
    table.
    """
    seabed = read_locked_seabed_markers(marker_table_path, config.seabed_marker_name)

    sign_convention: Dict[str, dict] = {}
    masks: Dict[str, object] = {}
    gap_results: Dict[str, object] = {}
    gaps: Dict[str, tuple] = {}
    stats: Dict[str, object] = {}
    profiles: Dict[str, object] = {}
    eligibility: Dict[str, object] = {}
    scenarios: Dict[str, tuple] = {}
    sensitivity: Dict[str, tuple] = {}

    for wk in sorted(frames):
        frame = frames[wk]
        sign_convention[wk] = verify_depth_sign_convention(frame)
        marker = seabed.get(wk)
        mdrt = None if marker is None else marker.mdrt_m
        tvd = None if marker is None else marker.tvd_m
        tvdss = None if marker is None else marker.tvdss_m

        # Two passes, and the order matters. The masks that feed gap
        # classification cannot themselves depend on the gap result, so the
        # first pass builds the eligibility mask, the gap pass consumes it,
        # and the second pass folds the resulting unresolved-region masks back
        # in. The eligibility mask is identical in both passes: gap
        # conditioning never adds or removes an ELIGIBLE MEASURED sample.
        provisional = build_density_masks(frame, config, seabed_mdrt_m=mdrt)
        gap_result = condition_density_gaps(
            frame, provisional, config, seabed_tvd_m=tvd)
        final_masks = build_density_masks(
            frame, config, seabed_mdrt_m=mdrt, gap_result=gap_result)

        well_stats = compute_density_qc_stats(
            frame, final_masks, config, seabed_mdrt_m=mdrt, seabed_tvd_m=tvd,
            seabed_tvdss_m=tvdss, gaps=gap_result.gaps)
        profile = build_stress_profile(frame, final_masks, gap_result, config)
        elig = derive_overburden_eligibility(
            well_stats, final_masks, gap_result, profile, config)

        masks[wk] = final_masks
        gap_results[wk] = gap_result
        gaps[wk] = gap_result.gaps
        stats[wk] = well_stats
        profiles[wk] = profile
        eligibility[wk] = elig
        scenarios[wk] = build_shallow_column_scenarios(
            well_stats, elig, profile, config)
        sensitivity[wk] = build_gap_threshold_sensitivity(
            frame, final_masks, well_stats, config, seabed_tvd_m=tvd)

    from p2mem.io.overburden_inventory import derive_overburden_issues
    issues = derive_overburden_issues(
        stats, eligibility, gap_results, profiles, config)

    return OverburdenRun(
        config=config, frames=frames, seabed=seabed, sign_convention=sign_convention,
        masks=masks, gaps=gaps, gap_results=gap_results, stats=stats,
        profiles=profiles, eligibility=eligibility, scenarios=scenarios,
        sensitivity=sensitivity, issues=issues)


def build_overburden_payloads(run: OverburdenRun) -> Dict[str, list]:
    """Build the complete, ordered Increment 7 payload set for export.

    The manifest is built LAST, from the CSV payloads this function has
    already produced, so its coverage block describes the records that are
    actually serialized rather than a reconstruction of them.
    """
    from p2mem.io.overburden_inventory import (
        build_density_availability_rows, build_density_gap_rows, build_density_qc_rows,
        build_gap_threshold_sensitivity_rows, build_overburden_eligibility_rows,
        build_overburden_issue_rows, build_overburden_manifest,
        build_shallow_column_scenario_rows, build_vertical_stress_profile_rows,
    )
    from p2mem.io.overburden_registry import (
        AVAILABILITY, ELIGIBILITY, GAPS, ISSUES, OVERBURDEN_MANIFEST_ARTIFACT,
        PROFILE, QC, SCENARIOS, SENSITIVITY,
    )

    payloads: Dict[str, list] = {
        AVAILABILITY: build_density_availability_rows(run.stats, run.frames),
        QC: build_density_qc_rows(run.stats, run.masks, run.config),
        GAPS: build_density_gap_rows(run.gaps),
        ELIGIBILITY: build_overburden_eligibility_rows(
            run.eligibility, run.profiles, run.config),
        PROFILE: build_vertical_stress_profile_rows(run.profiles, run.config),
        SCENARIOS: build_shallow_column_scenario_rows(run.scenarios),
        SENSITIVITY: build_gap_threshold_sensitivity_rows(run.sensitivity),
        ISSUES: build_overburden_issue_rows(run.issues),
    }
    manifest = build_overburden_manifest(
        run.stats, run.eligibility, run.gap_results, run.profiles, run.scenarios,
        run.sensitivity, run.issues, run.frames, run.config,
        depth_convention_verified=run.depth_convention_verified,
        output_payloads=payloads)
    payloads[OVERBURDEN_MANIFEST_ARTIFACT] = manifest
    return payloads


---

## Tests

All fixtures below are small, fictional and synthetic. **No real or private project LAS,
deviation, checkshot, or formation-top file is read, referenced, copied, or packaged by any
test.**

#### `tests/synthetic_inc7.py` - fictional well frames built from the REAL locked dataclasses

In [ ]:
%%writefile tests/synthetic_inc7.py
"""
Small, fictional, fully synthetic fixtures for the Increment 7 tests.

NOTHING here is derived from, or resembles, an approved project file. Every
array is hand-written so that the expected answer can be computed by hand and
compared against the implementation, which is the only way an analytical test
proves anything.

The well frames built here are real `p2mem.wellframe_models.WellFrame` /
`CurveSlot` objects, not stand-ins: a test that passes against a mock proves
nothing about the locked structure the production path actually receives.
"""

from __future__ import annotations

import numpy as np

from p2mem.wellframe_models import CurveSlot, WellFrame

__all__ = [
    "readonly", "make_curve", "make_frame", "constant_density_frame",
    "two_layer_frame", "gap_frame", "STANDARD_G",
]

#: The exact gravity every analytical expectation in these tests is computed
#: with. Kept here so that a change to the configured value cannot silently
#: move an "analytical" expectation with it.
STANDARD_G = 9.80665


def readonly(arr):
    view = np.asarray(arr).view()
    view.setflags(write=False)
    return view


def make_curve(values, *, canonical_name="RHOB_kg_m3", canonical_unit="kg/m3",
               conversion_function="gcc_to_kgm3", source_curve_name="RHOB",
               raw_mnemonic="RHOB", raw_unit="g/cc",
               source_filename="synthetic.las"):
    values = np.asarray(values, dtype=np.float64)
    valid = np.isfinite(values)
    return CurveSlot(
        canonical_name=canonical_name,
        source_curve_name=source_curve_name,
        raw_mnemonic=raw_mnemonic,
        raw_unit=raw_unit,
        canonical_unit=canonical_unit,
        conversion_function=conversion_function,
        source_filename=source_filename,
        evidence_class="measured",
        values=readonly(values),
        valid_mask=readonly(valid),
        n_samples=int(values.size),
        valid_count=int(np.count_nonzero(valid)),
        valid_fraction=(float(np.count_nonzero(valid)) / values.size
                        if values.size else 0.0),
    )


def make_frame(*, well_key="SYNTH_1", md, tvd=None, tvdss=None, density=None,
               datum_elevation_m=25.0, depth_valid_mask=None,
               survey_md_min_m=None, survey_md_max_m=None,
               depth_map_status="fully_mapped_within_survey_coverage",
               curves=None, extra_curves=None):
    """Build one synthetic `WellFrame`.

    `tvd` defaults to `md` (a vertical well). `tvdss` is always derived as
    `tvd - datum_elevation_m`, matching the project convention exactly, so a
    test that wants to violate the convention must say so explicitly by
    passing `tvdss`.
    """
    md = np.asarray(md, dtype=np.float64)
    tvd = md.copy() if tvd is None else np.asarray(tvd, dtype=np.float64)
    tvdss = (tvd - datum_elevation_m) if tvdss is None else np.asarray(
        tvdss, dtype=np.float64)
    n = int(md.size)
    if depth_valid_mask is None:
        depth_valid_mask = np.isfinite(tvd) & np.isfinite(tvdss)
    depth_valid_mask = np.asarray(depth_valid_mask, dtype=bool)

    slots = {}
    if curves is None and density is not None:
        slots["RHOB_kg_m3"] = make_curve(density)
    elif curves is not None:
        slots.update(curves)
    if extra_curves:
        slots.update(extra_curves)

    finite_md = md[np.isfinite(md)]
    return WellFrame(
        well_key=well_key,
        source_las_filename="synthetic.las",
        source_survey_filename="synthetic_dev.txt",
        n_samples=n,
        MD_m=readonly(md),
        TVD_m=readonly(tvd),
        TVDSS_m=readonly(tvdss),
        depth_valid_mask=readonly(depth_valid_mask),
        depth_basis_used="petrel_source_trace",
        interpolation_method="piecewise_linear_station_interpolation",
        datum_elevation_m=float(datum_elevation_m),
        depth_map_status=depth_map_status,
        survey_md_min_m=(float(np.min(finite_md)) if survey_md_min_m is None
                         else float(survey_md_min_m)),
        survey_md_max_m=(float(np.max(finite_md)) if survey_md_max_m is None
                         else float(survey_md_max_m)),
        las_md_min_m=float(np.min(finite_md)) if finite_md.size else 0.0,
        las_md_max_m=float(np.max(finite_md)) if finite_md.size else 0.0,
        n_depth_unmapped=int(n - np.count_nonzero(depth_valid_mask)),
        n_extrapolated=0,
        curves=slots,
    )


def constant_density_frame(rho=2000.0, n=11, step=10.0, top=1000.0,
                           datum_elevation_m=25.0, **kw):
    """A vertical well with CONSTANT density over a uniform depth grid.

    Analytical expectation: sigma_v increment over the whole column is
    `rho * g * (n - 1) * step`, exactly, because trapezoidal quadrature is
    exact for a constant integrand.
    """
    md = top + step * np.arange(n, dtype=np.float64)
    return make_frame(md=md, density=np.full(n, float(rho)),
                      datum_elevation_m=datum_elevation_m, **kw)


def two_layer_frame(rho_upper=2000.0, rho_lower=2500.0, n_upper=6, n_lower=6,
                    step=10.0, top=1000.0, datum_elevation_m=25.0, **kw):
    """Two constant-density layers meeting at one shared node.

    Analytical expectation: the total increment is the sum of the two layers'
    `rho * g * h`, because each layer is a constant integrand and the shared
    node contributes no interval of its own.
    """
    n = n_upper + n_lower - 1
    md = top + step * np.arange(n, dtype=np.float64)
    rho = np.concatenate([
        np.full(n_upper, float(rho_upper)),
        np.full(n_lower - 1, float(rho_lower)),
    ])
    return make_frame(md=md, density=rho, datum_elevation_m=datum_elevation_m, **kw)


def gap_frame(gap_slices, rho=2000.0, n=41, step=1.0, top=1000.0,
              datum_elevation_m=25.0, **kw):
    """A constant-density well with NaN density over the given index slices."""
    md = top + step * np.arange(n, dtype=np.float64)
    density = np.full(n, float(rho))
    for sl in gap_slices:
        density[sl] = np.nan
    return make_frame(md=md, density=density,
                      datum_elevation_m=datum_elevation_m, **kw)


#### `tests/helpers_inc7.py` - the shipped configuration and the production pipeline, shared by every test

In [ ]:
%%writefile tests/helpers_inc7.py
"""
Shared Increment 7 test helpers: the loaded configuration and the ONE
mask/gap/stats/profile/eligibility pipeline the production workflow uses.

`prepared` deliberately calls the SAME sequence as
`p2mem.io.overburden_workflow.run_overburden_workflow`. A test helper that
assembled the pieces in a different order would let the tests pass while the
production path did something else.
"""

from __future__ import annotations

import os
from pathlib import Path

import pytest

from p2mem.density_qc import (
    build_density_masks, compute_density_qc_stats, condition_density_gaps,
    load_overburden_config,
)
from p2mem.overburden import build_stress_profile, derive_overburden_eligibility

PROJECT_ROOT = Path(__file__).resolve().parent.parent
CONFIG_PATH = str(PROJECT_ROOT / "config" / "overburden_stress.yml")


@pytest.fixture(scope="session")
def overburden_config():
    """The real, packaged Increment 7 configuration.

    Session-scoped and frozen: the tests exercise the SHIPPED policy, not a
    convenient one invented for testing.
    """
    return load_overburden_config(CONFIG_PATH)


def prepared(frame, config, *, seabed_mdrt_m=None, seabed_tvd_m=None,
             seabed_tvdss_m=None, threshold_tvd_m=None):
    """Run the production pipeline for one frame and return every stage."""
    provisional = build_density_masks(frame, config, seabed_mdrt_m=seabed_mdrt_m)
    gaps = condition_density_gaps(
        frame, provisional, config, seabed_tvd_m=seabed_tvd_m,
        threshold_tvd_m=threshold_tvd_m)
    masks = build_density_masks(
        frame, config, seabed_mdrt_m=seabed_mdrt_m, gap_result=gaps)
    stats = compute_density_qc_stats(
        frame, masks, config, seabed_mdrt_m=seabed_mdrt_m,
        seabed_tvd_m=seabed_tvd_m, seabed_tvdss_m=seabed_tvdss_m,
        gaps=gaps.gaps)
    profile = build_stress_profile(frame, masks, gaps, config)
    eligibility = derive_overburden_eligibility(stats, masks, gaps, profile, config)
    return {
        "masks": masks, "gaps": gaps, "stats": stats,
        "profile": profile, "eligibility": eligibility,
    }


#### `tests/test_density_qc.py` - configuration, unit confirmation, the twelve masks, EXACT screening boundaries, gaps

In [ ]:
%%writefile tests/test_density_qc.py
"""
Increment 7 - configuration loading, unit/provenance confirmation, the twelve
explicit masks, screening-bound behaviour at EXACT boundaries, gap
classification, and gap conditioning.

Boundary tests here use the exact configured bound values, not values near
them: an inclusive/exclusive slip at a bound is a silent scientific change,
and only an exact-boundary test catches it.
"""

from __future__ import annotations

from dataclasses import replace
import math
import os

import numpy as np
import pytest
import yaml

from p2mem.density_qc import (
    DensityUnitError, build_density_masks, classify_density_gaps,
    compute_density_qc_stats, condition_density_gaps, load_overburden_config,
    resolve_density_slot,
)
from p2mem.overburden_models import (
    DENSITY_MASK_NAMES,
    GAP_CLASS_LONG_INTERNAL,
    GAP_CLASS_SHALLOW,
    GAP_CLASS_SHORT_INTERNAL,
    GAP_CLASS_TERMINAL,
    GAP_CLASS_UNMAPPED_DEPTH,
    GAP_DISPOSITION_BRIDGED,
    GAP_DISPOSITION_UNRESOLVED,
    SEABED_BASIS_LOCKED_MARKER,
    SEABED_BASIS_NOT_DETERMINABLE,
    DensityGapRecord,
    DensityMaskSet,
    GapConditioningResult,
    OverburdenConfigError,
    OverburdenInputError,
    readonly,
)

from helpers_inc7 import CONFIG_PATH, overburden_config, prepared  # noqa: E402
from synthetic_inc7 import (  # noqa: E402
    constant_density_frame, gap_frame, make_curve, make_frame,
)


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

def test_packaged_config_loads_and_declares_the_expected_policy(overburden_config):
    c = overburden_config
    assert c.increment == 7
    assert c.canonical_curve_name == "RHOB_kg_m3"
    assert c.accepted_canonical_unit == "kg/m3"
    assert c.expected_conversion_function == "gcc_to_kgm3"
    assert c.bounds_are_inclusive is True
    assert c.gravity_m_s2 == 9.80665
    assert c.scenario_names == ("low", "base", "high")
    assert c.scenario_high_percentile == 5.0
    assert c.short_gap_max_tvd_m in c.sensitivity_thresholds_tvd_m
    assert "vertical_depth" in c.integration_method


def _write_config(tmp_path, mutate):
    with open(CONFIG_PATH, encoding="utf-8") as fh:
        raw = yaml.safe_load(fh)
    mutate(raw)
    path = tmp_path / "overburden_stress.yml"
    path.write_text(yaml.safe_dump(raw, sort_keys=False), encoding="utf-8")
    return str(path)


def test_missing_config_file_is_a_typed_error(tmp_path):
    with pytest.raises(OverburdenConfigError, match="not found"):
        load_overburden_config(str(tmp_path / "nope.yml"))


def test_unknown_top_level_key_is_rejected_not_ignored(tmp_path):
    path = _write_config(tmp_path, lambda r: r.update({"surprise": 1}))
    with pytest.raises(OverburdenConfigError, match="unknown top-level key"):
        load_overburden_config(path)


def test_unknown_key_inside_a_known_section_is_rejected(tmp_path):
    path = _write_config(
        tmp_path, lambda r: r["screening_bounds"].update({"surprise": 1}))
    with pytest.raises(OverburdenConfigError, match="unknown key"):
        load_overburden_config(path)


def test_missing_required_section_key_is_rejected(tmp_path):
    path = _write_config(tmp_path, lambda r: r["integration"].pop("gravity_m_s2"))
    with pytest.raises(OverburdenConfigError, match="required key"):
        load_overburden_config(path)


@pytest.mark.parametrize("key", [
    "bridge_across_seabed_allowed", "bridge_outside_survey_coverage_allowed",
    "bridge_long_gaps_allowed", "shallow_gap_uses_internal_gap_rule",
])
def test_hard_scientific_invariants_cannot_be_switched_on(tmp_path, key):
    path = _write_config(tmp_path, lambda r: r["gap_conditioning"].update({key: True}))
    with pytest.raises(OverburdenConfigError, match="hard scientific invariant"):
        load_overburden_config(path)


def test_original_values_immutable_cannot_be_switched_off(tmp_path):
    path = _write_config(
        tmp_path,
        lambda r: r["density_source"].update({"original_values_immutable": False}))
    with pytest.raises(OverburdenConfigError, match="never modifies"):
        load_overburden_config(path)


def test_base_scenario_cannot_be_declared_a_best_estimate(tmp_path):
    path = _write_config(
        tmp_path,
        lambda r: r["shallow_column_scenarios"].update(
            {"base_is_not_a_best_estimate": False}))
    with pytest.raises(OverburdenConfigError, match="best estimate"):
        load_overburden_config(path)


def test_assumed_fraction_disclosure_cannot_be_switched_off(tmp_path):
    path = _write_config(
        tmp_path,
        lambda r: r["shallow_column_scenarios"].update(
            {"require_assumed_fraction_disclosure": False}))
    with pytest.raises(OverburdenConfigError, match="originates in the assumption"):
        load_overburden_config(path)


def test_cross_well_seabed_transfer_cannot_be_switched_on(tmp_path):
    path = _write_config(
        tmp_path,
        lambda r: r["water_column"].update({"cross_well_seabed_transfer_allowed": True}))
    with pytest.raises(OverburdenConfigError, match="belongs to the well"):
        load_overburden_config(path)


@pytest.mark.parametrize("value", [float("nan"), float("inf"), float("-inf")])
def test_non_finite_configuration_value_is_rejected(tmp_path, value):
    path = _write_config(
        tmp_path, lambda r: r["integration"].update({"gravity_m_s2": value}))
    with pytest.raises(OverburdenConfigError):
        load_overburden_config(path)


@pytest.mark.parametrize("value", [True, "9.81", None, [9.81]])
def test_wrongly_typed_configuration_value_is_rejected(tmp_path, value):
    path = _write_config(
        tmp_path, lambda r: r["integration"].update({"gravity_m_s2": value}))
    with pytest.raises(OverburdenConfigError):
        load_overburden_config(path)


def test_inverted_screening_bounds_are_rejected(tmp_path):
    path = _write_config(
        tmp_path,
        lambda r: r["screening_bounds"].update(
            {"rhob_min_kg_m3": 3500.0, "rhob_max_kg_m3": 1000.0}))
    with pytest.raises(OverburdenConfigError, match="strictly less"):
        load_overburden_config(path)


def test_approved_threshold_must_appear_in_the_sensitivity_list(tmp_path):
    path = _write_config(
        tmp_path,
        lambda r: r["gap_conditioning"].update({"short_gap_max_tvd_m": 7.5}))
    with pytest.raises(OverburdenConfigError, match="must appear in"):
        load_overburden_config(path)


def test_duplicate_configuration_key_is_rejected(tmp_path):
    path = tmp_path / "dup.yml"
    with open(CONFIG_PATH, encoding="utf-8") as fh:
        text = fh.read()
    path.write_text(text + "\nincrement: 7\n", encoding="utf-8")
    with pytest.raises(OverburdenConfigError, match="duplicate"):
        load_overburden_config(str(path))


def test_scenario_names_must_be_exactly_low_base_high(tmp_path):
    path = _write_config(
        tmp_path,
        lambda r: r["shallow_column_scenarios"].update({"scenario_names": ["base"]}))
    with pytest.raises(OverburdenConfigError, match="low/base/high"):
        load_overburden_config(path)


def test_schema_version_is_closed_not_merely_nonempty(tmp_path):
    path = _write_config(
        tmp_path, lambda r: r.update({"schema_version": "7.0.1-typo"}))
    with pytest.raises(OverburdenConfigError, match="exactly '7.0.1'"):
        load_overburden_config(path)


def test_fractional_minimum_sample_count_is_rejected_not_truncated(tmp_path):
    path = _write_config(
        tmp_path,
        lambda r: r["eligibility"].update(
            {"min_eligible_samples_for_increment": 2.9}))
    with pytest.raises(OverburdenConfigError, match="must be an integer"):
        load_overburden_config(path)


@pytest.mark.parametrize("value", [True, "2", 2.0, None])
def test_non_integer_minimum_sample_count_is_rejected(tmp_path, value):
    path = _write_config(
        tmp_path,
        lambda r: r["eligibility"].update(
            {"min_eligible_samples_for_increment": value}))
    with pytest.raises(OverburdenConfigError, match="must be an integer"):
        load_overburden_config(path)


def test_status_vocabulary_and_order_are_closed(tmp_path):
    path = _write_config(
        tmp_path,
        lambda r: r["eligibility"].update(
            {"statuses": ["a", "b", "c", "d"]}))
    with pytest.raises(OverburdenConfigError, match="must equal the four declared"):
        load_overburden_config(path)


def test_numeric_not_implemented_entry_is_rejected_not_coerced(tmp_path):
    path = _write_config(tmp_path, lambda r: r.update({"not_implemented": [1]}))
    with pytest.raises(OverburdenConfigError, match="not coerced"):
        load_overburden_config(path)


def test_duplicate_not_implemented_entry_is_rejected(tmp_path):
    def mutate(raw):
        raw["not_implemented"].append(raw["not_implemented"][0])

    path = _write_config(tmp_path, mutate)
    with pytest.raises(OverburdenConfigError, match="duplicate"):
        load_overburden_config(path)


@pytest.mark.parametrize("key", ["low_basis", "base_basis", "high_basis"])
def test_scenario_basis_tokens_are_closed(tmp_path, key):
    path = _write_config(
        tmp_path,
        lambda r: r["shallow_column_scenarios"].update({key: "unreviewed_basis"}))
    with pytest.raises(OverburdenConfigError, match=key):
        load_overburden_config(path)


@pytest.mark.parametrize("value", [0.1, 4.999, 50.0, 95.0, 99.9])
def test_loader_rejects_high_percentile_other_than_the_stored_p05(
        tmp_path, value):
    path = _write_config(
        tmp_path,
        lambda r: r["shallow_column_scenarios"].update(
            {"high_percentile": value}))
    with pytest.raises(OverburdenConfigError, match="exactly 5.0"):
        load_overburden_config(path)


def test_direct_config_replace_remains_the_supported_frozen_variant_path(
        overburden_config):
    changed = replace(overburden_config, bridge_short_internal_gaps=False)
    assert changed.bridge_short_internal_gaps is False
    assert overburden_config.bridge_short_internal_gaps is True


@pytest.mark.parametrize("value", [0.1, 4.999, 50.0, 95.0, 99.9])
def test_direct_config_constructor_rejects_high_percentile_other_than_p05(
        overburden_config, value):
    with pytest.raises(OverburdenConfigError, match="exactly 5.0"):
        replace(overburden_config, scenario_high_percentile=value)


@pytest.mark.parametrize("field_name,value", [
    ("schema_version", "anything"),
    ("increment", 7.0),
    ("increment", True),
    ("bridge_short_internal_gaps", "false"),
    ("bounds_are_inclusive", 1),
    ("scenarios_enabled", np.bool_(True)),
    ("absolute_requires_uninterrupted_column", "true"),
    ("min_eligible_samples_for_increment", 2.9),
    ("min_eligible_samples_for_increment", True),
])
def test_direct_config_constructor_rejects_closed_or_scalar_type_bypasses(
        overburden_config, field_name, value):
    with pytest.raises(OverburdenConfigError):
        replace(overburden_config, **{field_name: value})


@pytest.mark.parametrize("field_name", [
    "source_filename", "assurance_tier", "canonical_curve_name",
    "accepted_canonical_unit", "expected_conversion_function",
    "integration_method", "seabed_marker_name",
])
@pytest.mark.parametrize("value", ["", "   ", 1, None])
def test_direct_config_constructor_rejects_invalid_string_fields(
        overburden_config, field_name, value):
    with pytest.raises(OverburdenConfigError):
        replace(overburden_config, **{field_name: value})


@pytest.mark.parametrize("field_name", [
    "rhob_min_kg_m3", "rhob_max_kg_m3", "short_gap_max_tvd_m",
    "shallow_gap_tolerance_tvd_m", "gravity_m_s2",
    "profile_report_step_tvdss_m", "seawater_density_kg_m3",
    "seawater_density_low_kg_m3", "seawater_density_high_kg_m3",
    "scenario_high_percentile",
])
@pytest.mark.parametrize("value", [True, "1.0", 1 + 0j, None, float("nan"),
                                    float("inf"), float("-inf")])
def test_direct_config_constructor_rejects_non_numeric_or_non_finite_scalars(
        overburden_config, field_name, value):
    with pytest.raises(OverburdenConfigError):
        replace(overburden_config, **{field_name: value})


@pytest.mark.parametrize("value", [
    [0.0, 10.0],
    (),
    (0.0, 10.0, 10.0),
    (10.0, 0.0),
    (-1.0, 10.0),
    (0.0, float("nan"), 10.0),
    (0.0, "10.0"),
])
def test_direct_config_constructor_rejects_malformed_threshold_tuple(
        overburden_config, value):
    with pytest.raises(OverburdenConfigError):
        replace(overburden_config, sensitivity_thresholds_tvd_m=value)


@pytest.mark.parametrize("field_name,value", [
    ("integration_method", "trapezoidal_in_measured_depth"),
    ("scenario_names", ["low", "base", "high"]),
    ("scenario_names", ("base",)),
    ("not_implemented", ["pore_pressure_prediction"]),
    ("not_implemented", (1,)),
    ("not_implemented", ("",)),
    ("not_implemented", ("pore_pressure_prediction", "pore_pressure_prediction")),
    ("shallow_gap_tolerance_tvd_m", -1.0),
])
def test_direct_config_constructor_rejects_malformed_policy_values(
        overburden_config, field_name, value):
    with pytest.raises(OverburdenConfigError):
        replace(overburden_config, **{field_name: value})


# ---------------------------------------------------------------------------
# Unit and provenance confirmation
# ---------------------------------------------------------------------------

def test_a_well_without_a_density_curve_resolves_to_none(overburden_config):
    frame = make_frame(md=np.arange(1000.0, 1051.0, 10.0), curves={})
    slot, unit_ok, conv_ok = resolve_density_slot(frame, overburden_config)
    assert slot is None and unit_ok is False and conv_ok is False


def test_an_unexpected_canonical_unit_is_refused_not_rescaled(overburden_config):
    md = np.arange(1000.0, 1051.0, 10.0)
    frame = make_frame(md=md, curves={
        "RHOB_kg_m3": make_curve(np.full(md.size, 2.5), canonical_unit="g/cm3")})
    with pytest.raises(DensityUnitError, match="no unit conversion of its own"):
        resolve_density_slot(frame, overburden_config)


def test_density_unit_error_is_a_typeerror_subclass():
    assert issubclass(DensityUnitError, TypeError)


def test_an_unexpected_conversion_function_is_reported_not_corrected(
        overburden_config):
    md = np.arange(1000.0, 1051.0, 10.0)
    frame = make_frame(md=md, curves={
        "RHOB_kg_m3": make_curve(np.full(md.size, 2400.0),
                                 conversion_function="identity")})
    slot, unit_ok, conv_ok = resolve_density_slot(frame, overburden_config)
    assert slot is not None and unit_ok is True and conv_ok is False
    out = prepared(frame, overburden_config)
    assert out["stats"].conversion_confirmed is False
    # It is reported, not repaired: the curve is still eligible.
    assert out["stats"].n_eligible == md.size


@pytest.mark.parametrize("values", [
    np.array([True, False, True]),
    np.array(["2400", "2400", "2400"]),
    np.array([2400 + 0j, 2400 + 0j, 2400 + 0j]),
    np.array([object(), object(), object()], dtype=object),
])
def test_ambiguous_density_dtype_is_a_typeerror(overburden_config, values):
    md = np.arange(1000.0, 1030.0, 10.0)
    frame = make_frame(md=md, curves={
        "RHOB_kg_m3": make_curve.__wrapped__(values) if hasattr(make_curve, "__wrapped__")
        else _raw_curve(values)})
    with pytest.raises(TypeError):
        build_density_masks(frame, overburden_config)


def _raw_curve(values):
    """A CurveSlot holding a deliberately ambiguous dtype array.

    `make_curve` casts to float64 on purpose, so this helper bypasses it to
    reach the dtype gate that protects the production path.
    """
    from p2mem.wellframe_models import CurveSlot
    arr = np.asarray(values)
    view = arr.view()
    view.setflags(write=False)
    mask = np.zeros(arr.size, dtype=bool)
    mask.setflags(write=False)
    return CurveSlot(
        canonical_name="RHOB_kg_m3", source_curve_name="RHOB", raw_mnemonic="RHOB",
        raw_unit="g/cc", canonical_unit="kg/m3", conversion_function="gcc_to_kgm3",
        source_filename="synthetic.las", evidence_class="measured",
        values=view, valid_mask=mask, n_samples=int(arr.size), valid_count=0,
        valid_fraction=0.0)


# ---------------------------------------------------------------------------
# Masks
# ---------------------------------------------------------------------------

def test_all_twelve_masks_are_present_and_full_length(overburden_config):
    frame = constant_density_frame(n=17)
    masks = prepared(frame, overburden_config)["masks"]
    for name in DENSITY_MASK_NAMES:
        arr = getattr(masks, name)
        if name == "below_seabed_sample":
            assert arr is None            # no seabed supplied in this fixture
            continue
        assert arr.dtype == np.bool_
        assert arr.size == 17
        assert not arr.flags.writeable


def test_below_seabed_mask_is_none_when_the_seabed_is_not_determinable(
        overburden_config):
    masks = prepared(constant_density_frame(), overburden_config)["masks"]
    assert masks.below_seabed_sample is None
    assert masks.seabed_resolved is False
    assert masks.counts()["below_seabed_sample"] is None


def test_below_seabed_mask_exists_when_the_seabed_is_resolved(overburden_config):
    md = np.arange(400.0, 601.0, 10.0)
    frame = make_frame(md=md, density=np.full(md.size, 2000.0))
    masks = prepared(frame, overburden_config, seabed_mdrt_m=500.0,
                     seabed_tvd_m=500.0, seabed_tvdss_m=475.0)["masks"]
    assert masks.seabed_resolved is True
    assert int(np.count_nonzero(masks.below_seabed_sample)) == 11


def test_eligibility_mask_is_a_strict_conjunction(overburden_config):
    frame = gap_frame([slice(5, 8)], n=21, step=1.0)
    masks = prepared(frame, overburden_config)["masks"]
    elig = np.asarray(masks.eligible_for_measured_integration)
    for name in ("finite_numeric_density", "unit_resolved",
                 "screening_range_plausible", "depth_mapping_valid",
                 "within_survey_coverage"):
        assert not np.any(elig & ~np.asarray(getattr(masks, name)))


def test_a_bridged_sample_is_never_also_an_eligible_measured_sample(
        overburden_config):
    frame = gap_frame([slice(10, 13)], n=41, step=1.0)
    masks = prepared(frame, overburden_config)["masks"]
    assert not np.any(np.asarray(masks.bridged_short_gap)
                      & np.asarray(masks.eligible_for_measured_integration))


def test_mask_set_rejects_a_writable_mask():
    with pytest.raises(OverburdenInputError, match="read-only"):
        DensityMaskSet(
            well_key="X", n_samples=2, seabed_resolved=False,
            source_value_present=np.ones(2, dtype=bool),
            finite_numeric_density=readonly(np.ones(2, dtype=bool)),
            unit_resolved=readonly(np.ones(2, dtype=bool)),
            screening_range_plausible=readonly(np.ones(2, dtype=bool)),
            below_seabed_sample=None,
            depth_mapping_valid=readonly(np.ones(2, dtype=bool)),
            within_survey_coverage=readonly(np.ones(2, dtype=bool)),
            eligible_for_measured_integration=readonly(np.ones(2, dtype=bool)),
            bridged_short_gap=readonly(np.zeros(2, dtype=bool)),
            unresolved_long_gap=readonly(np.zeros(2, dtype=bool)),
            unresolved_shallow_column=readonly(np.zeros(2, dtype=bool)),
            unresolved_terminal_column=readonly(np.zeros(2, dtype=bool)))


def test_mask_set_rejects_an_eligibility_mask_that_is_not_a_conjunction():
    ones, zeros = readonly(np.ones(2, dtype=bool)), readonly(np.zeros(2, dtype=bool))
    with pytest.raises(OverburdenInputError, match="must be a conjunction"):
        DensityMaskSet(
            well_key="X", n_samples=2, seabed_resolved=False,
            source_value_present=ones, finite_numeric_density=zeros,
            unit_resolved=ones, screening_range_plausible=ones,
            below_seabed_sample=None, depth_mapping_valid=ones,
            within_survey_coverage=ones, eligible_for_measured_integration=ones,
            bridged_short_gap=zeros, unresolved_long_gap=zeros,
            unresolved_shallow_column=zeros, unresolved_terminal_column=zeros)


def test_mask_set_rejects_a_seabed_flag_that_disagrees_with_its_mask():
    ones, zeros = readonly(np.ones(2, dtype=bool)), readonly(np.zeros(2, dtype=bool))
    with pytest.raises(OverburdenInputError, match="seabed"):
        DensityMaskSet(
            well_key="X", n_samples=2, seabed_resolved=True,
            source_value_present=ones, finite_numeric_density=ones,
            unit_resolved=ones, screening_range_plausible=ones,
            below_seabed_sample=None, depth_mapping_valid=ones,
            within_survey_coverage=ones, eligible_for_measured_integration=ones,
            bridged_short_gap=zeros, unresolved_long_gap=zeros,
            unresolved_shallow_column=zeros, unresolved_terminal_column=zeros)


def test_mask_construction_does_not_mutate_the_source_density_array(
        overburden_config):
    frame = gap_frame([slice(5, 8)], n=21, step=1.0)
    before = np.array(frame.curve("RHOB_kg_m3").values, copy=True)
    build_density_masks(frame, overburden_config)
    after = np.asarray(frame.curve("RHOB_kg_m3").values)
    assert np.array_equal(np.isnan(before), np.isnan(after))
    assert np.array_equal(before[~np.isnan(before)], after[~np.isnan(after)])


# ---------------------------------------------------------------------------
# Screening bounds - EXACT boundary behaviour
# ---------------------------------------------------------------------------

def test_values_exactly_on_both_bounds_are_accepted(overburden_config):
    lo, hi = overburden_config.rhob_min_kg_m3, overburden_config.rhob_max_kg_m3
    md = np.arange(1000.0, 1030.0, 10.0)
    frame = make_frame(md=md, density=np.array([lo, 0.5 * (lo + hi), hi]))
    out = prepared(frame, overburden_config)
    assert out["stats"].n_in_screening_band == 3
    assert out["stats"].n_screening_bound_failures == 0


def test_values_just_outside_both_bounds_are_masked_out(overburden_config):
    lo, hi = overburden_config.rhob_min_kg_m3, overburden_config.rhob_max_kg_m3
    md = np.arange(1000.0, 1040.0, 10.0)
    frame = make_frame(md=md, density=np.array([
        np.nextafter(lo, -np.inf), lo, hi, np.nextafter(hi, np.inf)]))
    stats = prepared(frame, overburden_config)["stats"]
    assert stats.n_in_screening_band == 2
    assert stats.n_below_screening_min == 1
    assert stats.n_above_screening_max == 1
    assert stats.n_screening_bound_failures == 2


def test_non_positive_density_is_counted_as_a_bound_failure(overburden_config):
    md = np.arange(1000.0, 1040.0, 10.0)
    frame = make_frame(md=md, density=np.array([-1.0, 0.0, 2400.0, 2500.0]))
    stats = prepared(frame, overburden_config)["stats"]
    assert stats.n_non_positive == 2
    assert stats.n_screening_bound_failures == 2
    assert stats.n_in_screening_band == 2


@pytest.mark.parametrize("bad", [np.nan, np.inf, -np.inf])
def test_non_finite_density_is_never_in_the_screening_band(overburden_config, bad):
    md = np.arange(1000.0, 1030.0, 10.0)
    frame = make_frame(md=md, density=np.array([2400.0, bad, 2400.0]))
    stats = prepared(frame, overburden_config)["stats"]
    assert stats.n_finite == 2
    assert stats.n_non_finite == 1
    assert stats.n_in_screening_band == 2


def test_screening_counts_partition_the_finite_samples(overburden_config):
    md = np.arange(1000.0, 1060.0, 10.0)
    frame = make_frame(md=md, density=np.array(
        [np.nan, -1.0, 500.0, 2400.0, 4000.0, np.inf]))
    s = prepared(frame, overburden_config)["stats"]
    assert s.n_in_screening_band + s.n_screening_bound_failures == s.n_finite
    assert s.n_finite + s.n_non_finite == s.n_samples


# ---------------------------------------------------------------------------
# Gaps
# ---------------------------------------------------------------------------

def test_shallow_gap_is_classified_and_never_bridged(overburden_config):
    md = np.arange(1000.0, 1101.0, 10.0)
    frame = make_frame(md=md, density=np.full(md.size, 2000.0))
    gr = prepared(frame, overburden_config, seabed_mdrt_m=500.0,
                  seabed_tvd_m=500.0, seabed_tvdss_m=475.0)["gaps"]
    shallow = [g for g in gr.gaps if g.gap_class == GAP_CLASS_SHALLOW]
    assert len(shallow) == 1
    assert shallow[0].disposition == GAP_DISPOSITION_UNRESOLVED
    assert math.isclose(shallow[0].thickness_tvd_m, 500.0)


def test_shallow_gap_is_not_bridged_at_any_threshold(overburden_config):
    md = np.arange(1000.0, 1101.0, 10.0)
    frame = make_frame(md=md, density=np.full(md.size, 2000.0))
    for threshold in (0.0, 10.0, 1000.0, 1.0e9):
        gr = prepared(frame, overburden_config, seabed_mdrt_m=500.0,
                      seabed_tvd_m=500.0, seabed_tvdss_m=475.0,
                      threshold_tvd_m=threshold)["gaps"]
        shallow = next(g for g in gr.gaps if g.gap_class == GAP_CLASS_SHALLOW)
        assert shallow.disposition == GAP_DISPOSITION_UNRESOLVED


def test_terminal_gap_is_classified_and_never_bridged(overburden_config):
    md = np.arange(1000.0, 1101.0, 10.0)
    rho = np.full(md.size, 2000.0)
    rho[-4:] = np.nan
    frame = make_frame(md=md, density=rho)
    gr = prepared(frame, overburden_config)["gaps"]
    terminal = [g for g in gr.gaps if g.gap_class == GAP_CLASS_TERMINAL]
    assert len(terminal) == 1
    assert terminal[0].n_samples == 4
    assert terminal[0].disposition == GAP_DISPOSITION_UNRESOLVED
    assert math.isclose(terminal[0].thickness_tvd_m, 40.0)


def test_a_gap_caused_only_by_missing_depth_mapping_is_its_own_class(
        overburden_config):
    md = np.arange(1000.0, 1041.0, 10.0)
    tvd = md.copy()
    tvd[2] = np.nan
    valid = np.isfinite(tvd)
    frame = make_frame(md=md, tvd=tvd, tvdss=tvd - 25.0,
                       density=np.full(md.size, 2000.0), depth_valid_mask=valid)
    gr = prepared(frame, overburden_config)["gaps"]
    classes = [g.gap_class for g in gr.gaps]
    assert GAP_CLASS_UNMAPPED_DEPTH in classes
    unmapped = next(g for g in gr.gaps if g.gap_class == GAP_CLASS_UNMAPPED_DEPTH)
    assert unmapped.disposition == GAP_DISPOSITION_UNRESOLVED


def test_gap_thicknesses_are_reported_in_both_md_and_tvd(overburden_config):
    """A deviated well: the MD gap is longer than the TVD gap."""
    md = np.arange(1000.0, 1026.0, 5.0)
    tvd = md * 0.8
    rho = np.full(md.size, 2000.0)
    rho[2] = np.nan
    frame = make_frame(md=md, tvd=tvd, density=rho)
    gr = prepared(frame, overburden_config)["gaps"]
    gap = next(g for g in gr.gaps if g.gap_class == GAP_CLASS_SHORT_INTERNAL)
    assert math.isclose(gap.thickness_md_m, 10.0)
    assert math.isclose(gap.thickness_tvd_m, 8.0)
    assert gap.thickness_tvd_m < gap.thickness_md_m


def test_gap_classification_is_deterministic(overburden_config):
    frame = gap_frame([slice(5, 7), slice(20, 24)], n=41, step=1.0)
    first = prepared(frame, overburden_config)["gaps"]
    for _ in range(3):
        again = prepared(frame, overburden_config)["gaps"]
        assert [(g.gap_index, g.gap_class, g.disposition, g.n_samples)
                for g in again.gaps] == [
            (g.gap_index, g.gap_class, g.disposition, g.n_samples)
            for g in first.gaps]


def test_bridging_is_refused_outside_locked_survey_coverage(overburden_config):
    """A gap whose samples fall outside survey MD coverage is not bridgeable."""
    md = np.arange(1000.0, 1041.0, 10.0)
    rho = np.full(md.size, 2000.0)
    rho[2] = np.nan
    frame = make_frame(md=md, density=rho,
                       survey_md_min_m=1000.0, survey_md_max_m=1015.0)
    gr = prepared(frame, overburden_config)["gaps"]
    assert gr.n_bridged_gaps == 0


@pytest.mark.parametrize("bad", [
    True, np.bool_(False), "10", 10 + 0j, np.nan, np.inf, -np.inf, -1.0,
])
@pytest.mark.parametrize("operation", ["classify", "condition"])
def test_public_gap_threshold_overrides_fail_closed_on_invalid_values(
        bad, operation, overburden_config):
    frame = gap_frame([slice(10, 20)], rho=2000.0, n=41, step=1.0)
    masks = build_density_masks(frame, overburden_config)
    fn = classify_density_gaps if operation == "classify" else condition_density_gaps
    with pytest.raises(OverburdenInputError, match="threshold_tvd_m"):
        fn(frame, masks, overburden_config, threshold_tvd_m=bad)


def test_infinite_threshold_cannot_turn_a_long_gap_into_a_bridged_gap(
        overburden_config):
    frame = gap_frame([slice(10, 30)], rho=2000.0, n=41, step=1.0)
    masks = build_density_masks(frame, overburden_config)
    with pytest.raises(OverburdenInputError, match="finite"):
        condition_density_gaps(
            frame, masks, overburden_config, threshold_tvd_m=float("inf"))


# ---------------------------------------------------------------------------
# Gap-record and conditioning invariants
# ---------------------------------------------------------------------------

def test_only_a_short_internal_gap_may_carry_a_bridged_disposition():
    for bad_class in (GAP_CLASS_SHALLOW, GAP_CLASS_TERMINAL, GAP_CLASS_LONG_INTERNAL,
                      GAP_CLASS_UNMAPPED_DEPTH):
        with pytest.raises(OverburdenInputError, match="may be bridged"):
            DensityGapRecord(
                well_key="X", gap_index=0, gap_class=bad_class,
                disposition=GAP_DISPOSITION_BRIDGED, n_samples=1,
                start_index=1, end_index=1, md_start_m=0.0, md_end_m=2.0,
                tvd_start_m=0.0, tvd_end_m=2.0, thickness_md_m=2.0,
                thickness_tvd_m=2.0, threshold_tvd_m=10.0,
                bounding_density_above_kg_m3=2000.0,
                bounding_density_below_kg_m3=2000.0)


def test_a_bridged_gap_must_record_both_bracketing_densities():
    with pytest.raises(OverburdenInputError, match="bracketing measured densities"):
        DensityGapRecord(
            well_key="X", gap_index=0, gap_class=GAP_CLASS_SHORT_INTERNAL,
            disposition=GAP_DISPOSITION_BRIDGED, n_samples=1, start_index=1,
            end_index=1, md_start_m=0.0, md_end_m=2.0, tvd_start_m=0.0,
            tvd_end_m=2.0, thickness_md_m=2.0, thickness_tvd_m=2.0,
            threshold_tvd_m=10.0, bounding_density_above_kg_m3=None,
            bounding_density_below_kg_m3=2000.0)


def test_a_gap_classified_short_may_not_exceed_its_threshold():
    with pytest.raises(OverburdenInputError, match="classified short"):
        DensityGapRecord(
            well_key="X", gap_index=0, gap_class=GAP_CLASS_SHORT_INTERNAL,
            disposition=GAP_DISPOSITION_UNRESOLVED, n_samples=1, start_index=1,
            end_index=1, md_start_m=0.0, md_end_m=99.0, tvd_start_m=0.0,
            tvd_end_m=99.0, thickness_md_m=99.0, thickness_tvd_m=99.0,
            threshold_tvd_m=10.0, bounding_density_above_kg_m3=None,
            bounding_density_below_kg_m3=None)


def test_a_gap_classified_long_must_exceed_its_threshold():
    with pytest.raises(OverburdenInputError, match="classified long"):
        DensityGapRecord(
            well_key="X", gap_index=0, gap_class=GAP_CLASS_LONG_INTERNAL,
            disposition=GAP_DISPOSITION_UNRESOLVED, n_samples=1, start_index=1,
            end_index=1, md_start_m=0.0, md_end_m=2.0, tvd_start_m=0.0,
            tvd_end_m=2.0, thickness_md_m=2.0, thickness_tvd_m=2.0,
            threshold_tvd_m=10.0, bounding_density_above_kg_m3=None,
            bounding_density_below_kg_m3=None)


@pytest.mark.parametrize("gaps,samples", [(1, 0), (0, 3)])
def test_conditioning_rejects_a_gap_count_that_disagrees_with_its_sample_count(
        gaps, samples):
    mask = np.zeros(4, dtype=bool)
    mask[:samples] = True
    with pytest.raises(OverburdenInputError):
        GapConditioningResult(
            well_key="X", threshold_tvd_m=10.0, bridging_enabled=True,
            conditioned_density_kg_m3=readonly(np.full(4, 2000.0)),
            bridged_mask=readonly(mask), n_bridged_gaps=gaps,
            n_bridged_samples=samples, bridged_thickness_md_m=0.0,
            bridged_thickness_tvd_m=0.0, n_long_gaps=0, n_long_gap_samples=0,
            long_gap_thickness_md_m=0.0, long_gap_thickness_tvd_m=0.0)


def test_conditioning_rejects_more_gaps_than_samples():
    mask = np.zeros(4, dtype=bool)
    mask[0] = True
    with pytest.raises(OverburdenInputError, match="cannot contain only"):
        GapConditioningResult(
            well_key="X", threshold_tvd_m=10.0, bridging_enabled=True,
            conditioned_density_kg_m3=readonly(np.full(4, 2000.0)),
            bridged_mask=readonly(mask), n_bridged_gaps=3, n_bridged_samples=1,
            bridged_thickness_md_m=0.0, bridged_thickness_tvd_m=0.0,
            n_long_gaps=0, n_long_gap_samples=0, long_gap_thickness_md_m=0.0,
            long_gap_thickness_tvd_m=0.0)


def test_conditioning_rejects_a_writable_conditioned_array():
    with pytest.raises(OverburdenInputError, match="read-only"):
        GapConditioningResult(
            well_key="X", threshold_tvd_m=10.0, bridging_enabled=True,
            conditioned_density_kg_m3=np.full(4, 2000.0),
            bridged_mask=readonly(np.zeros(4, dtype=bool)), n_bridged_gaps=0,
            n_bridged_samples=0, bridged_thickness_md_m=0.0,
            bridged_thickness_tvd_m=0.0, n_long_gaps=0, n_long_gap_samples=0,
            long_gap_thickness_md_m=0.0, long_gap_thickness_tvd_m=0.0)


def test_conditioning_rejects_bridged_gaps_while_bridging_is_disabled():
    mask = np.zeros(4, dtype=bool)
    mask[1] = True
    with pytest.raises(OverburdenInputError, match="bridging is disabled"):
        GapConditioningResult(
            well_key="X", threshold_tvd_m=10.0, bridging_enabled=False,
            conditioned_density_kg_m3=readonly(np.full(4, 2000.0)),
            bridged_mask=readonly(mask), n_bridged_gaps=1, n_bridged_samples=1,
            bridged_thickness_md_m=1.0, bridged_thickness_tvd_m=1.0,
            n_long_gaps=0, n_long_gap_samples=0, long_gap_thickness_md_m=0.0,
            long_gap_thickness_tvd_m=0.0)


@pytest.mark.parametrize("bad", [True, "10", 10 + 0j, np.nan, np.inf, -np.inf, -1.0])
def test_conditioning_result_constructor_rejects_invalid_thresholds(
        bad, overburden_config):
    frame = constant_density_frame()
    good = prepared(frame, overburden_config)["gaps"]
    with pytest.raises(OverburdenInputError, match="threshold_tvd_m"):
        replace(good, threshold_tvd_m=bad)


@pytest.mark.parametrize("bad", [True, "10", 10 + 0j, np.nan, np.inf, -np.inf, -1.0])
def test_density_gap_record_constructor_rejects_invalid_thresholds(bad):
    with pytest.raises(OverburdenInputError, match="threshold_tvd_m"):
        DensityGapRecord(
            well_key="X", gap_index=0, gap_class=GAP_CLASS_LONG_INTERNAL,
            disposition=GAP_DISPOSITION_UNRESOLVED, n_samples=1,
            start_index=1, end_index=1, md_start_m=0.0, md_end_m=20.0,
            tvd_start_m=0.0, tvd_end_m=20.0, thickness_md_m=20.0,
            thickness_tvd_m=20.0, threshold_tvd_m=bad,
            bounding_density_above_kg_m3=2000.0,
            bounding_density_below_kg_m3=2000.0)


# ---------------------------------------------------------------------------
# QC statistics
# ---------------------------------------------------------------------------

def test_qc_statistics_report_the_correct_seabed_basis(overburden_config):
    frame = constant_density_frame()
    assert (prepared(frame, overburden_config)["stats"].seabed_basis
            == SEABED_BASIS_NOT_DETERMINABLE)
    md = np.arange(400.0, 601.0, 10.0)
    frame2 = make_frame(md=md, density=np.full(md.size, 2000.0))
    stats = prepared(frame2, overburden_config, seabed_mdrt_m=500.0,
                     seabed_tvd_m=500.0, seabed_tvdss_m=475.0)["stats"]
    assert stats.seabed_basis == SEABED_BASIS_LOCKED_MARKER
    assert stats.seabed_mdrt_m == 500.0


def test_qc_statistics_are_deterministic(overburden_config):
    frame = gap_frame([slice(5, 8), slice(20, 24)], n=41, step=1.0)
    first = prepared(frame, overburden_config)["stats"]
    for _ in range(3):
        again = prepared(frame, overburden_config)["stats"]
        assert again == first


def test_percentiles_are_measured_over_finite_samples_only(overburden_config):
    md = np.arange(1000.0, 1050.0, 10.0)
    frame = make_frame(md=md, density=np.array([2000.0, np.nan, 2200.0, 2400.0,
                                                2600.0]))
    stats = prepared(frame, overburden_config)["stats"]
    assert stats.rhob_min_kg_m3 == 2000.0
    assert stats.rhob_max_kg_m3 == 2600.0
    assert stats.rhob_median_kg_m3 == 2300.0


def test_high_scenario_percentile_uses_only_eligible_density_samples(
        overburden_config):
    md = np.arange(0.0, 100.0, 1.0)
    rho = np.full(md.size, 2400.0)
    rho[:50] = 1100.0
    rho[50:60] = np.nan
    frame = make_frame(md=md, density=rho)
    out = prepared(
        frame, overburden_config, seabed_mdrt_m=50.0,
        seabed_tvd_m=50.0, seabed_tvdss_m=25.0)
    stats = out["stats"]
    assert stats.n_finite == 90
    assert stats.n_eligible == 40
    assert stats.rhob_p05_kg_m3 == 1100.0
    assert stats.rhob_eligible_p05_kg_m3 == 2400.0


#### `tests/test_overburden.py` - analytical integration, sign convention, eligibility derivation, scenarios, sensitivity

In [ ]:
%%writefile tests/test_overburden.py
"""
Increment 7 - vertical-stress integration, sign convention, eligibility
derivation, scenarios, and gap-threshold sensitivity.

Every analytical expectation here is computed by hand from `rho * g * h`, not
from the implementation. A test that compared the code against itself would
prove only that the code is deterministic.
"""

from __future__ import annotations

from dataclasses import replace
import math

import numpy as np
import pytest

from p2mem.density_qc import build_density_masks, condition_density_gaps
from p2mem.overburden import (
    SIGN_CONVENTION_ID,
    build_gap_threshold_sensitivity,
    build_shallow_column_scenarios,
    build_stress_profile,
    derive_overburden_eligibility,
    integrate_vertical_stress,
    pa_to_mpa,
    uniform_column_stress_pa,
    verify_depth_sign_convention,
    water_column_stress_pa,
)
from p2mem.overburden_models import (
    REASON_INSUFFICIENT_ELIGIBLE_SAMPLES,
    REASON_INTERNAL_GAP_EXCEEDS_LIMIT,
    REASON_RHOB_NOT_AVAILABLE,
    REASON_SEABED_DATUM_UNRESOLVED,
    REASON_SHALLOW_COLUMN_UNRESOLVED,
    REASON_TERMINAL_COLUMN_UNRESOLVED,
    STATUS_ABSOLUTE,
    STATUS_NOT_ELIGIBLE,
    STATUS_PARTIAL_ONLY,
    STATUS_SENSITIVITY_ONLY,
    OverburdenInputError,
    StressPartition,
    VerticalStressProfile,
)

from helpers_inc7 import overburden_config, prepared  # noqa: E402
from synthetic_inc7 import (  # noqa: E402
    STANDARD_G, constant_density_frame, gap_frame, make_frame, two_layer_frame,
)


# ---------------------------------------------------------------------------
# Analytical integration
# ---------------------------------------------------------------------------

def test_constant_density_integration_is_exact():
    """A constant integrand: trapezoidal quadrature must be EXACT."""
    z = np.arange(0.0, 101.0, 1.0)
    rho = np.full(z.size, 2000.0)
    cum, n_zero = integrate_vertical_stress(rho, z, STANDARD_G)
    assert n_zero == 0
    assert cum[0] == 0.0
    assert math.isclose(float(cum[-1]), 2000.0 * STANDARD_G * 100.0, rel_tol=1e-12)
    # Every intermediate value must also be exact, not merely the total.
    for i in (1, 17, 50, 99):
        assert math.isclose(float(cum[i]), 2000.0 * STANDARD_G * float(z[i]),
                            rel_tol=1e-12)


def test_two_layer_integration_matches_hand_computed_sum():
    z = np.concatenate([np.arange(0.0, 51.0, 1.0), np.arange(51.0, 101.0, 1.0)])
    rho = np.where(z <= 50.0, 2000.0, 2500.0)
    cum, _ = integrate_vertical_stress(rho, z, STANDARD_G)
    # Layer 1 is exact. The 50->51 m interval straddles the contrast and the
    # trapezoid averages the two densities, which is the correct value for a
    # piecewise-linear density profile through those two nodes.
    upper = 2000.0 * STANDARD_G * 50.0
    straddle = 0.5 * (2000.0 + 2500.0) * STANDARD_G * 1.0
    lower = 2500.0 * STANDARD_G * 49.0
    assert math.isclose(float(cum[-1]), upper + straddle + lower, rel_tol=1e-12)


def test_water_column_plus_formation_sum_is_the_hand_computed_total():
    water = water_column_stress_pa(500.0, 1025.0, STANDARD_G)
    assert math.isclose(water, 1025.0 * STANDARD_G * 500.0, rel_tol=1e-15)
    z = np.arange(500.0, 601.0, 1.0)
    rho = np.full(z.size, 2200.0)
    cum, _ = integrate_vertical_stress(rho, z, STANDARD_G)
    total = water + float(cum[-1])
    assert math.isclose(
        total, 1025.0 * STANDARD_G * 500.0 + 2200.0 * STANDARD_G * 100.0,
        rel_tol=1e-12)


def test_integration_in_tvd_differs_from_integration_in_md_for_a_deviated_well():
    """The whole point of integrating in TVD, demonstrated numerically."""
    md = np.arange(0.0, 101.0, 1.0)
    tvd = md * 0.8            # 36.87 degrees from vertical, constant
    rho = np.full(md.size, 2400.0)
    cum_tvd, _ = integrate_vertical_stress(rho, tvd, STANDARD_G)
    cum_md, _ = integrate_vertical_stress(rho, md, STANDARD_G)
    assert math.isclose(float(cum_tvd[-1]), 2400.0 * STANDARD_G * 80.0, rel_tol=1e-12)
    assert math.isclose(float(cum_md[-1]), 2400.0 * STANDARD_G * 100.0, rel_tol=1e-12)
    # Integrating in MD would overstate the stress by exactly 1/0.8.
    assert float(cum_md[-1]) > float(cum_tvd[-1])
    assert math.isclose(float(cum_md[-1]) / float(cum_tvd[-1]), 1.25, rel_tol=1e-12)


def test_repeated_vertical_coordinate_contributes_exactly_zero_and_is_counted():
    z = np.array([0.0, 10.0, 10.0, 20.0])
    rho = np.array([2000.0, 2000.0, 2000.0, 2000.0])
    cum, n_zero = integrate_vertical_stress(rho, z, STANDARD_G)
    assert n_zero == 1
    assert float(cum[1]) == float(cum[2])           # the repeat adds nothing
    assert math.isclose(float(cum[-1]), 2000.0 * STANDARD_G * 20.0, rel_tol=1e-12)


def test_zero_thickness_whole_column_gives_exactly_zero_stress():
    z = np.zeros(5)
    rho = np.full(5, 2500.0)
    cum, n_zero = integrate_vertical_stress(rho, z, STANDARD_G)
    assert n_zero == 4
    assert float(cum[-1]) == 0.0


def test_decreasing_vertical_coordinate_is_rejected_not_sorted():
    z = np.array([0.0, 10.0, 5.0, 20.0])
    rho = np.full(4, 2000.0)
    with pytest.raises(OverburdenInputError, match="negative vertical increment"):
        integrate_vertical_stress(rho, z, STANDARD_G)


def test_fully_reversed_vertical_coordinate_is_rejected():
    z = np.array([100.0, 75.0, 50.0, 25.0])
    with pytest.raises(OverburdenInputError):
        integrate_vertical_stress(np.full(4, 2000.0), z, STANDARD_G)


@pytest.mark.parametrize("bad", [np.nan, np.inf, -np.inf])
def test_non_finite_density_reaches_no_integral(bad):
    rho = np.array([2000.0, bad, 2000.0])
    with pytest.raises(OverburdenInputError, match="non-finite density"):
        integrate_vertical_stress(rho, np.array([0.0, 1.0, 2.0]), STANDARD_G)


@pytest.mark.parametrize("bad", [np.nan, np.inf, -np.inf])
def test_non_finite_vertical_depth_reaches_no_integral(bad):
    z = np.array([0.0, bad, 2.0])
    with pytest.raises(OverburdenInputError, match="non-finite vertical-depth"):
        integrate_vertical_stress(np.full(3, 2000.0), z, STANDARD_G)


@pytest.mark.parametrize("value", [0.0, -1.0, -2500.0])
def test_non_positive_density_is_rejected(value):
    rho = np.array([2000.0, value, 2000.0])
    with pytest.raises(OverburdenInputError, match="non-positive density"):
        integrate_vertical_stress(rho, np.array([0.0, 1.0, 2.0]), STANDARD_G)


@pytest.mark.parametrize("arr", [
    np.array([True, False, True]),
    np.array(["a", "b", "c"]),
    np.array([b"a", b"b", b"c"]),
    np.array([1 + 2j, 2 + 0j, 3 + 0j]),
    np.array([object(), object(), object()], dtype=object),
])
def test_ambiguous_dtype_density_raises_typeerror(arr):
    with pytest.raises(TypeError):
        integrate_vertical_stress(arr, np.array([0.0, 1.0, 2.0]), STANDARD_G)


@pytest.mark.parametrize("arr", [
    np.array([True, False, True]),
    np.array(["0", "1", "2"]),
    np.array([0 + 0j, 1 + 0j, 2 + 0j]),
])
def test_ambiguous_dtype_depth_raises_typeerror(arr):
    with pytest.raises(TypeError):
        integrate_vertical_stress(np.full(3, 2000.0), arr, STANDARD_G)


def test_boolean_gravity_is_a_typeerror_not_a_silent_one():
    with pytest.raises(TypeError):
        integrate_vertical_stress(np.full(3, 2000.0), np.array([0.0, 1.0, 2.0]), True)


@pytest.mark.parametrize("gravity", ["9.80665", b"9.80665", 9.80665 + 0j, [9.80665]])
def test_non_real_scalar_gravity_is_rejected_without_coercion(gravity):
    with pytest.raises(TypeError, match="real numeric scalar"):
        integrate_vertical_stress(
            np.full(3, 2000.0), np.array([0.0, 1.0, 2.0]), gravity)


@pytest.mark.parametrize("g", [0.0, -9.80665, float("nan"), float("inf")])
def test_invalid_gravity_is_rejected(g):
    with pytest.raises(OverburdenInputError):
        integrate_vertical_stress(np.full(3, 2000.0), np.array([0.0, 1.0, 2.0]), g)


def test_length_mismatch_is_never_reconciled_by_truncation():
    with pytest.raises(OverburdenInputError, match="never truncates"):
        integrate_vertical_stress(np.full(5, 2000.0), np.arange(4.0), STANDARD_G)


def test_single_sample_cannot_form_a_trapezoid():
    with pytest.raises(OverburdenInputError, match="at least two samples"):
        integrate_vertical_stress(np.array([2000.0]), np.array([0.0]), STANDARD_G)


def test_integration_does_not_mutate_its_inputs():
    rho = np.array([2000.0, 2100.0, 2200.0])
    z = np.array([0.0, 1.0, 2.0])
    rho_before, z_before = rho.copy(), z.copy()
    integrate_vertical_stress(rho, z, STANDARD_G)
    assert np.array_equal(rho, rho_before)
    assert np.array_equal(z, z_before)


def test_returned_cumulative_array_is_read_only():
    cum, _ = integrate_vertical_stress(
        np.full(3, 2000.0), np.array([0.0, 1.0, 2.0]), STANDARD_G)
    assert not cum.flags.writeable
    with pytest.raises(ValueError):
        cum[0] = 1.0


def test_integration_is_deterministic_across_repeated_calls():
    rho = np.linspace(1900.0, 2600.0, 257)
    z = np.linspace(0.0, 256.0, 257)
    first, _ = integrate_vertical_stress(rho, z, STANDARD_G)
    for _ in range(3):
        again, _ = integrate_vertical_stress(rho, z, STANDARD_G)
        assert np.array_equal(np.asarray(first), np.asarray(again))


# ---------------------------------------------------------------------------
# Uniform-column helpers
# ---------------------------------------------------------------------------

def test_uniform_column_matches_rho_g_h():
    assert math.isclose(uniform_column_stress_pa(120.0, 1800.0, STANDARD_G),
                        1800.0 * STANDARD_G * 120.0, rel_tol=1e-15)


def test_zero_thickness_uniform_column_is_zero():
    assert uniform_column_stress_pa(0.0, 1025.0, STANDARD_G) == 0.0


def test_negative_thickness_column_is_rejected():
    with pytest.raises(OverburdenInputError):
        uniform_column_stress_pa(-1.0, 1025.0, STANDARD_G)


@pytest.mark.parametrize("args", [
    ("10", 2000.0, STANDARD_G),
    (10.0, "2000", STANDARD_G),
    (10.0, 2000.0, "9.80665"),
    (10.0 + 0j, 2000.0, STANDARD_G),
])
def test_uniform_column_rejects_coercible_non_real_scalars(args):
    with pytest.raises(TypeError, match="real numeric scalar"):
        uniform_column_stress_pa(*args)


def test_pa_to_mpa_is_an_exact_decimal_factor():
    assert pa_to_mpa(1.0e6) == 1.0
    assert pa_to_mpa(None) is None


# ---------------------------------------------------------------------------
# Sign convention
# ---------------------------------------------------------------------------

def test_sign_convention_is_verified_from_the_frames_own_arrays():
    frame = constant_density_frame(datum_elevation_m=25.0)
    ev = verify_depth_sign_convention(frame)
    assert ev["verified"] is True
    assert ev["sign_convention_id"] == SIGN_CONVENTION_ID
    assert ev["datum_elevation_m"] == 25.0
    assert ev["max_tvd_minus_tvdss_deviation_m"] < 1e-9
    assert ev["tvd_increases_downward"] is True


def test_sign_convention_rejects_a_tvdss_that_is_not_tvd_minus_datum():
    md = np.arange(1000.0, 1101.0, 10.0)
    # Deliberately wrong: TVDSS built with the OPPOSITE sign of the datum.
    frame = make_frame(md=md, tvd=md, tvdss=md + 25.0, datum_elevation_m=25.0,
                       density=np.full(md.size, 2000.0))
    with pytest.raises(OverburdenInputError, match="datum elevation"):
        verify_depth_sign_convention(frame)


def test_sign_convention_rejects_a_trajectory_that_rises_with_measured_depth():
    md = np.array([1000.0, 1010.0, 1020.0, 1030.0])
    tvd = np.array([1000.0, 1010.0, 1005.0, 1030.0])
    frame = make_frame(md=md, tvd=tvd, datum_elevation_m=25.0,
                       density=np.full(4, 2000.0))
    with pytest.raises(OverburdenInputError, match="TVD decreases"):
        verify_depth_sign_convention(frame)


def test_sign_convention_needs_at_least_two_mapped_samples():
    md = np.array([1000.0, 1010.0])
    frame = make_frame(md=md, density=np.full(2, 2000.0),
                       depth_valid_mask=np.array([True, False]))
    with pytest.raises(OverburdenInputError, match="fewer than two"):
        verify_depth_sign_convention(frame)


# ---------------------------------------------------------------------------
# Profiles built from real well frames
# ---------------------------------------------------------------------------

def test_profile_over_a_constant_density_frame_is_analytical(overburden_config):
    frame = constant_density_frame(rho=2000.0, n=11, step=10.0)
    p = prepared(frame, overburden_config, seabed_mdrt_m=None)
    profile = p["profile"]
    assert profile is not None
    assert profile.n_nodes == 11
    assert profile.n_intervals == 10
    assert profile.integration_coordinate == "tvdss_m"
    assert math.isclose(profile.total_measured_increment_pa,
                        2000.0 * overburden_config.gravity_m_s2 * 100.0,
                        rel_tol=1e-12)
    assert profile.total_bridged_increment_pa == 0.0
    assert profile.column_truncated_at_unresolved_gap is False


def test_profile_over_a_two_layer_frame_is_analytical(overburden_config):
    frame = two_layer_frame(rho_upper=2000.0, rho_lower=2500.0,
                            n_upper=6, n_lower=6, step=10.0)
    profile = prepared(frame, overburden_config, seabed_mdrt_m=None)["profile"]
    g = overburden_config.gravity_m_s2
    upper = 2000.0 * g * 50.0
    straddle = 0.5 * (2000.0 + 2500.0) * g * 10.0
    lower = 2500.0 * g * 40.0
    assert math.isclose(profile.total_measured_increment_pa,
                        upper + straddle + lower, rel_tol=1e-12)


def test_profile_is_zero_at_the_first_node_by_definition(overburden_config):
    profile = prepared(constant_density_frame(), overburden_config,
                       seabed_mdrt_m=None)["profile"]
    assert float(profile.cumulative_measured_increment_pa[0]) == 0.0


def test_profile_arrays_are_read_only(overburden_config):
    profile = prepared(constant_density_frame(), overburden_config,
                       seabed_mdrt_m=None)["profile"]
    for arr in (profile.md_m, profile.tvd_m, profile.tvdss_m,
                profile.density_kg_m3, profile.cumulative_measured_increment_pa):
        assert not arr.flags.writeable


def test_no_profile_when_only_one_valid_sample_exists(overburden_config):
    md = np.arange(1000.0, 1051.0, 10.0)
    rho = np.full(md.size, np.nan)
    rho[2] = 2200.0
    frame = make_frame(md=md, density=rho)
    assert prepared(frame, overburden_config, seabed_mdrt_m=None)["profile"] is None


def test_no_profile_when_density_is_entirely_invalid(overburden_config):
    md = np.arange(1000.0, 1051.0, 10.0)
    frame = make_frame(md=md, density=np.full(md.size, np.nan))
    assert prepared(frame, overburden_config, seabed_mdrt_m=None)["profile"] is None


def test_no_profile_when_the_well_has_no_density_curve(overburden_config):
    md = np.arange(1000.0, 1051.0, 10.0)
    frame = make_frame(md=md, curves={})
    out = prepared(frame, overburden_config, seabed_mdrt_m=None)
    assert out["profile"] is None
    assert REASON_RHOB_NOT_AVAILABLE in out["eligibility"].limiting_reasons
    assert out["eligibility"].status == STATUS_NOT_ELIGIBLE


def test_profile_never_extrapolates_beyond_the_eligible_interval(overburden_config):
    """The profile's span equals the eligible samples' own span, exactly."""
    md = np.arange(1000.0, 1101.0, 10.0)
    rho = np.full(md.size, 2000.0)
    rho[:3] = np.nan
    rho[-2:] = np.nan
    frame = make_frame(md=md, density=rho)
    profile = prepared(frame, overburden_config, seabed_mdrt_m=None)["profile"]
    assert profile.top_tvd_m == 1030.0
    assert profile.base_tvd_m == 1080.0
    assert profile.n_nodes == 6


def test_survey_coverage_truncates_the_eligible_interval(overburden_config):
    """Samples outside locked survey MD coverage are never integrated."""
    md = np.arange(1000.0, 1101.0, 10.0)
    frame = make_frame(md=md, density=np.full(md.size, 2000.0),
                       survey_md_min_m=1020.0, survey_md_max_m=1080.0)
    out = prepared(frame, overburden_config, seabed_mdrt_m=None)
    profile = out["profile"]
    assert profile.top_tvd_m == 1020.0
    assert profile.base_tvd_m == 1080.0
    assert out["stats"].n_samples_outside_survey_coverage == 4


# ---------------------------------------------------------------------------
# Gaps and truncation
# ---------------------------------------------------------------------------

def test_short_internal_gap_is_bridged_and_reported_separately(overburden_config):
    frame = gap_frame([slice(20, 23)], rho=2000.0, n=41, step=1.0)
    out = prepared(frame, overburden_config, seabed_mdrt_m=None)
    gr, profile = out["gaps"], out["profile"]
    assert gr.n_bridged_gaps == 1
    assert gr.n_bridged_samples == 3
    assert profile.total_bridged_increment_pa > 0.0
    # Constant density either side: bridging reproduces the same constant, so
    # the integral must still be the exact analytical value over 40 m.
    assert math.isclose(profile.total_measured_increment_pa,
                        2000.0 * overburden_config.gravity_m_s2 * 40.0,
                        rel_tol=1e-12)


def test_multiple_short_gaps_in_one_column_are_all_bridged(overburden_config):
    frame = gap_frame([slice(10, 12), slice(20, 23), slice(30, 31)],
                      rho=2000.0, n=41, step=1.0)
    gr = prepared(frame, overburden_config, seabed_mdrt_m=None)["gaps"]
    assert gr.n_bridged_gaps == 3
    assert gr.n_bridged_samples == 6
    assert gr.n_long_gaps == 0


def test_long_internal_gap_is_never_bridged_and_truncates_the_column(
        overburden_config):
    # 15 m of gap against a 10 m approved threshold.
    frame = gap_frame([slice(10, 25)], rho=2000.0, n=41, step=1.0)
    out = prepared(frame, overburden_config, seabed_mdrt_m=None)
    gr, profile, elig = out["gaps"], out["profile"], out["eligibility"]
    assert gr.n_bridged_gaps == 0
    assert gr.n_long_gaps == 1
    assert profile.column_truncated_at_unresolved_gap is True
    assert profile.n_eligible_samples_below_truncation == 16
    # Only the 10 m above the gap is integrated: no silent join across it.
    assert math.isclose(profile.total_measured_increment_pa,
                        2000.0 * overburden_config.gravity_m_s2 * 9.0,
                        rel_tol=1e-12)
    assert REASON_INTERNAL_GAP_EXCEEDS_LIMIT in elig.limiting_reasons
    assert elig.column_uninterrupted is False


def test_unresolved_short_gap_truncates_when_bridging_is_disabled(
        overburden_config):
    """Regression: no trapezoid may jump over an unresolved short gap."""
    config = replace(overburden_config, bridge_short_internal_gaps=False)
    frame = gap_frame([slice(10, 13)], rho=2000.0, n=21, step=1.0)
    out = prepared(frame, config, seabed_mdrt_m=None)
    gap = next(g for g in out["gaps"].gaps if g.gap_class == "short_internal_gap")
    profile = out["profile"]
    assert gap.disposition == "unresolved_not_bridged"
    assert out["gaps"].n_bridged_gaps == 0
    assert profile.column_truncated_at_unresolved_gap is True
    assert profile.n_nodes == 10
    assert profile.n_eligible_samples_below_truncation == 8
    assert math.isclose(
        profile.total_measured_increment_pa,
        2000.0 * config.gravity_m_s2 * 9.0,
        rel_tol=1e-12,
    )
    assert out["eligibility"].column_uninterrupted is False
    assert "internal_gap_unresolved" in out["eligibility"].limiting_reasons


def test_unmapped_internal_gap_also_truncates_the_profile(overburden_config):
    md = np.arange(1000.0, 1021.0, 1.0)
    depth_valid = np.ones(md.size, dtype=bool)
    depth_valid[10:13] = False
    tvd = md.copy()
    tvdss = tvd - 25.0
    tvd[10:13] = np.nan
    tvdss[10:13] = np.nan
    frame = make_frame(
        md=md, tvd=tvd, tvdss=tvdss, density=np.full(md.size, 2000.0),
        depth_valid_mask=depth_valid,
    )
    out = prepared(frame, overburden_config, seabed_mdrt_m=None)
    gap = next(g for g in out["gaps"].gaps
               if g.gap_class == "gap_caused_by_missing_depth_mapping")
    assert gap.disposition == "unresolved_not_bridged"
    assert out["profile"].column_truncated_at_unresolved_gap is True
    assert out["profile"].n_nodes == 10
    assert out["eligibility"].column_uninterrupted is False


def test_an_isolated_invalid_sample_is_a_one_sample_internal_gap(
        overburden_config):
    frame = gap_frame([slice(15, 16)], rho=2000.0, n=41, step=1.0)
    gr = prepared(frame, overburden_config, seabed_mdrt_m=None)["gaps"]
    bridged = [g for g in gr.gaps if g.disposition == "bridged_linear_in_tvd"]
    assert len(bridged) == 1
    assert bridged[0].n_samples == 1


def test_gap_conditioning_does_not_touch_the_original_density_array(
        overburden_config):
    frame = gap_frame([slice(20, 23)], rho=2000.0, n=41, step=1.0)
    original = np.array(frame.curve("RHOB_kg_m3").values, copy=True)
    out = prepared(frame, overburden_config, seabed_mdrt_m=None)
    after = np.asarray(frame.curve("RHOB_kg_m3").values)
    assert np.array_equal(np.isnan(original), np.isnan(after))
    assert np.array_equal(original[~np.isnan(original)], after[~np.isnan(after)])
    # The conditioned array is a DIFFERENT array with the gap filled.
    cond = np.asarray(out["gaps"].conditioned_density_kg_m3)
    assert np.isnan(original[21]) and np.isfinite(cond[21])


def test_bridged_values_are_linear_in_vertical_depth(overburden_config):
    md = np.arange(1000.0, 1011.0, 1.0)
    rho = np.full(md.size, np.nan)
    rho[0] = 2000.0
    rho[10] = 3000.0
    frame = make_frame(md=md, density=rho)
    out = prepared(frame, overburden_config, seabed_mdrt_m=None)
    cond = np.asarray(out["gaps"].conditioned_density_kg_m3)
    # Linear ramp from 2000 to 3000 over 10 m: node i is 2000 + 100*i.
    for i in range(11):
        assert math.isclose(float(cond[i]), 2000.0 + 100.0 * i, rel_tol=1e-12)


# ---------------------------------------------------------------------------
# Eligibility derivation
# ---------------------------------------------------------------------------

def test_absolute_status_when_density_reaches_the_seabed(overburden_config):
    """The only configuration in which an absolute curve is defensible."""
    md = np.arange(500.0, 1001.0, 10.0)
    frame = make_frame(md=md, density=np.full(md.size, 2000.0),
                       datum_elevation_m=25.0)
    out = prepared(frame, overburden_config, seabed_mdrt_m=500.0,
                   seabed_tvd_m=500.0, seabed_tvdss_m=475.0)
    elig = out["eligibility"]
    assert elig.status == STATUS_ABSOLUTE
    assert elig.limiting_reasons == ()
    assert elig.absolute_stress_supported is True


def test_shallow_gap_downgrades_absolute_to_sensitivity_only(overburden_config):
    md = np.arange(1000.0, 1501.0, 10.0)
    frame = make_frame(md=md, density=np.full(md.size, 2000.0))
    out = prepared(frame, overburden_config, seabed_mdrt_m=500.0,
                   seabed_tvd_m=500.0, seabed_tvdss_m=475.0)
    elig = out["eligibility"]
    assert elig.status == STATUS_SENSITIVITY_ONLY
    assert REASON_SHALLOW_COLUMN_UNRESOLVED in elig.limiting_reasons
    assert elig.absolute_stress_supported is False
    assert math.isclose(elig.shallow_unresolved_thickness_tvd_m, 500.0)


def test_unresolved_seabed_gives_partial_measured_increment_only(
        overburden_config):
    frame = constant_density_frame()
    elig = prepared(frame, overburden_config, seabed_mdrt_m=None)["eligibility"]
    assert elig.status == STATUS_PARTIAL_ONLY
    assert REASON_SEABED_DATUM_UNRESOLVED in elig.limiting_reasons
    assert elig.water_column_thickness_tvd_m is None


def test_status_is_derived_from_evidence_not_from_the_well_name(
        overburden_config):
    """The SAME arrays under two different well keys derive the same status."""
    md = np.arange(1000.0, 1101.0, 10.0)
    rho = np.full(md.size, 2000.0)
    a = prepared(make_frame(well_key="ALPHA_1", md=md, density=rho),
                 overburden_config, seabed_mdrt_m=None)["eligibility"]
    b = prepared(make_frame(well_key="OMEGA_9", md=md, density=rho),
                 overburden_config, seabed_mdrt_m=None)["eligibility"]
    assert a.status == b.status
    assert a.limiting_reasons == b.limiting_reasons


def test_limiting_reasons_are_sorted_deduplicated_enumerated_codes(
        overburden_config):
    frame = gap_frame([slice(10, 25)], rho=2000.0, n=41, step=1.0)
    elig = prepared(frame, overburden_config, seabed_mdrt_m=None)["eligibility"]
    assert list(elig.limiting_reasons) == sorted(set(elig.limiting_reasons))
    assert all(isinstance(r, str) and " " not in r for r in elig.limiting_reasons)


def test_terminal_gap_is_reported_but_does_not_block_an_absolute_status(
        overburden_config):
    md = np.arange(500.0, 1001.0, 10.0)
    rho = np.full(md.size, 2000.0)
    rho[-3:] = np.nan
    frame = make_frame(md=md, density=rho, datum_elevation_m=25.0)
    elig = prepared(frame, overburden_config, seabed_mdrt_m=500.0,
                    seabed_tvd_m=500.0, seabed_tvdss_m=475.0)["eligibility"]
    assert elig.status == STATUS_ABSOLUTE
    assert elig.terminal_unresolved_thickness_tvd_m == 30.0


def test_not_eligible_when_fewer_than_two_eligible_samples(overburden_config):
    md = np.arange(1000.0, 1051.0, 10.0)
    rho = np.full(md.size, np.nan)
    rho[1] = 2000.0
    elig = prepared(make_frame(md=md, density=rho), overburden_config,
                    seabed_mdrt_m=None)["eligibility"]
    assert elig.status == STATUS_NOT_ELIGIBLE
    assert REASON_INSUFFICIENT_ELIGIBLE_SAMPLES in elig.limiting_reasons


# ---------------------------------------------------------------------------
# Scenarios
# ---------------------------------------------------------------------------

def test_scenarios_are_published_only_for_sensitivity_only_wells(
        overburden_config):
    partial = prepared(constant_density_frame(), overburden_config,
                       seabed_mdrt_m=None)
    assert build_shallow_column_scenarios(
        partial["stats"], partial["eligibility"], partial["profile"],
        overburden_config) == ()


def test_scenarios_bracket_low_base_high_and_disclose_all_three_fractions(
        overburden_config):
    md = np.arange(1000.0, 1501.0, 10.0)
    frame = make_frame(md=md, density=np.full(md.size, 2000.0))
    out = prepared(frame, overburden_config, seabed_mdrt_m=500.0,
                   seabed_tvd_m=500.0, seabed_tvdss_m=475.0)
    scen = build_shallow_column_scenarios(
        out["stats"], out["eligibility"], out["profile"], overburden_config)
    by_name = {s.scenario_name: s for s in scen}
    assert {"low", "base", "high"} <= set(by_name)
    lo, base, hi = by_name["low"], by_name["base"], by_name["high"]
    # The bracket is ordered and the base is exactly the midpoint.
    assert (lo.assumed_shallow_density_kg_m3
            < base.assumed_shallow_density_kg_m3
            < hi.assumed_shallow_density_kg_m3)
    assert math.isclose(
        base.assumed_shallow_density_kg_m3,
        0.5 * (lo.assumed_shallow_density_kg_m3 + hi.assumed_shallow_density_kg_m3),
        rel_tol=1e-12)
    assert lo.partition.total_pa < base.partition.total_pa < hi.partition.total_pa
    for s in (lo, base, hi):
        assert 0.0 <= s.assumed_fraction_of_total <= 1.0
        assert 0.0 <= s.conditioned_fraction_of_total <= 1.0
        assert 0.0 <= s.measured_fraction_of_total <= 1.0
        assert math.isclose(
            s.assumed_fraction_of_total + s.conditioned_fraction_of_total
            + s.measured_fraction_of_total, 1.0,
            rel_tol=1e-9, abs_tol=1e-9)


def test_scenario_low_bound_is_the_configured_seawater_density(
        overburden_config):
    md = np.arange(1000.0, 1501.0, 10.0)
    frame = make_frame(md=md, density=np.full(md.size, 2000.0))
    out = prepared(frame, overburden_config, seabed_mdrt_m=500.0,
                   seabed_tvd_m=500.0, seabed_tvdss_m=475.0)
    scen = {s.scenario_name: s for s in build_shallow_column_scenarios(
        out["stats"], out["eligibility"], out["profile"], overburden_config)}
    assert (scen["low"].assumed_shallow_density_kg_m3
            == overburden_config.seawater_density_kg_m3)
    assert (scen["high"].assumed_shallow_density_kg_m3
            == out["stats"].rhob_eligible_p05_kg_m3)


def test_high_scenario_ignores_finite_density_below_the_eligible_domain(
        overburden_config):
    md = np.arange(0.0, 101.0, 1.0)
    rho = np.full(md.size, 2400.0)
    rho[:40] = 1100.0
    rho[40:50] = np.nan
    out = prepared(
        make_frame(md=md, density=rho), overburden_config,
        seabed_mdrt_m=40.0, seabed_tvd_m=40.0, seabed_tvdss_m=15.0)
    scenarios = {s.scenario_name: s for s in build_shallow_column_scenarios(
        out["stats"], out["eligibility"], out["profile"], overburden_config)}
    assert out["stats"].rhob_p05_kg_m3 == 1100.0
    assert out["stats"].rhob_eligible_p05_kg_m3 == 2400.0
    assert scenarios["high"].assumed_shallow_density_kg_m3 == 2400.0


def test_scenario_partition_components_sum_exactly_to_the_total(
        overburden_config):
    md = np.arange(1000.0, 1501.0, 10.0)
    frame = make_frame(md=md, density=np.full(md.size, 2000.0))
    out = prepared(frame, overburden_config, seabed_mdrt_m=500.0,
                   seabed_tvd_m=500.0, seabed_tvdss_m=475.0)
    for s in build_shallow_column_scenarios(
            out["stats"], out["eligibility"], out["profile"], overburden_config):
        p = s.partition
        assert math.isclose(
            p.water_column_pa + p.measured_formation_pa + p.bridged_gap_pa
            + p.unresolved_shallow_pa, p.total_pa, rel_tol=1e-12, abs_tol=1e-6)


def test_scenario_fraction_accounting_includes_a_nonzero_bridged_component(
        overburden_config):
    md = np.arange(0.0, 101.0, 1.0)
    rho = np.full(md.size, 2400.0)
    rho[:50] = np.nan
    rho[70:72] = np.nan
    frame = make_frame(md=md, density=rho)
    out = prepared(
        frame, overburden_config, seabed_mdrt_m=40.0,
        seabed_tvd_m=40.0, seabed_tvdss_m=15.0)
    scenarios = build_shallow_column_scenarios(
        out["stats"], out["eligibility"], out["profile"], overburden_config)
    assert scenarios
    assert out["profile"].total_bridged_increment_pa > 0.0
    for scenario in scenarios:
        total = scenario.partition.total_pa
        assert scenario.conditioned_fraction_of_total > 0.0
        assert math.isclose(
            scenario.conditioned_fraction_of_total,
            scenario.partition.bridged_gap_pa / total, rel_tol=1e-12)
        assert math.isclose(
            scenario.assumed_fraction_of_total
            + scenario.conditioned_fraction_of_total
            + scenario.measured_fraction_of_total,
            1.0, rel_tol=1e-12, abs_tol=1e-12)


def test_scenario_constructor_rejects_fraction_accounting_that_drops_bridging(
        overburden_config):
    md = np.arange(1000.0, 1501.0, 10.0)
    out = prepared(
        make_frame(md=md, density=np.full(md.size, 2000.0)),
        overburden_config, seabed_mdrt_m=500.0,
        seabed_tvd_m=500.0, seabed_tvdss_m=475.0)
    scenario = build_shallow_column_scenarios(
        out["stats"], out["eligibility"], out["profile"], overburden_config)[0]
    with pytest.raises(OverburdenInputError, match="fractions must sum to 1"):
        replace(scenario, measured_fraction_of_total=0.0)


def test_seawater_variants_move_only_the_water_component(overburden_config):
    md = np.arange(1000.0, 1501.0, 10.0)
    frame = make_frame(md=md, density=np.full(md.size, 2000.0))
    out = prepared(frame, overburden_config, seabed_mdrt_m=500.0,
                   seabed_tvd_m=500.0, seabed_tvdss_m=475.0)
    scen = {s.scenario_name: s for s in build_shallow_column_scenarios(
        out["stats"], out["eligibility"], out["profile"], overburden_config)}
    base, low, high = (scen["base"], scen["base_seawater_low"],
                       scen["base_seawater_high"])
    assert low.partition.water_column_pa < base.partition.water_column_pa
    assert high.partition.water_column_pa > base.partition.water_column_pa
    for other in (low, high):
        assert (other.partition.unresolved_shallow_pa
                == base.partition.unresolved_shallow_pa)
        assert (other.partition.measured_formation_pa
                == base.partition.measured_formation_pa)


def test_a_partition_with_an_unresolved_component_cannot_report_a_total():
    with pytest.raises(OverburdenInputError, match="unresolved component is not zero"):
        StressPartition(water_column_pa=None, measured_formation_pa=1.0,
                        bridged_gap_pa=0.0, unresolved_shallow_pa=None,
                        total_pa=1.0)


def test_a_partition_total_must_equal_the_sum_of_its_components():
    with pytest.raises(OverburdenInputError, match="components sum to"):
        StressPartition(water_column_pa=1.0, measured_formation_pa=1.0,
                        bridged_gap_pa=0.0, unresolved_shallow_pa=1.0,
                        total_pa=99.0)


def test_a_partition_rejects_a_negative_component():
    with pytest.raises(OverburdenInputError, match="must be >= 0"):
        StressPartition(water_column_pa=-1.0, measured_formation_pa=1.0,
                        bridged_gap_pa=0.0, unresolved_shallow_pa=None,
                        total_pa=None)


# ---------------------------------------------------------------------------
# Constructor invariants
# ---------------------------------------------------------------------------

def test_profile_rejects_a_decreasing_cumulative_series(overburden_config):
    from synthetic_inc7 import readonly
    arrs = dict(
        md_m=readonly(np.array([0.0, 1.0])), tvd_m=readonly(np.array([0.0, 1.0])),
        tvdss_m=readonly(np.array([0.0, 1.0])),
        density_kg_m3=readonly(np.array([2000.0, 2000.0])),
        bridged_mask=readonly(np.array([False, False])),
        cumulative_measured_increment_pa=readonly(np.array([0.0, -1.0])))
    with pytest.raises(OverburdenInputError, match="non-decreasing"):
        VerticalStressProfile(
            well_key="X", n_nodes=2, integration_coordinate="tvdss_m",
            gravity_m_s2=STANDARD_G, total_measured_increment_pa=-1.0,
            total_bridged_increment_pa=0.0, n_zero_thickness_intervals=0,
            n_intervals=1, top_tvd_m=0.0, base_tvd_m=1.0, top_tvdss_m=0.0,
            base_tvdss_m=1.0, column_truncated_at_unresolved_gap=False,
            n_eligible_samples_below_truncation=0, **arrs)


def test_profile_rejects_a_measured_depth_integration_coordinate():
    from synthetic_inc7 import readonly
    arrs = dict(
        md_m=readonly(np.array([0.0, 1.0])), tvd_m=readonly(np.array([0.0, 1.0])),
        tvdss_m=readonly(np.array([0.0, 1.0])),
        density_kg_m3=readonly(np.array([2000.0, 2000.0])),
        bridged_mask=readonly(np.array([False, False])),
        cumulative_measured_increment_pa=readonly(np.array([0.0, 1.0])))
    with pytest.raises(OverburdenInputError, match="integration_coordinate"):
        VerticalStressProfile(
            well_key="X", n_nodes=2, integration_coordinate="md_m",
            gravity_m_s2=STANDARD_G, total_measured_increment_pa=1.0,
            total_bridged_increment_pa=0.0, n_zero_thickness_intervals=0,
            n_intervals=1, top_tvd_m=0.0, base_tvd_m=1.0, top_tvdss_m=0.0,
            base_tvdss_m=1.0, column_truncated_at_unresolved_gap=False,
            n_eligible_samples_below_truncation=0, **arrs)


# ---------------------------------------------------------------------------
# Gap-threshold sensitivity
# ---------------------------------------------------------------------------

def test_gap_threshold_sensitivity_covers_every_configured_threshold(
        overburden_config):
    frame = gap_frame([slice(10, 15)], rho=2000.0, n=41, step=1.0)
    out = prepared(frame, overburden_config, seabed_mdrt_m=None)
    rows = build_gap_threshold_sensitivity(
        frame, out["masks"], out["stats"], overburden_config)
    assert [r.threshold_tvd_m for r in rows] == list(
        overburden_config.sensitivity_thresholds_tvd_m)
    assert sum(1 for r in rows if r.is_approved_threshold) == 1


def test_a_larger_threshold_bridges_at_least_as_many_gaps(overburden_config):
    frame = gap_frame([slice(10, 13), slice(20, 35)], rho=2000.0, n=61, step=1.0)
    out = prepared(frame, overburden_config, seabed_mdrt_m=None)
    rows = build_gap_threshold_sensitivity(
        frame, out["masks"], out["stats"], overburden_config)
    counts = [r.n_bridged_gaps for r in rows]
    assert counts == sorted(counts)


def test_zero_threshold_bridges_nothing(overburden_config):
    frame = gap_frame([slice(10, 13)], rho=2000.0, n=41, step=1.0)
    out = prepared(frame, overburden_config, seabed_mdrt_m=None)
    rows = build_gap_threshold_sensitivity(
        frame, out["masks"], out["stats"], overburden_config)
    zero = next(r for r in rows if r.threshold_tvd_m == 0.0)
    assert zero.n_bridged_gaps == 0
    assert zero.n_bridged_samples == 0


@pytest.mark.parametrize("bad", [True, "10", 10 + 0j, np.nan, np.inf, -np.inf, -1.0])
def test_gap_sensitivity_constructor_rejects_invalid_thresholds(
        bad, overburden_config):
    frame = gap_frame([slice(10, 13)], rho=2000.0, n=41, step=1.0)
    out = prepared(frame, overburden_config)
    good = build_gap_threshold_sensitivity(
        frame, out["masks"], out["stats"], overburden_config)[0]
    with pytest.raises(OverburdenInputError, match="threshold_tvd_m"):
        replace(good, threshold_tvd_m=bad)


#### `tests/test_overburden_policy.py` - closed schemas, authorization, transactional export, and the locked-engine faithfulness harness

In [ ]:
%%writefile tests/test_overburden_policy.py
"""
Increment 7 - output-policy engine, closed schemas, authorization, transactional
export, and the FAITHFULNESS harness against the locked Increment 6.1.7 engine.

The adversarial section deliberately mutates the records that are ACTUALLY
serialized. A schema test that validated a reconstruction of the payload would
prove nothing about what reaches disk.
"""

from __future__ import annotations

import copy
import csv
import json
import os
from pathlib import Path

import pytest

from p2mem.io import output_policy as locked
from p2mem.io.overburden_policy import (
    CATEGORY_ENUMERATED_CODE_LIST,
    OutputAuthorizationError,
    PolicyBundle,
    FieldPolicy,
    authorize_occurrence,
    bundle_from_locked_increment6,
    canonicalize_emitted_records,
    collect_string_fields,
    export_authorized_outputs,
    validate_artifact_schema,
    validate_emitted_records,
)
from p2mem.io.overburden_registry import (
    ASSURANCE_TIER_VALUE, AVAILABILITY, ELIGIBILITY, GAPS, ISSUES,
    OVERBURDEN_ARTIFACTS, OVERBURDEN_BUNDLE, OVERBURDEN_CSV_SCHEMAS,
    OVERBURDEN_MANIFEST_ARTIFACT, OVERBURDEN_STATEMENTS, PROFILE, QC, SCENARIOS,
    SENSITIVITY,
)

from helpers_inc7 import PROJECT_ROOT  # noqa: E402


# ---------------------------------------------------------------------------
# Structural guarantees of the bundle itself
# ---------------------------------------------------------------------------

def test_nine_declared_artifacts_with_one_json_manifest():
    assert len(OVERBURDEN_ARTIFACTS) == 9
    assert OVERBURDEN_MANIFEST_ARTIFACT in OVERBURDEN_ARTIFACTS
    assert sum(1 for a in OVERBURDEN_ARTIFACTS if a.endswith(".json")) == 1
    assert sum(1 for a in OVERBURDEN_ARTIFACTS if a.endswith(".csv")) == 8


def test_notebook_gate_uses_the_declared_scenario_basis_field():
    """Prevent a gate-only field-name drift from escaping notebook parity.

    ``%%writefile`` parity covers source files but not the later completion
    gate.  This test binds that gate's literal lookup to the real emitted CSV
    schema, so a typo cannot survive merely because the notebook is valid JSON.
    """
    expected = "assumed_density_basis"
    assert expected in OVERBURDEN_CSV_SCHEMAS[SCENARIOS].columns
    assert "assumption_basis" not in OVERBURDEN_CSV_SCHEMAS[SCENARIOS].columns

    path = PROJECT_ROOT / "07_Density_QC_and_Overburden_Stress_Framework.ipynb"
    notebook = json.loads(path.read_text(encoding="utf-8"))
    gate_cells = [
        "".join(cell.get("source", []))
        for cell in notebook["cells"]
        if (cell.get("cell_type") == "code"
            and not "".join(cell.get("source", [])).startswith("%%writefile ")
            and "INCREMENT 7.0.4 COMPLETION GATE" in
            "".join(cell.get("source", [])))
    ]
    assert len(gate_cells) == 1
    gate_source = gate_cells[0]
    assert f'row["{expected}"]' in gate_source
    assert 'row["assumption_basis"]' not in gate_source


def test_every_csv_string_field_is_classified_exactly_once():
    """The bundle constructor enforces this; assert it holds for the shipped one."""
    for artifact, schema in OVERBURDEN_CSV_SCHEMAS.items():
        classified = {f for (a, f) in OVERBURDEN_BUNDLE.field_policy if a == artifact}
        assert classified == set(schema.string_fields)


def test_a_bundle_with_an_unclassified_string_column_cannot_be_built():
    schemas = dict(OVERBURDEN_CSV_SCHEMAS)
    policies = [p for p in OVERBURDEN_BUNDLE.field_policy.values()
                if not (p.artifact == ISSUES and p.field == "severity")]
    with pytest.raises(ValueError, match="differ"):
        PolicyBundle(
            name="broken", field_policies=policies,
            statements=OVERBURDEN_BUNDLE.statements.values(),
            templates=OVERBURDEN_BUNDLE.templates.values(),
            labels=[lab for b in OVERBURDEN_BUNDLE.labels.values() for lab in b.values()],
            csv_schemas=schemas,
            manifest_artifact=OVERBURDEN_MANIFEST_ARTIFACT,
            json_object_keys=OVERBURDEN_BUNDLE.json_object_keys,
            json_list_item_kinds=OVERBURDEN_BUNDLE.json_list_item_kinds,
            json_boolean_paths=OVERBURDEN_BUNDLE.json_boolean_paths,
            json_integer_paths=OVERBURDEN_BUNDLE.json_integer_paths,
            json_number_paths=OVERBURDEN_BUNDLE.json_number_paths,
            json_nullable_paths=OVERBURDEN_BUNDLE.json_nullable_paths,
            approved_well_keys=OVERBURDEN_BUNDLE.approved_well_keys,
            field_types=OVERBURDEN_BUNDLE.field_types)


def test_no_csv_schema_declares_a_duplicate_or_untyped_column():
    for artifact, schema in OVERBURDEN_CSV_SCHEMAS.items():
        assert len(set(schema.columns)) == len(schema.columns), artifact
        typed = schema.integer_fields | schema.number_fields | schema.boolean_fields
        assert typed <= set(schema.columns), artifact


def test_an_enumerated_code_list_must_declare_its_vocabulary():
    with pytest.raises(ValueError, match="closed vocabulary"):
        FieldPolicy("a.csv", "f", CATEGORY_ENUMERATED_CODE_LIST)


def test_an_unknown_category_is_still_rejected():
    with pytest.raises(ValueError, match="unknown category"):
        FieldPolicy("a.csv", "f", "free_prose")


# ---------------------------------------------------------------------------
# The added enumerated_code_list category
# ---------------------------------------------------------------------------

def _occ(artifact, field, value):
    from p2mem.io.output_policy import FieldOccurrence
    return FieldOccurrence(artifact, field, value, "row[0]." + field)


@pytest.mark.parametrize("value", [
    "none",
    "seabed_datum_unresolved",
    "internal_gap_exceeds_limit;shallow_density_column_unresolved",
])
def test_a_sorted_list_of_known_codes_authorizes(value):
    ok, auth, reason = authorize_occurrence(
        OVERBURDEN_BUNDLE, _occ(ELIGIBILITY, "limiting_reasons", value))
    assert ok, reason
    assert auth[0] == "enumerated_code_list"


@pytest.mark.parametrize("value,fragment", [
    ("not_a_real_code", "closed vocabulary"),
    ("seabed_datum_unresolved;not_a_real_code", "closed vocabulary"),
    ("seabed_datum_unresolved;seabed_datum_unresolved", "repeat"),
    ("shallow_density_column_unresolved;internal_gap_exceeds_limit", "sorted"),
    ("seabed_datum_unresolved;", "empty element"),
    ("The interval is unresolved.", "closed vocabulary"),
])
def test_a_malformed_code_list_is_refused(value, fragment):
    ok, _, reason = authorize_occurrence(
        OVERBURDEN_BUNDLE, _occ(ELIGIBILITY, "limiting_reasons", value))
    assert not ok
    assert fragment in reason


def test_an_enumerated_code_list_counts_as_controlled():
    from p2mem.io.overburden_policy import CONTROLLED_CATEGORIES
    assert CATEGORY_ENUMERATED_CODE_LIST in CONTROLLED_CATEGORIES


# ---------------------------------------------------------------------------
# Faithfulness to the LOCKED Increment 6.1.7 engine
# ---------------------------------------------------------------------------

def _locked_increment6_payloads():
    out_dir = PROJECT_ROOT / "outputs" / "06_petrophysics_eligibility"
    payloads = {}
    for name in locked.OUTPUT_ARTIFACTS:
        path = out_dir / name
        if not path.exists():
            return None
        if name.endswith(".json"):
            payloads[name] = json.loads(path.read_text(encoding="utf-8"))
        else:
            with open(path, newline="", encoding="utf-8") as fh:
                payloads[name] = list(csv.DictReader(fh))
    return payloads


def test_generalized_engine_reproduces_the_locked_engine_on_real_increment6_records():
    """The faithfulness harness.

    Runs the Increment 7 engine over the LOCKED Increment 6 registries and the
    REAL packaged Increment 6 artifacts, and requires the same decision on
    every single occurrence. This is what makes "generalized, not
    reimplemented" a checkable claim rather than a comment.
    """
    payloads = _locked_increment6_payloads()
    if payloads is None:
        pytest.skip("packaged Increment 6 outputs are not present in this tree")
    wells = set(locked.APPROVED_WELL_KEYS)
    bundle = bundle_from_locked_increment6()

    mine = validate_emitted_records(
        bundle, payloads, well_keys=wells, serialization_stage="post")
    theirs = locked.validate_emitted_records(
        payloads, well_keys=wells, serialization_stage="post")

    for attr in ("n_string_field_occurrences", "n_controlled_occurrences",
                 "n_structural_occurrences", "n_unguaranteed_occurrences",
                 "n_unclassified_fields", "n_unauthorized_controlled_fields",
                 "n_field_kind_mismatches", "n_schema_violations",
                 "distinct_statements_used", "distinct_templates_used",
                 "distinct_labels_used"):
        assert getattr(mine, attr) == getattr(theirs, attr), attr
    assert mine.ok is theirs.ok is True
    assert mine.n_string_field_occurrences > 0
    assert (canonicalize_emitted_records(bundle, payloads, serialization_stage="post")
            == locked.canonicalize_emitted_records(payloads, serialization_stage="post"))


def test_generalized_engine_reproduces_the_locked_engines_rejections():
    """The same faithfulness claim, for the cases that must FAIL."""
    payloads = _locked_increment6_payloads()
    if payloads is None:
        pytest.skip("packaged Increment 6 outputs are not present in this tree")
    wells = set(locked.APPROVED_WELL_KEYS)
    bundle = bundle_from_locked_increment6()

    mutations = []
    p = copy.deepcopy(payloads)
    p["gr_endpoint_scenarios.csv"][0]["description"] = "An unauthorized sentence."
    mutations.append(p)
    p = copy.deepcopy(payloads)
    for row in p["gr_endpoint_scenarios.csv"]:
        row.pop("description")
    mutations.append(p)
    p = copy.deepcopy(payloads)
    for row in p["gr_endpoint_scenarios.csv"]:
        row["rogue_column"] = "x"
    mutations.append(p)
    p = copy.deepcopy(payloads)
    p.pop("gr_endpoint_scenarios.csv")
    mutations.append(p)

    for payload in mutations:
        mine = validate_emitted_records(
            bundle, payload, well_keys=wells, serialization_stage="post")
        theirs = locked.validate_emitted_records(
            payload, well_keys=wells, serialization_stage="post")
        assert mine.ok is False and theirs.ok is False
        assert len(mine.violations) == len(theirs.violations)


# ---------------------------------------------------------------------------
# Increment 7 payloads: adversarial schema and transaction probes
# ---------------------------------------------------------------------------

@pytest.fixture(scope="module")
def payloads_and_wells(tmp_path_factory):
    """The REAL Increment 7 payloads, built from synthetic well frames.

    Synthetic frames are used deliberately: these probes are about the export
    path, and must run in a tree that carries no approved project data.
    """
    import numpy as np
    from p2mem.density_qc import load_overburden_config
    from p2mem.io.overburden_workflow import build_overburden_payloads, run_overburden_workflow
    from synthetic_inc7 import make_frame

    config = load_overburden_config(str(PROJECT_ROOT / "config" / "overburden_stress.yml"))
    frames = {}
    for i, key in enumerate(sorted(OVERBURDEN_BUNDLE.approved_well_keys)):
        md = np.arange(1000.0 + 10.0 * i, 1201.0 + 10.0 * i, 10.0)
        rho = np.full(md.size, 2000.0 + 50.0 * i)
        if i == 1:
            rho[5:7] = np.nan          # a bridgeable short gap
        frames[key] = make_frame(well_key=key, md=md, density=rho)
    marker_table = tmp_path_factory.mktemp("marker_table") / "markers.csv"
    marker_table.write_text(
        "well_key,canonical_marker_name,MDRT_reconciled_m,TVD_survey_m,"
        "TVDSS_survey_corrected_m\n", encoding="utf-8")
    run = run_overburden_workflow(
        frames, config, marker_table_path=str(marker_table))
    return build_overburden_payloads(run), set(frames)


def test_the_real_payload_set_validates_clean(payloads_and_wells):
    payloads, wells = payloads_and_wells
    report = validate_emitted_records(
        OVERBURDEN_BUNDLE, payloads, well_keys=wells, serialization_stage="pre")
    assert report.ok, report.violations[:2]
    assert report.n_schema_violations == 0
    assert report.n_unclassified_fields == 0
    assert report.n_unauthorized_controlled_fields == 0
    assert report.n_string_field_occurrences > 0
    assert set(report.artifacts_inspected) == set(OVERBURDEN_ARTIFACTS)


def test_export_writes_every_declared_artifact(tmp_path, payloads_and_wells):
    payloads, wells = payloads_and_wells
    before, after = export_authorized_outputs(
        OVERBURDEN_BUNDLE, tmp_path / "out", payloads, well_keys=wells)
    assert before.ok and after.ok
    assert sorted(p.name for p in (tmp_path / "out").iterdir()) == list(
        OVERBURDEN_ARTIFACTS)


def test_default_json_serializer_emits_explicit_utf8_lf_bytes(
        tmp_path, payloads_and_wells):
    payloads, wells = payloads_and_wells
    out = tmp_path / "out"
    export_authorized_outputs(OVERBURDEN_BUNDLE, out, payloads, well_keys=wells)
    actual = (out / OVERBURDEN_MANIFEST_ARTIFACT).read_bytes()
    expected = (json.dumps(
        payloads[OVERBURDEN_MANIFEST_ARTIFACT], indent=2, sort_keys=True) + "\n"
    ).encode("utf-8")
    assert actual == expected
    assert actual.endswith(b"\n")
    assert b"\r\n" not in actual


def test_default_json_serializer_does_not_use_platform_text_newlines(
        tmp_path, payloads_and_wells, monkeypatch):
    payloads, wells = payloads_and_wells

    def forbidden_write_text(*args, **kwargs):
        raise AssertionError("default JSON serialization must use explicit bytes")

    monkeypatch.setattr(Path, "write_text", forbidden_write_text)
    before, after = export_authorized_outputs(
        OVERBURDEN_BUNDLE, tmp_path / "out", payloads, well_keys=wells)
    assert before.ok and after.ok


def test_export_round_trip_is_canonically_identical(tmp_path, payloads_and_wells):
    payloads, wells = payloads_and_wells
    out = tmp_path / "out"
    export_authorized_outputs(OVERBURDEN_BUNDLE, out, payloads, well_keys=wells)
    written = {}
    for name in OVERBURDEN_ARTIFACTS:
        path = out / name
        if name.endswith(".json"):
            written[name] = json.loads(path.read_text(encoding="utf-8"))
        else:
            with open(path, newline="", encoding="utf-8") as fh:
                written[name] = list(csv.DictReader(fh))
    assert (canonicalize_emitted_records(OVERBURDEN_BUNDLE, payloads,
                                         serialization_stage="pre")
            == canonicalize_emitted_records(OVERBURDEN_BUNDLE, written,
                                            serialization_stage="post"))


def _mutations(payloads):
    out = []

    def add(fn):
        p = copy.deepcopy(payloads)
        fn(p)
        out.append(p)

    # An unauthorized sentence in a registered-statement column.
    add(lambda p: p[ELIGIBILITY][0].__setitem__(
        "limitations", "Everything is fine here."))
    # A near-miss on a registered statement: one character changed.
    add(lambda p: p[ELIGIBILITY][0].__setitem__(
        "limitations", OVERBURDEN_STATEMENTS["eligibility_limitations"].text + " "))
    # A status value that is not an approved label.
    add(lambda p: p[ELIGIBILITY][0].__setitem__("overburden_status", "looks_fine"))
    # A limiting-reason list that is not sorted.
    add(lambda p: p[ELIGIBILITY][0].__setitem__(
        "limiting_reasons", "terminal_density_column_unresolved;seabed_datum_unresolved"))
    # A deleted column.
    add(lambda p: [row.pop("limitations") for row in p[ELIGIBILITY]])
    # An added column.
    add(lambda p: [row.__setitem__("rogue", "x") for row in p[ELIGIBILITY]])
    # A renamed column.
    add(lambda p: [row.__setitem__("well_key_renamed", row.pop("well_key"))
                   for row in p[ELIGIBILITY]])
    # Reordered columns: order is contractual for a CSV.
    add(lambda p: p.__setitem__(
        ELIGIBILITY, [dict(reversed(list(row.items()))) for row in p[ELIGIBILITY]]))
    # Wrong types where a number is declared.
    for bad in ("2000", None, True):
        add(lambda p, b=bad: p[ELIGIBILITY][0].__setitem__("gravity_m_s2", b))
    # A non-finite number.
    add(lambda p: p[ELIGIBILITY][0].__setitem__("gravity_m_s2", float("nan")))
    # A float where an integer count is declared.
    add(lambda p: p[ELIGIBILITY][0].__setitem__("n_eligible_samples", 3.0))
    # A string where a boolean is declared.
    add(lambda p: p[ELIGIBILITY][0].__setitem__("seabed_resolved", "True"))
    # An unknown well identifier.
    add(lambda p: p[ELIGIBILITY][0].__setitem__("well_key", "Rogue_Well_9"))
    # An unknown artifact in the payload set.
    add(lambda p: p.__setitem__("rogue_artifact.csv", []))
    # A missing declared artifact.
    add(lambda p: p.pop(SCENARIOS))
    # An unknown key inside the manifest.
    add(lambda p: p[OVERBURDEN_MANIFEST_ARTIFACT].__setitem__("rogue_key", 1))
    # A missing key inside the manifest.
    add(lambda p: p[OVERBURDEN_MANIFEST_ARTIFACT].pop("gravity_m_s2"))
    # A manifest well entry for a well that was never evaluated.
    add(lambda p: p[OVERBURDEN_MANIFEST_ARTIFACT]["wells"].__setitem__(
        "Rogue_Well_9", {}))
    return out


def test_every_adversarial_mutation_fails_closed_and_writes_nothing(
        tmp_path, payloads_and_wells):
    payloads, wells = payloads_and_wells
    for i, mutated in enumerate(_mutations(payloads)):
        target = tmp_path / f"case_{i}"
        with pytest.raises(OutputAuthorizationError):
            export_authorized_outputs(
                OVERBURDEN_BUNDLE, target, mutated, well_keys=wells)
        assert not target.exists() or not list(target.iterdir())


def test_a_rejected_export_leaves_prior_official_outputs_untouched(
        tmp_path, payloads_and_wells):
    payloads, wells = payloads_and_wells
    out = tmp_path / "out"
    export_authorized_outputs(OVERBURDEN_BUNDLE, out, payloads, well_keys=wells)
    before = {p.name: p.read_bytes() for p in out.iterdir()}

    bad = copy.deepcopy(payloads)
    bad[ELIGIBILITY][0]["limitations"] = "Unauthorized replacement text."
    with pytest.raises(OutputAuthorizationError):
        export_authorized_outputs(OVERBURDEN_BUNDLE, out, bad, well_keys=wells)
    after = {p.name: p.read_bytes() for p in out.iterdir()}
    assert after == before


def test_a_stale_undeclared_artifact_in_the_destination_is_refused(
        tmp_path, payloads_and_wells):
    payloads, wells = payloads_and_wells
    out = tmp_path / "out"
    out.mkdir()
    (out / "stale_result.csv").write_text("stale\n", encoding="utf-8")
    with pytest.raises(OutputAuthorizationError, match="stale"):
        export_authorized_outputs(OVERBURDEN_BUNDLE, out, payloads, well_keys=wells)
    assert (out / "stale_result.csv").read_text(encoding="utf-8") == "stale\n"


def test_a_figures_directory_is_not_treated_as_stale(tmp_path, payloads_and_wells):
    payloads, wells = payloads_and_wells
    out = tmp_path / "out"
    (out / "figures").mkdir(parents=True)
    export_authorized_outputs(OVERBURDEN_BUNDLE, out, payloads, well_keys=wells)
    assert (out / "figures").is_dir()


def test_a_serializer_that_mutates_an_authorized_value_is_caught(
        tmp_path, payloads_and_wells):
    """The post-serialization comparison, probed with an AUTHORIZED substitute.

    The replacement value is itself a registered statement, so it passes
    authorization. Only the canonical record comparison can catch it.
    """
    payloads, wells = payloads_and_wells
    out = tmp_path / "out"
    export_authorized_outputs(OVERBURDEN_BUNDLE, out, payloads, well_keys=wells)
    baseline = {p.name: p.read_bytes() for p in out.iterdir()}

    replacement = OVERBURDEN_STATEMENTS["qc_limitations"].text

    def mutating_writer(target, payload):
        value = copy.deepcopy(payload)
        if target.name == ELIGIBILITY:
            value[0]["limitations"] = replacement
        if target.suffix == ".json":
            target.write_text(json.dumps(value, indent=2, sort_keys=True) + "\n",
                              encoding="utf-8")
        else:
            schema = OVERBURDEN_CSV_SCHEMAS[target.name]
            with open(target, "w", newline="", encoding="utf-8") as fh:
                w = csv.DictWriter(fh, fieldnames=list(schema.columns))
                w.writeheader()
                w.writerows(value)

    with pytest.raises(OutputAuthorizationError):
        export_authorized_outputs(
            OVERBURDEN_BUNDLE, out, payloads, well_keys=wells, writer=mutating_writer)
    assert {p.name: p.read_bytes() for p in out.iterdir()} == baseline


def test_a_serializer_that_reorders_rows_is_caught(tmp_path, payloads_and_wells):
    payloads, wells = payloads_and_wells

    def reordering_writer(target, payload):
        value = copy.deepcopy(payload)
        if target.name == ELIGIBILITY:
            value = list(reversed(value))
        if target.suffix == ".json":
            target.write_text(json.dumps(value, indent=2, sort_keys=True) + "\n",
                              encoding="utf-8")
        else:
            schema = OVERBURDEN_CSV_SCHEMAS[target.name]
            with open(target, "w", newline="", encoding="utf-8") as fh:
                w = csv.DictWriter(fh, fieldnames=list(schema.columns))
                w.writeheader()
                w.writerows(value)

    out = tmp_path / "out"
    with pytest.raises(OutputAuthorizationError, match="canonical record mismatch"):
        export_authorized_outputs(
            OVERBURDEN_BUNDLE, out, payloads, well_keys=wells, writer=reordering_writer)
    assert not out.exists() or not list(out.iterdir())


def test_a_failing_serializer_leaves_no_residue(tmp_path, payloads_and_wells):
    payloads, wells = payloads_and_wells
    out = tmp_path / "out"
    out.mkdir()
    sentinel = out / ELIGIBILITY
    sentinel.write_bytes(b"approved baseline bytes\n")

    def failing_writer(_target, _payload):
        raise RuntimeError("simulated serializer failure")

    with pytest.raises(OutputAuthorizationError, match="serializer failed"):
        export_authorized_outputs(
            OVERBURDEN_BUNDLE, out, payloads, well_keys=wells, writer=failing_writer)
    assert sentinel.read_bytes() == b"approved baseline bytes\n"
    assert {p.name for p in out.iterdir()} == {ELIGIBILITY}
    # No staging or backup residue anywhere beside the destination.
    assert not [p for p in out.parent.iterdir()
                if p.name.startswith(".") and p.is_dir()]


def test_a_writer_that_omits_an_artifact_is_caught(tmp_path, payloads_and_wells):
    payloads, wells = payloads_and_wells

    def skipping_writer(target, payload):
        if target.name == SCENARIOS:
            return
        if target.suffix == ".json":
            target.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n",
                              encoding="utf-8")
        else:
            schema = OVERBURDEN_CSV_SCHEMAS[target.name]
            with open(target, "w", newline="", encoding="utf-8") as fh:
                w = csv.DictWriter(fh, fieldnames=list(schema.columns))
                w.writeheader()
                w.writerows(payload)

    out = tmp_path / "out"
    with pytest.raises(OutputAuthorizationError, match="wrong artifact inventory"):
        export_authorized_outputs(
            OVERBURDEN_BUNDLE, out, payloads, well_keys=wells, writer=skipping_writer)


def test_export_is_byte_deterministic_across_two_destinations(
        tmp_path, payloads_and_wells):
    payloads, wells = payloads_and_wells
    a, b = tmp_path / "a", tmp_path / "b"
    export_authorized_outputs(OVERBURDEN_BUNDLE, a, payloads, well_keys=wells)
    export_authorized_outputs(OVERBURDEN_BUNDLE, b, payloads, well_keys=wells)
    for name in OVERBURDEN_ARTIFACTS:
        assert (a / name).read_bytes() == (b / name).read_bytes(), name


def test_no_absolute_path_leaks_into_any_emitted_artifact(payloads_and_wells):
    payloads, _ = payloads_and_wells
    blob = json.dumps(payloads, sort_keys=True, default=str)
    assert ":\\" not in blob
    assert "/home/" not in blob
    assert "/content/" not in blob
    assert str(PROJECT_ROOT) not in blob


def test_the_serialization_stage_is_pinned_not_auto(payloads_and_wells):
    """`auto` must never be relied on in the export gate.

    Probed behaviourally: a payload whose every numeric value happens to be a
    string would be mis-detected as post-serialization by `auto`, and the
    explicit `pre` stage must reject it.
    """
    payloads, wells = payloads_and_wells
    schema = OVERBURDEN_CSV_SCHEMAS[ELIGIBILITY]
    typed = schema.integer_fields | schema.number_fields | schema.boolean_fields
    stringified = copy.deepcopy(payloads)
    for row in stringified[ELIGIBILITY]:
        for field in typed:
            row[field] = "" if row[field] is None else str(row[field])

    # `auto` inspects the values and concludes this artifact is already
    # serialized, so it accepts it. That is exactly why a production gate may
    # not run on `auto`: post-serialization bytes would sail through a check
    # that is supposed to be validating typed, pre-serialization records.
    auto = validate_emitted_records(
        OVERBURDEN_BUNDLE, stringified, well_keys=wells, serialization_stage="auto")
    pinned = validate_emitted_records(
        OVERBURDEN_BUNDLE, stringified, well_keys=wells, serialization_stage="pre")
    assert auto.n_schema_violations == 0
    assert pinned.n_schema_violations >= len(typed)
    assert pinned.n_schema_violations > auto.n_schema_violations
    with pytest.raises(OutputAuthorizationError):
        import tempfile
        with tempfile.TemporaryDirectory() as td:
            export_authorized_outputs(
                OVERBURDEN_BUNDLE, os.path.join(td, "out"), stringified,
                well_keys=wells)


# ---------------------------------------------------------------------------
# Collection and schema helpers
# ---------------------------------------------------------------------------

def test_collection_is_schema_driven_not_value_driven(payloads_and_wells):
    """A numeric-looking string in a prose column is still prose."""
    payloads, wells = payloads_and_wells
    mutated = copy.deepcopy(payloads)
    mutated[ELIGIBILITY][0]["limitations"] = "12345"
    occurrences = collect_string_fields(
        OVERBURDEN_BUNDLE, ELIGIBILITY, mutated[ELIGIBILITY], wells)
    assert any(o.field == "limitations" and o.value == "12345" for o in occurrences)
    ok, _, _ = authorize_occurrence(
        OVERBURDEN_BUNDLE, _occ(ELIGIBILITY, "limitations", "12345"))
    assert not ok


def test_an_empty_string_in_a_required_column_is_refused():
    ok, _, reason = authorize_occurrence(
        OVERBURDEN_BUNDLE, _occ(ELIGIBILITY, "limitations", ""))
    assert not ok and "empty string" in reason


def test_an_undeclared_artifact_has_no_schema():
    violations = validate_artifact_schema(OVERBURDEN_BUNDLE, "nope.csv", [])
    assert violations and "no declared schema" in violations[0]["reason"]


def test_the_scenario_template_refuses_a_prose_substitution():
    tpl = OVERBURDEN_BUNDLE.templates["scenario_basis"]
    good = {"assumed_density_kg_m3": "1500.0000",
            "unresolved_thickness_m": "500.0000",
            "assumed_fraction_pct": "62.50",
            "conditioned_fraction_pct": "2.50",
            "measured_fraction_pct": "35.00"}
    ok, auth, _ = authorize_occurrence(
        OVERBURDEN_BUNDLE, _occ(SCENARIOS, "scenario_basis", tpl.render(good)))
    assert ok and auth[0] == "template"
    bad = dict(good, assumed_fraction_pct="most of it")
    ok, _, reason = authorize_occurrence(
        OVERBURDEN_BUNDLE, _occ(SCENARIOS, "scenario_basis", tpl.render(bad)))
    assert not ok and "declared type" in reason


def test_the_mnemonic_template_accepts_only_an_integer_ordinal():
    tpl = OVERBURDEN_BUNDLE.templates["empty_mnemonic_ordinal"]
    ok, _, _ = authorize_occurrence(
        OVERBURDEN_BUNDLE, _occ(AVAILABILITY, "raw_mnemonic",
                                tpl.render({"ordinal": "6"})))
    assert ok
    ok, _, _ = authorize_occurrence(
        OVERBURDEN_BUNDLE, _occ(AVAILABILITY, "raw_mnemonic",
                                tpl.render({"ordinal": "six"})))
    assert not ok


def test_every_registered_statement_is_actually_reachable_from_a_field():
    """No dead prose in the registry: each statement is declared on a field."""
    declared = set()
    for policy in OVERBURDEN_BUNDLE.field_policy.values():
        declared.update(policy.statement_ids)
    assert declared == set(OVERBURDEN_STATEMENTS)


def test_every_approved_label_belongs_to_a_declared_field_kind():
    kinds = {p.field_kind for p in OVERBURDEN_BUNDLE.field_policy.values()
             if p.category == "typed_label"}
    assert set(OVERBURDEN_BUNDLE.labels) == kinds


def test_a_label_approved_under_one_field_kind_authorizes_nothing_under_another():
    ok, _, reason = authorize_occurrence(
        OVERBURDEN_BUNDLE, _occ(ELIGIBILITY, "overburden_status",
                                "bridged_linear_in_tvd"))
    assert not ok
    assert "approved only as" in reason


def test_the_assurance_tier_matches_the_locked_package_declaration():
    import p2mem
    assert ASSURANCE_TIER_VALUE == p2mem.ASSURANCE_TIER


#### `tests/test_overburden_inventory.py` - export builders, manifest, issue derivation, end-to-end determinism

In [ ]:
%%writefile tests/test_overburden_inventory.py
"""
Increment 7 - export builders, manifest construction, issue derivation, and
end-to-end determinism of the workflow.

All fixtures here are synthetic. The tests assert on the records that the
export path actually serializes.
"""

from __future__ import annotations

import copy
import json
import math

import numpy as np
import pytest

from p2mem.density_qc import load_overburden_config
from p2mem.io.overburden_inventory import (
    build_density_availability_rows, build_density_gap_rows, build_density_qc_rows,
    build_gap_threshold_sensitivity_rows, build_overburden_eligibility_rows,
    build_overburden_issue_rows, build_shallow_column_scenario_rows,
    build_vertical_stress_profile_rows, derive_overburden_issues,
)
from p2mem.io.overburden_policy import (
    export_authorized_outputs, validate_emitted_records,
)
from p2mem.io.overburden_registry import (
    AVAILABILITY, ELIGIBILITY, GAPS, ISSUE_CODES, ISSUES, NOT_IMPLEMENTED_TOKENS,
    OVERBURDEN_ARTIFACTS, OVERBURDEN_BUNDLE, OVERBURDEN_CSV_SCHEMAS,
    OVERBURDEN_MANIFEST_ARTIFACT, PROFILE, QC, SCENARIOS, SENSITIVITY,
)
from p2mem.io.overburden_workflow import (
    SeabedMarkerError, build_overburden_payloads, read_locked_seabed_markers,
    run_overburden_workflow,
)
from p2mem.overburden_models import (
    STATUS_ABSOLUTE, STATUS_PARTIAL_ONLY, STATUS_SENSITIVITY_ONLY,
)

from helpers_inc7 import CONFIG_PATH, overburden_config  # noqa: E402
from synthetic_inc7 import make_frame  # noqa: E402


# ---------------------------------------------------------------------------
# A small synthetic four-well set exercising three different statuses
# ---------------------------------------------------------------------------

@pytest.fixture(scope="module")
def synthetic_project(tmp_path_factory):
    """Four synthetic wells and a synthetic locked-marker table.

    Two wells carry a seabed marker (one reaching it, one far below it) and
    two do not, so the derived-status ladder is exercised end to end without
    touching any approved project data.
    """
    config = load_overburden_config(CONFIG_PATH)
    root = tmp_path_factory.mktemp("synthetic_project")
    markers = root / "markers.csv"
    markers.write_text(
        "well_key,canonical_marker_name,MDRT_reconciled_m,TVD_survey_m,"
        "TVDSS_survey_corrected_m\n"
        "Boreas_1,Sea Bed,500.0,500.0,475.0\n"
        "Poseidon_2,Sea Bed,500.0,500.0,475.0\n",
        encoding="utf-8")

    frames = {}
    # Reaches its own seabed: the only configuration supporting an absolute curve.
    md = np.arange(500.0, 1001.0, 10.0)
    frames["Boreas_1"] = make_frame(
        well_key="Boreas_1", md=md, density=np.full(md.size, 2000.0),
        datum_elevation_m=25.0)
    # Seabed known, log starts far below it: sensitivity-only.
    md = np.arange(2000.0, 2501.0, 10.0)
    frames["Poseidon_2"] = make_frame(
        well_key="Poseidon_2", md=md, density=np.full(md.size, 2300.0),
        datum_elevation_m=25.0)
    # No seabed marker: partial measured increment only.
    md = np.arange(1500.0, 2001.0, 10.0)
    rho = np.full(md.size, 2400.0)
    rho[10:12] = np.nan                     # a bridgeable short gap
    frames["Poseidon_North_1"] = make_frame(
        well_key="Poseidon_North_1", md=md, density=rho, datum_elevation_m=22.0)
    # No seabed marker, and a long internal gap.
    md = np.arange(1500.0, 2001.0, 10.0)
    rho = np.full(md.size, 2500.0)
    rho[20:25] = np.nan                     # 60 m TVD: far beyond the threshold
    frames["Proteus_1ST2"] = make_frame(
        well_key="Proteus_1ST2", md=md, density=rho, datum_elevation_m=21.8)

    run = run_overburden_workflow(frames, config, marker_table_path=str(markers))
    payloads = build_overburden_payloads(run)
    return {"config": config, "frames": frames, "run": run, "payloads": payloads,
            "root": root, "markers": str(markers)}


def test_the_synthetic_set_exercises_three_derived_statuses(synthetic_project):
    statuses = {wk: e.status
                for wk, e in synthetic_project["run"].eligibility.items()}
    assert statuses["Boreas_1"] == STATUS_ABSOLUTE
    assert statuses["Poseidon_2"] == STATUS_SENSITIVITY_ONLY
    assert statuses["Poseidon_North_1"] == STATUS_PARTIAL_ONLY
    assert statuses["Proteus_1ST2"] == STATUS_PARTIAL_ONLY


# ---------------------------------------------------------------------------
# Seabed marker reading
# ---------------------------------------------------------------------------

def test_a_missing_locked_marker_table_fails_closed(tmp_path):
    with pytest.raises(SeabedMarkerError, match="missing or not a file"):
        read_locked_seabed_markers(str(tmp_path / "absent.csv"), "Sea Bed")


def test_a_well_absent_from_the_marker_table_gets_no_seabed(synthetic_project):
    markers = read_locked_seabed_markers(synthetic_project["markers"], "Sea Bed")
    assert set(markers) == {"Boreas_1", "Poseidon_2"}
    assert "Poseidon_North_1" not in markers


def test_a_malformed_matching_marker_row_fails_closed(tmp_path):
    path = tmp_path / "markers.csv"
    path.write_text(
        "well_key,canonical_marker_name,MDRT_reconciled_m,TVD_survey_m,"
        "TVDSS_survey_corrected_m\n"
        "A_1,Sea Bed,not_a_number,500.0,475.0\n"
        "B_1,Sea Bed,500.0,500.0,475.0\n",
        encoding="utf-8")
    with pytest.raises(SeabedMarkerError, match="row 2 is malformed"):
        read_locked_seabed_markers(str(path), "Sea Bed")


def test_duplicate_matching_marker_rows_fail_closed(tmp_path):
    path = tmp_path / "markers.csv"
    path.write_text(
        "well_key,canonical_marker_name,MDRT_reconciled_m,TVD_survey_m,"
        "TVDSS_survey_corrected_m\n"
        "A_1,Sea Bed,500.0,500.0,475.0\n"
        "A_1,Sea Bed,501.0,501.0,476.0\n",
        encoding="utf-8")
    with pytest.raises(SeabedMarkerError, match="duplicate"):
        read_locked_seabed_markers(str(path), "Sea Bed")


@pytest.mark.parametrize("bad", ["NaN", "Inf", "-Inf"])
def test_non_finite_matching_marker_values_fail_closed(tmp_path, bad):
    path = tmp_path / "markers.csv"
    path.write_text(
        "well_key,canonical_marker_name,MDRT_reconciled_m,TVD_survey_m,"
        "TVDSS_survey_corrected_m\n"
        f"A_1,Sea Bed,{bad},500.0,475.0\n",
        encoding="utf-8")
    with pytest.raises(SeabedMarkerError, match="row 2 is malformed"):
        read_locked_seabed_markers(str(path), "Sea Bed")


def test_header_only_valid_marker_table_is_an_explicit_empty_mapping(tmp_path):
    path = tmp_path / "markers.csv"
    path.write_text(
        "well_key,canonical_marker_name,MDRT_reconciled_m,TVD_survey_m,"
        "TVDSS_survey_corrected_m\n",
        encoding="utf-8")
    assert read_locked_seabed_markers(str(path), "Sea Bed") == {}


def test_locked_seabed_reader_preserves_small_survey_roundoff(tmp_path):
    path = tmp_path / "markers.csv"
    path.write_text(
        "well_key,canonical_marker_name,MDRT_reconciled_m,TVD_survey_m,"
        "TVDSS_survey_corrected_m\n"
        "Boreas_1,Sea Bed,513.7,513.7000114393325,491.9000114393325\n",
        encoding="utf-8")
    marker = read_locked_seabed_markers(str(path), "Sea Bed")["Boreas_1"]
    assert marker.mdrt_m == 513.7
    assert marker.tvd_m == 513.7000114393325


# ---------------------------------------------------------------------------
# Row builders: shape, order, and content
# ---------------------------------------------------------------------------

def test_every_csv_payload_has_exactly_its_schemas_ordered_columns(
        synthetic_project):
    payloads = synthetic_project["payloads"]
    for name, schema in OVERBURDEN_CSV_SCHEMAS.items():
        for row in payloads[name]:
            assert tuple(row.keys()) == schema.columns, name


def test_rows_are_ordered_by_well_key_then_by_index(synthetic_project):
    payloads = synthetic_project["payloads"]
    for name in (AVAILABILITY, QC, ELIGIBILITY):
        keys = [row["well_key"] for row in payloads[name]]
        assert keys == sorted(keys), name
    gap_keys = [(r["well_key"], r["gap_index"]) for r in payloads[GAPS]]
    assert gap_keys == sorted(gap_keys)
    prof_keys = [(r["well_key"], r["node_index"]) for r in payloads[PROFILE]]
    assert prof_keys == sorted(prof_keys)


def test_scenario_rows_appear_in_declared_scenario_order(synthetic_project):
    from p2mem.overburden_models import VALID_SCENARIO_NAMES
    order = {n: i for i, n in enumerate(VALID_SCENARIO_NAMES)}
    rows = synthetic_project["payloads"][SCENARIOS]
    assert rows, "the synthetic set must produce at least one scenario"
    positions = [order[r["scenario_name"]] for r in rows]
    assert positions == sorted(positions)


def test_scenarios_are_produced_only_for_the_sensitivity_only_well(
        synthetic_project):
    rows = synthetic_project["payloads"][SCENARIOS]
    assert {r["well_key"] for r in rows} == {"Poseidon_2"}
    assert len(rows) == 5


def test_qc_and_manifest_export_the_eligible_population_p05(synthetic_project):
    run = synthetic_project["run"]
    payloads = synthetic_project["payloads"]
    qc_by_well = {row["well_key"]: row for row in payloads[QC]}
    manifest = payloads[OVERBURDEN_MANIFEST_ARTIFACT]
    for well_key, stats in run.stats.items():
        assert (qc_by_well[well_key]["rhob_eligible_p05_kg_m3"]
                == stats.rhob_eligible_p05_kg_m3)
        assert (manifest["wells"][well_key]["rhob_eligible_p05_kg_m3"]
                == stats.rhob_eligible_p05_kg_m3)


def test_scenario_csv_and_manifest_export_identical_fraction_triplets(
        synthetic_project):
    payloads = synthetic_project["payloads"]
    rows = payloads[SCENARIOS]
    manifest_rows = payloads[OVERBURDEN_MANIFEST_ARTIFACT]["wells"][
        "Poseidon_2"]["shallow_column_scenarios"]
    assert len(rows) == len(manifest_rows)
    for row, manifest_row in zip(rows, manifest_rows):
        for field in ("assumed_fraction_of_total", "conditioned_fraction_of_total",
                      "measured_fraction_of_total"):
            assert row[field] == manifest_row[field]


def test_profile_nodes_are_selected_existing_samples_never_interpolated(
        synthetic_project):
    run = synthetic_project["run"]
    for row in synthetic_project["payloads"][PROFILE]:
        profile = run.profiles[row["well_key"]]
        tvdss = np.asarray(profile.tvdss_m)
        rho = np.asarray(profile.density_kg_m3)
        hits = np.flatnonzero(tvdss == row["tvdss_m"])
        assert hits.size >= 1
        assert float(rho[hits[0]]) == row["rhob_kg_m3"]


def test_profile_reporting_always_includes_the_first_and_last_node(
        synthetic_project):
    run = synthetic_project["run"]
    rows = synthetic_project["payloads"][PROFILE]
    for wk, profile in run.profiles.items():
        if profile is None:
            continue
        mine = [r for r in rows if r["well_key"] == wk]
        assert mine[0]["tvdss_m"] == profile.top_tvdss_m
        assert mine[-1]["tvdss_m"] == profile.base_tvdss_m
        assert mine[0]["cumulative_measured_increment_pa"] == 0.0
        assert math.isclose(mine[-1]["cumulative_measured_increment_pa"],
                            profile.total_measured_increment_pa, rel_tol=1e-12)


def test_a_bridged_profile_node_is_labelled_as_conditioned_not_measured(
        synthetic_project):
    rows = synthetic_project["payloads"][PROFILE]
    sources = {r["density_source"] for r in rows}
    assert sources <= {"measured_rhob", "bridged_linear_in_tvd"}
    for r in rows:
        if r["density_source"] == "bridged_linear_in_tvd":
            assert r["evidence_class"] == "assumed_configured"
        else:
            assert r["evidence_class"] == "measured"


def test_megapascal_columns_are_exactly_the_pascal_columns_over_1e6(
        synthetic_project):
    for row in synthetic_project["payloads"][ELIGIBILITY]:
        if row["measured_increment_pa"] is not None:
            assert math.isclose(row["measured_increment_mpa"],
                                row["measured_increment_pa"] / 1.0e6, rel_tol=1e-15)
    for row in synthetic_project["payloads"][SCENARIOS]:
        assert math.isclose(row["total_stress_mpa"],
                            row["total_stress_pa"] / 1.0e6, rel_tol=1e-15)


def test_scenario_rows_disclose_all_three_fractions_in_their_basis_text(
        synthetic_project):
    for row in synthetic_project["payloads"][SCENARIOS]:
        for field in ("assumed_fraction_of_total", "conditioned_fraction_of_total",
                      "measured_fraction_of_total"):
            pct = f"{100.0 * row[field]:.2f}"
            assert pct in row["scenario_basis"]
        assert math.isclose(
            row["assumed_fraction_of_total"]
            + row["conditioned_fraction_of_total"]
            + row["measured_fraction_of_total"], 1.0,
            rel_tol=1e-12, abs_tol=1e-12)
        assert "not a calibrated value" in row["scenario_basis"]


def test_every_sensitivity_threshold_appears_for_every_well(synthetic_project):
    config = synthetic_project["config"]
    rows = synthetic_project["payloads"][SENSITIVITY]
    for wk in synthetic_project["frames"]:
        mine = [r for r in rows if r["well_key"] == wk]
        assert [r["threshold_tvd_m"] for r in mine] == list(
            config.sensitivity_thresholds_tvd_m)
        assert sum(1 for r in mine if r["is_approved_threshold"]) == 1


def test_no_numpy_scalar_reaches_a_payload(synthetic_project):
    """The schema's pre-serialization type check accepts only exact int/float."""
    for name, schema in OVERBURDEN_CSV_SCHEMAS.items():
        for row in synthetic_project["payloads"][name]:
            for field in schema.integer_fields:
                assert row[field] is None or type(row[field]) is int, (name, field)
            for field in schema.number_fields:
                assert row[field] is None or type(row[field]) is float, (name, field)
            for field in schema.boolean_fields:
                assert type(row[field]) is bool, (name, field)


# ---------------------------------------------------------------------------
# Issues
# ---------------------------------------------------------------------------

def test_every_derived_issue_code_is_declared(synthetic_project):
    for row in synthetic_project["payloads"][ISSUES]:
        assert row["code"] in ISSUE_CODES
        assert row["severity"] in ("INFO", "WARNING", "ERROR")


def test_issue_rows_are_deterministically_sorted(synthetic_project):
    rows = synthetic_project["payloads"][ISSUES]
    keys = [(r["severity"], r["code"], r["context"]) for r in rows]
    assert keys == sorted(keys)


def test_issues_are_derived_from_measurements_not_from_well_names(
        synthetic_project):
    codes = {(r["context"], r["code"]) for r in synthetic_project["payloads"][ISSUES]}
    assert ("Poseidon_2", "SHALLOW_DENSITY_COLUMN_UNRESOLVED") in codes
    assert ("Poseidon_North_1", "SEABED_DATUM_UNRESOLVED") in codes
    assert ("Proteus_1ST2", "LONG_INTERNAL_GAP_PRESENT") in codes
    # The well that reaches its seabed raises none of those.
    assert ("Boreas_1", "SHALLOW_DENSITY_COLUMN_UNRESOLVED") not in codes
    assert ("Boreas_1", "ABSOLUTE_OVERBURDEN_NOT_SUPPORTED") not in codes


def test_an_absent_density_curve_raises_an_error_issue(tmp_path, overburden_config):
    md = np.arange(1000.0, 1101.0, 10.0)
    frames = {"Boreas_1": make_frame(well_key="Boreas_1", md=md, curves={})}
    marker_table = tmp_path / "markers.csv"
    marker_table.write_text(
        "well_key,canonical_marker_name,MDRT_reconciled_m,TVD_survey_m,"
        "TVDSS_survey_corrected_m\n", encoding="utf-8")
    run = run_overburden_workflow(frames, overburden_config,
                                  marker_table_path=str(marker_table))
    codes = {i.code for i in run.issues}
    assert "DENSITY_CURVE_ABSENT" in codes


def test_issue_text_carries_no_path_separator_or_newline(synthetic_project):
    for row in synthetic_project["payloads"][ISSUES]:
        for field in ("context", "message"):
            assert "\n" not in row[field]
            assert "\\" not in row[field]
            assert "://" not in row[field]


# ---------------------------------------------------------------------------
# Manifest
# ---------------------------------------------------------------------------

def test_manifest_counts_agree_with_the_derived_statuses(synthetic_project):
    m = synthetic_project["payloads"][OVERBURDEN_MANIFEST_ARTIFACT]
    assert m["n_wells_evaluated"] == 4
    assert m["n_wells_absolute_supported"] == 1
    assert m["n_wells_screening_sensitivity_only"] == 1
    assert m["n_wells_partial_measured_only"] == 2
    assert m["n_wells_not_eligible"] == 0
    total = (m["n_wells_absolute_supported"] + m["n_wells_screening_sensitivity_only"]
             + m["n_wells_partial_measured_only"] + m["n_wells_not_eligible"])
    assert total == m["n_wells_evaluated"]


def test_manifest_declares_every_assumption_explicitly(synthetic_project):
    m = synthetic_project["payloads"][OVERBURDEN_MANIFEST_ARTIFACT]
    config = synthetic_project["config"]
    reg = m["assumption_register"]
    assert reg["seawater_density_kg_m3"] == config.seawater_density_kg_m3
    assert reg["rhob_min_kg_m3"] == config.rhob_min_kg_m3
    assert reg["rhob_max_kg_m3"] == config.rhob_max_kg_m3
    assert reg["short_gap_max_tvd_m"] == config.short_gap_max_tvd_m
    assert m["gravity_m_s2"] == config.gravity_m_s2
    assert m["depth_convention_verified"] is True


def test_manifest_declares_that_no_calibration_data_exist(synthetic_project):
    cal = synthetic_project["payloads"][OVERBURDEN_MANIFEST_ARTIFACT][
        "calibration_data_available"]
    for key, value in cal.items():
        if key == "statement":
            continue
        assert value is False, key


def test_manifest_named_lithology_flag_is_derived_from_the_actual_records(
        synthetic_project):
    m = synthetic_project["payloads"][OVERBURDEN_MANIFEST_ARTIFACT]
    cov = m["lithology_validation"]["emitted_field_coverage"]
    assert m["named_lithology_assigned"] is False
    assert cov["n_string_field_occurrences"] > 0
    assert cov["n_unclassified_fields"] == 0
    assert cov["n_unauthorized_controlled_fields"] == 0
    assert cov["violations"] == 0
    assert m["lithology_validation"]["violations"] == []
    # The coverage block describes the CSV records, so the manifest itself is
    # correctly absent from the inspected set at that point.
    assert set(cov["artifacts_inspected"]) == set(OVERBURDEN_CSV_SCHEMAS)


def test_manifest_lists_every_not_implemented_method(synthetic_project):
    m = synthetic_project["payloads"][OVERBURDEN_MANIFEST_ARTIFACT]
    assert list(m["methods_not_implemented"]) == list(NOT_IMPLEMENTED_TOKENS)
    for token in ("pore_pressure_prediction", "eaton_method", "bowers_method",
                  "effective_stress_calculation", "horizontal_stress_calculation",
                  "wellbore_stability_analysis", "named_lithology_assignment"):
        assert token in m["methods_not_implemented"]


def test_manifest_well_entries_match_the_eligibility_table(synthetic_project):
    m = synthetic_project["payloads"][OVERBURDEN_MANIFEST_ARTIFACT]
    rows = {r["well_key"]: r for r in synthetic_project["payloads"][ELIGIBILITY]}
    assert set(m["wells"]) == set(rows)
    for wk, entry in m["wells"].items():
        assert entry["overburden_status"] == rows[wk]["overburden_status"]
        assert entry["n_eligible"] == rows[wk]["n_eligible_samples"]
        expected = rows[wk]["limiting_reasons"]
        actual = ";".join(entry["limiting_reasons"]) or "none"
        assert actual == expected


# ---------------------------------------------------------------------------
# Determinism and non-mutation
# ---------------------------------------------------------------------------

def test_the_whole_workflow_is_deterministic(synthetic_project):
    config, frames = synthetic_project["config"], synthetic_project["frames"]
    first = json.dumps(synthetic_project["payloads"], sort_keys=True, default=str)
    for _ in range(2):
        run = run_overburden_workflow(
            frames, config, marker_table_path=synthetic_project["markers"])
        again = json.dumps(build_overburden_payloads(run), sort_keys=True, default=str)
        assert again == first


def test_the_workflow_does_not_mutate_the_input_frames(synthetic_project):
    frames = synthetic_project["frames"]
    snapshot = {
        wk: (np.array(fr.MD_m, copy=True), np.array(fr.TVD_m, copy=True),
             np.array(fr.TVDSS_m, copy=True),
             np.array(fr.curve("RHOB_kg_m3").values, copy=True)
             if fr.curve("RHOB_kg_m3") else None)
        for wk, fr in frames.items()}
    run = run_overburden_workflow(
        frames, synthetic_project["config"],
        marker_table_path=synthetic_project["markers"])
    build_overburden_payloads(run)
    for wk, fr in frames.items():
        md, tvd, tvdss, rho = snapshot[wk]
        assert np.array_equal(np.asarray(fr.MD_m), md)
        assert np.array_equal(np.asarray(fr.TVD_m), tvd)
        assert np.array_equal(np.asarray(fr.TVDSS_m), tvdss)
        if rho is not None:
            after = np.asarray(fr.curve("RHOB_kg_m3").values)
            assert np.array_equal(np.isnan(after), np.isnan(rho))
            assert np.array_equal(after[~np.isnan(after)], rho[~np.isnan(rho)])


def test_two_independent_roots_produce_byte_identical_artifacts(
        tmp_path, synthetic_project):
    payloads = synthetic_project["payloads"]
    wells = set(synthetic_project["frames"])
    a, b = tmp_path / "root_a" / "out", tmp_path / "root_b" / "out"
    export_authorized_outputs(OVERBURDEN_BUNDLE, a, payloads, well_keys=wells)
    export_authorized_outputs(OVERBURDEN_BUNDLE, b, payloads, well_keys=wells)
    for name in OVERBURDEN_ARTIFACTS:
        assert (a / name).read_bytes() == (b / name).read_bytes(), name


def test_building_the_payloads_twice_does_not_change_them(synthetic_project):
    run = run_overburden_workflow(
        synthetic_project["frames"], synthetic_project["config"],
        marker_table_path=synthetic_project["markers"])
    first = build_overburden_payloads(run)
    snapshot = copy.deepcopy(first)
    second = build_overburden_payloads(run)
    assert first == snapshot
    assert second == snapshot


#### Step 6 - Install the package in editable mode

In [ ]:
!pip install -e . --quiet
import importlib
import p2mem
importlib.reload(p2mem)
print("p2mem version:", p2mem.__version__)
if p2mem.__version__ != "0.7.4":
    raise RuntimeError(
        f"Expected p2mem 0.7.4 after this increment's install, got {p2mem.__version__!r}."
    )

#### Step 7 - Run the COMPLETE combined test suite (locked Increment 1-6.1.7 plus new Increment 7)

`FULL_TEST_SUITE_PASSED` defaults to `False` and is set only on a zero return code, so a
skipped or crashed run fails the completion gate cleanly rather than being read as a pass.

In [ ]:
import subprocess
import sys

FULL_TEST_SUITE_PASSED = False
TEST_COUNTS = {"passed": None, "failed": None, "collected": None}

_proc = subprocess.run([sys.executable, "-m", "pytest", "-v", "tests"],
                       capture_output=True, text=True, cwd=PROJECT_ROOT)
print(_proc.stdout[-24000:])
if _proc.stderr.strip():
    print("STDERR:", _proc.stderr[-4000:])

import re as _re
_m = _re.search(r"(\d+) passed", _proc.stdout)
if _m:
    TEST_COUNTS["passed"] = int(_m.group(1))
_m = _re.search(r"(\d+) failed", _proc.stdout)
TEST_COUNTS["failed"] = int(_m.group(1)) if _m else 0

if _proc.returncode == 0:
    FULL_TEST_SUITE_PASSED = True
    print(f"\nComplete combined test suite PASSED "
          f"({TEST_COUNTS['passed']} passed, {TEST_COUNTS['failed']} failed).")
else:
    raise RuntimeError(
        f"Combined test suite FAILED with return code {_proc.returncode}. Increment 7 "
        f"requires the complete suite - every locked Increment 1-6.1.7 test plus every new "
        f"Increment 7 test - to pass with zero failures. FULL_TEST_SUITE_PASSED remains "
        f"False. Stopping here rather than reporting completion on an unverified suite."
    )

#### Step 7b - Confirm the LOCKED Increment 6.1.7 test subset still passes on its own

The combined run above already includes these, but running the locked subset in isolation
proves the Increment 7 additions did not change a locked test's outcome by side effect.

In [ ]:
LOCKED_TEST_FILES = [
    "tests/test_units.py", "tests/test_las.py", "tests/test_trajectory.py",
    "tests/test_deviation.py", "tests/test_depth_mapping.py",
    "tests/test_deviation_inventory.py", "tests/test_checkshot.py",
    "tests/test_time_depth.py", "tests/test_checkshot_inventory.py",
    "tests/test_tops.py", "tests/test_tops_inventory.py", "tests/test_wellframe.py",
    "tests/test_petrophysics.py", "tests/test_method_eligibility.py",
]
LOCKED_SUBSET_PASSED = False
LOCKED_SUBSET_COUNT = None
_proc2 = subprocess.run([sys.executable, "-m", "pytest", "-q"] + LOCKED_TEST_FILES,
                        capture_output=True, text=True, cwd=PROJECT_ROOT)
print(_proc2.stdout[-6000:])
_m = _re.search(r"(\d+) passed", _proc2.stdout)
if _m:
    LOCKED_SUBSET_COUNT = int(_m.group(1))
LOCKED_SUBSET_PASSED = (_proc2.returncode == 0)
print(f"\nLocked Increment 1-6.1.7 test subset: "
      f"{'PASSED' if LOCKED_SUBSET_PASSED else 'FAILED'} "
      f"({LOCKED_SUBSET_COUNT} passed)")

---
#### Step 8 - Load the approved data through the LOCKED loaders

Nothing here re-parses a LAS file, re-reads a deviation survey, or re-derives a depth
mapping. Increment 7 consumes the locked layers exactly as they are.

In [ ]:
from p2mem.io.las import load_file_contract_config, load_wells
from p2mem.io.deviation import load_deviation_contract_config, load_deviation_surveys
from p2mem.petrophysics import load_petrophysics_eligibility_config
from p2mem.wellframe import assemble_well_frames

las_contracts = load_file_contract_config(os.path.join("config", "las_curve_contracts.yml"))
las_results, las_failures = load_wells(LAS_PATHS, las_contracts)

dev_contracts = load_deviation_contract_config(
    os.path.join("config", "deviation_survey_contracts.yml"))
dev_results, dev_failures = load_deviation_surveys(SURVEY_PATHS, dev_contracts)

pcfg = load_petrophysics_eligibility_config(
    os.path.join("config", "petrophysics_eligibility.yml"))
frames, frame_failures = assemble_well_frames(
    las_results, dev_results, las_paths=LAS_PATHS, survey_paths=SURVEY_PATHS,
    gr_family_by_well={w: d.gr_family_canonical_name for w, d in pcfg.wells.items()})

print("LAS loaded:    ", sorted(las_results), "| failures:", sorted(las_failures))
print("Surveys loaded:", sorted(dev_results), "| failures:", sorted(dev_failures))
print("Well frames:   ", sorted(frames), "| failures:", sorted(frame_failures))
if las_failures or dev_failures or frame_failures:
    raise RuntimeError(
        f"Not every approved file loaded. LAS failures: {sorted(las_failures)}; "
        f"survey failures: {sorted(dev_failures)}; frame failures: "
        f"{sorted(frame_failures)}. Increment 7 requires all four wells."
    )
for wk in sorted(frames):
    fr = frames[wk]
    print(f"  {wk:18s} n={fr.n_samples:6d}  {fr.depth_map_status}  "
          f"unmapped={fr.n_depth_unmapped}  extrapolated={fr.n_extrapolated}  "
          f"basis={fr.depth_basis_used}  datum_elev={fr.datum_elevation_m:.4f} m")

#### Step 9 - Load the Increment 7 configuration and verify the depth SIGN CONVENTION

The sign convention is **verified**, not assumed. `verify_depth_sign_convention` re-derives it
from each well frame's own arrays: TVD must increase with MD, `TVDSS` must equal
`TVD - datum_elevation` to floating-point tolerance, and `dTVD` must equal `dTVDSS` exactly.
Any failure raises rather than proceeding.

In [ ]:
from p2mem.density_qc import load_overburden_config
from p2mem.overburden import DEPTH_CONVENTION_STATEMENT, SIGN_CONVENTION_ID
from p2mem.overburden import verify_depth_sign_convention

config = load_overburden_config(os.path.join("config", "overburden_stress.yml"))
print(f"config schema {config.schema_version} | increment {config.increment}")
print(f"  screening band          : [{config.rhob_min_kg_m3:.1f}, "
      f"{config.rhob_max_kg_m3:.1f}] kg/m3 (inclusive={config.bounds_are_inclusive})")
print(f"  short-gap threshold     : {config.short_gap_max_tvd_m:.2f} m TVD "
      f"(sensitivity: {list(config.sensitivity_thresholds_tvd_m)})")
print(f"  shallow-gap tolerance   : {config.shallow_gap_tolerance_tvd_m:.2f} m TVD")
print(f"  gravity (ASSUMED)       : {config.gravity_m_s2} m/s2")
print(f"  seawater density (ASSUMED): {config.seawater_density_kg_m3} kg/m3 "
      f"[{config.seawater_density_low_kg_m3}, {config.seawater_density_high_kg_m3}]")
print(f"  integration method      : {config.integration_method}")

print(f"\nDEPTH CONVENTION ({SIGN_CONVENTION_ID}):")
print(" ", DEPTH_CONVENTION_STATEMENT)
print()
SIGN_EVIDENCE = {}
for wk in sorted(frames):
    ev = verify_depth_sign_convention(frames[wk])
    SIGN_EVIDENCE[wk] = ev
    print(f"  {wk:18s} verified={ev['verified']}  "
          f"max|TVD-TVDSS-datum| = {ev['max_tvd_minus_tvdss_deviation_m']:.3e} m  "
          f"max|dTVD-dTVDSS| = {ev['max_dtvd_minus_dtvdss_deviation_m']:.3e} m  "
          f"min dTVD = {ev['min_tvd_increment_m']:.6f} m")

#### Step 10 - Run the Increment 7 workflow on the real four-well data

This is the same `run_overburden_workflow` the tests exercise. The seabed datum is read only
from the LOCKED Increment 5 survey-corrected marker table, for the well it was picked in.

In [ ]:
from p2mem.io.overburden_workflow import (
    LOCKED_MARKER_TABLE, build_overburden_payloads, run_overburden_workflow,
)
from p2mem.overburden import pa_to_mpa

run = run_overburden_workflow(frames, config, marker_table_path=LOCKED_MARKER_TABLE)

print("Seabed markers read from the LOCKED Increment 5 output:")
for wk in sorted(frames):
    m = run.seabed.get(wk)
    if m is None:
        print(f"  {wk:18s} NOT DETERMINABLE - no approved formation-top file, so no "
              f"seabed marker exists in the locked output")
    else:
        print(f"  {wk:18s} MDRT={m.mdrt_m:.4f} m  TVD={m.tvd_m:.4f} m  "
              f"TVDSS={m.tvdss_m:.4f} m")

#### Step 11 - The measured density inventory, per well

Factual counts only. Nothing here is corrected, rescaled or interpreted.

In [ ]:
for wk in sorted(run.stats):
    s = run.stats[wk]
    print(f"\n{wk}")
    print(f"  curve {s.canonical_curve_name} (source {s.source_curve_name!r}, raw unit "
          f"{s.raw_unit!r} -> {s.canonical_unit!r} via {s.conversion_function!r}); "
          f"unit_resolved={s.unit_resolved} conversion_confirmed={s.conversion_confirmed}")
    print(f"  total LAS samples          : {s.n_samples}")
    print(f"  finite RHOB                : {s.n_finite}   non-finite: {s.n_non_finite}")
    print(f"  non-positive               : {s.n_non_positive}")
    print(f"  below screening min        : {s.n_below_screening_min}   above max: "
          f"{s.n_above_screening_max}")
    print(f"  screening-bound failures   : {s.n_screening_bound_failures}")
    print(f"  depth-mapped / unmapped    : {s.n_depth_mapped} / {s.n_depth_unmapped}")
    print(f"  outside survey coverage    : {s.n_samples_outside_survey_coverage}")
    print(f"  ELIGIBLE for integration   : {s.n_eligible}")
    if s.rhob_min_kg_m3 is not None:
        print(f"  finite min/p05/median/p95/max: {s.rhob_min_kg_m3:.4f} / "
              f"{s.rhob_p05_kg_m3:.4f} / {s.rhob_median_kg_m3:.4f} / "
              f"{s.rhob_p95_kg_m3:.4f} / {s.rhob_max_kg_m3:.4f} kg/m3")
        print(f"  eligible-population P05   : {s.rhob_eligible_p05_kg_m3:.4f} kg/m3")
        print(f"  headroom to upper bound    : "
              f"{config.rhob_max_kg_m3 - s.rhob_max_kg_m3:.4f} kg/m3")
    print(f"  first valid MD / TVD / TVDSS: {s.first_valid_md_m:.4f} / "
          f"{s.first_valid_tvd_m:.4f} / {s.first_valid_tvdss_m:.4f} m")
    print(f"  last  valid MD / TVD / TVDSS: {s.last_valid_md_m:.4f} / "
          f"{s.last_valid_tvd_m:.4f} / {s.last_valid_tvdss_m:.4f} m")
    print(f"  gross coverage MD / TVD    : {s.gross_coverage_md_m:.4f} / "
          f"{s.gross_coverage_tvd_m:.4f} m   (median MD step "
          f"{s.median_md_step_m:.5f} m)")
    print(f"  seabed basis               : {s.seabed_basis}")
    if s.shallow_gap_tvd_m is not None:
        print(f"  seabed -> first valid RHOB : {s.shallow_gap_tvd_m:.4f} m TVD "
              f"(UNMEASURED)")
    print(f"  last valid -> well bottom  : "
          f"{'n/a' if s.terminal_gap_tvd_m is None else f'{s.terminal_gap_tvd_m:.4f} m TVD'}")
    print(f"  internal gaps              : {s.n_internal_gaps} "
          f"({s.n_internal_gap_samples} samples)")

#### Step 12 - The twelve explicit masks, per well

`below_seabed_sample` is reported as `None` - **not determinable** - for a well with no
approved formation tops. An all-False or all-True array would each assert something this
project cannot support.

In [ ]:
from p2mem.overburden_models import DENSITY_MASK_NAMES

for wk in sorted(run.masks):
    counts = run.masks[wk].counts()
    print(f"\n{wk}  (n={run.stats[wk].n_samples})")
    for name in DENSITY_MASK_NAMES:
        value = counts[name]
        shown = "NOT DETERMINABLE" if value is None else f"{value:8d}"
        print(f"    {name:38s} {shown}")

#### Step 13 - Gap inventory and gap conditioning

The five structurally distinct cases are kept apart. A shallow seabed-to-log gap is an
**unmeasured column**, not a dropout, and is never bridged at any threshold.

In [ ]:
for wk in sorted(run.gap_results):
    gr = run.gap_results[wk]
    print(f"\n{wk}  threshold={gr.threshold_tvd_m:.2f} m TVD")
    print(f"  bridged : {gr.n_bridged_gaps} gap(s), {gr.n_bridged_samples} sample(s), "
          f"{gr.bridged_thickness_md_m:.4f} m MD / {gr.bridged_thickness_tvd_m:.4f} m TVD")
    print(f"  long    : {gr.n_long_gaps} gap(s), {gr.n_long_gap_samples} sample(s), "
          f"{gr.long_gap_thickness_md_m:.4f} m MD / {gr.long_gap_thickness_tvd_m:.4f} m TVD")
    for g in gr.gaps:
        t_md = "n/a" if g.thickness_md_m is None else f"{g.thickness_md_m:10.4f}"
        t_tvd = "n/a" if g.thickness_tvd_m is None else f"{g.thickness_tvd_m:10.4f}"
        print(f"    [{g.gap_index}] {g.gap_class:38s} n={g.n_samples:6d} "
              f"dMD={t_md} dTVD={t_tvd}  -> {g.disposition}")

#### Step 14 - Measured vertical-stress increments and DERIVED method eligibility

The increment is the integral of `rho*g*dz` in true vertical depth, from each well's **first
eligible density sample** down. It is zero at that first sample by definition and excludes
every column above it. **It is not an absolute vertical stress.**

In [ ]:
for wk in sorted(run.eligibility):
    e = run.eligibility[wk]
    p = run.profiles[wk]
    print(f"\n{wk}")
    print(f"  DERIVED STATUS  : {e.status}")
    print(f"  limiting reasons: {list(e.limiting_reasons) or ['none']}")
    print(f"  seabed resolved : {e.seabed_resolved}  ({e.seabed_basis})")
    print(f"  eligible samples: {e.n_eligible_samples}  bridged: {e.n_bridged_samples}")
    if p is not None:
        print(f"  integrated span : TVDSS {p.top_tvdss_m:.4f} -> {p.base_tvdss_m:.4f} m "
              f"({p.n_nodes} nodes, {p.n_intervals} intervals, "
              f"{p.n_zero_thickness_intervals} zero-thickness)")
        print(f"  MEASURED increment : {pa_to_mpa(p.total_measured_increment_pa):.4f} MPa "
              f"({p.total_measured_increment_pa:.1f} Pa)")
        print(f"  of which conditioned (bridged): "
              f"{pa_to_mpa(p.total_bridged_increment_pa):.6f} MPa")
        if p.column_truncated_at_unresolved_gap:
            print(f"  ** column TRUNCATED at the first unresolved long gap; "
                  f"{p.n_eligible_samples_below_truncation} eligible sample(s) below it "
                  f"are EXCLUDED rather than joined across unknown density **")
    if e.water_column_thickness_tvd_m is not None:
        print(f"  water column    : {e.water_column_thickness_tvd_m:.4f} m TVD "
              f"(ASSUMED seawater density)")
    if e.shallow_unresolved_thickness_tvd_m is not None:
        print(f"  UNMEASURED shallow column: "
              f"{e.shallow_unresolved_thickness_tvd_m:.4f} m TVD")
    print(f"  absolute overburden supported: {e.absolute_stress_supported}")

#### Step 15 - The unresolved shallow-column screening bracket

Published only where the missing thickness is **quantifiable** - that is, where the seabed
datum is resolved. Read the bracket, never a single row. The base case is an arithmetic
midpoint with **no evidentiary support** and is never a best estimate.

In [ ]:
_any = False
for wk in sorted(run.scenarios):
    scen = run.scenarios[wk]
    if not scen:
        print(f"\n{wk}: NO scenario published "
              f"(status {run.eligibility[wk].status}) - the unmeasured thickness is not "
              f"quantifiable without a seabed datum, so bracketing it would be arithmetic "
              f"without content.")
        continue
    _any = True
    print(f"\n{wk}  (unresolved shallow column "
          f"{scen[0].unresolved_thickness_tvd_m:.2f} m TVD, water column "
          f"{scen[0].water_column_thickness_tvd_m:.2f} m TVD)")
    print(f"    {'scenario':20s} {'rho_shallow':>12s} {'seawater':>9s} {'water':>8s} "
          f"{'shallow':>9s} {'measured':>9s} {'bridged':>8s} {'TOTAL':>9s} "
          f"{'assumed%':>9s} {'cond.%':>8s} {'meas.%':>8s}")
    for sc in scen:
        pt = sc.partition
        print(f"    {sc.scenario_name:20s} {sc.assumed_shallow_density_kg_m3:12.2f} "
              f"{sc.seawater_density_kg_m3:9.2f} {pa_to_mpa(pt.water_column_pa):8.3f} "
              f"{pa_to_mpa(pt.unresolved_shallow_pa):9.3f} "
              f"{pa_to_mpa(pt.measured_formation_pa):9.3f} "
              f"{pa_to_mpa(pt.bridged_gap_pa):8.4f} {pa_to_mpa(pt.total_pa):9.3f} "
              f"{100.0 * sc.assumed_fraction_of_total:8.2f}% "
              f"{100.0 * sc.conditioned_fraction_of_total:7.2f}% "
              f"{100.0 * sc.measured_fraction_of_total:7.2f}%")
    lo = min(s.partition.total_pa for s in scen)
    hi = max(s.partition.total_pa for s in scen)
    print(f"    BRACKET WIDTH: {pa_to_mpa(hi - lo):.3f} MPa "
          f"({pa_to_mpa(lo):.3f} to {pa_to_mpa(hi):.3f} MPa), i.e. a factor of "
          f"{hi / lo:.3f}. These are SCENARIOS, not results.")
if not _any:
    print("\nNo well in this dataset supports a shallow-column scenario.")

#### Step 16 - How much does the answer depend on the approved gap threshold?

Every candidate threshold is re-run end to end - conditioning, integration and status
derivation - so the effect of the approved choice is visible rather than assumed.

In [ ]:
for wk in sorted(run.sensitivity):
    print(f"\n{wk}")
    print(f"    {'thr(m TVD)':>11s} {'bridged':>9s} {'samples':>8s} {'long':>5s} "
          f"{'usable':>8s} {'increment(MPa)':>16s}  status")
    for t in run.sensitivity[wk]:
        inc = ("n/a" if t.total_measured_increment_pa is None
               else f"{pa_to_mpa(t.total_measured_increment_pa):.4f}")
        mark = "*" if t.is_approved_threshold else " "
        print(f"    {t.threshold_tvd_m:10.2f}{mark} {t.n_bridged_gaps:9d} "
              f"{t.n_bridged_samples:8d} {t.n_long_gaps:5d} "
              f"{t.n_eligible_or_bridged_samples:8d} {inc:>16s}  {t.derived_status}")
print("\n* = the approved threshold. Nothing in this table selects a preferred value, and a "
      "threshold that bridges more gaps is not thereby better.")

#### Step 17 - Does this dataset support an absolute overburden-stress curve?

Answered from the measurements above, not asserted.

In [ ]:
from p2mem.overburden_models import (
    STATUS_ABSOLUTE, STATUS_NOT_ELIGIBLE, STATUS_PARTIAL_ONLY, STATUS_SENSITIVITY_ONLY,
)

_by_status = {}
for wk, e in run.eligibility.items():
    _by_status.setdefault(e.status, []).append(wk)

print("DERIVED overburden eligibility, from measured coverage alone:")
for status in (STATUS_ABSOLUTE, STATUS_SENSITIVITY_ONLY, STATUS_PARTIAL_ONLY,
               STATUS_NOT_ELIGIBLE):
    print(f"  {status:34s} {sorted(_by_status.get(status, []))}")

ABSOLUTE_WELLS = sorted(_by_status.get(STATUS_ABSOLUTE, []))
if not ABSOLUTE_WELLS:
    print("\nNO well in this dataset supports an absolute vertical overburden-stress curve.")
    print("The reason is measured, not assumed: every well's density log begins far below")
    print("its own seabed, and the interval between carries no density measurement at all:")
    for wk in sorted(run.eligibility):
        e = run.eligibility[wk]
        if e.shallow_unresolved_thickness_tvd_m is not None:
            print(f"  {wk:18s} {e.shallow_unresolved_thickness_tvd_m:9.2f} m TVD "
                  f"UNMEASURED between the seabed and the first eligible sample")
        else:
            print(f"  {wk:18s} seabed datum NOT DETERMINABLE, so the unmeasured "
                  f"thickness cannot even be quantified")
    print("\nThis is reported as a partial measured increment or a transparent screening")
    print("bracket. It is NOT filled with a fitted density trend, and no single absolute")
    print("curve is presented as measured truth.")

---
#### Step 18 - Build every record, validate it, and publish transactionally

`export_authorized_outputs` validates exact schemas and content authorization at the
**pinned** pre-serialization stage, writes the complete candidate set into an isolated
sibling directory, re-reads and re-validates the bytes at the pinned post-serialization
stage, and compares every typed field and row before publishing. Any failure leaves the
official destination unchanged.

In [ ]:
from p2mem.io.overburden_policy import export_authorized_outputs
from p2mem.io.overburden_registry import OVERBURDEN_ARTIFACTS, OVERBURDEN_BUNDLE

OUT_DIR = os.path.join(PROJECT_ROOT, "outputs", "07_density_overburden")
FIG_DIR = os.path.join(OUT_DIR, "figures")

payloads = build_overburden_payloads(run)
for name in OVERBURDEN_ARTIFACTS:
    p = payloads[name]
    print(f"  {name:42s} {len(p) if isinstance(p, list) else 'json object'}")

# The production exporter owns the staging path: no notebook-specific writer is
# passed, so nothing can write into OUT_DIR before the post-serialization checks.
before, after = export_authorized_outputs(
    OVERBURDEN_BUNDLE, OUT_DIR, payloads, well_keys=set(frames))

print(f"\nSchema-driven authorization: {before.n_string_field_occurrences} string field "
      f"occurrence(s) across {len(before.artifacts_inspected)} artifact(s); "
      f"{before.n_controlled_occurrences} controlled, {before.n_structural_occurrences} "
      f"structural, {before.n_unguaranteed_occurrences} outside the guarantee; "
      f"{before.n_schema_violations} schema violation(s), "
      f"{before.n_unclassified_fields} unclassified, "
      f"{before.n_unauthorized_controlled_fields} unauthorized, "
      f"{before.n_field_kind_mismatches} field-kind mismatch(es).")
print(f"Post-serialization schema/content/canonical round trip: "
      f"{'OK' if after.ok else 'FAILED'}")
print(f"Registry usage: {before.distinct_statements_used} statement(s), "
      f"{before.distinct_templates_used} template(s), {before.distinct_labels_used} "
      f"label(s).")

manifest = payloads["density_overburden_manifest.json"]
_cov = manifest["lithology_validation"]["emitted_field_coverage"]
print(f"\nDERIVED named-lithology validation: {_cov['n_string_field_occurrences']} emitted "
      f"occurrence(s) -> named_lithology_assigned="
      f"{manifest['named_lithology_assigned']}, "
      f"{_cov['violations']} violation(s)")

print("\nOutputs written to:", OUT_DIR)
for fn in OVERBURDEN_ARTIFACTS:
    p = os.path.join(OUT_DIR, fn)
    print(f"  {'OK ' if os.path.exists(p) else 'MISSING'} {fn} "
          f"({os.path.getsize(p) if os.path.exists(p) else 0} bytes)")

#### Step 19 - QC figures

Four figures. The first three are factual; the fourth is explicitly labelled as a scenario
bracket, not a result.

In [ ]:
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

from p2mem import ASSURANCE_TIER
from p2mem.overburden import pa_to_mpa
from p2mem.overburden_models import (
    STATUS_ABSOLUTE, STATUS_NOT_ELIGIBLE, STATUS_PARTIAL_ONLY,
    STATUS_SENSITIVITY_ONLY,
)

STATUS_COLOR = {
    STATUS_ABSOLUTE: "#1b7837",
    STATUS_SENSITIVITY_ONLY: "#d95f02",
    STATUS_PARTIAL_ONLY: "#7570b3",
    STATUS_NOT_ELIGIBLE: "#999999",
}


def figure_01(run, fig_dir):
    """Vertical density coverage per well, against the seabed where known."""
    wells = sorted(run.stats)
    fig, ax = plt.subplots(figsize=(11.5, 7.0))
    for i, wk in enumerate(wells):
        s, e = run.stats[wk], run.eligibility[wk]
        x = i
        if s.seabed_tvdss_m is not None:
            ax.bar(x, s.seabed_tvdss_m, bottom=0.0, width=0.62,
                   color="#9ecae1", edgecolor="black", linewidth=0.4,
                   label="Water column (ASSUMED density)" if i == 0 else None)
            top = s.seabed_tvdss_m
            if e.eligible_top_tvdss_m is not None:
                ax.bar(x, e.eligible_top_tvdss_m - top, bottom=top, width=0.62,
                       color="#fdd0a2", edgecolor="black", linewidth=0.4, hatch="//",
                       label="UNMEASURED shallow column" if i == 0 else None)
        if e.eligible_top_tvdss_m is not None and e.eligible_base_tvdss_m is not None:
            ax.bar(x, e.eligible_base_tvdss_m - e.eligible_top_tvdss_m,
                   bottom=e.eligible_top_tvdss_m, width=0.62,
                   color="#31a354", edgecolor="black", linewidth=0.4,
                   label="MEASURED density (integrated)" if i == 0 else None)
        if s.last_valid_tvdss_m is not None and e.eligible_base_tvdss_m is not None:
            extra = s.last_valid_tvdss_m - e.eligible_base_tvdss_m
            if extra > 0:
                ax.bar(x, extra, bottom=e.eligible_base_tvdss_m, width=0.62,
                       color="#c7e9c0", edgecolor="black", linewidth=0.4, hatch="xx",
                       label="Eligible but below an unresolved gap"
                             if i == 0 else None)
        if e.terminal_unresolved_thickness_tvd_m and s.last_valid_tvdss_m is not None:
            ax.bar(x, e.terminal_unresolved_thickness_tvd_m,
                   bottom=s.last_valid_tvdss_m, width=0.62,
                   color="#f0f0f0", edgecolor="black", linewidth=0.4, hatch="..",
                   label="Terminal unmeasured column" if i == 0 else None)
        ax.text(x, 5480, e.status.replace("_", " "), ha="center", va="top",
                fontsize=8.0, color=STATUS_COLOR[e.status], fontweight="bold")
        if s.seabed_tvdss_m is None:
            ax.text(x, 1600, "seabed\nNOT DETERMINABLE", ha="center", va="center",
                    fontsize=8.0, color="#b2182b", style="italic")
    ax.set_xticks(range(len(wells)))
    ax.set_xticklabels(wells, fontsize=9)
    ax.set_ylabel("TVDSS (m, positive downward from MSL)")
    ax.invert_yaxis()
    ax.set_ylim(bottom=5600, top=-320)
    ax.grid(axis="y", alpha=0.25, linestyle=":")
    ax.legend(loc="upper right", fontsize=8, framealpha=0.95)
    ax.set_title(
        "Increment 7 Figure 1 - Vertical density coverage per well\n"
        "Hatched intervals carry NO measured density. "
        f"{ASSURANCE_TIER}", fontsize=10)
    fig.tight_layout()
    path = os.path.join(fig_dir, "fig01_density_coverage_and_unmeasured_column.png")
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return path


def figure_02(run, config, fig_dir):
    """Recorded bulk density against depth, with the screening band shown."""
    wells = sorted(run.stats)
    fig, axes = plt.subplots(1, len(wells), figsize=(14.0, 6.6), sharey=True)
    for ax, wk in zip(np.atleast_1d(axes), wells):
        frame = run.frames[wk]
        masks = run.masks[wk]
        slot = frame.curve(config.canonical_curve_name)
        tvdss = np.asarray(frame.TVDSS_m, dtype=np.float64)
        if slot is not None:
            rho = np.asarray(slot.values, dtype=np.float64)
            elig = np.asarray(masks.eligible_for_measured_integration, dtype=bool)
            ax.plot(rho[elig], tvdss[elig], lw=0.4, color="#2166ac")
        ax.axvline(config.rhob_min_kg_m3, color="#b2182b", lw=0.9, ls="--")
        ax.axvline(config.rhob_max_kg_m3, color="#b2182b", lw=0.9, ls="--")
        s = run.stats[wk]
        if s.seabed_tvdss_m is not None:
            ax.axhline(s.seabed_tvdss_m, color="#08519c", lw=1.1, ls="-.")
            ax.text(1050, s.seabed_tvdss_m - 40, "seabed", fontsize=7,
                    color="#08519c")
        ax.set_title(f"{wk}\n{run.eligibility[wk].status.replace('_', ' ')}",
                     fontsize=8.5, color=STATUS_COLOR[run.eligibility[wk].status])
        ax.set_xlabel("RHOB (kg/m3)")
        ax.set_xlim(900, 3650)
        ax.grid(alpha=0.25, linestyle=":")
    np.atleast_1d(axes)[0].set_ylabel("TVDSS (m, positive downward from MSL)")
    np.atleast_1d(axes)[0].invert_yaxis()
    fig.suptitle(
        "Increment 7 Figure 2 - Recorded bulk density as measured, with the "
        "CONFIGURED screening plausibility band (dashed)\n"
        "No value is clipped, rescaled, smoothed or replaced. "
        f"{ASSURANCE_TIER}", fontsize=10)
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    path = os.path.join(fig_dir, "fig02_density_qc_and_screening_band.png")
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return path


def figure_03(run, fig_dir):
    """The measured vertical-stress INCREMENT, which is not an absolute stress."""
    fig, ax = plt.subplots(figsize=(9.0, 7.4))
    styles = ["-", "--", "-.", ":"]
    for i, wk in enumerate(sorted(run.profiles)):
        p = run.profiles[wk]
        if p is None:
            continue
        ax.plot(np.asarray(p.cumulative_measured_increment_pa) / 1.0e6,
                np.asarray(p.tvdss_m), lw=1.6, ls=styles[i % len(styles)],
                color=STATUS_COLOR[run.eligibility[wk].status],
                label=f"{wk} ({run.eligibility[wk].status.replace(chr(95), chr(32))})")
        if p.column_truncated_at_unresolved_gap:
            ax.plot([pa_to_mpa(p.total_measured_increment_pa)], [p.base_tvdss_m],
                    marker="x", ms=9, mew=2.0, color="#b2182b")
            ax.annotate("truncated at an\nunresolved long gap",
                        (pa_to_mpa(p.total_measured_increment_pa), p.base_tvdss_m),
                        textcoords="offset points", xytext=(-12, 22), fontsize=7.5,
                        color="#b2182b", ha="right")
    ax.set_xlabel("Cumulative MEASURED vertical-stress increment (MPa)\n"
                  "zero at each well's first eligible density sample - NOT an "
                  "absolute vertical stress")
    ax.set_ylabel("TVDSS (m, positive downward from MSL)")
    ax.invert_yaxis()
    ax.grid(alpha=0.25, linestyle=":")
    ax.legend(loc="upper right", fontsize=8)
    ax.set_title(
        "Increment 7 Figure 3 - Measured vertical-stress increment, integrated "
        f"in TRUE VERTICAL DEPTH\n{ASSURANCE_TIER}", fontsize=10)
    fig.tight_layout()
    path = os.path.join(fig_dir, "fig03_measured_vertical_stress_increment.png")
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return path


def figure_04(run, fig_dir):
    """How much of a scenario total is assumption rather than measurement."""
    rows = []
    for wk in sorted(run.scenarios):
        for sc in run.scenarios[wk]:
            if sc.scenario_name in ("low", "base", "high"):
                rows.append((wk, sc))
    fig, (ax, ax2) = plt.subplots(
        1, 2, figsize=(13.0, 7.2), gridspec_kw={"width_ratios": [2.0, 1.0]})
    if rows:
        labels = [f"{wk}\n{sc.scenario_name}" for wk, sc in rows]
        x = np.arange(len(rows))
        water = np.array([pa_to_mpa(sc.partition.water_column_pa) for _, sc in rows])
        shallow = np.array([pa_to_mpa(sc.partition.unresolved_shallow_pa)
                            for _, sc in rows])
        measured = np.array([pa_to_mpa(sc.partition.measured_formation_pa)
                             for _, sc in rows])
        bridged = np.array([pa_to_mpa(sc.partition.bridged_gap_pa) for _, sc in rows])
        ax.bar(x, water, color="#9ecae1", edgecolor="black", linewidth=0.4,
               label="Water column (ASSUMED seawater density)")
        ax.bar(x, shallow, bottom=water, color="#fdd0a2", edgecolor="black",
               linewidth=0.4, hatch="//",
               label="Unresolved shallow column (ASSUMED density)")
        ax.bar(x, measured, bottom=water + shallow, color="#31a354",
               edgecolor="black", linewidth=0.4, label="MEASURED formation density")
        ax.bar(x, bridged, bottom=water + shallow + measured, color="#a1d99b",
               edgecolor="black", linewidth=0.4, hatch="..",
               label="Conditioned bridged gap")
        for i, (_, sc) in enumerate(rows):
            ax.text(i, pa_to_mpa(sc.partition.total_pa) + 1.5,
                    f"{100.0 * sc.assumed_fraction_of_total:.0f}%\nassumed",
                    ha="center", fontsize=7.5, color="#b2182b", fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=8)
        ax.set_ylabel("Vertical stress at the base of the measured column (MPa)")
        ax.set_ylim(0, 1.28 * float(np.max(water + shallow + measured + bridged)))
        ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.10), ncol=2,
                  fontsize=8, frameon=False)
        ax.grid(axis="y", alpha=0.25, linestyle=":")
    else:
        ax.text(0.5, 0.5, "No well supports a shallow-column scenario",
                ha="center", va="center", transform=ax.transAxes)
    ax.set_title("SCENARIOS, not results - read the bracket, never a single bar",
                 fontsize=9.5)

    wells = sorted(run.eligibility)
    counts = {}
    for wk in wells:
        counts[run.eligibility[wk].status] = counts.get(
            run.eligibility[wk].status, 0) + 1
    order = [STATUS_ABSOLUTE, STATUS_SENSITIVITY_ONLY, STATUS_PARTIAL_ONLY,
             STATUS_NOT_ELIGIBLE]
    ax2.barh(range(len(order)), [counts.get(s, 0) for s in order],
             color=[STATUS_COLOR[s] for s in order], edgecolor="black",
             linewidth=0.4)
    ax2.set_yticks(range(len(order)))
    ax2.set_yticklabels([s.replace("_", " ") for s in order], fontsize=8)
    ax2.set_xlabel("wells")
    ax2.set_xticks(range(0, len(wells) + 1))
    ax2.grid(axis="x", alpha=0.25, linestyle=":")
    ax2.set_title("DERIVED overburden eligibility", fontsize=9.5)
    fig.suptitle(
        "Increment 7 Figure 4 - Unresolved shallow-column screening bracket and "
        f"derived method eligibility\n{ASSURANCE_TIER}", fontsize=10)
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    path = os.path.join(fig_dir, "fig04_shallow_column_sensitivity_and_eligibility.png")
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return path


def build_all_figures(run, config, fig_dir):
    os.makedirs(fig_dir, exist_ok=True)
    return [
        figure_01(run, fig_dir),
        figure_02(run, config, fig_dir),
        figure_03(run, fig_dir),
        figure_04(run, fig_dir),
    ]

FIGURE_PATHS = build_all_figures(run, config, FIG_DIR)
for _p in FIGURE_PATHS:
    print('OK', os.path.basename(_p), os.path.getsize(_p), 'bytes')

#### Step 20 - Display the summary tables

In [ ]:
import pandas as pd

pd.set_option("display.width", 240)
pd.set_option("display.max_columns", 60)

for _name, _cols in [
    ("density_availability_inventory.csv",
     ["well_key", "n_samples", "n_finite_rhob", "n_screening_bound_failures",
      "n_eligible", "first_valid_tvdss_m", "last_valid_tvdss_m",
      "gross_coverage_tvd_m"]),
    ("overburden_eligibility_summary.csv",
     ["well_key", "overburden_status", "limiting_reasons",
      "shallow_unresolved_thickness_tvd_m", "measured_thickness_tvd_m",
      "measured_increment_mpa", "column_truncated_at_unresolved_gap",
      "absolute_stress_supported"]),
    ("shallow_column_scenarios.csv",
     ["well_key", "scenario_name", "assumed_shallow_density_kg_m3",
      "total_stress_mpa", "assumed_fraction_of_total",
      "conditioned_fraction_of_total", "measured_fraction_of_total"]),
]:
    _df = pd.read_csv(os.path.join(OUT_DIR, _name))
    print("\n" + "=" * 100)
    print(_name)
    print("=" * 100)
    print(_df[_cols].to_string(index=False))

#### Step 21 - Regenerate from an independent second root and compare byte for byte

Determinism is demonstrated, not claimed: the complete workflow is re-run into a temporary
second destination and every published artifact is compared byte for byte.

The comparison covers the nine deterministic CSV/JSON artifacts. Rendered PNG figures are
excluded on purpose - their bytes depend on the installed matplotlib version, so they are
not a determinism claim this project can honestly make.

In [ ]:
import shutil
import tempfile

DETERMINISM_OK = False
_tmp_root = tempfile.mkdtemp(prefix="inc7_determinism_")
try:
    _second_out = os.path.join(_tmp_root, "07_density_overburden")
    _run2 = run_overburden_workflow(
        frames, config, marker_table_path=LOCKED_MARKER_TABLE)
    _payloads2 = build_overburden_payloads(_run2)
    export_authorized_outputs(
        OVERBURDEN_BUNDLE, _second_out, _payloads2, well_keys=set(frames))
    _diffs = []
    for fn in OVERBURDEN_ARTIFACTS:
        a = open(os.path.join(OUT_DIR, fn), "rb").read()
        b = open(os.path.join(_second_out, fn), "rb").read()
        if a != b:
            _diffs.append(fn)
    DETERMINISM_OK = not _diffs
    print(f"Independent second regeneration: "
          f"{len(OVERBURDEN_ARTIFACTS) - len(_diffs)}/{len(OVERBURDEN_ARTIFACTS)} "
          f"artifacts byte-identical")
    if _diffs:
        print("  DIFFERING:", _diffs)
finally:
    shutil.rmtree(_tmp_root, ignore_errors=True)

---
## Increment 7 completion gate

Every condition is printed explicitly, with its measured value where it has one.

In [ ]:
print("=" * 78)
print("INCREMENT 7.0.4 COMPLETION GATE")
print("=" * 78)

import copy
import csv as _csv
import json as _json
import numpy as np
import tempfile as _tempfile
from pathlib import Path as _Path
from dataclasses import replace as _dc_replace

from p2mem.io.overburden_policy import (
    OutputAuthorizationError, canonicalize_emitted_records,
    validate_emitted_records,
)
from p2mem.io.overburden_registry import (
    ELIGIBILITY, OVERBURDEN_CSV_SCHEMAS, OVERBURDEN_MANIFEST_ARTIFACT,
    OVERBURDEN_STATEMENTS,
)
from p2mem.overburden_models import (
    DENSITY_MASK_NAMES, GAP_DISPOSITION_UNRESOLVED, OverburdenConfigError,
    OverburdenInputError,
    OverburdenInputError,
    OverburdenInputError,
    OverburdenInputError,
    OverburdenInputError,
    OverburdenInputError,
    OverburdenInputError,
    OverburdenInputError,
    SCENARIO_BASIS_BASE, SCENARIO_BASIS_HIGH, SCENARIO_BASIS_LOW,
    VALID_LIMITING_REASONS, VALID_OVERBURDEN_STATUSES,
)

_EXPECTED_FIGURES = [
    "fig01_density_coverage_and_unmeasured_column.png",
    "fig02_density_qc_and_screening_band.png",
    "fig03_measured_vertical_stress_increment.png",
    "fig04_shallow_column_sensitivity_and_eligibility.png",
]


def _read_written_payloads():
    out = {}
    for name in OVERBURDEN_ARTIFACTS:
        path = os.path.join(OUT_DIR, name)
        if name.endswith(".json"):
            with open(path, encoding="utf-8") as fh:
                out[name] = _json.load(fh)
        else:
            with open(path, newline="", encoding="utf-8") as fh:
                out[name] = list(_csv.DictReader(fh))
    return out


def _canonical_round_trip_holds():
    written = _read_written_payloads()
    return (canonicalize_emitted_records(
                OVERBURDEN_BUNDLE, payloads, serialization_stage="pre")
            == canonicalize_emitted_records(
                OVERBURDEN_BUNDLE, written, serialization_stage="post"))


def _schema_adversarial_probes_hold():
    mutations = []

    def add(fn):
        p = copy.deepcopy(payloads)
        fn(p)
        mutations.append(p)

    add(lambda p: p[ELIGIBILITY][0].__setitem__("limitations", "Everything is fine."))
    add(lambda p: p[ELIGIBILITY][0].__setitem__("overburden_status", "looks_fine"))
    add(lambda p: p[ELIGIBILITY][0].__setitem__(
        "limiting_reasons",
        "terminal_density_column_unresolved;seabed_datum_unresolved"))
    add(lambda p: [row.pop("limitations") for row in p[ELIGIBILITY]])
    add(lambda p: [row.__setitem__("rogue", "x") for row in p[ELIGIBILITY]])
    add(lambda p: p.__setitem__(
        ELIGIBILITY, [dict(reversed(list(r.items()))) for r in p[ELIGIBILITY]]))
    for bad in ("2000", None, True, float("nan")):
        add(lambda p, b=bad: p[ELIGIBILITY][0].__setitem__("gravity_m_s2", b))
    add(lambda p: p[ELIGIBILITY][0].__setitem__("n_eligible_samples", 3.0))
    add(lambda p: p[ELIGIBILITY][0].__setitem__("well_key", "Rogue_Well_9"))
    add(lambda p: p.__setitem__("rogue_artifact.csv", []))
    add(lambda p: p.pop("shallow_column_scenarios.csv"))
    add(lambda p: p[OVERBURDEN_MANIFEST_ARTIFACT].__setitem__("rogue_key", 1))
    add(lambda p: p[OVERBURDEN_MANIFEST_ARTIFACT].pop("gravity_m_s2"))
    add(lambda p: p[OVERBURDEN_MANIFEST_ARTIFACT]["wells"].__setitem__("Rogue_9", {}))

    for payload in mutations:
        with _tempfile.TemporaryDirectory() as td:
            target = os.path.join(td, "out")
            try:
                export_authorized_outputs(
                    OVERBURDEN_BUNDLE, target, payload, well_keys=set(frames))
            except OutputAuthorizationError:
                if os.path.exists(target) and os.listdir(target):
                    return False
            else:
                return False
    return True


def _transaction_adversarial_probes_hold():
    replacement = OVERBURDEN_STATEMENTS["qc_limitations"].text

    def _write(target, value):
        if target.suffix == ".json":
            target.write_text(_json.dumps(value, indent=2, sort_keys=True) + "\n",
                              encoding="utf-8")
        else:
            schema = OVERBURDEN_CSV_SCHEMAS[target.name]
            with open(target, "w", newline="", encoding="utf-8") as fh:
                w = _csv.DictWriter(fh, fieldnames=list(schema.columns))
                w.writeheader()
                w.writerows(value)

    def mutating_writer(target, payload):
        value = copy.deepcopy(payload)
        if target.name == ELIGIBILITY:
            value[0]["limitations"] = replacement
        _write(target, value)

    def reordering_writer(target, payload):
        value = copy.deepcopy(payload)
        if target.name == ELIGIBILITY:
            value = list(reversed(value))
        _write(target, value)

    def failing_writer(_target, _payload):
        raise RuntimeError("simulated serializer failure")

    def skipping_writer(target, payload):
        if target.name == "shallow_column_scenarios.csv":
            return
        _write(target, payload)

    for writer in (mutating_writer, reordering_writer, failing_writer, skipping_writer):
        with _tempfile.TemporaryDirectory() as td:
            target = _Path(td) / "out"
            target.mkdir()
            sentinel = target / ELIGIBILITY
            sentinel.write_bytes(b"approved baseline bytes\n")
            try:
                export_authorized_outputs(
                    OVERBURDEN_BUNDLE, target, payloads, well_keys=set(frames),
                    writer=writer)
            except OutputAuthorizationError:
                if (sentinel.read_bytes() != b"approved baseline bytes\n"
                        or {p.name for p in target.iterdir()} != {sentinel.name}):
                    return False
                if [p for p in _Path(td).iterdir()
                        if p.name.startswith(".") and p.is_dir()]:
                    return False
            else:
                return False
    return True


def _stale_artifact_is_refused():
    with _tempfile.TemporaryDirectory() as td:
        target = _Path(td) / "out"
        target.mkdir()
        (target / "stale_result.csv").write_text("stale\n", encoding="utf-8")
        try:
            export_authorized_outputs(
                OVERBURDEN_BUNDLE, target, payloads, well_keys=set(frames))
        except OutputAuthorizationError:
            return (target / "stale_result.csv").read_text(encoding="utf-8") == "stale\n"
    return False


def _unresolved_short_gap_regression_holds():
    """An unresolved short gap must truncate, never become one long trapezoid."""
    import importlib.util as _importlib_util
    from p2mem.density_qc import (
        build_density_masks as _build_masks,
        compute_density_qc_stats as _compute_stats,
        condition_density_gaps as _condition_gaps,
    )
    from p2mem.overburden import (
        build_stress_profile as _build_profile,
        derive_overburden_eligibility as _derive_eligibility,
    )

    spec = _importlib_util.spec_from_file_location(
        "_inc7_gate_synthetic", _Path(PROJECT_ROOT) / "tests" / "synthetic_inc7.py")
    module = _importlib_util.module_from_spec(spec)
    spec.loader.exec_module(module)
    frame = module.gap_frame([slice(10, 13)], rho=2000.0, n=21, step=1.0)
    cfg = _dc_replace(config, bridge_short_internal_gaps=False)
    provisional = _build_masks(frame, cfg, seabed_mdrt_m=None)
    gaps = _condition_gaps(frame, provisional, cfg, seabed_tvd_m=None)
    masks = _build_masks(frame, cfg, seabed_mdrt_m=None, gap_result=gaps)
    stats = _compute_stats(
        frame, masks, cfg, seabed_mdrt_m=None, seabed_tvd_m=None,
        seabed_tvdss_m=None, gaps=gaps.gaps)
    profile = _build_profile(frame, masks, gaps, cfg)
    eligibility = _derive_eligibility(stats, masks, gaps, profile, cfg)
    gap = next(g for g in gaps.gaps if g.gap_class == "short_internal_gap")
    analytical_pa = 2000.0 * cfg.gravity_m_s2 * 9.0
    return (
        gap.disposition == GAP_DISPOSITION_UNRESOLVED
        and profile.column_truncated_at_unresolved_gap is True
        and profile.n_nodes == 10
        and profile.n_eligible_samples_below_truncation == 8
        and abs(profile.total_measured_increment_pa - analytical_pa) < 1e-8
        and eligibility.n_unresolved_internal_gaps == 1
        and eligibility.n_unresolved_long_gaps == 0
        and eligibility.column_uninterrupted is False
        and "internal_gap_unresolved" in eligibility.limiting_reasons
    )


def _public_config_constructor_contract_holds():
    """Direct construction and dataclasses.replace must not bypass the loader."""
    invalid_variants = (
        {"schema_version": "anything"},
        {"increment": 7.0},
        {"bridge_short_internal_gaps": "false"},
        {"min_eligible_samples_for_increment": 2.9},
        {"not_implemented": (1,)},
        {"scenario_high_percentile": 50.0},
        {"sensitivity_thresholds_tvd_m": (0.0, float("nan"), 10.0)},
    )
    for changes in invalid_variants:
        try:
            _dc_replace(config, **changes)
        except OverburdenConfigError:
            continue
        return False
    valid_variant = _dc_replace(config, bridge_short_internal_gaps=False)
    return valid_variant.bridge_short_internal_gaps is False


def _no_absolute_path_in_any_artifact():
    blob = _json.dumps(payloads, sort_keys=True, default=str)
    return not any(t in blob for t in (":\\", "/content/", "/home/", PROJECT_ROOT))


def _every_status_is_derived_not_hardcoded():
    """No well identifier can drive a branch anywhere in the derivation path.

    The four approved WELL KEYS are the only identifiers a code path could
    branch on, so their absence from every derivation module AND from the
    configuration is the operative check. The spaced well names are checked
    separately, in the code modules only: the configuration's own file header
    names the project, which is itself named after a well, and no branch can
    read a comment.
    """
    _code_files = [
        os.path.join(PROJECT_ROOT, "p2mem", "overburden.py"),
        os.path.join(PROJECT_ROOT, "p2mem", "overburden_models.py"),
        os.path.join(PROJECT_ROOT, "p2mem", "density_qc.py"),
        os.path.join(PROJECT_ROOT, "p2mem", "io", "overburden_workflow.py"),
        os.path.join(PROJECT_ROOT, "p2mem", "io", "overburden_inventory.py"),
    ]
    _code_src = "".join(
        _Path(p).read_text(encoding="utf-8") for p in _code_files)
    _config_src = _Path(
        os.path.join(PROJECT_ROOT, "config", "overburden_stress.yml")
    ).read_text(encoding="utf-8")
    _keys = ("Boreas_1", "Poseidon_2", "Poseidon_North_1", "Proteus_1ST2")
    if any(k in _code_src or k in _config_src for k in _keys):
        return False
    return not any(n in _code_src for n in ("Boreas 1", "Poseidon 2",
                                            "Poseidon North 1", "Proteus 1ST2"))


def _original_density_arrays_are_unmodified():
    for wk, fr in frames.items():
        slot = fr.curve(config.canonical_curve_name)
        if slot is None:
            continue
        raw = np.asarray(las_results[wk].canonical_data[config.canonical_curve_name])
        cur = np.asarray(slot.values)
        if not np.array_equal(np.isnan(raw), np.isnan(cur)):
            return False
        if not np.array_equal(raw[~np.isnan(raw)], cur[~np.isnan(cur)]):
            return False
        if slot.values.flags.writeable:
            return False
    return True


def _no_increment8_module_is_importable():
    import importlib
    for name in ("p2mem.pore_pressure", "p2mem.effective_stress", "p2mem.eaton",
                 "p2mem.bowers", "p2mem.nct", "p2mem.elastic", "p2mem.rock_strength",
                 "p2mem.horizontal_stress", "p2mem.mud_window",
                 "p2mem.wellbore_stability"):
        try:
            importlib.import_module(name)
        except ImportError:
            continue
        return False
    return True



def _eligible_p05_contract_holds():
    for wk, stats in run.stats.items():
        eligible = np.asarray(run.masks[wk].eligible_for_measured_integration, dtype=bool)
        slot = frames[wk].curve(config.canonical_curve_name)
        values = np.asarray(slot.values, dtype=float)
        expected = (float(np.percentile(values[eligible], 5.0))
                    if np.count_nonzero(eligible) else None)
        if stats.rhob_eligible_p05_kg_m3 != expected:
            return False
        high = next((s for s in run.scenarios[wk] if s.scenario_name == "high"), None)
        if high is not None and high.assumed_shallow_density_kg_m3 != expected:
            return False
    return True


def _threshold_override_contract_holds():
    import importlib.util as _importlib_util
    from p2mem.density_qc import (
        build_density_masks as _build_masks,
        classify_density_gaps as _classify_gaps,
        condition_density_gaps as _condition_gaps,
    )
    spec = _importlib_util.spec_from_file_location(
        "_inc7_gate_threshold", _Path(PROJECT_ROOT) / "tests" / "synthetic_inc7.py")
    module = _importlib_util.module_from_spec(spec)
    spec.loader.exec_module(module)
    frame = module.gap_frame([slice(5, 15)], rho=2000.0, n=21, step=1.0)
    masks = _build_masks(frame, config)
    for bad in (True, "10", 10 + 0j, np.nan, np.inf, -np.inf, -1.0):
        for operation in (_classify_gaps, _condition_gaps):
            try:
                operation(frame, masks, config, threshold_tvd_m=bad)
            except OverburdenInputError:
                continue
            return False
    return True


def _scenario_fraction_contract_holds():
    for wk in frames:
        for sc in run.scenarios[wk]:
            total = sc.partition.total_pa
            if total is None or total <= 0.0:
                return False
            expected = (
                (sc.partition.water_column_pa + sc.partition.unresolved_shallow_pa) / total,
                sc.partition.bridged_gap_pa / total,
                sc.partition.measured_formation_pa / total,
            )
            actual = (
                sc.assumed_fraction_of_total,
                sc.conditioned_fraction_of_total,
                sc.measured_fraction_of_total,
            )
            if not np.allclose(actual, expected, rtol=1e-12, atol=1e-12):
                return False
            if not np.isclose(sum(actual), 1.0, rtol=1e-12, atol=1e-12):
                return False
    return True


def _locked_seabed_reader_fails_closed():
    from p2mem.io.overburden_workflow import (
        SeabedMarkerError as _SeabedMarkerError,
        read_locked_seabed_markers as _read_markers,
    )
    header = ("well_key,canonical_marker_name,MDRT_reconciled_m,TVD_survey_m,"
              "TVDSS_survey_corrected_m\n")
    cases = (
        None,
        header + "A,Sea Bed,not_a_number,500,475\n",
        header + "A,Sea Bed,500,500,475\nA,Sea Bed,501,501,476\n",
        header + "A,Sea Bed,NaN,500,475\n",
    )
    with _tempfile.TemporaryDirectory() as td:
        for i, content in enumerate(cases):
            path = _Path(td) / f"case_{i}.csv"
            if content is not None:
                path.write_bytes(content.encode("utf-8"))
            try:
                _read_markers(str(path), "Sea Bed")
            except _SeabedMarkerError:
                continue
            return False
    return True


def _json_bytes_are_platform_stable_lf():
    path = _Path(OUT_DIR) / OVERBURDEN_MANIFEST_ARTIFACT
    actual = path.read_bytes()
    expected = (_json.dumps(
        payloads[OVERBURDEN_MANIFEST_ARTIFACT], indent=2, sort_keys=True
    ) + "\n").encode("utf-8")
    return actual == expected and actual.endswith(b"\n") and b"\r\n" not in actual

_status_counts = {}
for _e in run.eligibility.values():
    _status_counts[_e.status] = _status_counts.get(_e.status, 0) + 1

gate_checks = {
    "High scenario uses P05 of eligible—not merely finite—density": (
        _eligible_p05_contract_holds()
        and "rhob_eligible_p05_kg_m3" in OVERBURDEN_CSV_SCHEMAS["density_qc_summary.csv"].columns
    ),
    "Invalid public gap-threshold overrides fail closed": (
        _threshold_override_contract_holds()
    ),
    "Scenario contribution fractions match their stress partitions": (
        _scenario_fraction_contract_holds()
        and "conditioned_fraction_of_total" in
        OVERBURDEN_CSV_SCHEMAS["shallow_column_scenarios.csv"].columns
    ),
    "Locked seabed ingestion rejects missing or ambiguous matching data": (
        _locked_seabed_reader_fails_closed()
    ),
    "JSON artifact is explicit UTF-8 with platform-stable LF bytes": (
        _json_bytes_are_platform_stable_lf()
    ),
    "Complete combined test suite passed with zero failures": (
        globals().get("FULL_TEST_SUITE_PASSED", False) is True
        and TEST_COUNTS.get("failed") == 0
    ),
    "Locked Increment 1-6.1.7 test subset passes in isolation": (
        globals().get("LOCKED_SUBSET_PASSED", False) is True
    ),
    "All 4 approved LAS files loaded (0 failures)": (
        len(las_results) == 4 and len(las_failures) == 0
    ),
    "All 4 approved deviation surveys loaded (0 failures)": (
        len(dev_results) == 4 and len(dev_failures) == 0
    ),
    "All 4 well frames assembled (0 failures)": (
        len(frames) == 4 and len(frame_failures) == 0
    ),
    "ZERO samples extrapolated in any well": all(
        fr.n_extrapolated == 0 for fr in frames.values()
    ),
    "Depth SIGN CONVENTION verified against every frame's own arrays": (
        len(SIGN_EVIDENCE) == 4
        and all(ev["verified"] is True for ev in SIGN_EVIDENCE.values())
        and all(ev["sign_convention_id"] == SIGN_CONVENTION_ID
                for ev in SIGN_EVIDENCE.values())
        and run.depth_convention_verified is True
    ),
    "Integration performed in TVDSS, never in measured depth": all(
        p.integration_coordinate == "tvdss_m"
        for p in run.profiles.values() if p is not None
    ),
    "All 12 explicit density masks built for every well": all(
        set(run.masks[wk].counts()) == set(DENSITY_MASK_NAMES) for wk in frames
    ),
    "below_seabed mask is NOT DETERMINABLE where no approved tops exist": all(
        (run.masks[wk].below_seabed_sample is None)
        == (wk not in run.seabed) for wk in frames
    ),
    "Original resolved RHOB arrays are unmodified and read-only": (
        _original_density_arrays_are_unmodified()
    ),
    "Conditioned density lives in a SEPARATELY NAMED array with its own mask": all(
        run.gap_results[wk].conditioned_density_kg_m3 is not
        (frames[wk].curve(config.canonical_curve_name).values
         if frames[wk].curve(config.canonical_curve_name) else None)
        and not run.gap_results[wk].conditioned_density_kg_m3.flags.writeable
        for wk in frames
    ),
    "No shallow or terminal gap was bridged, at any threshold": all(
        g.disposition != "bridged_linear_in_tvd"
        for wk in frames for g in run.gap_results[wk].gaps
        if g.gap_class != "short_internal_gap"
    ),
    "No long internal gap was bridged": all(
        g.disposition == "unresolved_not_bridged"
        for wk in frames for g in run.gap_results[wk].gaps
        if g.gap_class == "long_internal_gap"
    ),
    "Gap counts and sample counts satisfy the constructor invariants": all(
        (run.gap_results[wk].n_bridged_gaps == 0)
        == (run.gap_results[wk].n_bridged_samples == 0)
        and (run.gap_results[wk].n_bridged_gaps == 0
             or run.gap_results[wk].n_bridged_samples
             >= run.gap_results[wk].n_bridged_gaps)
        and (run.gap_results[wk].n_long_gaps == 0)
        == (run.gap_results[wk].n_long_gap_samples == 0)
        for wk in frames
    ),

    "Unresolved short internal gap truncates integration (direct regression)": (
        _unresolved_short_gap_regression_holds()
    ),
    "Every unresolved internal gap is counted and makes the column interrupted": all(
        e.n_unresolved_internal_gaps >= e.n_unresolved_long_gaps
        and e.column_uninterrupted == (e.n_unresolved_internal_gaps == 0)
        for e in run.eligibility.values()
    ),

    "Public OverburdenConfig constructor matches loader type safety": (
        _public_config_constructor_contract_holds()
    ),
    "Scenario bases are explicit conditional/illustrative assumptions": (
        config.schema_version == "7.0.1"
        and {row["assumed_density_basis"] for row in payloads["shallow_column_scenarios.csv"]}
        == {SCENARIO_BASIS_LOW, SCENARIO_BASIS_BASE, SCENARIO_BASIS_HIGH}
    ),
    "Every eligibility status is one of the four declared codes": (
        set(_status_counts) <= set(VALID_OVERBURDEN_STATUSES)
        and sum(_status_counts.values()) == 4
    ),
    "Every limiting reason is an enumerated code, sorted and deduplicated": all(
        set(e.limiting_reasons) <= set(VALID_LIMITING_REASONS)
        and list(e.limiting_reasons) == sorted(set(e.limiting_reasons))
        for e in run.eligibility.values()
    ),
    "Status derivation contains NO well name anywhere in code or config": (
        _every_status_is_derived_not_hardcoded()
    ),
    "Gap-threshold sensitivity reported for every well at every threshold": all(
        [t.threshold_tvd_m for t in run.sensitivity[wk]]
        == list(config.sensitivity_thresholds_tvd_m)
        and sum(1 for t in run.sensitivity[wk] if t.is_approved_threshold) == 1
        for wk in frames
    ),
    "Scenarios published ONLY where the unmeasured thickness is quantifiable": all(
        bool(run.scenarios[wk]) == (run.eligibility[wk].status == STATUS_SENSITIVITY_ONLY)
        for wk in frames
    ),
    "Every scenario partitions assumed, conditioned and measured fractions": all(
        0.0 <= sc.assumed_fraction_of_total <= 1.0
        and 0.0 <= sc.conditioned_fraction_of_total <= 1.0
        and 0.0 <= sc.measured_fraction_of_total <= 1.0
        and abs(sc.assumed_fraction_of_total
                + sc.conditioned_fraction_of_total
                + sc.measured_fraction_of_total - 1.0) < 1e-12
        and sc.partition.total_pa is not None
        for wk in frames for sc in run.scenarios[wk]
    ),
    "No absolute sigma_v is claimed for a well whose shallow column is unmeasured": all(
        (e.absolute_stress_supported is False)
        for e in run.eligibility.values()
        if (e.shallow_unresolved_thickness_tvd_m or 0.0)
        > config.shallow_gap_tolerance_tvd_m or not e.seabed_resolved
    ),
    "Every emitted output string field is classified (0 unclassified)": (
        before.n_unclassified_fields == 0 and after.n_unclassified_fields == 0
        and before.n_string_field_occurrences > 0
    ),
    "Every emitted controlled field is authorized (0 unauthorized, 0 kind mismatch)": (
        before.n_unauthorized_controlled_fields == 0
        and before.n_field_kind_mismatches == 0
        and after.n_unauthorized_controlled_fields == 0
        and after.n_field_kind_mismatches == 0
    ),
    "Exact artifact schemas pass before AND after serialization": (
        before.ok and after.ok
        and before.n_schema_violations == 0 and after.n_schema_violations == 0
        and len(before.artifacts_inspected) == len(OVERBURDEN_ARTIFACTS) == 9
    ),
    "Every typed field and row survives serialization unchanged": (
        _canonical_round_trip_holds()
    ),
    "Missing, unknown, reordered, empty and type-mutated fields fail closed": (
        _schema_adversarial_probes_hold()
    ),
    "Writer mutation, row reordering, omission and failure are failure-atomic": (
        _transaction_adversarial_probes_hold()
    ),
    "A stale undeclared artifact in the destination is refused": (
        _stale_artifact_is_refused()
    ),
    "ZERO named lithology assigned (DERIVED from the emitted records)": (
        manifest["named_lithology_assigned"] is False
        and manifest["lithology_validation"]["violations"] == []
        and _cov["n_string_field_occurrences"] > 0
        and _cov["n_unclassified_fields"] == 0
    ),
    "No absolute path leaks into any emitted artifact": (
        _no_absolute_path_in_any_artifact()
    ),
    "9 deterministic output artifacts declared and present": (
        len(OVERBURDEN_ARTIFACTS) == 9
        and all(os.path.exists(os.path.join(OUT_DIR, fn))
                for fn in OVERBURDEN_ARTIFACTS)
    ),
    "4 QC figures present": all(
        os.path.exists(os.path.join(FIG_DIR, fn)) for fn in _EXPECTED_FIGURES
    ),
    "Regeneration from an independent second root is byte-identical": (
        globals().get("DETERMINISM_OK", False) is True
    ),
    "Increment 8 is NOT started (no later-phase module is importable)": (
        _no_increment8_module_is_importable()
    ),
}

for label, passed in gate_checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}] {label}")
print(f"\n{sum(1 for v in gate_checks.values() if v)}/{len(gate_checks)} gate checks passed")

if all(gate_checks.values()):
    print("\nIncrement 7.0.4 is complete. Stopping here per the approved scope.")
    print()
    print("MEASURED FACTS:")
    for wk in sorted(run.eligibility):
        e = run.eligibility[wk]
        print(f"  {wk:18s} {e.status:34s} eligible={e.n_eligible_samples:6d} "
              f"increment="
              f"{'n/a' if e.measured_increment_pa is None else f'{pa_to_mpa(e.measured_increment_pa):8.4f} MPa'}")
    print()
    print("Not implemented (explicitly out of scope): pore-pressure prediction, sonic or")
    print("resistivity NCT fitting, Eaton, Bowers, equivalent-depth and drilling-exponent")
    print("methods, effective-stress calculation, dynamic or static elastic-property")
    print("modelling, rock-strength modelling, horizontal-stress calculation, stress")
    print("calibration, mud-window calculation, breakout, tensile-fracture and")
    print("wellbore-stability analysis, named lithology assignment, and mineralogical")
    print("interpretation. Increment 8 has NOT been started.")
else:
    raise RuntimeError("Increment 7 completion gate FAILED - see failed check(s) above.")